In [ ]:
# ============================================================
# Input fertilizer
# Processing NPKGRIDSv1.08_maize, Calculating the Wheat synthetic N
# Transform from .nc to .tif
# ============================================================

import numpy as np
import xarray as xr
import rasterio
from rasterio.transform import from_origin
import pandas as pd
from pathlib import Path

# ======================================
# Set File path
# ======================================
input_nc = Path("Z:/staff/Qiang_S/09_Datasets/01N_Input/01Global_SyntheticFertilizer/01DownloadDatasets/01Fer_Wheat/01NPKGRIDS_Fer_Wheat/NPKGRIDSv1.08_wheat.nc")
output_dir = Path("Z:/staff/Qiang_S/09_Datasets/01N_Input/01Global_SyntheticFertilizer/03TempData/02NPKGRIDS_Fer_Wheat_Temp")
output_dir.mkdir(parents=True, exist_ok=True)

output_tif = output_dir / "NPKGRIDS_FerN_Wheat_2020.tif"

# open NetCDF file
ds = xr.open_dataset(input_nc)

# Extract and squeeze out redundant dimensions, such as time.
nrat = ds['Nrate'].squeeze()
lat = ds['lat'].values
lon = ds['lon'].values

# ======================================
# 🚨 Check if the order of the lat (latitude from north to south) and nrat data needs to be reversed.
# ======================================
if lat[0] < lat[-1]:
    lat = lat[::-1]
    nrat = nrat[::-1, :]

# ======================================
# 🚨 Check if the lon (from west to east) needs to be flipped.
# ======================================
if lon[0] > lon[-1]:
    lon = lon[::-1]
    nrat = nrat[:, ::-1]

# Calculate resolution
res_x = abs(lon[1] - lon[0])
res_y = abs(lat[1] - lat[0])

# Create a transform using the top left corner as the origin.
transform = from_origin(lon.min(), lat.max(), res_x, res_y)

# Write GeoTIFF
with rasterio.open(
    output_tif,
    'w',
    driver='GTiff',
    height=nrat.shape[0],
    width=nrat.shape[1],
    count=1,
    dtype=nrat.dtype,
    crs='EPSG:4326',
    transform=transform
) as dst:
    dst.write(nrat.values, 1)

print(f"✅ GeoTIFF finished：{output_tif}")

In [ ]:
# ============================================================
# Input fertilizer
# dataset 2 Processing NPKGRIDSv1.08_wheat
# Processing NPKGRIDSv1.08_wheat
# Tools，very useful, Unify into global data., 0.083333333
# 3 arcmin → 5 arcmin
# ============================================================
import os
import numpy as np
import rasterio
from rasterio.transform import from_origin
from rasterio.warp import reproject, Resampling

# ===Parameter settings ===
res = 0.083333333
crs = "EPSG:4326"

# path
input_dir = "Z:/staff/Qiang_S/09_Datasets/01N_Input/01Global_SyntheticFertilizer/03TempData/02NPKGRIDS_Fer_Wheat_Step1Temp"          # 输入目录
output_dir = "Z:/staff/Qiang_S/09_Datasets/01N_Input/01Global_SyntheticFertilizer/03TempData/02NPKGRIDS_Fer_Wheat_Step2Temp"   # 输出目录

# Calculate global grid size
width = int(360 / res)
height = int(180 / res)

#Start constructing the affine transformation from the top left corner.
transform = from_origin(-180, 90, res, res)

# Global output profile (no nodata value)
global_profile = {
    'driver': 'GTiff',
    'height': height,
    'width': width,
    'count': 1,
    'dtype': 'float32',
    'crs': crs,
    'transform': transform,
    'nodata': -9999  # nodata
}

os.makedirs(output_dir, exist_ok=True)

# === Batch processing ===
for root, _, files in os.walk(input_dir):
    for file in files:
        if file.endswith(".tif"):
            input_path = os.path.join(root, file)
            output_path = os.path.join(output_dir, file)

            with rasterio.open(input_path) as src:
                dst_array = np.full((height, width), np.nan, dtype='float32')

                reproject(
                    source=rasterio.band(src, 1),
                    destination=dst_array,
                    src_transform=src.transform,
                    src_crs=src.crs,
                    dst_transform=transform,
                    dst_crs=crs,
                    resampling=Resampling.average, # this is an important set
                    dst_nodata= -9999
                )

                with rasterio.open(output_path, "w", **global_profile) as dst:
                    dst.write(dst_array, 1)

print("✅ All images have been standardized to a global scale; output complete!")

In [ ]:
# ============================================================
# Input fertilizer
# dataset 2 Processing NPKGRIDSv1.08_wheat
# useful tools,🎉🎉🎉🎉🎉，one datasets, first Multiply ratio and Remove specific values
# ============================================================

from osgeo import gdal
import numpy as np
import os
import glob

# =====================================================
# User Parameters
# =====================================================

# Input / Output folders
input_folder = r"Z:/staff/Qiang_S/09_Datasets/01N_Input/01Global_SyntheticFertilizer/03TempData/02NPKGRIDS_Fer_Wheat_Step2Temp"
output_folder = r"Z:/staff/Qiang_S/09_Datasets/01N_Input/01Global_SyntheticFertilizer/02StandardizedDatasets/01NPKGRIDS_Fer_Wheat_Standardized"
os.makedirs(output_folder, exist_ok=True)

# ---- Step 1: Multiply ratio ----
ratio = 1     # e.g. 6, 0.001, 10.5

# ---- Step 2: Remove specific values ----
replace_values = [-1]   # values to be treated as invalid

# ---- Step 3: Remove value range ----
range_min = -2
range_max = 0       # large number

# ---- Final NoData value ----
nodata_value = -9999

# =====================================================
# Batch Processing
# =====================================================
tif_files = glob.glob(os.path.join(input_folder, "*.tif"))
print(f"Found {len(tif_files)} GeoTIFF files")

for i, in_file in enumerate(tif_files, 1):
    fname = os.path.basename(in_file)
    out_file = os.path.join(output_folder, fname)

    print(f"[{i}/{len(tif_files)}] Processing: {fname}")

    ds = gdal.Open(in_file)
    band = ds.GetRasterBand(1)

    # Read original nodata
    ori_nodata = band.GetNoDataValue()

    arr = band.ReadAsArray().astype(np.float32)

    # -------------------------------------------------
    # 1) Convert original nodata → NaN
    # -------------------------------------------------
    if ori_nodata is not None:
        arr[arr == ori_nodata] = np.nan

    # -------------------------------------------------
    # 2) Multiply by ratio
    # -------------------------------------------------
    arr = arr * ratio

    # -------------------------------------------------
    # 3) Replace specific values → NaN
    # -------------------------------------------------
    for v in replace_values:
        arr[arr == v] = np.nan

    # -------------------------------------------------
    # 4) Replace values within range → NaN
    #    Condition: range_min < value < range_max
    # -------------------------------------------------
    mask_range = (arr > range_min) & (arr < range_max)
    arr[mask_range] = np.nan

    # -------------------------------------------------
    # 5) Convert all NaN → unified nodata
    # -------------------------------------------------
    arr = np.where(np.isnan(arr), nodata_value, arr)

    # -------------------------------------------------
    # 6) Write output GeoTIFF
    # -------------------------------------------------
    driver = gdal.GetDriverByName("GTiff")
    out_ds = driver.Create(
        out_file,
        ds.RasterXSize,
        ds.RasterYSize,
        1,
        gdal.GDT_Float32,
        options=["COMPRESS=DEFLATE", "PREDICTOR=2", "TILED=YES"]
    )

    out_ds.SetGeoTransform(ds.GetGeoTransform())
    out_ds.SetProjection(ds.GetProjection())

    out_band = out_ds.GetRasterBand(1)
    out_band.SetNoDataValue(nodata_value)
    out_band.WriteArray(arr)

    out_ds.FlushCache()
    out_ds = None
    ds = None

print("\n✅ ALL FILES FINISHED")
print(f"Output directory: {output_folder}")

In [ ]:
# =====================================================
# Input fertilizer
# dataset 2 Processing NPKGRIDSv1.08_wheat
# data fusion, FAO and NPKGRIDSv1.08_wheat, new data 2016,2017,2018, 2019🎉🎉---
# =====================================================
import geopandas as gpd
import rasterio
import rasterio.mask
import numpy as np
import pandas as pd
from tqdm import tqdm
import os
from rasterio.features import geometry_mask

# ============================================================
# Part 1 -- input1 parameters
# ============================================================
shp_path = r"Z:\staff\Qiang_S\09_Datasets\09OtherDataUsed\GDAM_GlobalCountryLevel_Shp/GlobalCountryLevel_Shp.shp"         # M49_Code was used as the ID
excel_path = r"Z:\staff\Qiang_S\09_Datasets\01N_Input\01Global_SyntheticFertilizer\01DownloadDatasets\09Fer_Total/FAOSTAT_FertilizerN_2015_2020.xlsx"       # FAO excel, contain yield...

# Wheat fer
# tif_base_path = r"Z:\staff\Qiang_S\09_Datasets\01N_Input\01Global_SyntheticFertilizer\02StandardizedDatasets\01LUH2v2_Fer_Wheat_Standardized/fertrate_2015.tif"

# Maize fer
# tif_base_path = r"Z:\staff\Qiang_S\09_Datasets\01N_Input\01Global_SyntheticFertilizer\02StandardizedDatasets\02LUH2v2_Fer_Maize_Standardized/fertrate_2015.tif"
# tif_base_path = r"Z:\staff\Qiang_S\09_Datasets\01N_Input\01Global_SyntheticFertilizer\02StandardizedDatasets\02NPKGRIDS_Fer_Maize_Standardized/NPKGRIDS_FerN_Maize_2020.tif"

# Rice fer
#tif_base_path = r"Z:\staff\Qiang_S\09_Datasets\01N_Input\01Global_SyntheticFertilizer\02StandardizedDatasets\03NPKGRIDS_Fer_Rice_Standardized/NPKGRIDS_FerN_Rice_2020.tif"
tif_base_path = r"Z:\staff\Qiang_S\09_Datasets\01N_Input\01Global_SyntheticFertilizer\02StandardizedDatasets\03LUH2v2_Fer_Rice_Standardized/fertrate_2015.tif"

# base year .tif
base_year = 2015                                   # base year

target_years = [2016, 2017]                     # want to generate the following years, multiple years are possible.
#target_years = [2018, 2019]

selected_item_code = 3102   # 

Ratio1Or1000 = 1  # GlobalCropYield5min, SAPM2010,SPAM2005，SPAM2000，GDHY2016_5min
# Ratio1Or1000 = 1000 # SPAM2020, GAEZ2020, GAEZ2015, GAEZ2010, GAEZ2000,
Pre_fileName = "fertrate_rice_I" #output file pre name

print_check_year = 2016  # You want to print the target year at a specific scale, such as 2019.
output_folder = r"Z:\staff\Qiang_S\09_Datasets\01N_Input\01Global_SyntheticFertilizer\02StandardizedDatasets\03LUH2v2_Fer_Rice_Standardized"   # output folder
nodata_value = -9999                                   # Set No data 

# ============================================================
# Part 2 -- Read data
# ============================================================

# read the base .tif
base_ds = rasterio.open(tif_base_path)
base_data = base_ds.read(1)
base_data = base_data.astype("float32")

# ---------------------------------------------------------
# Part 3 -- Build a dictionary of country scaling ratios (including printouts for proofreading).
# ---------------------------------------------------------

# Read Excel
df_excel = pd.read_excel(excel_path)

# Keep only the relevant columns
required_cols = ["Area Code (M49)", "Item Code (CPC)", "Year", "Value"]
df_excel = df_excel[required_cols]

# Filter selected crops（Item Code）
df_excel = df_excel[df_excel["Item Code (CPC)"] == selected_item_code]

# Filtering base year
df_base = df_excel[df_excel["Year"] == base_year]
base_dict = dict(zip(df_base["Area Code (M49)"], df_base["Value"]))

scaling_dict = {}

for ty in target_years:

    df_t = df_excel[df_excel["Year"] == ty]
    t_dict = dict(zip(df_t["Area Code (M49)"], df_t["Value"]))

    scaling_dict[ty] = {}
    common_countries = set(base_dict.keys()).intersection(set(t_dict.keys()))

    for c in common_countries:

        base_val = base_dict.get(c, None)
        t_val = t_dict.get(c, None)

        if base_val is None or t_val is None:
            continue
        if np.isnan(base_val) or np.isnan(t_val):
            continue
        if base_val == 0:
            continue

        scaling_dict[ty][c] = t_val / base_val

    print(f"📌 Target year: {ty}, valid countries: {len(scaling_dict[ty])}")

# ---------------------------------------------------------
# Additional print: Check the percentage value of a specific target_year.
# ---------------------------------------------------------
if print_check_year in scaling_dict:
    print("\n==========Check（Year：{}） ==========".format(print_check_year))
    test_dict = scaling_dict[print_check_year]

    if len(test_dict) == 0:
        print(f"⚠️ No countries can be counted. {print_check_year} scaling ratio (data may be missing)")
    else:
        for country_code, ratio in list(test_dict.items())[:20]:  # Print only the first 20
            print(f"M49: {country_code} → ratio: {ratio:.4f}")

        print("\n（Only the first 20 items are displayed）")

else:
    print(f"⚠️ {print_check_year} The data is not in target_years, so the verification data cannot be printed.")
    

# ---------------------------------------------------------
# Part 4-- Pre-generate "Country → Raster Mask" mapping
# ---------------------------------------------------------

print("\n=== Part 5: Country masks ===")

# read shapefile
gdf = gpd.read_file(shp_path)

# check item
if "M49_Code" not in gdf.columns:
    raise ValueError("❌ ERROR: Shapefile did't find M49_Code！")

# Reading the base year raster for spatial reference
with rasterio.open(tif_base_path) as src:
    raster_meta = src.meta
    transform = src.transform
    raster_shape = (src.height, src.width)

# Pre-generate a mask for each country (True = masked, False = valid pixels).
country_masks = {}

for idx, row in gdf.iterrows():
    m49 = row["M49_Code"]
    geom = row.geometry

    if geom is None:
        continue

    mask = geometry_mask(
        geometries=[geom],
        out_shape=raster_shape,
        transform=transform,
        invert=True  # True 表示 geom 内部为 True
    )

    # 保存 mask
    country_masks[m49] = mask

print(f"✔ Total number of country masks generated: {len(country_masks)}")

# ---------------------------------------------------------
# Effective pixel count for the top 10 countries' masks (for inspection)
# ---------------------------------------------------------
print("\n=== Mask check（top 10）===")
for i, (cid, m) in enumerate(country_masks.items()):
    if i >= 10:
        break
    print(f"M49 {cid}: Number of effective pixels = {m.sum()}")

print("=== Part 5 done ===\n")

# ---------------------------------------------------------
# Part 5-- Scaling the raster by country ratio → Outputting a tif file for target_years
# ---------------------------------------------------------
print("\n=== Part 5: Generate target year TIF ===")

with rasterio.open(tif_base_path) as src:
    base_data = src.read(1).astype(float)
    nodata = src.nodata if src.nodata is not None else -9999
    base_data[base_data == nodata] = np.nan

# Iterate through each target year
for ty in target_years:

    print(f"\n--- processing year {ty} ---")
    ratio_dict = scaling_dict[ty]

    # Create output array
    out_arr = np.full_like(base_data, np.nan)

    # country one by one.
    for cid, mask in country_masks.items():

        if cid not in ratio_dict:
            # No proportion → Fill in nan
            continue

        ratio = ratio_dict[cid]

        # Printing the first few countries for inspection
        if np.random.rand() < 0.005:  # Randomly print a small number of countries
            print(f"  M49 {cid}: Ratio={ratio:.4f}, mask pix={mask.sum()}")
        # ----
        # Scaling
        out_arr[mask] = base_data[mask] * ratio  * Ratio1Or1000

    # --- Check the output array ---
    valid_values = out_arr[~np.isnan(out_arr)]
    print(f"  ✔ Number of effective pixels: {len(valid_values)}")

    if len(valid_values) > 0:
        print(f"  ✔ Scaling results statistics: min={valid_values.min():.4f}, max={valid_values.max():.4f}, mean={valid_values.mean():.4f}")

        # The first 20 values ​​are printed for proofreading.
        check_vals = valid_values[:20]
        print("✔ Preview of the first 20 pixel values:", check_vals)
    else:
        print("⚠️ Warning: Output is empty! Please check Part 5 masks or Excel scale.")

    # Write tif and naming
    out_path = os.path.join(output_folder, f"{Pre_fileName}_{ty}.tif")
    meta = raster_meta.copy()
    meta.update(dtype="float32", nodata=-9999)

    with rasterio.open(out_path, "w", **meta) as dst:
        tmp = out_arr.copy()
        tmp[np.isnan(tmp)] = -9999
        dst.write(tmp.astype("float32"), 1)

    print(f"  ✔ All Done: {out_path}")

print("\n=== Part 5 Done ===")

In [ ]:
path = r"Z:\staff\Qiang_S\09_Datasets\01N_Input\01Global_SyntheticFertilizer\01DownloadDatasets\01Fer_Wheat\01LUH2v2_Fer_Wheat/fertilizer_application_histsoc_whe_1850-2015.nc"
path = path.replace("\\", "/")
print(path)

In [ ]:
# ========================================
# Input fertilizer
# Dataset 3 LUH2v2
# # Show the Information
# ========================================

import xarray as xr
ds = xr.open_dataset("Z:/staff/Qiang_S/09_Datasets/01N_Input/01Global_SyntheticFertilizer/01DownloadDatasets/01Fer_Wheat/01LUH2v2_Fer_Wheat/fertilizer_application_histsoc_whe_1850-2015.nc", decode_times=False)
print(ds)

In [ ]:
# =====================================================
# ---Input fertilizer---
# ---Dataset 3 LUH2v2---
# ---Step2 transform formate, .nc to .tif
# =====================================================
import os
import numpy as np
import xarray as xr
import rasterio
from rasterio.transform import from_origin

# =========================
# User configuration
# =========================
nc_path = r"Z:/staff/Qiang_S/09_Datasets/01N_Input/01Global_SyntheticFertilizer/01DownloadDatasets/01Fer_Wheat/01LUH2v2_Fer_Wheat/fertilizer_application_histsoc_whe_1850-2015.nc"
out_dir = r"Z:/staff/Qiang_S/09_Datasets/01N_Input/01Global_SyntheticFertilizer/03TempData/LUH2v2_fer_WheatStep1Temp"

start_year = 1995
end_year   = 2015

var_name = "fertrate"
nodata_value = -9999

os.makedirs(out_dir, exist_ok=True)

# =========================
# Open NetCDF
# =========================
ds = xr.open_dataset(nc_path)

data = ds[var_name]          # (time, lat, lon)
lats = ds["latitude"].values
lons = ds["longitude"].values
years = ds["time"].values

# =========================
# Calculate spatial info
# =========================
res_x = abs(lons[1] - lons[0])
res_y = abs(lats[0] - lats[1])

# Top-left corner
transform = from_origin(
    lons.min() - res_x / 2,
    lats.max() + res_y / 2,
    res_x,
    res_y
)

# =========================
# Loop through years
# =========================
for year in range(start_year, end_year + 1):

    if year not in years:
        print(f"⚠ Year {year} not found, skipped")
        continue

    print(f"Processing year {year}")

    # Extract one year
    arr = data.sel(time=year).values.astype(np.float32)

    # Handle NaN → NoData
    arr = np.where(np.isnan(arr), nodata_value, arr)

    out_tif = os.path.join(out_dir, f"fertrate_{year}.tif")

    profile = {
        "driver": "GTiff",
        "height": arr.shape[0],
        "width": arr.shape[1],
        "count": 1,
        "dtype": "float32",
        "crs": "EPSG:4326",
        "transform": transform,
        "nodata": nodata_value,
        "compress": "LZW"
    }

    with rasterio.open(out_tif, "w", **profile) as dst:
        dst.write(arr, 1)

    print(f"✔ Saved: {out_tif}")

print("\n✅ All years finished")

In [ ]:
# =====================================================
# ---Input fertilizer---
# ---Dataset 3 LUH2v2---
# ---Step3 Resemple---useful tool--🎉🎉🎉🎉🎉🎉🎉🎉🎉
# =====================================================
import os
import glob
import numpy as np
import rasterio
from rasterio.warp import reproject, Resampling
from rasterio.transform import from_origin

# =========================
# User configuration
# =========================
input_dir  = r"Z:/staff/Qiang_S/09_Datasets/01N_Input/01Global_SyntheticFertilizer/03TempData/LUH2v2_fer_WheatStep1Temp"
output_dir = r"Z:/staff/Qiang_S/09_Datasets/01N_Input/01Global_SyntheticFertilizer/02StandardizedDatasets/01LUH2v2_Fer_Wheat_Standardized"

os.makedirs(output_dir, exist_ok=True)

nodata = -9999

# 5 arcmin resolution
out_res = 5.0 / 60.0  # 0.08333333333333333

# Global extent
xmin, ymin, xmax, ymax = -180, -90, 180, 90
out_width  = int((xmax - xmin) / out_res)   # 4320
out_height = int((ymax - ymin) / out_res)   # 2160

out_transform = from_origin(xmin, ymax, out_res, out_res)

# =========================
# Batch resampling
# =========================
tif_files = sorted(glob.glob(os.path.join(input_dir, "*.tif")))
print(f"Found {len(tif_files)} files")

for tif in tif_files:
    fname = os.path.basename(tif)
    out_tif = os.path.join(output_dir, fname)

    print(f"Resampling: {fname}")

    with rasterio.open(tif) as src:
        src_data = src.read(1)
        src_transform = src.transform
        src_crs = src.crs
        src_nodata = src.nodata

        # Prepare output array
        dst_data = np.full(
            (out_height, out_width),
            nodata,
            dtype=np.float32
        )

        # Reproject / resample
        reproject(
            source=src_data,
            destination=dst_data,
            src_transform=src_transform,
            src_crs=src_crs,
            src_nodata=src_nodata,
            dst_transform=out_transform,
            dst_crs="EPSG:4326",
            dst_nodata=nodata,
            resampling=Resampling.average   # resample, bilinear, nearest, average等
        )

        profile = {
            "driver": "GTiff",
            "height": out_height,
            "width": out_width,
            "count": 1,
            "dtype": "float32",
            "crs": "EPSG:4326",
            "transform": out_transform,
            "nodata": nodata,
            "compress": "LZW"
        }

        with rasterio.open(out_tif, "w", **profile) as dst:
            dst.write(dst_data, 1)

    print(f"✔ Saved: {out_tif}")

print("\n✅ All files resampled to 5 arcmin")

In [ ]:

# ============================================================
# Global Wheat NUE - GeoTIFF Grid Standardization
# Match mask by year | Nearest-neighbor gap filling
# JupyterLab - Single Cell
# ============================================================

import os
import re
import csv
import traceback
from pathlib import Path
from datetime import datetime

import numpy as np
import rasterio
from rasterio.warp import reproject
from rasterio.enums import Resampling
from scipy.ndimage import distance_transform_edt
from tqdm.auto import tqdm


# ============================================================
# STEP 1. CONFIGURATION
# ============================================================

CONFIG = {
    # Input root directory (all subfolders searched recursively)
    "input_root": r"Z:\staff\Qiang_S\09_Datasets\02N_Output\01Global_Yield_Wheat\02StandardizedDatasets",

    # Mask directory (all subfolders searched recursively)
    # Example: wheat_1995.tif, wheat_1996.tif, ...
    "mask_root": r"Z:\staff\Qiang_S\09_Datasets\03GlobalNUE_Results\01Global_NUE_Wheat\01WheatNUE_V6_M6_150Temp",

    # Output root directory
    "output_root": r"Z:\staff\Qiang_S\09_Datasets\99ForTest",

    # Years to process (inclusive)
    "year_start": 2015,
    "year_end": 2020,

    # Output NoData
    "out_nodata": -9999.0,

    # Mask validity:
    # "positive" -> valid where mask > 0
    # "nonzero"  -> valid where mask != 0
    # "not_nodata" -> all valid mask pixels
    "mask_valid_rule": "positive",

    # Spatial resampling of continuous input raster values:
    # "nearest" preserves original values
    # "bilinear" interpolates spatially
    "resampling": "nearest",

    # Which pixels are treated as missing input data?
    "input_zero_is_nodata": False,
    "input_negative_is_nodata": False,

    # If no valid source pixels exist inside mask:
    # "nodata" -> output remains -9999
    # "error" -> skip file with error
    "all_missing_policy": "nodata",

    # Whether to overwrite existing outputs
    "overwrite": False,

    # Maximum memory use during scipy distance transform
    # This implementation processes a full raster at once.
    "print_details": True,
}


# ============================================================
# STEP 2. UTILITIES
# ============================================================

def get_year(path):
    """
    Identify year from the final four digits in filename stem.
    Example: GCSN_Wheat_1996.tif -> 1996
    """
    stem = Path(path).stem
    match = re.search(r"(\d{4})$", stem)
    return int(match.group(1)) if match else None


def find_tiffs(root):
    """Find all .tif and .tiff files recursively."""
    root = Path(root)
    return sorted(
        p for p in root.rglob("*")
        if p.is_file() and p.suffix.lower() in {".tif", ".tiff"}
    )


def same_grid(a, b):
    return (
        a.crs == b.crs
        and a.width == b.width
        and a.height == b.height
        and a.transform.almost_equals(b.transform)
    )


def get_mask_valid(mask_ds, rule):
    """
    Read mask and construct Boolean valid domain.
    """
    mask = mask_ds.read(1, masked=True)
    data = mask.filled(np.nan).astype(np.float32)

    valid = np.isfinite(data)

    if rule == "positive":
        valid &= data > 0
    elif rule == "nonzero":
        valid &= data != 0
    elif rule == "not_nodata":
        pass
    else:
        raise ValueError(f"Unknown mask rule: {rule}")

    return valid


def nearest_fill_inside_mask(arr, mask_valid):
    """
    Fill missing pixels inside the mask using nearest VALID
    source pixel INSIDE the mask.

    Outside-mask pixels are never used as fill sources.
    """
    valid_source = mask_valid & np.isfinite(arr)
    missing = mask_valid & ~np.isfinite(arr)

    filled = arr.copy()

    if not missing.any():
        return filled, 0, int(valid_source.sum())

    if not valid_source.any():
        return filled, 0, 0

    # EDT finds nearest zero in ~valid_source,
    # i.e. nearest valid source pixel.
    indices = distance_transform_edt(
        ~valid_source,
        return_distances=False,
        return_indices=True
    )

    rows, cols = np.where(missing)

    filled[rows, cols] = arr[
        indices[0, rows, cols],
        indices[1, rows, cols]
    ]

    return filled, len(rows), int(valid_source.sum())


def align_input_to_mask(src, mask_ds, cfg):
    """
    Align a source raster to the mask's exact grid.
    """
    height = mask_ds.height
    width = mask_ds.width

    dst = np.full((height, width), np.nan, dtype=np.float32)

    method = {
        "nearest": Resampling.nearest,
        "bilinear": Resampling.bilinear
    }[cfg["resampling"]]

    # Read source as masked data: source NoData -> NaN
    arr = src.read(1, masked=True).filled(np.nan).astype(np.float32)

    arr[~np.isfinite(arr)] = np.nan

    if cfg["input_zero_is_nodata"]:
        arr[arr == 0] = np.nan

    if cfg["input_negative_is_nodata"]:
        arr[arr < 0] = np.nan

    reproject(
        source=arr,
        destination=dst,
        src_transform=src.transform,
        src_crs=src.crs,
        src_nodata=np.nan,
        dst_transform=mask_ds.transform,
        dst_crs=mask_ds.crs,
        dst_nodata=np.nan,
        resampling=method,
        init_dest_nodata=True,
        num_threads=2
    )

    return dst


def process_one_file(input_path, mask_path, output_path, cfg):
    """
    Process one GeoTIFF.
    """

    with rasterio.open(mask_path) as mask_ds, \
         rasterio.open(input_path) as src:

        if src.crs is None or mask_ds.crs is None:
            raise ValueError("Input or mask raster has no CRS")

        mask_valid = get_mask_valid(
            mask_ds, cfg["mask_valid_rule"]
        )

        arr = align_input_to_mask(src, mask_ds, cfg)

        # Restrict to mask.
        arr[~mask_valid] = np.nan

        # Before filling
        missing_before = int(
            np.count_nonzero(mask_valid & ~np.isfinite(arr))
        )

        # Fill only from valid pixels inside mask.
        arr, filled_count, valid_source_count = \
            nearest_fill_inside_mask(arr, mask_valid)

        if valid_source_count == 0 and missing_before > 0:
            if cfg["all_missing_policy"] == "error":
                raise ValueError(
                    "No valid input pixels inside the mask"
                )

        # Ensure outside mask = NoData
        arr[~mask_valid] = np.nan

        remaining_missing = int(
            np.count_nonzero(mask_valid & ~np.isfinite(arr))
        )

        output = np.where(
            np.isfinite(arr),
            arr,
            cfg["out_nodata"]
        ).astype(np.float32)

        # Use mask as spatial reference.
        profile = mask_ds.profile.copy()
        profile.update(
            driver="GTiff",
            count=1,
            dtype="float32",
            nodata=cfg["out_nodata"],
            compress="LZW",
            predictor=3,
            BIGTIFF="IF_SAFER"
        )

        # Avoid incompatible inherited mask profile settings.
        profile.pop("photometric", None)
        profile.pop("alpha", None)
        profile.pop("colorinterp", None)

        output_path.parent.mkdir(
            parents=True, exist_ok=True
        )

        with rasterio.open(output_path, "w", **profile) as dst:
            dst.write(output, 1)

        return {
            "status": "SUCCESS",
            "mask_valid_pixels": int(mask_valid.sum()),
            "valid_source_pixels": valid_source_count,
            "missing_before_fill": missing_before,
            "filled_pixels": filled_count,
            "remaining_missing": remaining_missing,
            "grid_already_matched": same_grid(src, mask_ds)
        }


# ============================================================
# STEP 3. DISCOVER INPUT AND MASK FILES
# ============================================================

input_root = Path(CONFIG["input_root"]).resolve()
mask_root = Path(CONFIG["mask_root"]).resolve()
output_root = Path(CONFIG["output_root"]).resolve()

if not input_root.is_dir():
    raise FileNotFoundError(f"Input directory not found: {input_root}")

if not mask_root.is_dir():
    raise FileNotFoundError(f"Mask directory not found: {mask_root}")

if input_root == output_root:
    raise ValueError("Input and output roots must differ")

# Prevent recursive inclusion of outputs.
if output_root.is_relative_to(input_root):
    raise ValueError(
        "Output root must not be inside input root"
    )

input_files = [
    p for p in find_tiffs(input_root)
    if get_year(p) is not None
    and CONFIG["year_start"] <= get_year(p) <= CONFIG["year_end"]
]

mask_files = find_tiffs(mask_root)

mask_by_year = {}

for p in mask_files:
    year = get_year(p)
    if year is None:
        continue

    if year in mask_by_year:
        raise ValueError(
            f"Multiple mask files found for year {year}:\n"
            f"  {mask_by_year[year]}\n  {p}"
        )

    mask_by_year[year] = p

print("=" * 65)
print("Global Wheat NUE - Spatial Grid Standardization")
print("=" * 65)
print("Input root: ", input_root)
print("Mask root:  ", mask_root)
print("Output root:", output_root)
print("Input files:", len(input_files))
print("Mask years: ", sorted(mask_by_year.keys()))
print("Started:    ", datetime.now())
print("=" * 65)


# ============================================================
# STEP 4. PROCESS ALL FILES
# ============================================================

logs = []

for input_path in tqdm(input_files, desc="Standardizing GeoTIFFs"):

    year = get_year(input_path)
    relative_path = input_path.relative_to(input_root)

    output_path = output_root / relative_path

    record = {
        "year": year,
        "input_file": str(input_path),
        "output_file": str(output_path),
        "mask_file": "",
        "status": "",
        "error": ""
    }

    try:
        mask_path = mask_by_year.get(year)

        if mask_path is None:
            record["status"] = "SKIPPED_NO_MASK"
            logs.append(record)
            continue

        record["mask_file"] = str(mask_path)

        if output_path.exists() and not CONFIG["overwrite"]:
            record["status"] = "SKIPPED_EXISTS"
            logs.append(record)
            continue

        info = process_one_file(
            input_path,
            mask_path,
            output_path,
            CONFIG
        )

        record.update(info)

        if CONFIG["print_details"]:
            print(
                f"\n[{year}] {relative_path}"
                f"\n  Filled: {info['filled_pixels']:,}"
                f"\n  Remaining missing: "
                f"{info['remaining_missing']:,}"
                f"\n  Grid matched: "
                f"{info['grid_already_matched']}"
            )

    except Exception as e:
        record["status"] = "ERROR"
        record["error"] = str(e)
        print(f"\nERROR: {input_path}\n{e}")

    logs.append(record)


# ============================================================
# STEP 5. SAVE PROCESS LOG
# ============================================================

output_root.mkdir(parents=True, exist_ok=True)

log_path = output_root / "Standardization_Log.csv"

log_fields = [
    "year",
    "input_file",
    "output_file",
    "mask_file",
    "status",
    "grid_already_matched",
    "mask_valid_pixels",
    "valid_source_pixels",
    "missing_before_fill",
    "filled_pixels",
    "remaining_missing",
    "error"
]

with open(log_path, "w", newline="", encoding="utf-8-sig") as f:
    writer = csv.DictWriter(f, fieldnames=log_fields)
    writer.writeheader()
    writer.writerows(logs)

from collections import Counter

counts = Counter(r["status"] for r in logs)

print("\n" + "=" * 65)
print("PROCESSING COMPLETED")
print("=" * 65)

for status, count in sorted(counts.items()):
    print(f"{status}: {count}")

print("Log:", log_path)
print("Finished:", datetime.now())
print("=" * 65)


In [ ]:
# ============================================================
# Processing 01HaNi_Fer_Wheat_Standardized--- Please refer to the Maize code!!!
# ---Step1--successful ------calculate the Mount of N---for specific crop Man N rate---
# --....---
# ---Step8--✅✅✅✅
# ============================================================

In [ ]:
# =====================================================
# -------Global Maize NUE---Based on median-------------
# =====================================================

In [ ]:
# =====================================================
# -------Global Wheat NUE---Based on median-------------
# ---Step 1----Load package----------
# =====================================================

import os
import sys
import glob
import math
import shutil
import tempfile
from collections import defaultdict
import numpy as np
import rasterio
from rasterio.enums import Resampling
from rasterio.warp import reproject, calculate_default_transform
from rasterio.windows import Window
from rasterio.transform import Affine
import geopandas as gpd
import pandas as pd
from scipy import ndimage
from tqdm import tqdm
from datetime import datetime

In [ ]:
# =====================================================
# -------Global Wheat NUE---Based on median-------------
# ---Step 1----Load package----------
# ---Step 2----Setting parameters----------
# =====================================================

# -------------------------- CONFIG --------------------------
CONFIG = {
    # Parameter folder list (Example: Multiple dataset folders can be listed, with each folder containing tif files organized by year).
    # You can put each "dataset" in a separate directory, for example /data/data_yield/ds1/....tif, /data/data_yield/ds2/...
    "data_yield_dirs": [
        r"Z:/staff/Qiang_S/09_Datasets/02N_Output/01Global_Yield_Wheat/02StandardizedDatasets/GAEZ_Wheat",
        r"Z:/staff/Qiang_S/09_Datasets/02N_Output/01Global_Yield_Wheat/02StandardizedDatasets/GDHY_wheat_5min",
        r"Z:/staff/Qiang_S/09_Datasets/02N_Output/01Global_Yield_Wheat/02StandardizedDatasets/GlobalCropYield5min_Yield",
        r"Z:/staff/Qiang_S/09_Datasets/02N_Output/01Global_Yield_Wheat/02StandardizedDatasets/GlobalWheatYield4km_ReProj5min_Mean",
        r"Z:/staff/Qiang_S/09_Datasets/02N_Output/01Global_Yield_Wheat/02StandardizedDatasets/SPAM_Wheat",
        # ...
    ],
    "data_F_dirs": [
        r"Z:/staff/Qiang_S/09_Datasets/01N_Input/01Global_SyntheticFertilizer/02StandardizedDatasets/01GCSN_Fer_Wheat_Standardized",
        r"Z:/staff/Qiang_S/09_Datasets/01N_Input/01Global_SyntheticFertilizer/02StandardizedDatasets/01HaNi_Fer_Wheat_Standardized",
        r"Z:/staff/Qiang_S/09_Datasets/01N_Input/01Global_SyntheticFertilizer/02StandardizedDatasets/01LUH2v2_Fer_Wheat_Standardized",
        r"Z:/staff/Qiang_S/09_Datasets/01N_Input/01Global_SyntheticFertilizer/02StandardizedDatasets/01NPKGRIDS_Fer_Wheat_Standardized",
    ],
    "data_M_dirs": [
        r"Z:/staff/Qiang_S/09_Datasets/01N_Input/02Global_Manure/02StandardizedDatasets/01GCSN_Man_Wheat_Standardized",
        r"Z:/staff/Qiang_S/09_Datasets/01N_Input/02Global_Manure/02StandardizedDatasets/01HaNi_Man_Wheat_Standardized",
    ],
     #"data_CR_dirs": [
        # r"D:/1_PrecisionAgriculture/1_Researching/01_Dataset/03Global_CropResidues/01Wheat/PKU1995_2020",
         #r"D:/1_PrecisionAgriculture/1_Researching/01_Dataset/03Global_CropResidues/01Wheat/RADAR4KIT1997_2020_5min",
     #],
    "data_D_dirs": [
        r"Z:/staff/Qiang_S/09_Datasets/01N_Input/03Global_Deposition/02StandardizedDatasets/4_WL_5min4Crops1995_2020",
    ],

    # Points.xlsx path (first column: serial number, second column: Lon, longitude, third column: Lat, fourth column: year; starting from the second row).
    "points_xlsx": r"Z:/staff/Qiang_S/09_Datasets/09OtherDataUsed/09_Meta_AnalysisFields/NUE_Global_Farms 20251103_calculateTopWheatForCal.xlsx",

    # Output path
    "output_dir": r"Z:/staff/Qiang_S/09_Datasets/03GlobalNUE_Results/01Global_NUE_Wheat/01WheatNUE_V6_20260320",

    # Constant settings
    "SN": 3.5,    #  Seed N，wheat 3.5
    "BNF": 5.0,   #  biological nitrogen fixation， 
    
    #  grain N concentration, wheat----V5_20260311---
    # "M": -0.0002105,
    # "N": 3.2333,
    # "P": 1.81,
    # "Q": 1.95,

    #  grain N concentration, wheat----V6_20260320---
    "M": -0.0002105,
    "N": 3.2333,
    "P": 1.90,
    "Q": 2.02, 
    

    # nodata value for outputs
    "out_nodata": -9999.0,

    # year range (inclusive)
    "year_start": 1995,
    "year_end": 2020,

    # tempfile（None -> outo create tempfile）
    "tmp_dir": None,

    # Internal block size (adjustable to control memory/I/O)
    "block_height": 512,
    "block_width": 512,

    # Whether to generate a temporary GeoTIFF for a single combination in the middle (Recommended: True)
    "write_combo_tifs": True,
}
# ------------------------ END CONFIG ------------------------
print("Current time:", datetime.now())

In [ ]:
# =====================================================
# -------Global Wheat NUE---Based on median-------------
# ---Step 1----Load package----------
# ---Step 2----Setting parameters----------
# ---Step 3--- Utility functions ----------------
# =====================================================

def ensure_dir(path):
    os.makedirs(path, exist_ok=True)

def year_from_filename(path):
    """
   As required, the last four characters of the filename are the year identifier, used to identify the year.
   For example, "something_2000.tif" -> 2000
    """
    base = os.path.splitext(os.path.basename(path))[0]
    if len(base) < 4:
        return None
    s = base[-4:]
    if s.isdigit():
        return int(s)
    return None

def collect_files_by_year(dirs):
    """
    Input: A list of `dirs`, each representing a dataset folder (containing year-based TIFF files).
    Output: A list of `datasets`, where each dataset is a dictionary: year -> filepath.
    """
    datasets = []
    for d in dirs:
        files = glob.glob(os.path.join(d, "*.tif")) + glob.glob(os.path.join(d, "*.tiff"))
        year_map = {}
        for f in files:
            y = year_from_filename(f)
            if y is not None:
                year_map[y] = f
        datasets.append(year_map)
    return datasets

def reproject_match(src_arr, src_profile, target_profile):
    """
    Reproject/resample src_arr(bands, h, w) onto the transform/width/height/crs of target_profile
    Return the reprojected array(bands, H, W)
    """
    dst_profile = target_profile.copy()
    dst_height = dst_profile['height']
    dst_width = dst_profile['width']
    dst_crs = dst_profile['crs']
    dst_transform = dst_profile['transform']
    dst_count = src_arr.shape[0]
    dst = np.zeros((dst_count, dst_height, dst_width), dtype=src_arr.dtype)

    # Using rasterio.reproject
    for i in range(dst_count):
        reproject(
            source=src_arr[i],
            destination=dst[i],
            src_transform=src_profile['transform'],
            src_crs=src_profile['crs'],
            dst_transform=dst_transform,
            dst_crs=dst_crs,
            resampling=Resampling.nearest
        )
    return dst

def read_raster_as_float(path):
    """Read a single file as a float32 and return (arr, profile) arr shape (1,h,w)."""
    with rasterio.open(path) as src:
        arr = src.read(1).astype('float32')
        profile = src.profile.copy()
    return arr, profile

def expand_to_stack(arr):
    """如果输入是 2D 返回 (1,h,w)"""
    if arr.ndim == 2:
        return np.expand_dims(arr, 0)
    return arr

def nearest_fill_2d(arr, nodata=np.nan):
    """
    Perform nearest-neighbor padding on a 2D array `arr` (float, NaN indicates invalid):
    Return the padded array (without modifying the original array), or return the original array if all elements are invalid.
    Implementation: Use `scipy.ndimage.distance_transform_edt(return_indices=True)`
    """
    a = np.array(arr, copy=True)
    mask_valid = ~np.isnan(a)
    if mask_valid.any() is False:
        return a  # Completely invalid
    if mask_valid.all():
        return a  # Completely valid
    # distance transform with indices of nearest valid pixel
    # create mask of invalid; distance_transform_edt on invalid mask returns distances to nearest valid
    invalid = ~mask_valid
    # return indices of nearest valid pixel
    _, inds = ndimage.distance_transform_edt(invalid, return_indices=True)
    # inds is shape (2, h, w) of indices of nearest False in invalid => nearest True in mask_valid
    row_inds = inds[0]
    col_inds = inds[1]
    filled = a[row_inds, col_inds]
    return filled

def window_generator(height, width, block_h, block_w):
    """Generate rasterio windows (rows) and traverse the raster blocks."""
    for i in range(0, height, block_h):
        h = min(block_h, height - i)
        for j in range(0, width, block_w):
            w = min(block_w, width - j)
            yield Window(j, i, w, h), i, j, h, w

# ---------------- Main processing ----------------
def main(config):
    # load config
    out_dir = config['output_dir']
    ensure_dir(out_dir)
    tmp_root = config['tmp_dir'] or tempfile.mkdtemp(prefix="nue_tmp_")
    ensure_dir(tmp_root)
    print("Temporary directory:", tmp_root)

    # collect datasets per parameter
    dy_ds = collect_files_by_year(config['data_yield_dirs'])
    dF_ds = collect_files_by_year(config['data_F_dirs'])
    dM_ds = collect_files_by_year(config['data_M_dirs'])
    #dCR_ds = collect_files_by_year(config['data_CR_dirs'])
    dD_ds = collect_files_by_year(config['data_D_dirs'])

    print("Collected datasets counts (yield,F,M,CR,D):", len(dy_ds), len(dF_ds), len(dM_ds), len(dD_ds))

    # read points
    points_df = pd.read_excel(config['points_xlsx'], engine='openpyxl', header=None)
    # expected: first row header? The user said "Read from the second line.", we'll treat first row as header if it exists.
    # Safer: treat first row as header row of Excel; but user said first column sequence etc; so we assume file includes header row.
    # We'll try to detect: if header row is string types, use header=0; else header=None.
    # For simplicity, assume columns: index, lon, lat, year starting from row 2 => skip first row (header)
    # We'll detect and parse robustly:
    # If first row values are strings -> header -> use header=0, else header=None
    # But we've read raw; now ensure correct extraction:
    # To be robust: find first row that appears numeric in columns 1..3 and treat entries from that row onwards.

    # Try to normalize to columns: seq, lon, lat, year
    pts = []
    # iterate rows starting at row index 1 to skip header possibly
    for idx in range(1, len(points_df)):
        try:
            seq = points_df.iat[idx, 0]
            lon = float(points_df.iat[idx, 1])
            lat = float(points_df.iat[idx, 2])
            year = int(points_df.iat[idx, 3])
            pts.append({'seq': seq, 'lon': lon, 'lat': lat, 'year': year})
        except Exception:
            # skip row if cannot parse
            continue
    if len(pts) == 0:
        # fallback: try reading all rows as data
        for idx in range(0, len(points_df)):
            try:
                seq = points_df.iat[idx, 0]
                lon = float(points_df.iat[idx, 1])
                lat = float(points_df.iat[idx, 2])
                year = int(points_df.iat[idx, 3])
                pts.append({'seq': seq, 'lon': lon, 'lat': lat, 'year': year})
            except Exception:
                continue

    print(f"Loaded {len(pts)} points from {config['points_xlsx']}")

    # Prepare points result structure: we'll append per-point rows later
    point_results_rows = []

    # For each year
    for year in range(config['year_start'], config['year_end'] + 1):
        print("\n=== Processing year", year, "===")
        # Build list of files available per dataset for this year
        # For each dataset list, get map year->filepath
        # For each dataset index in dy_ds etc, check if year in that dataset
        dy_files = [(i, ds[year]) for i, ds in enumerate(dy_ds) if year in ds]
        dF_files = [(i, ds[year]) for i, ds in enumerate(dF_ds) if year in ds]
        dM_files = [(i, ds[year]) for i, ds in enumerate(dM_ds) if year in ds]
        #dCR_files = [(i, ds[year]) for i, ds in enumerate(dCR_ds) if year in ds]
        dD_files = [(i, ds[year]) for i, ds in enumerate(dD_ds) if year in ds]

        #if not (dy_files and dF_files and dM_files and dCR_files and dD_files):
        if not (dy_files and dF_files and dM_files and dD_files):    
            print(f"Year {year}: not all parameter groups have files; skipping.")
            continue

        # Cartesian product combinations: iterate indices
        import itertools
        #combos = list(itertools.product(dy_files, dF_files, dM_files, dCR_files, dD_files))
        combos = list(itertools.product(dy_files, dF_files, dM_files, dD_files))
        n_combos = len(combos)
        print(f"Year {year}: {n_combos} combinations to compute.")

        # Prepare list to store paths of combo result tifs for later aggregation
        combo_tif_paths = []

        # Use first data_yield file as spatial reference (we will reproject other rasters if required)
        # But since different datasets may have different transforms, we will base on each data_yield file per combo.
        combo_counter = 0
        for combo in tqdm(combos, desc=f"Year {year} combos"):
            # combo is tuple of ((idx, dy_path), (idx,dF_path), ...)
            dy_path = combo[0][1]
            dF_path = combo[1][1]
            dM_path = combo[2][1]
        
            #dCR_path = combo[3][1]
            #dD_path = combo[4][1]
            dD_path = combo[3][1]
            
            combo_counter += 1

            # Read data_yield as reference
            try:
                with rasterio.open(dy_path) as src_dy:
                    ref_profile = src_dy.profile.copy()
                    ref_arr = src_dy.read(1).astype('float32')
                    ref_nodata = src_dy.nodata
                    if ref_nodata is None:
                        # assume negative or 0?
                        ref_nodata = np.nan
                    # convert nodata to nan for computation
                    if ref_nodata is not None:
                        ref_arr = np.where(ref_arr == ref_nodata, np.nan, ref_arr)
            except Exception as e:
                print("Error reading data_yield", dy_path, e)
                continue

            # If other rasters have different shape/transform/crs, reproject to reference
            def read_and_match(path):
                with rasterio.open(path) as src:
                    arr = src.read(1).astype('float32')
                    nodata = src.nodata
                    if nodata is not None:
                        arr = np.where(arr == nodata, np.nan, arr)
                    profile = src.profile.copy()
                    # If profile doesn't match ref_profile, reproject
                    need_reproject = (profile.get('crs') != ref_profile.get('crs') or
                                      profile.get('transform') != ref_profile.get('transform') or
                                      profile.get('width') != ref_profile.get('width') or
                                      profile.get('height') != ref_profile.get('height'))
                    if need_reproject:
                        # reproject to ref_profile
                        # create destination array shape
                        dst = np.full((ref_profile['height'], ref_profile['width']), np.nan, dtype='float32')
                        reproject(
                            source=arr,
                            destination=dst,
                            src_transform=profile['transform'],
                            src_crs=profile['crs'],
                            dst_transform=ref_profile['transform'],
                            dst_crs=ref_profile['crs'],
                            resampling=Resampling.nearest,
                            src_nodata=nodata,
                            dst_nodata=np.nan
                        )
                        arr = dst
                        profile = ref_profile.copy()
                    return arr, profile

            try:
                arr_F, pF = read_and_match(dF_path)
                arr_M, pM = read_and_match(dM_path)
                #arr_CR, pCR = read_and_match(dCR_path)
                arr_D, pD = read_and_match(dD_path)
            except Exception as e:
                print("Error reading other rasters in combo:", e)
                continue

            # Prepare arrays: ref_arr, arr_F, arr_M, arr_CR, arr_D
            # Apply rules:
            # - data_yield only compute where >0; else nodata
            # - arr_F: if NaN, fill with nearest neighbor
            # - other arr_*: NaN -> 0
            dy = ref_arr.copy()
            # treat zeros/negatives as not participating? The rule says data_yield>0 participates.
            mask_participate = ~np.isnan(dy) & (dy > 0)

            if not np.any(mask_participate):
                # no pixels to compute
                continue

            # fill data_F nearest
            # but only for pixels where participate True and arr_F is nan -> we fill globally to avoid per-pixel
            # We'll fill entire arr_F via nearest fill
            if np.isnan(arr_F).all():
                # no valid data_F anywhere -> treat as zeros? But problem says choose nearest value; if none exist treat as 0
                arr_F_filled = np.zeros_like(arr_F)
            else:
                arr_F_filled = nearest_fill_2d(arr_F)

                # wherever arr_F_filled still nan (shouldn't), set 0
                arr_F_filled = np.where(np.isnan(arr_F_filled), 0.0, arr_F_filled)

            # for other parameters missing -> 0
            arr_M = np.where(np.isnan(arr_M), 0.0, arr_M)
            #arr_CR = np.where(np.isnan(arr_CR), 0.0, arr_CR)
            arr_D = np.where(np.isnan(arr_D), 0.0, arr_D)

            # Compute C = M * data_yield + N ; clip to [P,Q]
            C = config['M'] * dy + config['N']
            C = np.where(np.isnan(C), np.nan, C)
            C = np.clip(C, config['P'], config['Q'])

            # Compute denominator
            #denom = arr_F_filled + arr_M + arr_CR + arr_D + config['SN'] + config['BNF']
            denom = arr_F_filled + arr_M + arr_D + config['SN'] + config['BNF']

            # compute NUE where mask_participate True and denom != 0 and C not nan
            numer = dy * C
            # Initialize result with nodata (NaN)
            result = np.full_like(dy, np.nan, dtype='float32')
            valid_calc = mask_participate & (~np.isnan(C)) & (~np.isnan(denom)) & (denom != 0)
            result[valid_calc] = numer[valid_calc] / denom[valid_calc]
            # set invalid to np.nan (already)

            # Now set nodata pixels to specified out_nodata for writing
            out_arr = np.where(np.isnan(result), config['out_nodata'], result).astype('float32')

            # Save combo result to temporary GeoTIFF (if configured)
            combo_name = f"combo_{year}_{combo_counter:04d}.tif"
            combo_path = os.path.join(tmp_root, combo_name)
            combo_tif_paths.append(combo_path)
            if config['write_combo_tifs']:
                # write with ref_profile but count=1, dtype=float32, nodata
                write_profile = ref_profile.copy()
                write_profile.update(
                    dtype='float32',
                    count=1,
                    compress='LZW',
                    nodata=config['out_nodata']
                )
                with rasterio.open(combo_path, 'w', **write_profile) as dst:
                    dst.write(out_arr, 1)
            else:
                # if not writing tifs, we still need to persist array (we'll use npy)
                np.save(combo_path + ".npy", out_arr)

        # end combos loop

        # If no combos produced results, skip
        if len(combo_tif_paths) == 0:
            print(f"Year {year}: no combo results produced.")
            continue

        # --- Aggregate per-pixel statistics across combo_tif_paths, block-wise ---
        # Use first combo for template
        template_path = combo_tif_paths[0]
        with rasterio.open(template_path) as tpl:
            tpl_profile = tpl.profile.copy()
            height = tpl_profile['height']
            width = tpl_profile['width']
            transform = tpl_profile['transform']
            crs = tpl_profile['crs']
            dtype = 'float32'

        # prepare output files paths for this year
        out_median = os.path.join(out_dir, f"NUE_median_{year}.tif")
        out_mean = os.path.join(out_dir, f"NUE_mean_{year}.tif")
        out_max = os.path.join(out_dir, f"NUE_max_{year}.tif")
        out_min = os.path.join(out_dir, f"NUE_min_{year}.tif")
        out_dif = os.path.join(out_dir, f"NUE_dif_{year}.tif")
        out_n = os.path.join(out_dir, f"NUE_n_{year}.tif")

        # create output datasets
        common_profile = tpl_profile.copy()
        common_profile.update(count=1, dtype='float32', compress='LZW', nodata=config['out_nodata'])
        outs = {}
        for p in [out_median, out_mean, out_max, out_min, out_dif, out_n]:
            with rasterio.open(p, 'w', **common_profile) as dst:
                dst.write(np.full((1, height, width), config['out_nodata'], dtype='float32'))
        # We'll re-open in 'r+' mode to write windows
        dst_median = rasterio.open(out_median, 'r+')
        dst_mean = rasterio.open(out_mean, 'r+')
        dst_max = rasterio.open(out_max, 'r+')
        dst_min = rasterio.open(out_min, 'r+')
        dst_dif = rasterio.open(out_dif, 'r+')
        dst_n = rasterio.open(out_n, 'r+')

        # For each block, read the same window from all combo tifs and compute stats
        n_combo = len(combo_tif_paths)
        print(f"Aggregating {n_combo} combo rasters block-wise...")
        for window, i_row, j_col, h, w in window_generator(height, width, config['block_height'], config['block_width']):
            # stack array shape (n_combo, h, w)
            stack = np.zeros((n_combo, h, w), dtype='float32')
            for k, path in enumerate(combo_tif_paths):
                with rasterio.open(path) as src:
                    arr = src.read(1, window=window)
                # convert nodata to nan for stats
                arr = np.where(arr == config['out_nodata'], np.nan, arr)
                stack[k] = arr

            # compute stats along axis 0
            # Count of valid values
            valid_mask = ~np.isnan(stack)
            n_valid = np.sum(valid_mask, axis=0).astype('float32')
            # when n_valid==0, set outputs nodata
            mean = np.nanmean(stack, axis=0)
            # for median: if even number of valid values, take mean of middle two: numpy.nanmedian already does that
            median = np.nanmedian(stack, axis=0)
            maxv = np.nanmax(stack, axis=0)
            minv = np.nanmin(stack, axis=0)
            dif = maxv - minv

            # replace NaNs with out_nodata
            mean = np.where(np.isnan(mean), config['out_nodata'], mean).astype('float32')
            median = np.where(np.isnan(median), config['out_nodata'], median).astype('float32')
            maxv = np.where(np.isnan(maxv), config['out_nodata'], maxv).astype('float32')
            minv = np.where(np.isnan(minv), config['out_nodata'], minv).astype('float32')
            dif = np.where(np.isnan(dif), config['out_nodata'], dif).astype('float32')
            n_valid_out = np.where(n_valid == 0, config['out_nodata'], n_valid).astype('float32')

            # write windows
            dst_mean.write(mean, 1, window=window)
            dst_median.write(median, 1, window=window)
            dst_max.write(maxv, 1, window=window)
            dst_min.write(minv, 1, window=window)
            dst_dif.write(dif, 1, window=window)
            dst_n.write(n_valid_out, 1, window=window)

        # close outputs
        dst_mean.close()
        dst_median.close()
        dst_max.close()
        dst_min.close()
        dst_dif.close()
        dst_n.close()

        print(f"Year {year} aggregated outputs written to {out_dir}")

        # --- Point extraction for this year ---
        # For every point with this year, sample all combo rasters at the point coordinates
        pts_this_year = [p for p in pts if p['year'] == year]
        if len(pts_this_year) > 0:
            print(f"Extracting values for {len(pts_this_year)} points for year {year} ...")
            # open all combo rasters for reading
            combo_srcs = [rasterio.open(p) for p in combo_tif_paths]
            for pt in pts_this_year:
                lon = pt['lon']
                lat = pt['lat']
                seq = pt['seq']
                values = []
                for src in combo_srcs:
                    try:
                        rowcol = src.index(lon, lat)
                        v = src.read(1)[rowcol[0], rowcol[1]]
                        if v == config['out_nodata']:
                            values.append(np.nan)
                        else:
                            values.append(float(v))
                    except Exception:
                        values.append(np.nan)
                # build row: seq, lon, lat, year, then all values
                row = {'seq': seq, 'lon': lon, 'lat': lat, 'year': year}
                for i, val in enumerate(values, 1):
                    row[f'val_{i}'] = val
                point_results_rows.append(row)
            for s in combo_srcs:
                s.close()

        # Clean up temporary combo tifs to save disk, optional: keep if needed
        # We'll delete them to avoid filling disk
        print(f"Cleaning up {len(combo_tif_paths)} temporary combo files...")
        for p in combo_tif_paths:
            try:
                os.remove(p)
            except Exception:
                pass

    # end years loop

    # --- Write Result.xlsx for points ---
    if len(point_results_rows) > 0:
        res_df = pd.DataFrame(point_results_rows)
        # sort by seq, year
        res_df = res_df.sort_values(['seq', 'year'])
        result_path = os.path.join(out_dir, "Result.xlsx")
        with pd.ExcelWriter(result_path, engine='openpyxl') as writer:
            res_df.to_excel(writer, index=False, sheet_name='Results')
        print("Point extraction results written to:", result_path)
    else:
        print("No point results to write.")

    # done
    print("All done.")
    # optional: remove tmp_root if created by us
    # if config['tmp_dir'] is None:
    #     shutil.rmtree(tmp_root)

In [ ]:
# =====================================================
# -------Global Wheat NUE---Based on median-------------
# ---Step 1----Load package----------
# ---Step 2----Setting parameters----------
# ---Step 3--- Utility functions ----------------
# ---Step 4--- main ----------------
# =====================================================
if __name__ == "__main__":
    # sanity checks on CONFIG
    for key in ['data_yield_dirs','data_F_dirs','data_M_dirs','data_D_dirs']:   
        if not isinstance(CONFIG[key], list):
            print(f"CONFIG error: {key} must be a list of dataset directories.")
            sys.exit(1)
    main(CONFIG)

print("Current time:", datetime.now())

In [ ]:
# =====================================================
# -------Global Wheat NUE---Based on median-------------
# ---Step 1----Load package----------
# ---Step 2----Setting parameters----------
# ---Step 3--- Utility functions ----------------
# ---Step 4--- main ----------------
# ---Step 5--- simple validation -🎇🎇🎇--------------
# ---Step 6--- Remove outliers, Useful tool,Pixels number should larger than 6, NUE more then 150; country mean + 2 standard deviations，set non, 
# =====================================================
import rasterio
import rasterio.mask
import numpy as np
import os
import geopandas as gpd
import pandas as pd
from rasterio.features import geometry_mask
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter


A_FOLDER       = r"Z:\staff\Qiang_S\09_Datasets\03GlobalNUE_Results\01Global_NUE_Wheat/01WheatNUE_V6_20260320"
MASK_FOLDER    = r"Z:\staff\Qiang_S\09_Datasets\03GlobalNUE_Results\01Global_NUE_Wheat/01WheatNUE_V6_20260320"

OUT_FOLDER     = r"Z:\staff\Qiang_S\09_Datasets\03GlobalNUE_Results\01Global_NUE_Wheat\01WheatNUE_V6_M6_150C"

SHP_PATH       = r"Z:/staff/Qiang_S/09_Datasets/09OtherDataUsed/GDAM_GlobalCountryLevel_Shp/GlobalCountryLevel_Shp.shp"
M49_FIELD      = "M49_Code"       # shp code
NAME_FIELD     = "COUNTRY"      # shp country name
EXCEL_OUT      = os.path.join(OUT_FOLDER, "outlier_removal_summary_V6_M6_150C.xlsx")

# ----year range----
START_YEAR     = 1995
END_YEAR       = 2020

NODATA         = -9999

# Step 1: 像素统计个数阈值（N < MIN_COUNT 则剔除）
MIN_COUNT      = 6

# Step 2: 硬性值域范围（保留区间：[LOWER_HARD, UPPER_HARD]）
LOWER_HARD     = 0.1
UPPER_HARD     = 150.0

# Step 3: 按国家标准差剔除
STD_FACTOR     = 2.0          # 标准差倍数
MAX_REMOVE_RATIO = 0.05       # 每国最多剔除比例（0.05 = 5%，即保留至少95%）

# =====================================================
os.makedirs(OUT_FOLDER, exist_ok=True)

# =====================================================
# 读取 SHP，统一投影为 EPSG:4326
# =====================================================
print("📂 Loading shapefile...")
gdf = gpd.read_file(SHP_PATH)
if gdf.crs is None or gdf.crs.to_epsg() != 4326:
    gdf = gdf.to_crs(epsg=4326)

# 按 m49 排序
gdf = gdf.sort_values(by=M49_FIELD).reset_index(drop=True)
country_list = gdf[M49_FIELD].tolist()
print(f"  Countries loaded: {len(gdf)}")

# =====================================================
# 主循环
# =====================================================
# records_year: 每年一条汇总
# records_country: 每年×每国 详细记录
records_year    = []
records_country = []

for year in range(START_YEAR, END_YEAR + 1):

    A_tif   = os.path.join(A_FOLDER,    f"NUE_median_{year}.tif")
    N_tif   = os.path.join(MASK_FOLDER, f"NUE_n_{year}.tif")
    OUT_tif = os.path.join(OUT_FOLDER,  f"NUE_median_Opt_{year}.tif")

    if not os.path.exists(A_tif):
        print(f"[{year}] ⚠ A file missing, skipped.")
        continue
    if not os.path.exists(N_tif):
        print(f"[{year}] ⚠ Mask file missing, skipped.")
        continue

    with rasterio.open(A_tif) as src_a:
        A       = src_a.read(1).astype(np.float32)
        profile = src_a.profile.copy()
        transform = src_a.transform
        crs       = src_a.crs
        height, width = A.shape

    with rasterio.open(N_tif) as src_n:
        N = src_n.read(1)

    if A.shape != N.shape:
        print(f"[{year}] ⚠ Shape mismatch, skipped.")
        continue

    # -----------------------------------------------
    # Step 1: 剔除统计个数不足的像素
    # -----------------------------------------------
    valid_mask = (A != NODATA)
    initial_count = int(np.sum(valid_mask))

    step1_mask    = (N < MIN_COUNT) & valid_mask
    step1_removed = int(np.sum(step1_mask))
    A[step1_mask] = NODATA
    valid_mask    = (A != NODATA)

    # -----------------------------------------------
    # Step 2: 剔除硬性值域外的像素
    # -----------------------------------------------
    step2_upper   = (A > UPPER_HARD) & valid_mask
    step2_lower   = (A < LOWER_HARD) & valid_mask
    step2_removed = int(np.sum(step2_upper) + np.sum(step2_lower))
    A[step2_upper] = NODATA
    A[step2_lower] = NODATA
    valid_mask = (A != NODATA)

    # -----------------------------------------------
    # Step 3: 按国家进行 2σ 剔除
    # -----------------------------------------------
    step3_total_removed = 0
    gdf_reproj = gdf.to_crs(crs) if gdf.crs.to_epsg() != (crs.to_epsg() if crs else 4326) else gdf

    for _, country_row in gdf_reproj.iterrows():
        m49  = country_row[M49_FIELD]
        name = country_row[NAME_FIELD] if NAME_FIELD and NAME_FIELD in country_row else str(m49)
        geom = country_row.geometry

        if geom is None or geom.is_empty:
            records_country.append(_empty_country_record(year, m49, name))
            continue

        # 生成该国掩膜（True = 国家范围内）
        try:
            country_mask_arr = ~geometry_mask(
                [geom],
                transform=transform,
                invert=False,
                out_shape=(height, width)
            )
        except Exception:
            records_country.append(_empty_country_record(year, m49, name))
            continue

        # 该国有效像素
        in_country_valid = country_mask_arr & valid_mask
        pixel_indices    = np.where(in_country_valid)
        n_valid          = len(pixel_indices[0])

        if n_valid == 0:
            records_country.append({
                "Year": year, "M49": m49, "Country": name,
                "Valid_before_step3": 0,
                "Mean": np.nan, "Std": np.nan, "Upper_sigma": np.nan,
                "Candidate_removed": 0, "Max_allowed_remove": 0,
                "Step3_removed": 0, "Step3_removed_%": 0.0,
            })
            continue

        vals        = A[pixel_indices]
        mean_val    = float(np.mean(vals))
        std_val     = float(np.std(vals))
        upper_sigma = mean_val + STD_FACTOR * std_val

        # 候选剔除像素
        candidate_mask = in_country_valid & (A > upper_sigma)
        n_candidate    = int(np.sum(candidate_mask))

        # 最多允许删除的个数（向下取整，保证至少保留 90%）
        max_allowed = int(np.floor(n_valid * MAX_REMOVE_RATIO))

        # 实际删除：若候选超过上限，按值从大到小只删前 max_allowed 个
        if n_candidate <= max_allowed:
            actual_removed = n_candidate
            A[candidate_mask] = NODATA
        else:
            actual_removed = max_allowed
            if max_allowed > 0:
                cand_rows, cand_cols = np.where(candidate_mask)
                cand_vals = A[cand_rows, cand_cols]
                # 按值降序，只取前 max_allowed 个
                sort_idx  = np.argsort(-cand_vals)[:max_allowed]
                A[cand_rows[sort_idx], cand_cols[sort_idx]] = NODATA

        valid_mask = (A != NODATA)   # 更新全局 valid_mask
        step3_total_removed += actual_removed

        records_country.append({
            "Year": year,
            "M49":  m49,
            "Country": name,
            "Valid_before_step3": n_valid,
            "Mean":        round(mean_val, 6),
            "Std":         round(std_val, 6),
            "Upper_sigma": round(upper_sigma, 6),
            "Candidate_removed":   n_candidate,
            "Max_allowed_remove":  max_allowed,
            "Step3_removed":       actual_removed,
            "Step3_removed_%":     round(actual_removed / n_valid * 100, 4) if n_valid > 0 else 0.0,
        })

    # -----------------------------------------------
    # 年度汇总
    # -----------------------------------------------
    final_count   = int(np.sum(A != NODATA))
    total_removed = initial_count - final_count
    removed_ratio = total_removed / initial_count * 100 if initial_count > 0 else 0

    print(
        f"[{year}]  初始={initial_count:,}  "
        f"Step1={step1_removed:,}  Step2={step2_removed:,}  Step3={step3_total_removed:,}  "
        f"最终={final_count:,}  总剔除={total_removed:,} ({removed_ratio:.2f}%)"
    )

    records_year.append({
        "Year":             year,
        "Initial_Valid":    initial_count,
        "Step1_removed":    step1_removed,
        "Step1_%":          round(step1_removed / initial_count * 100, 4) if initial_count else 0,
        "Step2_removed":    step2_removed,
        "Step2_%":          round(step2_removed / initial_count * 100, 4) if initial_count else 0,
        "Step3_removed":    step3_total_removed,
        "Step3_%":          round(step3_total_removed / initial_count * 100, 4) if initial_count else 0,
        "Final_Valid":      final_count,
        "Total_Removed":    total_removed,
        "Removed_Ratio_%":  round(removed_ratio, 4),
    })

    # 写出栅格
    profile.update(dtype="float32", nodata=NODATA, compress="lzw")
    with rasterio.open(OUT_tif, "w", **profile) as dst:
        dst.write(A.astype(np.float32), 1)


def _empty_country_record(year, m49, name):
    return {
        "Year": year, "M49": m49, "Country": name,
        "Valid_before_step3": 0, "Mean": np.nan, "Std": np.nan,
        "Upper_sigma": np.nan, "Candidate_removed": 0,
        "Max_allowed_remove": 0, "Step3_removed": 0, "Step3_removed_%": 0.0,
    }


# =====================================================
# 导出 Excel
# =====================================================
if not records_year:
    print("\n⚠ No records to export.")
else:
    df_year    = pd.DataFrame(records_year)
    df_country = pd.DataFrame(records_country).sort_values(["Year", "M49"]).reset_index(drop=True)

    wb = Workbook()

    # ── 样式辅助 ──────────────────────────────────────
    thin   = Side(style="thin", color="AAAAAA")
    border = Border(left=thin, right=thin, top=thin, bottom=thin)

    def hdr_cell(ws, row, col, value, bg="2F75B6", fg="FFFFFF"):
        c = ws.cell(row=row, column=col, value=value)
        c.font      = Font(bold=True, color=fg, name="Arial", size=10)
        c.fill      = PatternFill("solid", start_color=bg)
        c.alignment = Alignment(horizontal="center", wrap_text=True)
        c.border    = border
        return c

    def data_cell(ws, row, col, value, bg="FFFFFF", fmt=None):
        c = ws.cell(row=row, column=col, value=value)
        c.font      = Font(name="Arial", size=10)
        c.fill      = PatternFill("solid", start_color=bg)
        c.alignment = Alignment(horizontal="center")
        c.border    = border
        if fmt:
            c.number_format = fmt
        return c

    # ── Sheet 1: 年度汇总 ─────────────────────────────
    ws1 = wb.active
    ws1.title = "Annual Summary"

    COL_COLORS_YEAR = {
        "Year": "FFFFFF",
        "Initial_Valid": "FFFFFF",
        "Step1_removed": "D6E4F0", "Step1_%": "D6E4F0",
        "Step2_removed": "D5E8D4", "Step2_%": "D5E8D4",
        "Step3_removed": "FFF2CC", "Step3_%": "FFF2CC",
        "Final_Valid": "FFFFFF",
        "Total_Removed": "F4CCCC", "Removed_Ratio_%": "F4CCCC",
    }

    cols_y = list(df_year.columns)
    for c_idx, col in enumerate(cols_y, 1):
        hdr_cell(ws1, 1, c_idx, col)

    for r_idx, row in enumerate(df_year.itertuples(index=False), 2):
        for c_idx, (col, val) in enumerate(zip(cols_y, row), 1):
            bg  = COL_COLORS_YEAR.get(col, "FFFFFF")
            fmt = "0.00" if "%" in col else ("#,##0" if col != "Year" else "0")
            data_cell(ws1, r_idx, c_idx, val, bg=bg, fmt=fmt)

    # 汇总行
    sr = len(df_year) + 2
    data_cell(ws1, sr, 1, "TOTAL/AVG", bg="D9D9D9").font = Font(bold=True, name="Arial", size=10)
    for c_idx, col in enumerate(cols_y[1:], 2):
        cl   = get_column_letter(c_idx)
        rng  = f"{cl}2:{cl}{len(df_year)+1}"
        fn   = f"=AVERAGE({rng})" if "%" in col else f"=SUM({rng})"
        fmt2 = "0.00" if "%" in col else "#,##0"
        c    = ws1.cell(row=sr, column=c_idx, value=fn)
        c.font   = Font(bold=True, name="Arial", size=10)
        c.fill   = PatternFill("solid", start_color="D9D9D9")
        c.alignment = Alignment(horizontal="center")
        c.border = border
        c.number_format = fmt2

    for c_idx, col in enumerate(cols_y, 1):
        ws1.column_dimensions[get_column_letter(c_idx)].width = max(len(col) + 4, 14)
    ws1.row_dimensions[1].height = 36
    ws1.freeze_panes = "B2"

    # ── Sheet 2: 国家明细 ─────────────────────────────
    ws2 = wb.create_sheet("Country Detail")
    cols_c = list(df_country.columns)

    COL_COLORS_CTY = {
        "Year": "FFFFFF", "M49": "FFFFFF", "Country": "FFFFFF",
        "Valid_before_step3": "FFFFFF",
        "Mean": "EAD1DC", "Std": "EAD1DC", "Upper_sigma": "EAD1DC",
        "Candidate_removed": "FFF2CC", "Max_allowed_remove": "FFF2CC",
        "Step3_removed": "F4CCCC", "Step3_removed_%": "F4CCCC",
    }

    for c_idx, col in enumerate(cols_c, 1):
        hdr_cell(ws2, 1, c_idx, col)

    for r_idx, row in enumerate(df_country.itertuples(index=False), 2):
        for c_idx, (col, val) in enumerate(zip(cols_c, row), 1):
            bg  = COL_COLORS_CTY.get(col, "FFFFFF")
            if "%" in col:
                fmt = "0.00"
            elif col in ("Mean", "Std", "Upper_sigma"):
                fmt = "0.000000"
            elif col not in ("Year", "M49", "Country"):
                fmt = "#,##0"
            else:
                fmt = None
            display_val = None if (isinstance(val, float) and np.isnan(val)) else val
            data_cell(ws2, r_idx, c_idx, display_val, bg=bg, fmt=fmt)

    for c_idx, col in enumerate(cols_c, 1):
        ws2.column_dimensions[get_column_letter(c_idx)].width = max(len(col) + 3, 12)
    ws2.column_dimensions["C"].width = 22
    ws2.row_dimensions[1].height = 36
    ws2.freeze_panes = "D2"

    # ── Sheet 3: 参数存档 ─────────────────────────────
    ws3 = wb.create_sheet("Parameters")
    params = [
        ("Parameter", "Value"),
        ("MIN_COUNT (Step1)", MIN_COUNT),
        ("LOWER_HARD (Step2)", LOWER_HARD),
        ("UPPER_HARD (Step2)", UPPER_HARD),
        ("STD_FACTOR (Step3)", STD_FACTOR),
        ("MAX_REMOVE_RATIO (Step3)", MAX_REMOVE_RATIO),
        ("M49_FIELD", M49_FIELD),
        ("SHP_PATH", SHP_PATH),
        ("START_YEAR", START_YEAR),
        ("END_YEAR", END_YEAR),
        ("NODATA", NODATA),
    ]
    for r_idx, (k, v) in enumerate(params, 1):
        for c_idx, val in enumerate([k, v], 1):
            c = ws3.cell(row=r_idx, column=c_idx, value=val)
            c.font   = Font(bold=(r_idx == 1), name="Arial", size=10,
                            color="FFFFFF" if r_idx == 1 else "000000")
            c.fill   = PatternFill("solid", start_color="2F75B6" if r_idx == 1 else "F5F5F5")
            c.border = border
            c.alignment = Alignment(horizontal="left")
    ws3.column_dimensions["A"].width = 28
    ws3.column_dimensions["B"].width = 50

    wb.save(EXCEL_OUT)
    print(f"\n📊 Excel saved: {EXCEL_OUT}")

print("\n🎯 All years processed.")

In [ ]:
# =====================================================
# -------Global Wheat NUE---Based on median-------------
# ---Step 1----Load package----------
# ---Step 2----Setting parameters----------
# ---Step 3--- Utility functions ----------------
# ---Step 4--- main ----------------
# ---Step 5--- simple validation -🎇🎇🎇--------------
# ---Step 6--- Remove outliers, Useful tool,Pixels number should larger than 6, NUE more then 150; country mean + 2 standard deviations，set non, 
# ---Step 7--- Check mean value
# =====================================================
import os
import glob
import rasterio
import numpy as np

# =========================
# Configuration
# =========================
INPUT_FOLDER = r"Z:/staff/Qiang_S/09_Datasets/03GlobalNUE_Results/01Global_NUE_Wheat\01WheatNUE_V6_M6_150C"
OUTPUT_TXT = r"Z:/staff/Qiang_S/09_Datasets/03GlobalNUE_Results/01Global_NUE_Wheat\01WheatNUE_V6_M6_150C/mean_by_yearV6_M6_150C.txt"

nodata = -9999

# =========================
# Collect results
# =========================
results = []

tif_files = sorted(glob.glob(os.path.join(INPUT_FOLDER, "*.tif")))

if not tif_files:
    raise RuntimeError("No tif files found!")

print(f"Found {len(tif_files)} tif files\n")

for tif in tif_files:
    fname = os.path.basename(tif)

    # --- Extract year from filename ---
    # e.g. A_Opt_1995.tif → 1995
    year = os.path.splitext(fname)[0].split("_")[-1]

    with rasterio.open(tif) as src:
        data = src.read(1).astype(np.float32)

    # Remove nodata
    valid = data != nodata
    valid_count = np.sum(valid)

    if valid_count == 0:
        mean_val = np.nan
        print(f"{year}: No valid pixels")
    else:
        mean_val = np.mean(data[valid])
        print(f"{year}: mean = {mean_val:.6f} (valid pixels: {valid_count})")

    results.append((year, mean_val))

# =========================
# Write to TXT
# =========================
with open(OUTPUT_TXT, "w", encoding="utf-8") as f:
    f.write("year,mean\n")
    for year, mean_val in results:
        if np.isnan(mean_val):
            f.write(f"{year},NaN\n")
        else:
            f.write(f"{year},{mean_val:.6f}\n")

print(f"\n✅ Finished! Results saved to:\n{OUTPUT_TXT}")

In [ ]:
# =====================================================
# -------Global Wheat NUE---Based on median-------------
# ---Step 1----Load package----------
# ---Step 2----Setting parameters----------
# ---Step 3--- Utility functions ----------------
# ---Step 4--- main ----------------
# ---Step 5--- simple validation -🎇🎇🎇--------------
# ---Step 6--- Remove outliers, Useful tool, mean + 2 standard deviations，set non, Pixels number should larger than 2
# ---Step 7--- Check mean value
# ---Step 8--- the normalized range of NUE
# =====================================================
import os
import glob
import numpy as np
import rasterio

# =========================
# Configuration
# =========================
A_FOLDER = r"Z:/staff/Qiang_S/09_Datasets/03GlobalNUE_Results/01Global_NUE_Wheat/01WheatNUE_V6_20260320"
B_FOLDER = r"Z:/staff/Qiang_S/09_Datasets/03GlobalNUE_Results/01Global_NUE_Wheat/01WheatNUE_V6_M6_150C"
OUT_FOLDER = r"Z:/staff/Qiang_S/09_Datasets/03GlobalNUE_Results/01Global_NUE_Wheat/01WheatNUE_V6_M6_150C_Nor"

nodata = -9999

os.makedirs(OUT_FOLDER, exist_ok=True)

# =========================
# Collect A files
# =========================
a_files = sorted(glob.glob(os.path.join(A_FOLDER, "NUE_dif_*.tif")))

if not a_files:
    raise RuntimeError("No NUE_dif_*.tif files found!")

print(f"Found {len(a_files)} year files\n")

# =========================
# Batch processing
# =========================
for a_path in a_files:
    fname = os.path.basename(a_path)

    # Extract year
    year = fname.replace("NUE_dif_", "").replace(".tif", "")

    b_path = os.path.join(B_FOLDER, f"NUE_median_Opt_{year}.tif")
    out_path = os.path.join(OUT_FOLDER, f"NUE_Nor_{year}.tif")

    if not os.path.exists(b_path):
        print(f"⚠ Missing B file for year {year}, skipped")
        continue

    print(f"Processing year {year}")

    with rasterio.open(a_path) as src_a, rasterio.open(b_path) as src_b:
        A = src_a.read(1).astype(np.float32)
        B = src_b.read(1).astype(np.float32)

        # Copy metadata from A
        profile = src_a.profile.copy()

    # =========================
    # Core calculation
    # =========================

    # Initialize output as nodata
    result = np.full(A.shape, nodata, dtype=np.float32)

    # Valid pixels:
    # B is valid AND not zero (avoid division by zero)
    valid = (B != nodata) & (~np.isnan(B)) & (B != 0)

    result[valid] = A[valid] / B[valid]

    # =========================
    # Export GeoTIFF
    # =========================
    profile.update({
        "dtype": "float32",
        "nodata": nodata,
        "compress": "lzw"
    })

    with rasterio.open(out_path, "w", **profile) as dst:
        dst.write(result, 1)

    print(f"✔ Saved: {out_path}")

print("\n✅ All years finished")

In [ ]:
# Calculate the proportion of each interval
# ----for myself use, will not upload to GitHub------

import os
import glob
import rasterio
import numpy as np

# =====================================================
# 1 参数设置（自行修改）
# =====================================================
tif_folder = r"Z:\staff\Qiang_S\09_Datasets\03GlobalNUE_Results\01Global_NUE_Wheat\01WheatNUE_V5_150_M6_Nor"

# 区间范围
# value_min = 0
# value_max = 100

# 区间个数
n_bins = 5

# =====================================================
# 2 生成区间
# =====================================================
bins = [0,0.2,0.5,1,2,3]

print("Value bins:")
for i in range(n_bins):
    print(f"[{bins[i]:.2f}, {bins[i+1]:.2f})")

print("\n----------------------------------")

# =====================================================
# 3 获取所有tif
# =====================================================
tif_files = glob.glob(os.path.join(tif_folder, "*.tif"))

if len(tif_files) == 0:
    raise ValueError("No tif files found")

# 用于统计所有tif总体分布
total_counts = np.zeros(n_bins)

# =====================================================
# 4 循环计算
# =====================================================
for tif in tif_files:

    with rasterio.open(tif) as src:
        data = src.read(1)

        nodata = src.nodata

    # 转换为float
    data = data.astype(float)

    # 去除NoData
    if nodata is not None:
        data[data == nodata] = np.nan

    data = data[np.isfinite(data)]

    # 统计直方图
    counts, _ = np.histogram(data, bins=bins)

    total_counts += counts

    total = counts.sum()

    print(f"\nFile: {os.path.basename(tif)}")

    for i in range(n_bins):
        ratio = counts[i] / total if total > 0 else 0

        print(
            f"[{bins[i]:.2f}, {bins[i+1]:.2f}) : "
            f"{counts[i]} pixels  ({ratio:.2%})"
        )

# =====================================================
# 5 总体统计
# =====================================================
print("\n==================================")
print("Overall statistics (all tif)")
print("==================================")

total_pixels = total_counts.sum()

for i in range(n_bins):

    ratio = total_counts[i] / total_pixels if total_pixels > 0 else 0

    print(
        f"[{bins[i]:.2f}, {bins[i+1]:.2f}) : "
        f"{int(total_counts[i])} pixels  ({ratio:.2%})"
    )

In [ ]:
# ---Figure---
# ---for myself use, will not upload to GitHub------

import matplotlib.pyplot as plt
import geopandas as gpd
import rasterio
import numpy as np
from matplotlib.ticker import FuncFormatter

plt.rcParams['font.family'] = 'Arial' # Global font

# 年份（6 个）
years = [1995, 2000, 2005, 2010, 2015, 2020]

tif_template = (
    "Z:/staff/Qiang_S/09_Datasets/03GlobalNUE_Results/01Global_NUE_Wheat/01WheatNUE_V6_M6_150C/NUE_median_Opt_{}.tif"  # Wheat
    #"Z:/staff/Qiang_S/09_Datasets/03GlobalNUE_Results/02Global_NUE_Maize/02MaizeNUE_V3_150_M4/NUE_median_Opt_{}.tif"  # Maize
    #"Z:/staff/Qiang_S/09_Datasets/03GlobalNUE_Results/03Global_NUE_Rice/03RiceNUE_V2_150_M4/NUE_median_Opt_{}.tif" # Rice    
)

shp_file = (
    "Z:/staff/Qiang_S/09_Datasets/09OtherDataUsed/GDAM_GlobalCountryLevel_Shp/GlobalCountryLevel_Shp.shp"
)

out_png = (
    "Z:/staff/Qiang_S/09_Datasets/03GlobalNUE_Results/01Global_NUE_Wheat/01WheatNUE_V6_M6_150C/WheatNUE_median_6yearsA.png"
)

nodata_value = -9999                  # <-- 空值标记
cmap_style = "RdYlBu"                   # <-- 颜色映射

# 数值范围控制
vmin, vmax = 0, 90                   # <-- 关心的数值范围
overall_min, overall_max = 0, 150     # <-- 数据总范围（用于 colorbar 显示）

# ===============================
# 定义经纬度格式化函数
# ===============================
def format_lon(x, pos):
    val = int(x)
    if val == 0: return "0°"
    return f"{abs(val)}°{'E' if val > 0 else 'W'}"

def format_lat(y, pos):
    val = int(y)
    if val == 0: return "0°"
    return f"{abs(val)}°{'N' if val > 0 else 'S'}"

# ===============================
# 3. 创建 2×3 子图
# 修改点：调整 figsize 以匹配全球地图的长宽比，减少由于比例不适配产生的留白
# ===============================
fig, axes = plt.subplots(
    nrows=3,
    ncols=2,
    figsize=(12, 9),  # ⭐ 修改：高度从15降为9，更紧凑
    sharex=True,
    sharey=True
)

# ⭐ 修改：手动调整子图间距，wsapce/hspace 设为极小值
plt.subplots_adjust(wspace=0.05, hspace=0.05)

axes = axes.flatten()
img_for_cbar = None 

# ===============================
# 4. 循环绘制每一年
# ===============================
for i, year in enumerate(years):
    ax = axes[i]
    
    # ... (读取数据的代码保持不变) ...
    tif_file = tif_template.format(year)
    with rasterio.open(tif_file) as src:
        raster = src.read(1).astype("float32")
        bounds = src.bounds
        
    raster = np.where(raster == nodata_value, np.nan, raster)

    # 绘图
    img = ax.imshow(
        raster,
        cmap=cmap_style,
        extent=[bounds.left, bounds.right, bounds.bottom, bounds.top],
        vmin=vmin,
        vmax=vmax
    )

    if img_for_cbar is None:
        img_for_cbar = img

    # 叠加国界 (线宽调细适合发表)
    gdf = gpd.read_file(shp_file)
    gdf.boundary.plot(ax=ax, color="black", linewidth=0.15)

    # 年份标注 (移到左下角或左上角，加个简单的框)
    ax.text(
        0.03, 0.95, str(year),
        transform=ax.transAxes,
        fontsize=12, 
        fontweight="bold",
        fontfamily= "Arial",
        ha="left", 
        va="top",
        bbox=dict(facecolor="white", edgecolor="none", alpha=0.7, pad=2)
    )

    # ===============================
    # ⭐ 修改：坐标轴控制 (外边框显示)
    # ===============================
    row, col = divmod(i, 2)

    # 设置刻度格式
    ax.xaxis.set_major_formatter(FuncFormatter(format_lon))
    ax.yaxis.set_major_formatter(FuncFormatter(format_lat))
    
    # 设置刻度间隔 (根据需要调整，这里设为经度60度，纬度30度一档)
    ax.set_xticks(np.arange(-120, 121, 60))
    ax.set_yticks(np.arange(-60, 61, 30))
    
    # 刻度线朝内
    ax.tick_params(direction='in', which='both', labelsize=10, labelcolor = '#333333', width=0.5)

    # ---边框控制 ---
    for edge in ['top', 'bottom', 'left', 'right']:
        ax.spines[edge].set_linewidth(0.5)  # 设置边框粗细
        ax.spines[edge].set_color('#333333')   # 设置边框颜色
    # -----------------------

    # ---强制设置经纬度刻度标签的字体样式---
    for label in ax.get_xticklabels():
        label.set_fontfamily('Arial')
        label.set_fontsize(10)
        label.set_fontweight('normal')  # 可选 'bold'

    for label in ax.get_yticklabels():
        label.set_fontfamily('Arial')
        label.set_fontsize(10)
        label.set_fontweight('normal')

# ===============================
# 5. 自定义 Colorbar (缩小尺寸)
# ===============================

# ⭐ 修改：不再使用 fig.colorbar(ax=axes)，而是添加一个新的轴
# [left, bottom, width, height] -> 坐标系是整个 Figure (0到1)
# 调整 bottom 参数可以将图例上下移动
cbar_ax = fig.add_axes([0.3, 0.05, 0.4, 0.02]) 

cbar = fig.colorbar(
    img_for_cbar,
    cax=cbar_ax,
    orientation="horizontal",
    extend="max"
)

cbar.set_label("Wheat NUE (%)", fontsize=12, fontfamily= "Arial", fontweight= "bold")
cbar.ax.tick_params(labelsize=10, direction="in", labelcolor = '#333333', width=0.5)

# 设置显示的刻度
cbar.set_ticks([0, 10, 20, 30, 40, 50, 60, 70, 80, 90])
# 新增：强制设置图例刻度数字的字体样式
for t in cbar.ax.get_xticklabels():
    t.set_fontfamily('Arial')
    t.set_fontsize(10)
    t.set_fontweight('normal') # 如果需要加粗可以改为 'bold'

img_for_cbar.set_clim(0, 90) # 保持您原来的clim逻辑

# ----保存---
plt.savefig(out_png, dpi=400, bbox_inches="tight")

# ⭐ 新增：导出 TIFF
out_tif = out_png.replace(".png", ".tif")
plt.savefig(out_tif, dpi=400, bbox_inches="tight", format="tiff")

# ⭐ 新增：导出 PDF
out_pdf = out_png.replace(".png", ".pdf")
plt.savefig(out_pdf, bbox_inches="tight", format="pdf")  # PDF 为矢量图，无需设置 dpi

plt.show()

In [ ]:
import matplotlib.pyplot as plt
import geopandas as gpd
import rasterio
import numpy as np
from matplotlib.ticker import FuncFormatter

# 全局字体设置
plt.rcParams['font.family'] = 'Arial'

# 年份
years = [1995, 2000, 2005, 2010, 2015, 2020]

# --- 路径保持您的定义 ---
tif_template = "Z:/staff/Qiang_S/09_Datasets/03GlobalNUE_Results/01Global_NUE_Wheat/01WheatNUE_V6_M6_150C/NUE_median_Opt_{}.tif"
shp_file = "Z:/staff/Qiang_S/09_Datasets/09OtherDataUsed/GDAM_GlobalCountryLevel_Shp/GlobalCountryLevel_Shp.shp"
out_png = "Z:/staff/Qiang_S/09_Datasets/03GlobalNUE_Results/01Global_NUE_Wheat/01WheatNUE_V6_M6_150C/WheatNUE_median_6yearsAU.png"

nodata_value = -9999
cmap_style = "RdYlBu"
vmin, vmax = 0, 90

# 格式化函数
def format_lon(x, pos):
    val = int(x)
    if val == 0: return "0°"
    return f"{abs(val)}°{'E' if val > 0 else 'W'}"

def format_lat(y, pos):
    val = int(y)
    if val == 0: return "0°"
    return f"{abs(val)}°{'N' if val > 0 else 'S'}"

# =========================================================
# 1. 修改排版：创建 2×3 子图 (横向3个，纵向2个)
# =========================================================
# =========================================================
# 1. 重新设定完美的 figsize (2.25 : 1 的比例)
# =========================================================
fig, axes = plt.subplots(
    nrows=2, 
    ncols=3, 
    figsize=(21, 6.5),
    sharex=True, 
    sharey=True
)

# =========================================================
# 2. 精确控制间距 (参数详解)
# =========================================================
plt.subplots_adjust(
    left=0.12,    # 增加左侧留白，确保 "Wheat NUE (%)" 不会撞到地图刻度
    right=0.88,   # 增加右侧留白，给垂直图例预留充足空间
    top=0.92,     # 顶部稍微留白
    bottom=0.08,  # 底部留白
    wspace=0.02,  # ⭐ 极小横向间距：让三张图几乎贴合，视觉连贯性最强
    hspace=0.05   # ⭐ 适中纵向间距：给经度标签 (0°, 60°E) 留出呼吸空间
)


axes = axes.flatten()
img_for_cbar = None 

# 加载国界 (放在循环外读取一次以提高效率)
gdf = gpd.read_file(shp_file)

# 2. 循环绘制
for i, year in enumerate(years):
    ax = axes[i]
    
    tif_file = tif_template.format(year)
    with rasterio.open(tif_file) as src:
        raster = src.read(1).astype("float32")
        bounds = src.bounds
        
    raster = np.where(raster == nodata_value, np.nan, raster)

    img = ax.imshow(
        raster,
        cmap=cmap_style,
        extent=[bounds.left, bounds.right, bounds.bottom, bounds.top],
        vmin=vmin,
        vmax=vmax
    )

    if img_for_cbar is None:
        img_for_cbar = img

    # 叠加国界
    gdf.boundary.plot(ax=ax, color="black", linewidth=0.15)

    # 年份标注
    ax.text(
        0.03, 0.95, str(year),
        transform=ax.transAxes,
        fontsize=12, fontweight="bold",
        ha="left", va="top",
        bbox=dict(facecolor="white", edgecolor="none", alpha=0.7, pad=2)
    )

    # 坐标轴控制
    ax.xaxis.set_major_formatter(FuncFormatter(format_lon))
    ax.yaxis.set_major_formatter(FuncFormatter(format_lat))
    ax.set_xticks(np.arange(-120, 121, 60))
    ax.set_yticks(np.arange(-60, 61, 30))
    ax.tick_params(direction='in', which='both', labelsize=9, labelcolor='#333333', width=0.5)

    # 边框
    for edge in ['top', 'bottom', 'left', 'right']:
        ax.spines[edge].set_linewidth(0.5)
        ax.spines[edge].set_color('#333333')

# =========================================================
# 3. 修改图例：放置在右侧，垂直居中
# [left, bottom, width, height] 
# =========================================================
# cbar_ax = fig.add_axes([0.91, 0.25, 0.015, 0.5]) # 右侧位置
# 右侧图例：[左, 底, 宽, 高]
# 将宽度设窄 (0.012)，高度设为子图总高度的 60% (0.5)
cbar_ax = fig.add_axes([0.91, 0.25, 0.012, 0.5]) 

cbar = fig.colorbar(
    img_for_cbar,
    cax=cbar_ax,
    orientation="vertical", # 垂直放置
    extend="max"
)

# 强制设置图例字体
for t in cbar.ax.get_yticklabels():
    t.set_fontfamily('Arial')
    t.set_fontsize(10)

# =========================================================
# 4. 修改图名：放在最左侧，垂直居中
# =========================================================
fig.text(
    0.03, 0.5, 'Wheat NUE (%)', 
    va='center', ha='center', 
    rotation='vertical', 
    fontsize=16, 
    fontweight='bold', 
    fontfamily='Arial'
)

# 保存与导出
plt.savefig(out_png, dpi=400, bbox_inches="tight")

out_tif = out_png.replace(".png", ".tif")
plt.savefig(out_tif, dpi=400, bbox_inches="tight", format="tiff")

out_pdf = out_png.replace(".png", ".pdf")
plt.savefig(out_pdf, bbox_inches="tight", format="pdf")

plt.show()

In [ ]:
#---Important Step---
#---Extract country mean value--Useful---single-process computing----

import os
import glob
import re
import geopandas as gpd
import rasterio
import pandas as pd
from rasterstats import zonal_stats

# ===============================
# USER CONFIG
# ===============================

SHP_PATH = r"Z:/staff/Qiang_S/09_Datasets/09OtherDataUsed/GDAM_GlobalCountryLevel_Shp/GlobalCountryLevel_Shp.shp"
# ---Wheat---
TIF_FOLDER = r"Z:\staff\Qiang_S\09_Datasets\03GlobalNUE_Results\01Global_NUE_Wheat\100Temp"
OUT_EXCEL = r"Z:\staff\Qiang_S\09_Datasets\03GlobalNUE_Results\01Global_NUE_Wheat\01WheatNUE_V6_M6_150C/Wheat_NUE_countrytempSingle.xlsx"

NODATA = -9999

# ===============================
# Load shapefile
# ===============================

print("Reading shapefile...")
gdf = gpd.read_file(SHP_PATH)

# ensure CRS
gdf = gdf.to_crs("EPSG:4326")

# ===============================
# Find all tif files
# ===============================

tif_files = sorted(glob.glob(os.path.join(TIF_FOLDER, "*.tif")))

print(f"Found {len(tif_files)} tif files")

# ===============================
# Output container
# ===============================

records = []

# ===============================
# Loop tif files
# ===============================

for tif in tif_files:

    fname = os.path.basename(tif)

    # extract year from filename (last 4 digits)
    m = re.search(r"(\d{4})\.tif$", fname)
    if not m:
        print(f"Skip (no year): {fname}")
        continue

    year = int(m.group(1))
    print(f"Processing year {year}")

    with rasterio.open(tif) as src:
        affine = src.transform
        nodata = src.nodata
        if nodata is None:
            nodata = NODATA

    # ===============================
    # Zonal statistics
    # ===============================

    stats = zonal_stats(
        gdf,
        tif,
        stats=["mean"],
        nodata=nodata,
        geojson_out=False
    )

    # ===============================
    # Collect results
    # ===============================

    for i, row in gdf.iterrows():

        mean_val = stats[i]["mean"]

        records.append({
            "M49_Code": row["M49_Code"],
            "COUNTRY": row["COUNTRY"],
            "year": year,
            "value": mean_val
        })

# ===============================
# Save Excel
# ===============================

df = pd.DataFrame(records)

df.to_excel(OUT_EXCEL, index=False)

print("\n✅ Finished!")
print(f"Saved to: {OUT_EXCEL}")

In [ ]:

# ============================================================
# Global GeoTIFF Grid Harmonization
#
# Purpose:
#   1. Recursively search all GeoTIFF files
#   2. Standardize global 5-arc-minute raster grids
#   3. Preserve original folder structure and filenames
#   4. Avoid resampling when only minor metadata errors exist
#   5. Use nearest-neighbor resampling when required
#   6. Standardize dtype to float32 and NoData to NaN
#   7. Generate a CSV processing log
#
# Target grid:
#   CRS: EPSG:4326
#   Extent: (-180, -90, 180, 90)
#   Size: 4320 columns x 2160 rows
#   Resolution: 1/12 degree
# ============================================================

from pathlib import Path
import csv
import time

import numpy as np
import rasterio

from rasterio.crs import CRS
from rasterio.transform import from_origin
from rasterio.enums import Resampling
from rasterio.vrt import WarpedVRT


# ============================================================
# 1. USER CONFIGURATION
# ============================================================

# Original dataset root directory
INPUT_ROOT = Path(
    #r"Z:\staff\Qiang_S\09_Datasets\01N_Input\01Global_SyntheticFertilizer\02StandardizedDatasets"
    #r"Z:\staff\Qiang_S\09_Datasets\01N_Input\02Global_Manure\02StandardizedDatasets"
    #r"Z:\staff\Qiang_S\09_Datasets\01N_Input\03Global_Deposition\02StandardizedDatasets"
    #r"Z:\staff\Qiang_S\09_Datasets\02N_Output\01Global_Yield_Wheat\02StandardizedDatasets"
    #r"Z:\staff\Qiang_S\09_Datasets\02N_Output\02Global_Yield_Maize\02StandardizedDatasets"
    #r"Z:\staff\Qiang_S\09_Datasets\02N_Output\03Global_Yield_Rice\02StandardizedDatasets"
    #r"Z:\staff\Qiang_S\09_Datasets\09OtherDataUsed\GCSN_HarvestedArea_1995_2020/02StandardizedDatasets"
    #r"Z:\staff\Qiang_S\09_Datasets\03GlobalNUE_Results\01Global_NUE_Wheat\02WheatNUE_M2_20261019_NUEMasked"
    r"Z:\staff\Qiang_S\09_Datasets\09OtherDataUsed\NatureC2025"
)

# New output root directory
# Must be separate from INPUT_ROOT
OUTPUT_ROOT = Path(
    #r"Z:\staff\Qiang_S\09_Datasets\01N_Input\01Global_SyntheticFertilizer\02StandardizedDatasets_GridAligned"
    #r"Z:\staff\Qiang_S\09_Datasets\01N_Input\02Global_Manure\02StandardizedDatasets_GridAligned"
    #r"Z:\staff\Qiang_S\09_Datasets\01N_Input\03Global_Deposition\02StandardizedDatasets_GridAligned"
    #r"Z:\staff\Qiang_S\09_Datasets\02N_Output\01Global_Yield_Wheat\02StandardizedDatasets_GridAligned"
    #r"Z:\staff\Qiang_S\09_Datasets\02N_Output\02Global_Yield_Maize\02StandardizedDatasets_GridAligned"
    #r"Z:\staff\Qiang_S\09_Datasets\02N_Output\03Global_Yield_Rice\02StandardizedDatasets_GridAligned"
    r"Z:\staff\Qiang_S\09_Datasets\09OtherDataUsed\NatureC2025_GridAligned"
)

# Overwrite existing output files?
OVERWRITE = False

# Maximum grid discrepancy allowed for metadata-only SNAP
# Unit: fraction of one target pixel
# Example: 0.03 = 3% of one pixel
SNAP_TOLERANCE_PIXELS = 0.03

# Output NoData
OUTPUT_NODATA = np.nan

# Optional additional source NoData values
# Use [-9999.0] only if these values represent missing data
ADDITIONAL_NODATA_VALUES = []

# TIFF compression
COMPRESSION = "deflate"
BLOCK_SIZE = 256

# Check output georeferencing after writing
VERIFY_OUTPUT = True


# ============================================================
# 2. DEFINE STANDARD GLOBAL GRID
# ============================================================

TARGET_CRS = CRS.from_epsg(4326)

TARGET_WIDTH = 4320
TARGET_HEIGHT = 2160

TARGET_LEFT = -180.0
TARGET_RIGHT = 180.0
TARGET_TOP = 90.0
TARGET_BOTTOM = -90.0

TARGET_RES = 1.0 / 12.0

TARGET_TRANSFORM = from_origin(
    TARGET_LEFT,
    TARGET_TOP,
    TARGET_RES,
    TARGET_RES
)

TARGET_BOUNDS = (
    TARGET_LEFT,
    TARGET_BOTTOM,
    TARGET_RIGHT,
    TARGET_TOP
)


# ============================================================
# 3. VALIDATE INPUT AND OUTPUT PATHS
# ============================================================

INPUT_ROOT = INPUT_ROOT.resolve()
OUTPUT_ROOT = OUTPUT_ROOT.resolve()

if not INPUT_ROOT.is_dir():
    raise FileNotFoundError(
        f"Input directory does not exist: {INPUT_ROOT}"
    )

if (
    OUTPUT_ROOT == INPUT_ROOT
    or INPUT_ROOT in OUTPUT_ROOT.parents
):
    raise ValueError(
        "OUTPUT_ROOT cannot be equal to or inside INPUT_ROOT. "
        "Please choose a separate output directory."
    )

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)


# ============================================================
# 4. CHECK GRID ALIGNMENT
# ============================================================

def calculate_grid_difference(src):
    """
    Calculate maximum source edge difference
    in units of target pixels.

    Only meaningful when CRS and raster orientation match.
    """

    if src.crs != TARGET_CRS:
        return float("inf")

    if src.width != TARGET_WIDTH:
        return float("inf")

    if src.height != TARGET_HEIGHT:
        return float("inf")

    # Require north-up, non-rotated raster
    if (
        src.transform.a <= 0
        or src.transform.e >= 0
        or abs(src.transform.b) > 1e-12
        or abs(src.transform.d) > 1e-12
    ):
        return float("inf")

    bounds = src.bounds

    differences = [
        abs(bounds.left - TARGET_LEFT),
        abs(bounds.right - TARGET_RIGHT),
        abs(bounds.top - TARGET_TOP),
        abs(bounds.bottom - TARGET_BOTTOM)
    ]

    max_difference_degrees = max(differences)

    return max_difference_degrees / TARGET_RES


def determine_processing_method(src):
    """
    SNAP:
        Correct metadata only without resampling.

    REPROJECT:
        Resample/reproject to the exact target grid.
    """

    if src.crs is None:
        raise ValueError(
            "Source CRS is missing. "
            "Cannot safely determine raster alignment."
        )

    difference = calculate_grid_difference(src)

    if difference <= SNAP_TOLERANCE_PIXELS:
        return "SNAP", difference

    return "REPROJECT", difference


# ============================================================
# 5. CREATE STANDARD OUTPUT PROFILE
# ============================================================

def create_output_profile(src):

    return {
        "driver": "GTiff",
        "width": TARGET_WIDTH,
        "height": TARGET_HEIGHT,
        "count": src.count,
        "dtype": "float32",
        "crs": TARGET_CRS,
        "transform": TARGET_TRANSFORM,
        "nodata": OUTPUT_NODATA,
        "compress": COMPRESSION,
        "predictor": 3,
        "tiled": True,
        "blockxsize": BLOCK_SIZE,
        "blockysize": BLOCK_SIZE,
        "BIGTIFF": "IF_SAFER"
    }


# ============================================================
# 6. COPY DATA IN BLOCKS
# ============================================================

def copy_raster_blocks(reader, dst):
    """
    Read and write raster blocks.

    Benefits:
      - Low memory use
      - Convert to float32
      - Standardize NoData to NaN
      - Preserve original finite pixel values
    """

    for _, window in dst.block_windows(1):

        # Read all bands in this window
        data = reader.read(
            window=window,
            masked=True,
            out_dtype="float32"
        )

        # Convert source masked pixels into NaN
        data = np.asarray(
            data.filled(np.nan),
            dtype=np.float32
        )

        # Replace NaN/Inf with output NoData
        data[~np.isfinite(data)] = np.nan

        # Optional additional NoData values
        for nd in ADDITIONAL_NODATA_VALUES:
            data[data == nd] = np.nan

        dst.write(data, window=window)


# ============================================================
# 7. PROCESS ONE GEOTIFF
# ============================================================

def process_geotiff(input_path, output_path):

    start_time = time.perf_counter()

    output_path.parent.mkdir(
        parents=True,
        exist_ok=True
    )

    with rasterio.open(input_path) as src:

        method, difference = determine_processing_method(src)

        profile = create_output_profile(src)

        # ----------------------------------------------------
        # Method A: SNAP
        # Metadata correction without resampling
        # ----------------------------------------------------

        if method == "SNAP":

            with rasterio.open(
                output_path,
                "w",
                **profile
            ) as dst:

                copy_raster_blocks(src, dst)

                # Preserve band descriptions
                for band_idx, desc in enumerate(
                    src.descriptions, start=1
                ):
                    if desc:
                        dst.set_band_description(
                            band_idx, desc
                        )

        # ----------------------------------------------------
        # Method B: REPROJECT
        # Align using nearest-neighbor sampling
        # ----------------------------------------------------

        else:

            with WarpedVRT(
                src,
                crs=TARGET_CRS,
                transform=TARGET_TRANSFORM,
                width=TARGET_WIDTH,
                height=TARGET_HEIGHT,
                resampling=Resampling.nearest,
                nodata=np.nan,
                dtype="float32"
            ) as vrt:

                with rasterio.open(
                    output_path,
                    "w",
                    **profile
                ) as dst:

                    copy_raster_blocks(vrt, dst)

                    for band_idx, desc in enumerate(
                        src.descriptions, start=1
                    ):
                        if desc:
                            dst.set_band_description(
                                band_idx, desc
                            )

        source_crs = str(src.crs)
        source_size = f"{src.width}x{src.height}"

        source_bounds = (
            f"{src.bounds.left:.10f},"
            f"{src.bounds.bottom:.10f},"
            f"{src.bounds.right:.10f},"
            f"{src.bounds.top:.10f}"
        )

    # --------------------------------------------------------
    # Validate output grid
    # --------------------------------------------------------

    if VERIFY_OUTPUT:

        with rasterio.open(output_path) as check:

            assert check.crs == TARGET_CRS
            assert check.width == TARGET_WIDTH
            assert check.height == TARGET_HEIGHT

            assert check.transform.almost_equals(
                TARGET_TRANSFORM,
                precision=12
            )

            assert check.dtypes == (
                "float32",
            ) * check.count

            assert np.isnan(check.nodata)

    elapsed = time.perf_counter() - start_time

    return {
        "method": method,
        "grid_difference_pixels": difference,
        "source_crs": source_crs,
        "source_size": source_size,
        "source_bounds": source_bounds,
        "elapsed_seconds": round(elapsed, 3)
    }


# ============================================================
# 8. FIND ALL TIFF FILES RECURSIVELY
# ============================================================

tif_files = sorted(
    p for p in INPUT_ROOT.rglob("*")
    if p.is_file()
    and p.suffix.lower() in (".tif", ".tiff")
)

print("=" * 70)
print("GLOBAL GEOTIFF GRID HARMONIZATION")
print("=" * 70)

print(f"Input root : {INPUT_ROOT}")
print(f"Output root: {OUTPUT_ROOT}")

print(f"Total GeoTIFF files: {len(tif_files)}")

print(f"Target CRS       : {TARGET_CRS}")
print(f"Target raster    : {TARGET_WIDTH} x {TARGET_HEIGHT}")
print(f"Target resolution: {TARGET_RES:.12f} degree")
print(f"SNAP tolerance   : {SNAP_TOLERANCE_PIXELS} pixels")

print("=" * 70)


# ============================================================
# 9. PREPARE PROCESSING LOG
# ============================================================

log_path = OUTPUT_ROOT / "Grid_Harmonization_Log.csv"

log_fields = [
    "relative_path",
    "method",
    "status",
    "grid_difference_pixels",
    "source_crs",
    "source_size",
    "source_bounds",
    "elapsed_seconds",
    "error"
]

statistics = {
    "SNAP": 0,
    "REPROJECT": 0,
    "SKIPPED": 0,
    "FAILED": 0
}


# ============================================================
# 10. BATCH PROCESSING
# ============================================================

with open(
    log_path,
    "w",
    newline="",
    encoding="utf-8-sig"
) as log_file:

    writer = csv.DictWriter(
        log_file,
        fieldnames=log_fields
    )

    writer.writeheader()

    for index, input_path in enumerate(
        tif_files, start=1
    ):

        # Preserve original relative directory structure
        relative_path = input_path.relative_to(INPUT_ROOT)

        output_path = OUTPUT_ROOT / relative_path

        print(
            f"\n[{index}/{len(tif_files)}] "
            f"{relative_path}"
        )

        log_row = {
            "relative_path": str(relative_path),
            "method": "",
            "status": "",
            "grid_difference_pixels": "",
            "source_crs": "",
            "source_size": "",
            "source_bounds": "",
            "elapsed_seconds": "",
            "error": ""
        }

        # ----------------------------------------------------
        # Skip existing output
        # ----------------------------------------------------

        if output_path.exists() and not OVERWRITE:

            statistics["SKIPPED"] += 1

            log_row["status"] = "SKIPPED"
            log_row["error"] = "Output already exists"

            print("  SKIPPED: Output already exists")

            writer.writerow(log_row)
            log_file.flush()

            continue

        # ----------------------------------------------------
        # Process
        # ----------------------------------------------------

        try:

            result = process_geotiff(
                input_path,
                output_path
            )

            method = result["method"]

            statistics[method] += 1

            log_row.update(result)
            log_row["status"] = "SUCCESS"

            print(
                f"  Method: {method} | "
                f"Time: {result['elapsed_seconds']:.2f}s"
            )

            print(f"  Saved: {output_path}")

        except Exception as e:

            statistics["FAILED"] += 1

            log_row["status"] = "FAILED"
            log_row["error"] = str(e)

            print(f"  ERROR: {e}")

            # Remove incomplete output
            if output_path.exists():
                try:
                    output_path.unlink()
                except OSError:
                    pass

        writer.writerow(log_row)
        log_file.flush()


# ============================================================
# 11. FINAL SUMMARY
# ============================================================

print("\n" + "=" * 70)
print("GRID HARMONIZATION COMPLETED")
print("=" * 70)

print(f"Total files : {len(tif_files)}")
print(f"SNAP        : {statistics['SNAP']}")
print(f"REPROJECT   : {statistics['REPROJECT']}")
print(f"SKIPPED     : {statistics['SKIPPED']}")
print(f"FAILED      : {statistics['FAILED']}")

print(f"\nOutput directory: {OUTPUT_ROOT}")
print(f"Processing log  : {log_path}")

print("\nStandard grid:")
print("CRS        : EPSG:4326")
print("Extent     : -180, -90, 180, 90")
print("Resolution : 0.083333333333 degree")
print("Columns    : 4320")
print("Rows       : 2160")
print("Data type  : float32")
print("NoData     : NaN")

print("=" * 70)

In [ ]:
# ============================================================
# Global Wheat NUE - Extract Country Mean Values
# Tool single
# NaN / NoData Fixed Version
#
# 1. Automatically identify GeoTIFF NoData
# 2. Exclude NaN, Inf, -Inf and -9999
# 3. Preserve all original valid values
# 4. Calculate country mean using zonal_stats
# 5. Export Excel with original structure
#
# Run directly in one JupyterLab Cell
# ============================================================

import os
import glob
import re

import numpy as np
import geopandas as gpd
import rasterio
import pandas as pd

from rasterstats import zonal_stats


# ============================================================
# 1. USER CONFIG
# ============================================================

SHP_PATH = (
    r"Z:/staff/Qiang_S/09_Datasets/"
    r"09OtherDataUsed/GDAM_GlobalCountryLevel_Shp/"
    r"GlobalCountryLevel_Shp.shp"
)

# Wheat NUE GeoTIFF folder
TIF_FOLDER = (
    r"Z:\staff\Qiang_S\09_Datasets"
    r"\03GlobalNUE_Results\01Global_NUE_Wheat"
    r"\02WheatNUE_M2_20261019_M2_M6_200C"
)

# Output Excel
OUT_EXCEL = (
    r"Z:\staff\Qiang_S\09_Datasets"
    r"\03GlobalNUE_Results\01Global_NUE_Wheat"
    r"\02WheatNUE_M2_20261019_M2_M6_200C"
    r"\Wheat_NUE_20261019countryM2.xlsx"
)

# Standardized NoData
NODATA = -9999.0

# Additional invalid values not recorded in GeoTIFF metadata
# Example: [-3.4028235e38]
ADDITIONAL_NODATA = []

# Country field names
M49_FIELD = "M49_Code"
COUNTRY_FIELD = "COUNTRY"

# Preserve original zonal statistics method
ALL_TOUCHED = False


# ============================================================
# 2. PREPARE OUTPUT
# ============================================================

os.makedirs(
    os.path.dirname(OUT_EXCEL),
    exist_ok=True
)


# ============================================================
# 3. LOAD SHAPEFILE
# ============================================================

print("=" * 70)
print("GLOBAL WHEAT NUE - COUNTRY MEAN EXTRACTION")
print("=" * 70)

print("\nReading shapefile...")

gdf = gpd.read_file(SHP_PATH)

if gdf.crs is None:
    raise ValueError(
        "Shapefile CRS is missing. "
        "Please define its original CRS."
    )

# Original CRS setting
gdf = gdf.to_crs("EPSG:4326")

print(f"Countries loaded: {len(gdf)}")

# Check required fields
for field in [M49_FIELD, COUNTRY_FIELD]:

    if field not in gdf.columns:
        raise ValueError(
            f"Required field not found: {field}"
        )


# ============================================================
# 4. FIND ALL TIF FILES
# ============================================================

tif_files = sorted(
    glob.glob(
        os.path.join(TIF_FOLDER, "*.tif")
    )
)

print(f"\nFound {len(tif_files)} tif files")

if len(tif_files) == 0:
    raise RuntimeError(
        "No GeoTIFF files found in TIF_FOLDER."
    )


# ============================================================
# 5. FUNCTION - READ AND CLEAN RASTER
# ============================================================

def read_raster_clean(src):
    """
    Read raster and standardize invalid pixels.

    Invalid pixels:
        1. Raster metadata NoData
        2. NaN
        3. Inf / -Inf
        4. -9999
        5. Additional user-defined NoData

    All invalid pixels become -9999.

    Original valid pixel values remain unchanged.
    """

    # Read raster with automatic metadata NoData mask
    masked_data = src.read(
        1,
        masked=True
    )

    # Convert to float32
    data = np.array(
        masked_data.data,
        dtype=np.float32,
        copy=True
    )

    # --------------------------------------------------------
    # Identify invalid pixels
    # --------------------------------------------------------

    # 1. Metadata-defined NoData
    metadata_mask = np.ma.getmaskarray(
        masked_data
    )

    # 2. NaN, positive infinity, negative infinity
    nonfinite_mask = ~np.isfinite(data)

    # 3. Explicit -9999
    nodata_mask = (data == NODATA)

    # Combine all masks
    invalid_mask = (
        metadata_mask
        | nonfinite_mask
        | nodata_mask
    )

    # 4. Additional NoData
    for value in ADDITIONAL_NODATA:

        invalid_mask |= (data == value)

    # --------------------------------------------------------
    # Standardize NoData
    # --------------------------------------------------------

    data[invalid_mask] = NODATA

    # --------------------------------------------------------
    # Statistics
    # --------------------------------------------------------

    total_pixels = data.size

    invalid_count = int(
        np.count_nonzero(invalid_mask)
    )

    valid_count = total_pixels - invalid_count

    return (
        data,
        valid_count,
        invalid_count,
        total_pixels
    )


# ============================================================
# 6. OUTPUT CONTAINER
# ============================================================

records = []

# Cache reprojected country boundaries by CRS
country_crs_cache = {}


# ============================================================
# 7. LOOP THROUGH TIFF FILES
# ============================================================

for index, tif in enumerate(tif_files, start=1):

    fname = os.path.basename(tif)

    # Extract year from filename
    # Example: NUE_median_Opt_1995.tif

    m = re.search(
        r"(\d{4})\.tif$",
        fname,
        re.IGNORECASE
    )

    if not m:

        print(f"Skip (no year): {fname}")

        continue

    year = int(m.group(1))

    print("\n" + "-" * 70)

    print(
        f"[{index}/{len(tif_files)}] "
        f"Processing year {year}"
    )

    # --------------------------------------------------------
    # 7.1 Read and clean raster
    # --------------------------------------------------------

    with rasterio.open(tif) as src:

        affine = src.transform

        raster_crs = src.crs

        if raster_crs is None:
            raise ValueError(
                f"Raster CRS missing: {fname}"
            )

        # Read data and standardize invalid pixels
        (
            raster_data,
            valid_count,
            invalid_count,
            total_pixels
        ) = read_raster_clean(src)

    # --------------------------------------------------------
    # 7.2 Diagnostic output
    # --------------------------------------------------------

    print(f"Raster shape: {raster_data.shape}")

    print(f"Raster CRS: {raster_crs}")

    print(f"Total pixels  : {total_pixels:,}")

    print(f"Valid pixels  : {valid_count:,}")

    print(f"Invalid pixels: {invalid_count:,}")

    if valid_count == 0:

        print(
            f"WARNING: No valid pixels for year {year}"
        )

    # --------------------------------------------------------
    # 7.3 Align country CRS to raster CRS
    # --------------------------------------------------------

    crs_key = raster_crs.to_wkt()

    if crs_key not in country_crs_cache:

        if gdf.crs == raster_crs:

            country_crs_cache[crs_key] = gdf

        else:

            country_crs_cache[crs_key] = (
                gdf.to_crs(raster_crs)
            )

    gdf_year = country_crs_cache[crs_key]

    # ========================================================
    # 7.4 Zonal statistics - ORIGINAL MEAN LOGIC
    # ========================================================

    # All invalid raster pixels are now -9999.
    #
    # zonal_stats will exclude -9999 automatically.
    #
    # No interpolation.
    # No pixel filling.
    # No change to original valid NUE values.

    stats = zonal_stats(
        gdf_year,
        raster_data,
        affine=affine,
        stats=["mean"],
        nodata=NODATA,
        all_touched=ALL_TOUCHED,
        geojson_out=False
    )

    # ========================================================
    # 7.5 Collect results
    # ========================================================

    country_valid_count = 0
    country_missing_count = 0

    for (_, row), stat in zip(
        gdf_year.iterrows(),
        stats
    ):

        mean_val = stat.get("mean")

        # ----------------------------------------------------
        # Ensure exported mean is finite
        # ----------------------------------------------------

        if (
            mean_val is None
            or not np.isfinite(mean_val)
        ):

            mean_val = None

            country_missing_count += 1

        else:

            mean_val = float(mean_val)

            country_valid_count += 1

        # Original Excel column structure
        records.append({
            "M49_Code": row[M49_FIELD],
            "COUNTRY": row[COUNTRY_FIELD],
            "year": year,
            "value": mean_val
        })

    print(
        f"Countries with valid mean: "
        f"{country_valid_count}"
    )

    print(
        f"Countries without valid pixels: "
        f"{country_missing_count}"
    )

    print(f"Year {year} finished.")


# ============================================================
# 8. SAVE EXCEL
# ============================================================

df = pd.DataFrame(
    records,
    columns=[
        "M49_Code",
        "COUNTRY",
        "year",
        "value"
    ]
)

if df.empty:

    raise RuntimeError(
        "No valid year files were processed."
    )

# Ensure no Inf is exported
df["value"] = pd.to_numeric(
    df["value"],
    errors="coerce"
)

df["value"] = df["value"].replace(
    [np.inf, -np.inf],
    np.nan
)

# Save
df.to_excel(
    OUT_EXCEL,
    index=False
)


# ============================================================
# 9. FINAL VALIDATION
# ============================================================

print("\n" + "=" * 70)
print("COUNTRY MEAN EXTRACTION COMPLETED")
print("=" * 70)

print(f"Total records: {len(df):,}")

print(
    "Valid country-year means: "
    f"{df['value'].notna().sum():,}"
)

print(
    "Missing country-year means: "
    f"{df['value'].isna().sum():,}"
)

print(
    f"Years processed: "
    f"{df['year'].nunique()}"
)

print(f"\nExcel saved to:\n{OUT_EXCEL}")

print("=" * 70)

In [ ]:
# =======2026.10.22=====================================================
# Global NUE - Parallel Outlier Processing Options 

# Step 1: N < MIN_COUNT or N missing -> REMOVE

# Step 2:
#   Upper threshold -> REMOVE / NEAREST
#   Lower threshold -> REMOVE / NEAREST
#
# Step 3:
#   Optional country-level mean + STD_FACTOR * STD
#   -> REMOVE / NEAREST
#   Maximum selected proportion per country: 5%
#
# NaN / NoData safe
# Parallel processing by year
# Output: GeoTIFF + Excel (3 original sheets)
#
# JupyterLab single-cell executable
# ============================================================

import os
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio

from rasterio.features import geometry_mask
from scipy.ndimage import distance_transform_edt
from joblib import Parallel, delayed

from openpyxl import Workbook
from openpyxl.styles import (
    Font, PatternFill, Alignment, Border, Side
)
from openpyxl.utils import get_column_letter

# ============================================================
# 1. USER CONFIGURATION
# ===========================================================
# ------------------------------------------------------------
# Input / output
# ------------------------------------------------------------

A_FOLDER = (
    r"Z:\staff\Qiang_S\09_Datasets\03GlobalNUE_Results\02Global_NUE_Maize\02MaizeNUE_M2_20261027V2_Median_Intersection"
    #r"Z:\staff\Qiang_S\09_Datasets\03GlobalNUE_Results\01Global_NUE_Wheat\02WheatNUE_M2_20261021V2_Median_Intersection"
    
)


MASK_FOLDER = (
    r"Z:\staff\Qiang_S\09_Datasets\03GlobalNUE_Results\02Global_NUE_Maize\02MaizeNUE_M2_20261027V2"
    #r"Z:\staff\Qiang_S\09_Datasets\03GlobalNUE_Results\01Global_NUE_Wheat\02WheatNUE_M2_20261021V2"
)

OUT_FOLDER = (
    r"Z:\staff\Qiang_S\09_Datasets\03GlobalNUE_Results\02Global_NUE_Maize\02MaizeNUE_M2_20261027V2_M6_150R"
    #r"Z:\staff\Qiang_S\09_Datasets\03GlobalNUE_Results\01Global_NUE_Wheat\02WheatNUE_M2_20261021V2_M6_150N"
)

SHP_PATH = (
    r"Z:\staff\Qiang_S\09_Datasets"
    r"\09OtherDataUsed\GDAM_GlobalCountryLevel_Shp"
    r"\GlobalCountryLevel_Shp.shp"
)

M49_FIELD = "M49_Code"
NAME_FIELD = "COUNTRY"

EXCEL_OUT = os.path.join(
    OUT_FOLDER,
    "outlier_removal_summary_V1_M6_150R.xlsx"
)

# ------------------------------------------------------------
# Years and parallel processing
# ------------------------------------------------------------

START_YEAR = 1995
END_YEAR = 2020

# None = all years
# Example: [1995, 1996, 2000, 2020]
YEARS_TO_RUN = None

PARALLEL_YEARS = 10

# ------------------------------------------------------------
# Common settings
# ------------------------------------------------------------

NODATA = -9999.0

# ------------------------------------------------------------
# Step 1: Minimum valid sample count
# ------------------------------------------------------------

MIN_COUNT = 6

# ------------------------------------------------------------
# Step 2: Hard thresholds
# ------------------------------------------------------------
LOWER_HARD = 0.01
UPPER_HARD = 1.5

# Options: "REMOVE" or "NEAREST"
UPPER_OUTLIER_ACTION = "REMOVE"
LOWER_OUTLIER_ACTION = "REMOVE"

# ------------------------------------------------------------
# Step 3: Country standard deviation
# ------------------------------------------------------------

# True = enable country standard deviation processing
# False = skip Step 3

APPLY_STD_REMOVAL = False #True

# Mean + STD_FACTOR * standard deviation
STD_FACTOR = 2.0

# Maximum selected outlier ratio per country
MAX_REMOVE_RATIO = 0.05

# NEW:
# "REMOVE"  = Set selected Step 3 outliers to NoData
# "NEAREST" = Replace with nearest qualified value inside the SAME country

STD_OUTLIER_ACTION = "REMOVE"

# ------------------------------------------------------------
# Nearest-neighbor settings
# ------------------------------------------------------------

# None = No maximum distance
# Example: 12 = maximum 12 raster pixels
MAX_NEAREST_DISTANCE_PIXELS = None

# If no qualified nearest value exists,
# the target becomes NoData.


# ============================================================
# 2. PARAMETER VALIDATION
# ============================================================

VALID_ACTIONS = {"REMOVE", "NEAREST"}

UPPER_OUTLIER_ACTION = UPPER_OUTLIER_ACTION.upper()
LOWER_OUTLIER_ACTION = LOWER_OUTLIER_ACTION.upper()
STD_OUTLIER_ACTION = STD_OUTLIER_ACTION.upper()

for name, action in [
    ("UPPER_OUTLIER_ACTION", UPPER_OUTLIER_ACTION),
    ("LOWER_OUTLIER_ACTION", LOWER_OUTLIER_ACTION),
    ("STD_OUTLIER_ACTION", STD_OUTLIER_ACTION)
]:
    if action not in VALID_ACTIONS:
        raise ValueError(
            f"{name} must be REMOVE or NEAREST"
        )

if LOWER_HARD >= UPPER_HARD:
    raise ValueError(
        "LOWER_HARD must be smaller than UPPER_HARD"
    )

if PARALLEL_YEARS < 1:
    raise ValueError("PARALLEL_YEARS must be >= 1")

if not (0 <= MAX_REMOVE_RATIO <= 1):
    raise ValueError(
        "MAX_REMOVE_RATIO must be between 0 and 1"
    )

if (
    MAX_NEAREST_DISTANCE_PIXELS is not None
    and MAX_NEAREST_DISTANCE_PIXELS < 0
):
    raise ValueError(
        "MAX_NEAREST_DISTANCE_PIXELS must be >= 0"
    )

if not os.path.isfile(SHP_PATH):
    raise FileNotFoundError(SHP_PATH)

os.makedirs(OUT_FOLDER, exist_ok=True)


# ============================================================
# 3. RASTER UTILITY FUNCTIONS
# ============================================================

def read_raster_fixed(src):
    """
    Read raster and standardize:
      Metadata NoData
      NaN / Inf
      -9999

    Valid values remain unchanged.
    """

    data = (
        src.read(1, masked=True)
        .astype(np.float32)
        .filled(np.nan)
    )

    data = np.asarray(
        data,
        dtype=np.float32
    )

    invalid = (
        ~np.isfinite(data)
        | (data == NODATA)
    )

    data[invalid] = NODATA

    return data


def get_valid_mask(data):

    return (
        np.isfinite(data)
        & (data != NODATA)
    )


def empty_country_record(year, m49, name):

    return {
        "Year": year,
        "M49": m49,
        "Country": name,
        "Valid_before_step3": 0,
        "Mean": np.nan,
        "Std": np.nan,
        "Upper_sigma": np.nan,
        "Candidate_removed": 0,
        "Max_allowed_remove": 0,
        "Step3_removed": 0,
        "Step3_removed_%": 0.0
    }


# ============================================================
# 4. NEAREST-NEIGHBOR REPLACEMENT
# ============================================================

def replace_by_nearest(
    raster,
    target_mask,
    donor_mask,
    max_distance=None
):
    """
    Replace selected target pixels with nearest donor values.

    Parameters
    ----------
    raster:
        2D NumPy array, modified in place.

    target_mask:
        True for pixels selected for replacement.

    donor_mask:
        True for qualified source pixels.

    max_distance:
        Maximum distance in pixel units.
        None = unlimited.

    Returns
    -------
    replaced_count:
        Number successfully replaced.

    failed_count:
        Number without an eligible donor.
        These pixels become NoData.

    Notes
    -----
    - Original donor values are copied exactly.
    - No averaging or interpolation.
    - Calculations use row/column Euclidean distance.
    - Processing uses a local bounding window.
    """

    target_rows, target_cols = np.where(target_mask)

    target_count = len(target_rows)

    if target_count == 0:
        return 0, 0

    donor_mask = (
        donor_mask
        & get_valid_mask(raster)
        & ~target_mask
    )

    donor_rows, donor_cols = np.where(donor_mask)

    # No qualified donors
    if len(donor_rows) == 0:

        raster[target_rows, target_cols] = NODATA

        return 0, target_count

    # --------------------------------------------------------
    # Bounding window containing targets and donors
    # --------------------------------------------------------

    row_min = int(min(
        target_rows.min(),
        donor_rows.min()
    ))

    row_max = int(max(
        target_rows.max(),
        donor_rows.max()
    )) + 1

    col_min = int(min(
        target_cols.min(),
        donor_cols.min()
    ))

    col_max = int(max(
        target_cols.max(),
        donor_cols.max()
    )) + 1

    local_donors = donor_mask[
        row_min:row_max,
        col_min:col_max
    ]

    local_rows = target_rows - row_min
    local_cols = target_cols - col_min

    # --------------------------------------------------------
    # Nearest eligible donor
    # --------------------------------------------------------

    if max_distance is None:

        nearest_indices = distance_transform_edt(
            ~local_donors,
            return_distances=False,
            return_indices=True
        )

        nearest_rows = (
            nearest_indices[
                0,
                local_rows,
                local_cols
            ] + row_min
        )

        nearest_cols = (
            nearest_indices[
                1,
                local_rows,
                local_cols
            ] + col_min
        )

        replacement_values = raster[
            nearest_rows,
            nearest_cols
        ].copy()

        raster[
            target_rows,
            target_cols
        ] = replacement_values

        return target_count, 0

    else:

        distances, nearest_indices = distance_transform_edt(
            ~local_donors,
            return_distances=True,
            return_indices=True
        )

        target_distances = distances[
            local_rows,
            local_cols
        ]

        within_distance = (
            target_distances <= max_distance
        )

        # Qualified targets
        valid_rows = target_rows[within_distance]
        valid_cols = target_cols[within_distance]

        nearest_rows = (
            nearest_indices[
                0,
                local_rows[within_distance],
                local_cols[within_distance]
            ] + row_min
        )

        nearest_cols = (
            nearest_indices[
                1,
                local_rows[within_distance],
                local_cols[within_distance]
            ] + col_min
        )

        replacement_values = raster[
            nearest_rows,
            nearest_cols
        ].copy()

        raster[
            valid_rows,
            valid_cols
        ] = replacement_values

        # Targets exceeding maximum distance
        failed_rows = target_rows[~within_distance]
        failed_cols = target_cols[~within_distance]

        raster[
            failed_rows,
            failed_cols
        ] = NODATA

        replaced_count = int(
            np.count_nonzero(within_distance)
        )

        failed_count = target_count - replaced_count

        return replaced_count, failed_count


# ============================================================
# 5. PROCESS ONE YEAR
# ============================================================

def process_one_year(year):

    country_records = []

    A_tif = os.path.join(
        A_FOLDER,
        f"NUE_M2_median_{year}.tif"
    )

    N_tif = os.path.join(
        MASK_FOLDER,
        f"NUE_M2_n_{year}.tif"
    )

    OUT_tif = os.path.join(
        OUT_FOLDER,
        f"NUE_median_Opt_{year}.tif"
    )

    try:

        if not os.path.exists(A_tif):
            return {
                "year": year,
                "status": "MISSING_A",
                "message": A_tif,
                "annual": None,
                "countries": []
            }

        if not os.path.exists(N_tif):
            return {
                "year": year,
                "status": "MISSING_N",
                "message": N_tif,
                "annual": None,
                "countries": []
            }

        # ----------------------------------------------------
        # Read country polygons within each process
        # ----------------------------------------------------

        gdf = gpd.read_file(SHP_PATH)

        if gdf.crs is None:
            raise ValueError("Shapefile CRS missing")

        if gdf.crs.to_epsg() != 4326:
            gdf = gdf.to_crs(epsg=4326)

        gdf = gdf.sort_values(
            by=M49_FIELD
        ).reset_index(drop=True)

        # ----------------------------------------------------
        # Read raster A
        # ----------------------------------------------------

        with rasterio.open(A_tif) as src_a:

            A = read_raster_fixed(src_a)

            profile = src_a.profile.copy()
            transform = src_a.transform
            crs = src_a.crs

            height, width = A.shape

        # ----------------------------------------------------
        # Read raster N
        # ----------------------------------------------------

        with rasterio.open(N_tif) as src_n:

            N = read_raster_fixed(src_n)

            if (
                src_n.crs != crs
                or src_n.transform != transform
            ):
                raise ValueError(
                    "A and N raster grids are not aligned"
                )

        if A.shape != N.shape:
            raise ValueError(
                f"Shape mismatch: A={A.shape}, N={N.shape}"
            )

        if crs is None:
            raise ValueError("Raster CRS missing")

        # ====================================================
        # STEP 1 - N < MIN_COUNT / Missing N
        # ====================================================

        valid_mask = get_valid_mask(A)

        initial_count = int(
            np.count_nonzero(valid_mask)
        )

        n_valid_mask = get_valid_mask(N)

        step1_mask = (
            valid_mask
            & (
                ~n_valid_mask
                | (N < MIN_COUNT)
            )
        )

        step1_removed = int(
            np.count_nonzero(step1_mask)
        )

        A[step1_mask] = NODATA

        valid_mask = get_valid_mask(A)

        # ====================================================
        # STEP 2 - HARD THRESHOLDS
        # ====================================================

        step2_upper = (
            (A > UPPER_HARD)
            & valid_mask
        )

        step2_lower = (
            (A < LOWER_HARD)
            & valid_mask
        )

        upper_count = int(
            np.count_nonzero(step2_upper)
        )

        lower_count = int(
            np.count_nonzero(step2_lower)
        )

        # Same reported field as original script:
        # Number of pixels identified outside hard thresholds
        step2_removed = upper_count + lower_count

        # ----------------------------------------------------
        # Determine remove / nearest targets
        # ----------------------------------------------------

        step2_remove_mask = np.zeros_like(
            valid_mask,
            dtype=bool
        )

        step2_nearest_mask = np.zeros_like(
            valid_mask,
            dtype=bool
        )

        if UPPER_OUTLIER_ACTION == "REMOVE":
            step2_remove_mask |= step2_upper
        else:
            step2_nearest_mask |= step2_upper

        if LOWER_OUTLIER_ACTION == "REMOVE":
            step2_remove_mask |= step2_lower
        else:
            step2_nearest_mask |= step2_lower

        # Qualified donor pixels are those inside
        # the original hard thresholds, after Step 1.

        step2_donors = (
            valid_mask
            & (A >= LOWER_HARD)
            & (A <= UPPER_HARD)
        )

        # ----------------------------------------------------
        # REMOVE
        # ----------------------------------------------------

        step2_direct_removed = int(
            np.count_nonzero(step2_remove_mask)
        )

        A[step2_remove_mask] = NODATA

        # ----------------------------------------------------
        # NEAREST
        # ----------------------------------------------------

        step2_replaced, step2_failed = replace_by_nearest(
            A,
            step2_nearest_mask,
            step2_donors,
            MAX_NEAREST_DISTANCE_PIXELS
        )

        step2_actual_removed = (
            step2_direct_removed
            + step2_failed
        )

        valid_mask = get_valid_mask(A)

        # ====================================================
        # STEP 3 - COUNTRY STANDARD DEVIATION
        # ====================================================

        # Reported identified/selected count
        step3_total_removed = 0

        # Actual deletion count
        step3_actual_removed = 0

        # Successful replacement count
        step3_total_replaced = 0

        if APPLY_STD_REMOVAL:

            if gdf.crs != crs:
                gdf_reproj = gdf.to_crs(crs)
            else:
                gdf_reproj = gdf

            # Preserve M49 country processing order
            for _, country_row in gdf_reproj.iterrows():

                m49 = country_row[M49_FIELD]

                name = (
                    country_row[NAME_FIELD]
                    if NAME_FIELD in country_row
                    else str(m49)
                )

                geom = country_row.geometry

                if geom is None or geom.is_empty:

                    country_records.append(
                        empty_country_record(
                            year, m49, name
                        )
                    )

                    continue

                # --------------------------------------------
                # Country raster mask
                # --------------------------------------------

                try:

                    country_mask_arr = ~geometry_mask(
                        [geom],
                        transform=transform,
                        invert=False,
                        out_shape=(height, width)
                    )

                except Exception:

                    country_records.append(
                        empty_country_record(
                            year, m49, name
                        )
                    )

                    continue

                # --------------------------------------------
                # Country valid pixels
                # --------------------------------------------

                in_country_valid = (
                    country_mask_arr
                    & valid_mask
                )

                pixel_indices = np.where(
                    in_country_valid
                )

                n_valid = len(pixel_indices[0])

                if n_valid == 0:

                    country_records.append(
                        empty_country_record(
                            year, m49, name
                        )
                    )

                    continue

                # --------------------------------------------
                # Mean and standard deviation
                # --------------------------------------------

                vals = A[pixel_indices]

                mean_val = float(
                    np.mean(
                        vals,
                        dtype=np.float64
                    )
                )

                std_val = float(
                    np.std(
                        vals,
                        dtype=np.float64
                    )
                )

                upper_sigma = (
                    mean_val
                    + STD_FACTOR * std_val
                )

                # --------------------------------------------
                # Identify Step 3 candidates
                # --------------------------------------------

                candidate_mask = (
                    in_country_valid
                    & (A > upper_sigma)
                )

                n_candidate = int(
                    np.count_nonzero(candidate_mask)
                )

                max_allowed = int(
                    np.floor(
                        n_valid * MAX_REMOVE_RATIO
                    )
                )

                # --------------------------------------------
                # Select outliers
                #
                # Same 5% cap and descending-value rule
                # as the original script
                # --------------------------------------------

                selected_mask = np.zeros_like(
                    valid_mask,
                    dtype=bool
                )

                if n_candidate <= max_allowed:

                    selected_mask = candidate_mask

                    actual_selected = n_candidate

                else:

                    actual_selected = max_allowed

                    if max_allowed > 0:

                        cand_rows, cand_cols = np.where(
                            candidate_mask
                        )

                        cand_vals = A[
                            cand_rows,
                            cand_cols
                        ]

                        sort_idx = np.argsort(
                            -cand_vals
                        )[:max_allowed]

                        selected_mask[
                            cand_rows[sort_idx],
                            cand_cols[sort_idx]
                        ] = True

                # --------------------------------------------
                # NEW: Step 3 action
                # --------------------------------------------

                if STD_OUTLIER_ACTION == "REMOVE":

                    A[selected_mask] = NODATA

                    country_replaced = 0
                    country_actual_removed = actual_selected

                else:

                    # Nearest-neighbor donors:
                    # 1. Inside the same country
                    # 2. Valid after Step 2
                    # 3. Not above the country's sigma limit
                    #
                    # Exclude ALL values above sigma as donors,
                    # not just selected outliers.

                    country_donors = (
                        in_country_valid
                        & (A <= upper_sigma)
                    )

                    (
                        country_replaced,
                        country_actual_removed
                    ) = replace_by_nearest(
                        A,
                        selected_mask,
                        country_donors,
                        MAX_NEAREST_DISTANCE_PIXELS
                    )

                # --------------------------------------------
                # Update global raster valid mask
                # --------------------------------------------

                valid_mask = get_valid_mask(A)

                # Preserve existing reported meaning:
                # Step3_removed = selected outlier count
                # even when action is NEAREST.

                step3_total_removed += actual_selected

                step3_actual_removed += (
                    country_actual_removed
                )

                step3_total_replaced += (
                    country_replaced
                )

                # --------------------------------------------
                # Original country Excel fields
                # --------------------------------------------

                country_records.append({
                    "Year": year,
                    "M49": m49,
                    "Country": name,
                    "Valid_before_step3": n_valid,
                    "Mean": round(mean_val, 6),
                    "Std": round(std_val, 6),
                    "Upper_sigma": round(
                        upper_sigma, 6
                    ),
                    "Candidate_removed": n_candidate,
                    "Max_allowed_remove": max_allowed,
                    "Step3_removed": actual_selected,
                    "Step3_removed_%": (
                        round(
                            actual_selected / n_valid * 100,
                            4
                        )
                        if n_valid > 0 else 0.0
                    )
                })

        # ====================================================
        # 6. ANNUAL SUMMARY
        # ====================================================

        final_count = int(
            np.count_nonzero(
                get_valid_mask(A)
            )
        )

        total_removed = (
            initial_count - final_count
        )

        removed_ratio = (
            total_removed / initial_count * 100
            if initial_count > 0 else 0
        )

        # True number of deleted pixels:
        # replacement does NOT count as actual deletion.

        expected_actual_removed = (
            step1_removed
            + step2_actual_removed
            + step3_actual_removed
        )

        count_check = (
            total_removed
            - expected_actual_removed
        )

        # Original Annual Summary column structure
        annual_record = {
            "Year": year,
            "Initial_Valid": initial_count,
            "Step1_removed": step1_removed,
            "Step1_%": (
                round(
                    step1_removed / initial_count * 100,
                    4
                )
                if initial_count else 0
            ),
            "Step2_removed": step2_removed,
            "Step2_%": (
                round(
                    step2_removed / initial_count * 100,
                    4
                )
                if initial_count else 0
            ),
            "Step3_removed": step3_total_removed,
            "Step3_%": (
                round(
                    step3_total_removed / initial_count * 100,
                    4
                )
                if initial_count else 0
            ),
            "Final_Valid": final_count,
            "Total_Removed": total_removed,
            "Removed_Ratio_%": round(
                removed_ratio, 4
            )
        }

        # ====================================================
        # 7. EXPORT GEOTIFF
        # ====================================================

        profile.update(
            dtype="float32",
            nodata=NODATA,
            compress="lzw"
        )

        with rasterio.open(
            OUT_tif,
            "w",
            **profile
        ) as dst:

            dst.write(
                A.astype(np.float32),
                1
            )

        # ----------------------------------------------------
        # Return results to main process
        # ----------------------------------------------------

        return {
            "year": year,
            "status": "SUCCESS",
            "message": "",
            "annual": annual_record,
            "countries": country_records,
            "check": count_check,
            "step2_replaced": step2_replaced,
            "step3_replaced": step3_total_replaced,
            "actual_removed": total_removed,
            "output": OUT_tif
        }

    except Exception as e:

        return {
            "year": year,
            "status": "FAILED",
            "message": repr(e),
            "annual": None,
            "countries": []
        }


# ============================================================
# 8. PARALLEL PROCESSING BY YEAR
# ============================================================

YEARS = (
    list(range(START_YEAR, END_YEAR + 1))
    if YEARS_TO_RUN is None
    else sorted(set(map(int, YEARS_TO_RUN)))
)

if not YEARS:
    raise ValueError("No years selected")

print("=" * 70)
print("GLOBAL NUE OUTLIER PROCESSING")
print("=" * 70)

print(f"Years: {YEARS}")
print(f"Parallel workers: {PARALLEL_YEARS}")

print(f"Step 1 MIN_COUNT: {MIN_COUNT}")

print(
    f"Step 2 Upper: {UPPER_HARD} "
    f"({UPPER_OUTLIER_ACTION})"
)

print(
    f"Step 2 Lower: {LOWER_HARD} "
    f"({LOWER_OUTLIER_ACTION})"
)

print(
    f"Step 3 Enabled: {APPLY_STD_REMOVAL}"
)

print(
    f"Step 3 Action: {STD_OUTLIER_ACTION}"
)

print(
    f"Step 3 STD_FACTOR: {STD_FACTOR}"
)

print("=" * 70, flush=True)


if PARALLEL_YEARS == 1 or len(YEARS) == 1:

    results = [
        process_one_year(year)
        for year in YEARS
    ]

else:

    results = Parallel(
        n_jobs=min(
            PARALLEL_YEARS,
            len(YEARS)
        ),
        backend="loky",
        batch_size=1,
        pre_dispatch="n_jobs",
        verbose=10
    )(
        delayed(process_one_year)(year)
        for year in YEARS
    )


# ============================================================
# 9. COLLECT PARALLEL RESULTS
# ============================================================

results.sort(
    key=lambda item: item["year"]
)

records_year = []
records_country = []

for result in results:

    year = result["year"]
    status = result["status"]

    if status == "SUCCESS":

        records_year.append(
            result["annual"]
        )

        records_country.extend(
            result["countries"]
        )

        a = result["annual"]

        print(
            f"[{year}] "
            f"Initial={a['Initial_Valid']:,} | "
            f"Step1={a['Step1_removed']:,} | "
            f"Step2={a['Step2_removed']:,} | "
            f"Step3={a['Step3_removed']:,} | "
            f"Step2_Replaced={result['step2_replaced']:,} | "
            f"Step3_Replaced={result['step3_replaced']:,} | "
            f"Final={a['Final_Valid']:,} | "
            f"Check={result['check']}"
        )

    else:

        print(
            f"[{year}] {status}: "
            f"{result['message']}"
        )


# ============================================================
# 10. EXCEL REPORT
# ============================================================

COUNTRY_COLS = [
    "Year",
    "M49",
    "Country",
    "Valid_before_step3",
    "Mean",
    "Std",
    "Upper_sigma",
    "Candidate_removed",
    "Max_allowed_remove",
    "Step3_removed",
    "Step3_removed_%"
]

if not records_year:

    print(
        "\nNo successful years. Excel not generated."
    )

else:

    df_year = (
        pd.DataFrame(records_year)
        .sort_values("Year")
        .reset_index(drop=True)
    )

    df_country = pd.DataFrame(
        records_country,
        columns=COUNTRY_COLS
    )

    if not df_country.empty:

        df_country = (
            df_country
            .sort_values(["Year", "M49"])
            .reset_index(drop=True)
        )

    wb = Workbook()

    thin = Side(
        style="thin",
        color="AAAAAA"
    )

    border = Border(
        left=thin,
        right=thin,
        top=thin,
        bottom=thin
    )

    # --------------------------------------------------------
    # Excel formatting helpers
    # --------------------------------------------------------

    def hdr_cell(
        ws, row, col, value,
        bg="2F75B6",
        fg="FFFFFF"
    ):

        c = ws.cell(
            row=row,
            column=col,
            value=value
        )

        c.font = Font(
            bold=True,
            color=fg,
            name="Arial",
            size=10
        )

        c.fill = PatternFill(
            "solid",
            start_color=bg
        )

        c.alignment = Alignment(
            horizontal="center",
            wrap_text=True
        )

        c.border = border

        return c


    def data_cell(
        ws, row, col, value,
        bg="FFFFFF",
        fmt=None
    ):

        if isinstance(value, np.generic):
            value = value.item()

        if (
            isinstance(value, float)
            and not np.isfinite(value)
        ):
            value = None

        c = ws.cell(
            row=row,
            column=col,
            value=value
        )

        c.font = Font(
            name="Arial",
            size=10
        )

        c.fill = PatternFill(
            "solid",
            start_color=bg
        )

        c.alignment = Alignment(
            horizontal="center"
        )

        c.border = border

        if fmt:
            c.number_format = fmt

        return c


    # ========================================================
    # SHEET 1: ANNUAL SUMMARY
    # ========================================================

    ws1 = wb.active

    ws1.title = "Annual Summary"

    col_colors_year = {
        "Year": "FFFFFF",
        "Initial_Valid": "FFFFFF",
        "Step1_removed": "D6E4F0",
        "Step1_%": "D6E4F0",
        "Step2_removed": "D5E8D4",
        "Step2_%": "D5E8D4",
        "Step3_removed": "FFF2CC",
        "Step3_%": "FFF2CC",
        "Final_Valid": "FFFFFF",
        "Total_Removed": "F4CCCC",
        "Removed_Ratio_%": "F4CCCC"
    }

    cols_y = list(df_year.columns)

    for ci, col in enumerate(cols_y, 1):
        hdr_cell(ws1, 1, ci, col)

    for ri, row in enumerate(
        df_year.itertuples(index=False),
        2
    ):

        for ci, (col, val) in enumerate(
            zip(cols_y, row),
            1
        ):

            fmt = (
                "0.00"
                if "%" in col
                else (
                    "#,##0"
                    if col != "Year"
                    else "0"
                )
            )

            data_cell(
                ws1,
                ri,
                ci,
                val,
                bg=col_colors_year.get(
                    col,
                    "FFFFFF"
                ),
                fmt=fmt
            )

    # Original TOTAL/AVG row
    sr = len(df_year) + 2

    data_cell(
        ws1,
        sr,
        1,
        "TOTAL/AVG",
        bg="D9D9D9"
    ).font = Font(
        bold=True,
        name="Arial",
        size=10
    )

    for ci, col in enumerate(
        cols_y[1:],
        2
    ):

        cl = get_column_letter(ci)

        rng = (
            f"{cl}2:"
            f"{cl}{len(df_year) + 1}"
        )

        formula = (
            f"=AVERAGE({rng})"
            if "%" in col
            else f"=SUM({rng})"
        )

        c = ws1.cell(
            row=sr,
            column=ci,
            value=formula
        )

        c.font = Font(
            bold=True,
            name="Arial",
            size=10
        )

        c.fill = PatternFill(
            "solid",
            start_color="D9D9D9"
        )

        c.alignment = Alignment(
            horizontal="center"
        )

        c.border = border

        c.number_format = (
            "0.00"
            if "%" in col
            else "#,##0"
        )

    for ci, col in enumerate(cols_y, 1):

        ws1.column_dimensions[
            get_column_letter(ci)
        ].width = max(
            len(col) + 4,
            14
        )

    ws1.row_dimensions[1].height = 36
    ws1.freeze_panes = "B2"

    # ========================================================
    # SHEET 2: COUNTRY DETAIL
    # ========================================================

    ws2 = wb.create_sheet(
        "Country Detail"
    )

    cols_c = COUNTRY_COLS

    col_colors_cty = {
        "Year": "FFFFFF",
        "M49": "FFFFFF",
        "Country": "FFFFFF",
        "Valid_before_step3": "FFFFFF",
        "Mean": "EAD1DC",
        "Std": "EAD1DC",
        "Upper_sigma": "EAD1DC",
        "Candidate_removed": "FFF2CC",
        "Max_allowed_remove": "FFF2CC",
        "Step3_removed": "F4CCCC",
        "Step3_removed_%": "F4CCCC"
    }

    for ci, col in enumerate(cols_c, 1):
        hdr_cell(ws2, 1, ci, col)

    for ri, row in enumerate(
        df_country.itertuples(index=False),
        2
    ):

        for ci, (col, val) in enumerate(
            zip(cols_c, row),
            1
        ):

            if "%" in col:
                fmt = "0.00"

            elif col in (
                "Mean",
                "Std",
                "Upper_sigma"
            ):
                fmt = "0.000000"

            elif col not in (
                "Year",
                "M49",
                "Country"
            ):
                fmt = "#,##0"

            else:
                fmt = None

            data_cell(
                ws2,
                ri,
                ci,
                val,
                bg=col_colors_cty.get(
                    col,
                    "FFFFFF"
                ),
                fmt=fmt
            )

    for ci, col in enumerate(cols_c, 1):

        ws2.column_dimensions[
            get_column_letter(ci)
        ].width = max(
            len(col) + 3,
            12
        )

    ws2.column_dimensions["C"].width = 22
    ws2.row_dimensions[1].height = 36
    ws2.freeze_panes = "D2"

    # ========================================================
    # SHEET 3: PARAMETERS
    # ========================================================

    ws3 = wb.create_sheet(
        "Parameters"
    )

    params = [
        ("Parameter", "Value"),
        ("MIN_COUNT (Step1)", MIN_COUNT),
        ("LOWER_HARD (Step2)", LOWER_HARD),
        ("UPPER_HARD (Step2)", UPPER_HARD),
        ("LOWER_OUTLIER_ACTION", LOWER_OUTLIER_ACTION),
        ("UPPER_OUTLIER_ACTION", UPPER_OUTLIER_ACTION),
        ("APPLY_STD_REMOVAL", APPLY_STD_REMOVAL),
        ("STD_FACTOR (Step3)", STD_FACTOR),
        ("STD_OUTLIER_ACTION", STD_OUTLIER_ACTION),
        ("MAX_REMOVE_RATIO (Step3)", MAX_REMOVE_RATIO),
        (
            "MAX_NEAREST_DISTANCE_PIXELS",
            "None" if MAX_NEAREST_DISTANCE_PIXELS is None
            else MAX_NEAREST_DISTANCE_PIXELS
        ),
        ("M49_FIELD", M49_FIELD),
        ("SHP_PATH", SHP_PATH),
        ("START_YEAR", START_YEAR),
        ("END_YEAR", END_YEAR),
        ("NODATA", NODATA),
        ("PARALLEL_YEARS", PARALLEL_YEARS),
        (
            "YEARS_TO_RUN",
            "All" if YEARS_TO_RUN is None
            else str(YEARS)
        )
    ]

    for ri, (k, v) in enumerate(
        params,
        1
    ):

        for ci, val in enumerate(
            (k, v),
            1
        ):

            c = ws3.cell(
                row=ri,
                column=ci,
                value=val
            )

            c.font = Font(
                bold=(ri == 1),
                name="Arial",
                size=10,
                color=(
                    "FFFFFF"
                    if ri == 1
                    else "000000"
                )
            )

            c.fill = PatternFill(
                "solid",
                start_color=(
                    "2F75B6"
                    if ri == 1
                    else "F5F5F5"
                )
            )

            c.border = border

            c.alignment = Alignment(
                horizontal="left"
            )

    ws3.column_dimensions["A"].width = 38
    ws3.column_dimensions["B"].width = 60

    # --------------------------------------------------------
    # Save Excel
    # --------------------------------------------------------

    wb.save(EXCEL_OUT)

    print(
        f"\nExcel saved: {EXCEL_OUT}"
    )

    # ========================================================
    # 11. FINAL VALIDATION
    # ========================================================

    check_rows = []

    for result in results:

        if result["status"] == "SUCCESS":

            check_rows.append({
                "Year": result["year"],
                "Step2_detected": (
                    result["annual"]["Step2_removed"]
                ),
                "Step3_detected": (
                    result["annual"]["Step3_removed"]
                ),
                "Step2_replaced": (
                    result["step2_replaced"]
                ),
                "Step3_replaced": (
                    result["step3_replaced"]
                ),
                "Actual_removed": (
                    result["actual_removed"]
                ),
                "Check": result["check"]
            })

    df_check = pd.DataFrame(
        check_rows
    )

    print("\n" + "=" * 70)
    print("FINAL VALIDATION SUMMARY")
    print("=" * 70)

    print(
        df_check.to_string(
            index=False
        )
    )

    if (df_check["Check"] == 0).all():

        print(
            "\nPASS: All annual pixel counts are consistent."
        )

    else:

        print(
            "\nWARNING: Pixel count inconsistency detected."
        )

print("\nAll requested years finished.")

In [ ]:
# ============================================================
# tool Parallel Country Mean Extraction
# Global NUE - 
# Parallel Processing + NaN / NoData Fix
#
# 1. Preserve original zonal_stats calculation
# 2. Correctly handle NaN / Inf / NoData / -9999
# 3. Parallel computation across years
# 4. Export results to Excel
#
# JupyterLab / Windows compatible (joblib - loky)
# ============================================================

import os
import glob
import re
import time

import numpy as np
import geopandas as gpd
import rasterio
import pandas as pd

from rasterstats import zonal_stats
from joblib import Parallel, delayed


# ============================================================
# 1. USER CONFIGURATION
# ============================================================

SHP_PATH = (
    r"Z:/staff/Qiang_S/09_Datasets/"
    r"09OtherDataUsed/GDAM_GlobalCountryLevel_Shp/"
    r"GlobalCountryLevel_Shp.shp"
)

# Input GeoTIFF folder
TIF_FOLDER = (
    # r"Z:\staff\Qiang_S\09_Datasets\03GlobalNUE_Results\01Global_NUE_Wheat\02WheatNUE_M2_20261021V2_M6_180C"
    r"Z:\staff\Qiang_S\09_Datasets\03GlobalNUE_Results\02Global_NUE_Maize\02MaizeNUE_M2_20261022V1_M6_180C"
)

# Output Excel
OUT_EXCEL = os.path.join(
    TIF_FOLDER,
    "Maize_NUE_countryV1_M6_180C.xlsx"
)

# Standard NoData used internally
NODATA = -9999.0

# Additional NoData values not recorded in TIFF metadata
ADDITIONAL_NODATA = []

# Number of parallel processes
N_WORKERS = 10

# Zonal statistics settings
# Keep original rasterstats default pixel inclusion rule
ALL_TOUCHED = False

# ============================================================
# 2. INITIALIZATION
# ============================================================

start_time = time.perf_counter()

print("=" * 70)
print("GLOBAL WHEAT NUE - COUNTRY MEAN")
print("=" * 70)

print(f"Input folder   : {TIF_FOLDER}")
print(f"Output Excel   : {OUT_EXCEL}")
print(f"Parallel jobs  : {N_WORKERS}")
print(f"Internal NoData: {NODATA}")

# ============================================================
# 3. LOAD COUNTRY SHAPEFILE
# ============================================================

print("\nReading shapefile...")

gdf = gpd.read_file(SHP_PATH)

if gdf.crs is None:
    raise ValueError(
        "Shapefile CRS is missing. "
        "Please define the correct CRS before processing."
    )

# Same CRS as original code
gdf = gdf.to_crs("EPSG:4326")

# Ensure sequential row index for stats matching
gdf = gdf.reset_index(drop=True)

# Check required columns
for field in ["M49_Code", "COUNTRY"]:
    if field not in gdf.columns:
        raise ValueError(
            f"Missing required shapefile field: {field}"
        )

print(f"Countries loaded: {len(gdf)}")

# ============================================================
# 4. FIND TIFF FILES
# ============================================================

tif_files = sorted(
    p for p in glob.glob(
        os.path.join(TIF_FOLDER, "*")
    )
    if p.lower().endswith((".tif", ".tiff"))
)

print(f"Found {len(tif_files)} GeoTIFF files")

if not tif_files:
    raise RuntimeError("No GeoTIFF files found!")

# Extract years
year_files = []

for tif in tif_files:

    fname = os.path.basename(tif)

    # Match the final four-digit year
    match = re.search(
        r"(\d{4})\.(?:tif|tiff)$",
        fname,
        re.IGNORECASE
    )

    if not match:
        print(f"Skipping file without year: {fname}")
        continue

    year = int(match.group(1))

    year_files.append((year, tif))

# Sort by year
year_files.sort(key=lambda x: (x[0], x[1]))

if not year_files:
    raise RuntimeError(
        "No files with valid four-digit years found."
    )

print(
    "Years to process:",
    [year for year, _ in year_files]
)

# ============================================================
# 5. PROCESS ONE YEAR
# ============================================================

def process_one_year(year, tif, countries):
    """
    Calculate country mean NUE for one year.

    Original method preserved:
        rasterstats.zonal_stats(stats=['mean'])

    Main fix:
        Standardize all invalid raster pixels
        before calling zonal_stats.

    Returns:
        year, records, valid_pixel_count
    """

    t0 = time.perf_counter()

    # --------------------------------------------------------
    # 5.1 Read GeoTIFF
    # --------------------------------------------------------

    with rasterio.open(tif) as src:

        if src.count != 1:
            raise ValueError(
                f"{os.path.basename(tif)}: "
                "Expected single-band GeoTIFF."
            )

        if src.crs is None:
            raise ValueError(
                f"{os.path.basename(tif)}: Missing CRS."
            )

        if src.crs != countries.crs:
            raise ValueError(
                f"{os.path.basename(tif)}: CRS mismatch. "
                f"Raster={src.crs}, Countries={countries.crs}"
            )

        affine = src.transform

        # Read with metadata-defined NoData mask
        data_masked = src.read(
            1,
            masked=True
        )

        # Convert masked values to NaN
        data = np.asarray(
            data_masked.astype(
                np.float32
            ).filled(np.nan),
            dtype=np.float32
        )

    # --------------------------------------------------------
    # 5.2 Fix NaN / Inf / NoData
    # --------------------------------------------------------

    # Recognize finite values
    valid = np.isfinite(data)

    # Recognize explicit -9999 NoData
    valid &= (data != NODATA)

    # Optional additional missing values
    for nd in ADDITIONAL_NODATA:
        valid &= (data != nd)

    valid_pixel_count = int(
        np.count_nonzero(valid)
    )

    # Replace all invalid pixels with one consistent NoData
    # Existing valid values are not modified
    data[~valid] = NODATA

    # --------------------------------------------------------
    # 5.3 Calculate country zonal statistics
    # --------------------------------------------------------

    # Keep original zonal_stats mean calculation.
    # Use the cleaned raster array instead of original TIFF
    # to prevent NaN from affecting mean values.

    stats = zonal_stats(
        countries,
        data,
        affine=affine,
        stats=["mean"],
        nodata=NODATA,
        geojson_out=False,
        all_touched=ALL_TOUCHED
    )

    # --------------------------------------------------------
    # 5.4 Collect country results
    # --------------------------------------------------------

    year_records = []

    for i, row in countries.iterrows():

        mean_val = stats[i].get("mean")

        # Keep missing country values as None
        # (Excel will display an empty cell)
        if mean_val is not None:

            if not np.isfinite(mean_val):
                mean_val = None
            else:
                mean_val = float(mean_val)

        year_records.append({
            "M49_Code": row["M49_Code"],
            "COUNTRY": row["COUNTRY"],
            "year": year,
            "value": mean_val
        })

    elapsed = time.perf_counter() - t0

    print(
        f"[{year}] Completed | "
        f"Valid pixels: {valid_pixel_count:,} | "
        f"Countries: {len(year_records)} | "
        f"Time: {elapsed:.1f}s",
        flush=True
    )

    return (
        year,
        year_records,
        valid_pixel_count
    )


# ============================================================
# 6. PARALLEL PROCESSING
# ============================================================

print("\n" + "=" * 70)
print("START PARALLEL COMPUTATION")
print("=" * 70)

n_jobs = min(
    N_WORKERS,
    len(year_files)
)

print(f"Running {n_jobs} parallel processes")

# joblib loky works with functions defined in
# Jupyter notebooks, including Windows environments.

parallel_results = Parallel(
    n_jobs=n_jobs,
    backend="loky",
    verbose=10,
    batch_size=1,
    pre_dispatch="n_jobs"
)(
    delayed(process_one_year)(
        year,
        tif,
        gdf
    )
    for year, tif in year_files
)

# ============================================================
# 7. COMBINE ALL YEARS
# ============================================================

print("\nCombining results...")

# Ensure results remain sorted by year
parallel_results.sort(
    key=lambda x: x[0]
)

records = []

for year, year_records, valid_count in parallel_results:

    records.extend(year_records)

    print(
        f"{year} | "
        f"Countries={len(year_records)} | "
        f"Valid pixels={valid_count:,}"
    )

# ============================================================
# 8. EXPORT EXCEL
# ============================================================

df = pd.DataFrame(
    records,
    columns=[
        "M49_Code",
        "COUNTRY",
        "year",
        "value"
    ]
)

# Same structure as original code
df.to_excel(
    OUT_EXCEL,
    index=False,
    engine="openpyxl"
)

# ============================================================
# 9. FINAL SUMMARY
# ============================================================

elapsed_total = time.perf_counter() - start_time

print("\n" + "=" * 70)
print("COUNTRY MEAN EXTRACTION COMPLETED")
print("=" * 70)

print(f"Processed TIFF files : {len(year_files)}")
print(f"Countries            : {len(gdf)}")
print(f"Total records        : {len(df):,}")
print(f"Parallel processes   : {n_jobs}")

print(
    f"Valid country means  : "
    f"{df['value'].notna().sum():,}"
)

print(
    f"Missing country means: "
    f"{df['value'].isna().sum():,}"
)

print(f"Total time           : {elapsed_total:.2f}s")
print(f"Saved to             : {OUT_EXCEL}")

print("=" * 70)


In [ ]:
# ================V 2026.10.27=====================C4NUPW=======================
# Global NUE - Parallel Country Statistics (5 Methods)
#
# Method 1: Percentile Trimmed Mean
# Method 2: 2 / 3 Standard Deviation Mean
# Method 3: Arithmetic Mean
# Method 4: Harvested-Area-Weighted Mean
# Method 5 (NEW):
#              SUM(Nup_i * HA_i)
# NUE_region = -------------------------
#              SUM((Nup_i/NUE_i) * HA_i)
#
# Core validity rule:
#   NUE must be valid and > 0
#   Nup must be valid and >= 0
#   HA must be valid and > 0
#   All three pixels must overlap spatially.
#
# Parallel years / Windows / JupyterLab
# No resampling, filling or interpolation
# ============================================================

import os
import re
import time
import traceback

from pathlib import Path
from itertools import repeat

import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
import ipywidgets as widgets

from rasterstats import gen_zonal_stats
from joblib import Parallel, delayed
from IPython.display import display, clear_output


# ============================================================
# 1. USER CONFIGURATION
# ============================================================

SHP_PATH = (
    r"Z:/staff/Qiang_S/09_Datasets/"
    r"09OtherDataUsed/GDAM_GlobalCountryLevel_Shp/"
    r"GlobalCountryLevel_Shp.shp"
)

# ------------------------------------------------------------
# NUE GeoTIFF folder
# ------------------------------------------------------------

TIF_FOLDER = (
    r"Z:\staff\Qiang_S\09_Datasets\03GlobalNUE_Results\02Global_NUE_Maize\02MaizeNUE_M2_20261027V2_M6_150R"
)

# ------------------------------------------------------------
# Harvested Area GeoTIFF folder
# ------------------------------------------------------------

AREA_FOLDER = (
    #r"Z:\staff\Qiang_S\09_Datasets\09OtherDataUsed\GCSN_HarvestedArea_1995_2020\02StandardizedDatasets\01WheatHarvestedArea"
    r"Z:\staff\Qiang_S\09_Datasets\09OtherDataUsed\GCSN_HarvestedArea_1995_2020\02StandardizedDatasets\02MaizeHarvestedArea"
)

# ------------------------------------------------------------
# NEW: Nup GeoTIFF folder
# ------------------------------------------------------------

NUP_FOLDER = (
    r"Z:\staff\Qiang_S\09_Datasets\03GlobalNUE_Results\02Global_NUE_Maize\02MaizeNUE_M2_20261027V2_Nup"
)

#CROP = "Wheat"
CROP = "Maize"

# Harvested Area examples:
#   Wheat_HarArea_1995.tif
#   Wheat_HarArea_2020.tif

AREA_FILE_PREFIX = f"{CROP}_HarArea_"

# Nup examples supported by default:
#   Wheat_Nup_1995.tif
#   Nup_Wheat_1995.tif
#   Wheat_Nuptake_1995.tif
#
# Match any file containing "nup"
# and ending with "_YYYY.tif".

NUP_FILENAME_REGEX = (
    r"^.*nup.*_(?P<year>(?:19|20)\d{2})"
    r"\.(?:tif|tiff)$"
)

# ------------------------------------------------------------
# Output Excel
# New name avoids overwriting original results.
# ------------------------------------------------------------

OUT_EXCEL = os.path.join(
    TIF_FOLDER,
    f"{CROP}_NUE_Country_Mean_WithNup.xlsx"
)

# ------------------------------------------------------------
# NoData settings
# ------------------------------------------------------------

NODATA = -9999.0
ADDITIONAL_NODATA = []

# ------------------------------------------------------------
# Parallel years
# ------------------------------------------------------------

N_WORKERS = 15

# None: all years
# Example: [1995, 1996, 2020]
YEARS_TO_RUN = None

# ------------------------------------------------------------
# Country raster inclusion rule
# ------------------------------------------------------------

ALL_TOUCHED = False

# Percentile / STD filtering only if N > 10
MIN_PIXELS_FOR_FILTER = 10

# ------------------------------------------------------------
# Area and Nup parameters
# ------------------------------------------------------------

AREA_THRESHOLD = 0.0

# If Nup is physically nonnegative,
# zero is allowed but contributes zero.
NUP_MIN_VALUE = 0.0

# NUE must be positive to calculate Nup/NUE.
NUE_MIN_VALUE = 0.0

# Require corresponding files when selected
REQUIRE_AREA_MATCH = True
REQUIRE_NUP_MATCH = True


# ============================================================
# 2. READ AND CLEAN RASTER
# ============================================================

def read_clean_raster(src):
    """
    Invalid:
      Metadata NoData
      NaN / Inf
      -9999
      ADDITIONAL_NODATA

    Return float32 raster with invalid pixels
    standardized to NODATA.
    """

    if src.count != 1:
        raise ValueError(
            f"Expected single-band raster: {src.name}"
        )

    masked = src.read(1, masked=True)

    data = np.array(
        masked.data,
        dtype=np.float32,
        copy=True
    )

    invalid = np.ma.getmaskarray(masked).copy()

    invalid |= ~np.isfinite(data)
    invalid |= (data == NODATA)

    for nd in ADDITIONAL_NODATA:
        invalid |= (data == nd)

    data[invalid] = NODATA

    valid_count = int(
        np.count_nonzero(~invalid)
    )

    return data, valid_count


# ============================================================
# 3. VERIFY GLOBAL RASTER ALIGNMENT
# ============================================================

def check_grid_alignment(reference, other, label):
    """
    All three rasters must have the same:
       CRS
       width / height
       affine transform

    No reprojection / resampling is performed.
    """

    if reference.crs is None or other.crs is None:
        raise ValueError(
            f"{label}: Missing CRS"
        )

    if reference.crs != other.crs:
        raise ValueError(
            f"{label}: CRS mismatch"
        )

    if (
        reference.width != other.width
        or reference.height != other.height
    ):
        raise ValueError(
            f"{label}: Raster dimensions mismatch"
        )

    if not reference.transform.almost_equals(
        other.transform,
        precision=1e-10
    ):
        raise ValueError(
            f"{label}: Pixel grid transform mismatch"
        )


def check_mini_alignment(
    reference_stat,
    other_stat,
    year,
    country,
    label
):
    """
    Verify that rasterstats used exactly the same
    cropped window for a given country.
    """

    if reference_stat is None or other_stat is None:
        raise ValueError(
            f"{year} {country}: Missing {label} statistic"
        )

    a = reference_stat.get("mini_raster_affine")
    b = other_stat.get("mini_raster_affine")

    x = reference_stat.get("mini_raster_array")
    y = other_stat.get("mini_raster_array")

    if a is None or b is None or x is None or y is None:
        raise ValueError(
            f"{year} {country}: Missing {label} mini raster"
        )

    if x.shape != y.shape:
        raise ValueError(
            f"{year} {country}: {label} mini shape mismatch"
        )

    if not a.almost_equals(b, precision=1e-10):
        raise ValueError(
            f"{year} {country}: {label} mini grid mismatch"
        )


# ============================================================
# 4. UTILITY: VALID PIXELS FROM A MASKED ARRAY
# ============================================================

def get_masked_values_and_valid(masked_array):
    """
    Return:
       values as float64
       valid boolean mask

    Polygon-outside pixels remain excluded.
    """

    ma = np.ma.asarray(masked_array)

    values = np.asarray(
        np.ma.getdata(ma),
        dtype=np.float64
    )

    valid = ~np.ma.getmaskarray(ma)
    valid = valid.copy()

    valid &= np.isfinite(values)
    valid &= (values != NODATA)

    for nd in ADDITIONAL_NODATA:
        valid &= (values != nd)

    return values, valid


# ============================================================
# 5. ORIGINAL COUNTRY STATISTICS
# ============================================================

def compute_country_metrics(masked_array, config):
    """
    Original algorithms unchanged:
      Arithmetic mean
      Percentile trimmed mean
      +/- k STD mean

    Small regions <=10 pixels:
      Return original mean for selected filters.
    """

    values, valid = get_masked_values_and_valid(
        masked_array
    )

    values = values[valid]

    n = values.size

    tag = config["trim_tag"]

    result = {
        "Valid_Pixels": int(n),
        "value": None,
        f"Mean_Trimmed_{tag}": None,
        f"N_Trimmed_{tag}": None,
        "Mean_STD_2": None,
        "N_STD_2": None,
        "Mean_STD_3": None,
        "N_STD_3": None
    }

    if n == 0:
        return result

    original_mean = float(
        np.mean(values, dtype=np.float64)
    )

    # --------------------------------------------------------
    # Arithmetic mean
    # --------------------------------------------------------

    if config["enable_all"]:
        result["value"] = original_mean

    # --------------------------------------------------------
    # Percentile trimmed mean
    # --------------------------------------------------------

    if config["enable_trimmed"]:

        mean_col = f"Mean_Trimmed_{tag}"
        count_col = f"N_Trimmed_{tag}"

        if n <= MIN_PIXELS_FOR_FILTER:

            result[mean_col] = original_mean
            result[count_col] = int(n)

        else:

            tail = (
                100.0 - config["keep_percent"]
            ) / 2.0

            p_low, p_high = np.percentile(
                values,
                [tail, 100.0 - tail]
            )

            selected = values[
                (values >= p_low)
                & (values <= p_high)
            ]

            result[count_col] = int(selected.size)

            if selected.size > 0:

                result[mean_col] = float(
                    np.mean(
                        selected,
                        dtype=np.float64
                    )
                )

    # --------------------------------------------------------
    # STD mean
    # --------------------------------------------------------

    if config["enable_std"]:

        if n <= MIN_PIXELS_FOR_FILTER:

            for k in config["std_multipliers"]:

                result[f"Mean_STD_{k}"] = original_mean
                result[f"N_STD_{k}"] = int(n)

        else:

            std_val = float(
                np.std(
                    values,
                    dtype=np.float64,
                    ddof=0
                )
            )

            for k in config["std_multipliers"]:

                lower = original_mean - k * std_val
                upper = original_mean + k * std_val

                selected = values[
                    (values >= lower)
                    & (values <= upper)
                ]

                result[f"N_STD_{k}"] = int(selected.size)

                if selected.size > 0:

                    result[f"Mean_STD_{k}"] = float(
                        np.mean(
                            selected,
                            dtype=np.float64
                        )
                    )

    return result


# ============================================================
# 6. ORIGINAL HARVESTED-AREA-WEIGHTED MEAN
# ============================================================

def compute_area_weighted_mean(
    nue_masked,
    area_masked
):
    """
    Mean = SUM(NUE * HA) / SUM(HA)

    Only common valid NUE and HA pixels.
    """

    output = {
        "Mean_Area_Weighted": None,
        "N_Area_Weighted": 0,
        "Harvested_Area_Used": 0.0
    }

    if nue_masked is None or area_masked is None:
        return output

    nue, valid_nue = get_masked_values_and_valid(
        nue_masked
    )

    area, valid_area = get_masked_values_and_valid(
        area_masked
    )

    if nue.shape != area.shape:
        raise ValueError(
            "NUE / Area mini rasters have different shapes"
        )

    common = (
        valid_nue
        & valid_area
        & (area > AREA_THRESHOLD)
    )

    n_common = int(np.count_nonzero(common))

    output["N_Area_Weighted"] = n_common

    if n_common == 0:
        return output

    a = area[common]
    x = nue[common]

    area_sum = float(
        np.sum(a, dtype=np.float64)
    )

    output["Harvested_Area_Used"] = area_sum

    if area_sum <= 0:
        return output

    output["Mean_Area_Weighted"] = float(
        np.sum(x * a, dtype=np.float64) / area_sum
    )

    return output


# ============================================================
# 7. NEW: REGIONAL NUE - RATIO OF SUMS
# ============================================================

def compute_nup_based_nue(
    nue_masked,
    nup_masked,
    area_masked
):
    """
    NUE_region =

        SUM(Nup_i * HA_i)
        -------------------------------
        SUM((Nup_i / NUE_i) * HA_i)

    IMPORTANT:
      1. NUE is the core mask.
      2. NUE must be finite and > 0.
      3. Nup must be finite and >= 0.
      4. Harvested Area must be finite and > 0.
      5. All three rasters must overlap spatially.
      6. Numerator and denominator use EXACTLY
         the same valid pixels.

    No interpolation or filling.

    Returns:
      NUE_Ratio_of_Sums
      N_Ratio_Common_Pixels
      Sum_Nup_x_HA
      Sum_Ninput_x_HA
    """

    output = {
        "NUE_Ratio_of_Sums": None,
        "N_Ratio_Common_Pixels": 0,
        "Sum_Nup_x_HA": 0.0,
        "Sum_Ninput_x_HA": 0.0
    }

    if (
        nue_masked is None
        or nup_masked is None
        or area_masked is None
    ):
        return output

    nue, nue_valid = get_masked_values_and_valid(
        nue_masked
    )

    nup, nup_valid = get_masked_values_and_valid(
        nup_masked
    )

    area, area_valid = get_masked_values_and_valid(
        area_masked
    )

    if not (
        nue.shape == nup.shape == area.shape
    ):
        raise ValueError(
            "NUE, Nup and Area mini rasters "
            "must have the same shape"
        )

    # --------------------------------------------------------
    # Core mask: NUE must exist and be positive
    # --------------------------------------------------------

    common = (
        nue_valid
        & (nue > NUE_MIN_VALUE)
        & nup_valid
        & (nup >= NUP_MIN_VALUE)
        & area_valid
        & (area > AREA_THRESHOLD)
    )

    n_initial = int(
        np.count_nonzero(common)
    )

    if n_initial == 0:
        return output

    # --------------------------------------------------------
    # Get exactly matched pixels
    # --------------------------------------------------------

    nue_common = nue[common]
    nup_common = nup[common]
    area_common = area[common]

    # --------------------------------------------------------
    # Numerator and denominator contributions
    # --------------------------------------------------------

    with np.errstate(
        divide="ignore",
        invalid="ignore",
        over="ignore"
    ):

        numerator_terms = (
            nup_common * area_common
        )

        denominator_terms = (
            (nup_common / nue_common)
            * area_common
        )

    # Extra numerical safety:
    # If an extreme input produces Inf/NaN during
    # calculation, remove that pixel from BOTH sums.

    finite_terms = (
        np.isfinite(numerator_terms)
        & np.isfinite(denominator_terms)
    )

    numerator_terms = numerator_terms[
        finite_terms
    ]

    denominator_terms = denominator_terms[
        finite_terms
    ]

    n_common = int(
        numerator_terms.size
    )

    output["N_Ratio_Common_Pixels"] = n_common

    if n_common == 0:
        return output

    # --------------------------------------------------------
    # Regional sums
    # --------------------------------------------------------

    sum_nup_ha = float(
        np.sum(
            numerator_terms,
            dtype=np.float64
        )
    )

    sum_ninput_ha = float(
        np.sum(
            denominator_terms,
            dtype=np.float64
        )
    )

    if not (
        np.isfinite(sum_nup_ha)
        and np.isfinite(sum_ninput_ha)
    ):
        raise ValueError(
            "Non-finite regional Nup/N-input sums"
        )

    output["Sum_Nup_x_HA"] = sum_nup_ha

    output["Sum_Ninput_x_HA"] = sum_ninput_ha

    # Denominator must be > 0
    if sum_ninput_ha <= 0:
        return output

    result = (
        sum_nup_ha / sum_ninput_ha
    )

    if np.isfinite(result):
        output["NUE_Ratio_of_Sums"] = float(result)

    return output


# ============================================================
# 8. LOAD COUNTRY SHAPEFILE
# ============================================================

def load_countries():

    if not os.path.isfile(SHP_PATH):
        raise FileNotFoundError(SHP_PATH)

    gdf = gpd.read_file(SHP_PATH)

    if gdf.crs is None:
        raise ValueError(
            "Country shapefile CRS is missing"
        )

    gdf = gdf.to_crs("EPSG:4326")

    gdf = gdf.reset_index(drop=True)

    for field in ["M49_Code", "COUNTRY"]:
        if field not in gdf.columns:
            raise ValueError(
                f"Missing shapefile field: {field}"
            )

    return gdf


# ============================================================
# 9. FIND YEARLY RASTERS
# ============================================================

def extract_year(filename):

    match = re.search(
        r"_(19\d{2}|20\d{2})\.(?:tif|tiff)$",
        Path(filename).name,
        flags=re.IGNORECASE
    )

    if match:
        return int(match.group(1))

    return None


def index_rasters(
    folder,
    filename_filter,
    recursive=True
):
    """
    Build year -> file path mapping.

    Raise on duplicate year files to prevent
    accidentally using a wrong crop or raster.
    """

    if not os.path.isdir(folder):
        raise FileNotFoundError(folder)

    root = Path(folder)

    iterator = (
        root.rglob("*")
        if recursive
        else root.iterdir()
    )

    file_index = {}

    for path in sorted(iterator):

        if not path.is_file():
            continue

        if path.suffix.lower() not in (".tif", ".tiff"):
            continue

        if not filename_filter(path.name):
            continue

        year = extract_year(path.name)

        if year is None:
            continue

        if year in file_index:
            raise ValueError(
                f"Duplicate raster for {year}:\n"
                f"{file_index[year]}\n{path}"
            )

        file_index[year] = str(path)

    return file_index


def find_year_files(config):

    # --------------------------------------------------------
    # NUE rasters
    # --------------------------------------------------------

    nue_files = index_rasters(
        TIF_FOLDER,
        filename_filter=lambda name: True,
        recursive=False
    )

    if YEARS_TO_RUN is not None:
        requested_years = set(
            map(int, YEARS_TO_RUN)
        )
    else:
        requested_years = None

    if requested_years is not None:

        nue_files = {
            y: p
            for y, p in nue_files.items()
            if y in requested_years
        }

    if not nue_files:
        raise RuntimeError(
            "No matching NUE GeoTIFF files found"
        )

    # --------------------------------------------------------
    # Harvested Area rasters
    # --------------------------------------------------------

    need_area = (
        config["enable_weighted"]
        or config["enable_ratio"]
    )

    area_files = {}

    if need_area:

        area_files = index_rasters(
            AREA_FOLDER,
            filename_filter=lambda name: (
                name.lower().startswith(
                    AREA_FILE_PREFIX.lower()
                )
            ),
            recursive=True
        )

    # --------------------------------------------------------
    # NEW: Nup rasters
    # --------------------------------------------------------

    nup_files = {}

    if config["enable_ratio"]:

        pattern = re.compile(
            NUP_FILENAME_REGEX,
            flags=re.IGNORECASE
        )

        nup_files = index_rasters(
            NUP_FOLDER,
            filename_filter=lambda name: (
                pattern.fullmatch(name) is not None
            ),
            recursive=True
        )

    # --------------------------------------------------------
    # Match by year
    # --------------------------------------------------------

    year_files = []

    for year in sorted(nue_files):

        area_path = area_files.get(year)
        nup_path = nup_files.get(year)

        if (
            need_area
            and REQUIRE_AREA_MATCH
            and area_path is None
        ):
            raise FileNotFoundError(
                f"No Harvested Area raster for {year}"
            )

        if (
            config["enable_ratio"]
            and REQUIRE_NUP_MATCH
            and nup_path is None
        ):
            raise FileNotFoundError(
                f"No Nup raster for {year}. "
                "Check NUP_FOLDER and NUP_FILENAME_REGEX."
            )

        year_files.append((
            year,
            nue_files[year],
            area_path,
            nup_path
        ))

    return year_files


# ============================================================
# 10. PROCESS ONE YEAR
# ============================================================

def process_one_year(
    year,
    nue_tif,
    area_tif,
    nup_tif,
    countries,
    config
):

    t0 = time.perf_counter()

    # --------------------------------------------------------
    # Read aligned rasters
    # --------------------------------------------------------

    with rasterio.open(nue_tif) as src_nue:

        if src_nue.crs is None:
            raise ValueError(
                f"{year}: NUE CRS is missing"
            )

        crs = src_nue.crs
        affine = src_nue.transform

        nue_data, nue_valid_count = (
            read_clean_raster(src_nue)
        )

        area_data = None
        nup_data = None

        need_area = (
            config["enable_weighted"]
            or config["enable_ratio"]
        )

        if need_area and area_tif is not None:

            with rasterio.open(area_tif) as src_area:

                check_grid_alignment(
                    src_nue,
                    src_area,
                    f"{year}: NUE / Area"
                )

                area_data, _ = read_clean_raster(
                    src_area
                )

        if (
            config["enable_ratio"]
            and nup_tif is not None
        ):

            with rasterio.open(nup_tif) as src_nup:

                check_grid_alignment(
                    src_nue,
                    src_nup,
                    f"{year}: NUE / Nup"
                )

                nup_data, _ = read_clean_raster(
                    src_nup
                )

    # --------------------------------------------------------
    # Country coordinate system
    # --------------------------------------------------------

    if countries.crs != crs:

        gdf_year = countries.to_crs(
            crs
        )

    else:

        gdf_year = countries

    # --------------------------------------------------------
    # Initialize one row per country
    # --------------------------------------------------------

    records = []

    for _, row in gdf_year.iterrows():

        records.append({
            "M49_Code": row["M49_Code"],
            "COUNTRY": row["COUNTRY"],
            "year": year,

            "Valid_Pixels": 0,

            "Mean_Area_Weighted": None,
            "N_Area_Weighted": 0,
            "Harvested_Area_Used": 0.0,

            "NUE_Ratio_of_Sums": None,
            "N_Ratio_Common_Pixels": None,
            "Sum_Nup_x_HA": None,
            "Sum_Ninput_x_HA": None
        })

    valid_positions = []

    for i, geom in enumerate(
        gdf_year.geometry
    ):

        if geom is not None and not geom.is_empty:
            valid_positions.append(i)

    geometries = (
        gdf_year.geometry
        .iloc[valid_positions]
        .tolist()
    )

    # --------------------------------------------------------
    # Zonal statistics for each country
    # --------------------------------------------------------

    if geometries:

        nue_iterator = gen_zonal_stats(
            geometries,
            nue_data,
            affine=affine,
            stats=["mean"],
            nodata=NODATA,
            all_touched=ALL_TOUCHED,
            geojson_out=False,
            raster_out=True
        )

        if area_data is not None:

            area_iterator = gen_zonal_stats(
                geometries,
                area_data,
                affine=affine,
                stats=["count"],
                nodata=NODATA,
                all_touched=ALL_TOUCHED,
                geojson_out=False,
                raster_out=True
            )

        else:

            area_iterator = repeat(None)

        if nup_data is not None:

            nup_iterator = gen_zonal_stats(
                geometries,
                nup_data,
                affine=affine,
                stats=["count"],
                nodata=NODATA,
                all_touched=ALL_TOUCHED,
                geojson_out=False,
                raster_out=True
            )

        else:

            nup_iterator = repeat(None)

        # ----------------------------------------------------
        # Country loop
        # ----------------------------------------------------

        for pos, stat_nue, stat_area, stat_nup in zip(
            valid_positions,
            nue_iterator,
            area_iterator,
            nup_iterator
        ):

            country_name = records[pos]["COUNTRY"]

            nue_mini = stat_nue.get(
                "mini_raster_array"
            )

            area_mini = (
                stat_area.get("mini_raster_array")
                if stat_area is not None
                else None
            )

            nup_mini = (
                stat_nup.get("mini_raster_array")
                if stat_nup is not None
                else None
            )

            # -----------------------------------------------
            # Methods 1-3: Original country statistics
            # -----------------------------------------------

            if nue_mini is not None:

                metrics = compute_country_metrics(
                    nue_mini,
                    config
                )

                # Preserve rasterstats original mean
                if (
                    config["enable_all"]
                    and stat_nue.get("mean") is not None
                ):

                    original_mean = float(
                        stat_nue["mean"]
                    )

                    if np.isfinite(original_mean):
                        metrics["value"] = original_mean

                records[pos].update(metrics)

            # -----------------------------------------------
            # Check NUE / Area mini-raster alignment
            # -----------------------------------------------

            if stat_area is not None:

                check_mini_alignment(
                    stat_nue,
                    stat_area,
                    year,
                    country_name,
                    "NUE / Area"
                )

            # -----------------------------------------------
            # Method 4: Original area-weighted mean
            # -----------------------------------------------

            if (
                config["enable_weighted"]
                and area_mini is not None
            ):

                weighted_metrics = (
                    compute_area_weighted_mean(
                        nue_mini,
                        area_mini
                    )
                )

                records[pos].update(
                    weighted_metrics
                )

            # -----------------------------------------------
            # Method 5: NEW Nup-based regional NUE
            # -----------------------------------------------

            if config["enable_ratio"]:

                if (
                    stat_area is not None
                    and stat_nup is not None
                ):

                    check_mini_alignment(
                        stat_nue,
                        stat_nup,
                        year,
                        country_name,
                        "NUE / Nup"
                    )

                    # Both Nup and Area use same
                    # country geometry / pixel grid.
                    check_mini_alignment(
                        stat_area,
                        stat_nup,
                        year,
                        country_name,
                        "Area / Nup"
                    )

                    ratio_metrics = compute_nup_based_nue(
                        nue_mini,
                        nup_mini,
                        area_mini
                    )

                    records[pos].update(
                        ratio_metrics
                    )

    elapsed = time.perf_counter() - t0

    print(
        f"[{year}] Completed | "
        f"Countries={len(records)} | "
        f"Valid NUE pixels={nue_valid_count:,} | "
        f"Time={elapsed:.1f}s",
        flush=True
    )

    return year, records, nue_valid_count


# ============================================================
# 11. MAIN PROCESSING
# ============================================================

def run_country_mean_comparison(config):

    start_time = time.perf_counter()

    if N_WORKERS < 1:
        raise ValueError(
            "N_WORKERS must be >= 1"
        )

    if not (
        0 < config["keep_percent"] <= 100
    ):
        raise ValueError(
            "Keep percentile must be within (0,100]"
        )

    print("=" * 75)
    print("GLOBAL NUE - COUNTRY STATISTICS")
    print("=" * 75)

    print(
        f"1. Percentile mean: "
        f"{config['enable_trimmed']}"
    )

    print(
        f"2. STD mean: "
        f"{config['enable_std']} "
        f"{config['std_multipliers']}"
    )

    print(
        f"3. Arithmetic mean: "
        f"{config['enable_all']}"
    )

    print(
        f"4. HA-weighted mean: "
        f"{config['enable_weighted']}"
    )

    print(
        f"5. Nup-based ratio of sums: "
        f"{config['enable_ratio']}"
    )

    print(f"Crop: {CROP}")

    # --------------------------------------------------------
    # Load inputs
    # --------------------------------------------------------

    countries = load_countries()

    year_files = find_year_files(
        config
    )

    n_jobs = min(
        N_WORKERS,
        len(year_files)
    )

    print(f"Countries: {len(countries)}")
    print(f"Years: {[item[0] for item in year_files]}")
    print(f"Parallel workers: {n_jobs}")

    # --------------------------------------------------------
    # Parallel years
    # --------------------------------------------------------

    if n_jobs == 1:

        parallel_results = [
            process_one_year(
                year,
                nue_path,
                area_path,
                nup_path,
                countries,
                config
            )
            for year, nue_path, area_path, nup_path
            in year_files
        ]

    else:

        parallel_results = Parallel(
            n_jobs=n_jobs,
            backend="loky",
            verbose=10,
            batch_size=1,
            pre_dispatch="n_jobs"
        )(
            delayed(process_one_year)(
                year,
                nue_path,
                area_path,
                nup_path,
                countries,
                config
            )
            for year, nue_path, area_path, nup_path
            in year_files
        )

    # --------------------------------------------------------
    # Combine all years
    # --------------------------------------------------------

    parallel_results.sort(
        key=lambda item: item[0]
    )

    records = []

    for _, year_records, _ in parallel_results:
        records.extend(year_records)

    df = pd.DataFrame(records)

    # --------------------------------------------------------
    # Original columns + NEW ratio columns
    # --------------------------------------------------------

    trim_tag = config["trim_tag"]

    output_columns = [
        "M49_Code",
        "COUNTRY",
        "year",

        # Original arithmetic mean
        "value",
        "Valid_Pixels",

        # Original percentile method
        f"Mean_Trimmed_{trim_tag}",
        f"N_Trimmed_{trim_tag}",

        # Original standard deviation methods
        "Mean_STD_2",
        "N_STD_2",
        "Mean_STD_3",
        "N_STD_3",

        # Original harvested-area-weighted method
        "Mean_Area_Weighted",
        "N_Area_Weighted",
        "Harvested_Area_Used",

        # NEW Nup-based regional NUE
        "NUE_Ratio_of_Sums",
        "N_Ratio_Common_Pixels",
        "Sum_Nup_x_HA",
        "Sum_Ninput_x_HA"
    ]

    for col in output_columns:

        if col not in df.columns:
            df[col] = np.nan

    df = df[output_columns]

    # --------------------------------------------------------
    # Save Excel
    # --------------------------------------------------------

    os.makedirs(
        os.path.dirname(OUT_EXCEL),
        exist_ok=True
    )

    if len(df) > 1_048_575:
        raise ValueError(
            "Excel worksheet row limit exceeded"
        )

    df.to_excel(
        OUT_EXCEL,
        index=False,
        engine="openpyxl",
        sheet_name="Country Mean Comparison"
    )

    # --------------------------------------------------------
    # Summary
    # --------------------------------------------------------

    elapsed = (
        time.perf_counter() - start_time
    )

    print("\n" + "=" * 75)
    print("COUNTRY STATISTICS COMPLETED")
    print("=" * 75)

    print(f"Total records: {len(df):,}")

    for col in [
        "value",
        f"Mean_Trimmed_{trim_tag}",
        "Mean_STD_2",
        "Mean_STD_3",
        "Mean_Area_Weighted",
        "NUE_Ratio_of_Sums"
    ]:

        print(
            f"{col}: "
            f"{df[col].notna().sum():,} "
            "valid country-year records"
        )

    print(f"\nTotal time: {elapsed:.2f}s")
    print(f"Excel saved: {OUT_EXCEL}")

    display(df.head(15))

    return df


# ============================================================
# 12. JUPYTERLAB INTERACTIVE CONTROLS
# ============================================================

# ------------------------------------------------------------
# Option 1: Percentile trimmed mean
# ------------------------------------------------------------

checkbox_trimmed = widgets.Checkbox(
    value=False,
    description="Option 1: Percentile Trimmed Mean",
    indent=False,
    layout=widgets.Layout(width="380px")
)

percent_widget = widgets.BoundedFloatText(
    value=95.0,
    min=1.0,
    max=100.0,
    step=1.0,
    description="Keep (%)",
    layout=widgets.Layout(width="230px")
)

# ------------------------------------------------------------
# Option 2: Standard deviation mean
# ------------------------------------------------------------

checkbox_std = widgets.Checkbox(
    value=False,
    description="Option 2: Standard Deviation Mean",
    indent=False,
    layout=widgets.Layout(width="380px")
)

checkbox_std2 = widgets.Checkbox(
    value=True,
    description="2 sigma",
    indent=False
)

checkbox_std3 = widgets.Checkbox(
    value=False,
    description="3 sigma",
    indent=False
)

# ------------------------------------------------------------
# Option 3: Arithmetic mean
# ------------------------------------------------------------

checkbox_all = widgets.Checkbox(
    value=True,
    description="Option 3: Arithmetic Mean",
    indent=False,
    layout=widgets.Layout(width="380px")
)

# ------------------------------------------------------------
# Option 4: Harvested-area-weighted mean
# ------------------------------------------------------------

checkbox_weighted = widgets.Checkbox(
    value=True,
    description="Option 4: Harvested-Area-Weighted Mean",
    indent=False,
    layout=widgets.Layout(width="450px")
)

# ------------------------------------------------------------
# Option 5: NEW Nup-based NUE
# ------------------------------------------------------------

checkbox_ratio = widgets.Checkbox(
    value=True,
    description="Option 5: Nup-based NUE (Ratio of Sums)",
    indent=False,
    layout=widgets.Layout(width="450px")
)

# ------------------------------------------------------------
# Run button
# ------------------------------------------------------------

run_button = widgets.Button(
    description="Run All Selected Methods",
    button_style="success",
    icon="play",
    layout=widgets.Layout(width="280px")
)

output_widget = widgets.Output()


# ============================================================
# 13. RUN BUTTON CALLBACK
# ============================================================

def on_run_clicked(button):

    enable_trimmed = checkbox_trimmed.value
    enable_std = checkbox_std.value
    enable_all = checkbox_all.value
    enable_weighted = checkbox_weighted.value
    enable_ratio = checkbox_ratio.value

    if not any([
        enable_trimmed,
        enable_std,
        enable_all,
        enable_weighted,
        enable_ratio
    ]):

        with output_widget:
            clear_output(wait=True)
            print("ERROR: Select at least one method.")
        return

    std_multipliers = []

    if checkbox_std2.value:
        std_multipliers.append(2)

    if checkbox_std3.value:
        std_multipliers.append(3)

    if enable_std and not std_multipliers:

        with output_widget:
            clear_output(wait=True)
            print(
                "ERROR: Select 2 sigma and/or 3 sigma."
            )
        return

    keep_percent = float(
        percent_widget.value
    )

    trim_tag = (
        f"{keep_percent:g}"
        .replace(".", "p")
    )

    config = {
        "enable_trimmed": enable_trimmed,
        "keep_percent": keep_percent,
        "trim_tag": trim_tag,

        "enable_std": enable_std,
        "std_multipliers": std_multipliers,

        "enable_all": enable_all,

        "enable_weighted": enable_weighted,

        "enable_ratio": enable_ratio
    }

    run_button.disabled = True

    with output_widget:

        clear_output(wait=True)

        try:

            global country_mean_results

            country_mean_results = (
                run_country_mean_comparison(
                    config
                )
            )

        except Exception:

            traceback.print_exc()

        finally:

            run_button.disabled = False


run_button.on_click(on_run_clicked)


# ============================================================
# 14. DISPLAY JUPYTER CONTROL PANEL
# ============================================================

control_panel = widgets.VBox([

    widgets.HTML(
        "<h3>Global NUE - Country Statistics Comparison</h3>"
    ),

    checkbox_trimmed,
    percent_widget,

    widgets.HTML("<hr>"),

    checkbox_std,

    widgets.HBox([
        checkbox_std2,
        checkbox_std3
    ]),

    widgets.HTML("<hr>"),

    checkbox_all,

    widgets.HTML("<hr>"),

    checkbox_weighted,

    widgets.HTML("<hr>"),

    checkbox_ratio,

    widgets.HTML("<hr>"),

    run_button
])

display(control_panel)
display(output_widget)

print(
    "Select one or more methods and click "
    "'Run All Selected Methods'."
)

In [ ]:
# ================V 2026.10.26============================================
# Global NUE - Parallel Country Mean Comparison c4HAWM
#
# Option 1: Percentile Trimmed Mean
# Option 2: Standard Deviation Mean (2/3 sigma)
# Option 3: Arithmetic Mean
# Option 4: Harvested-Area-Weighted Mean (NEW)
#
# Weighted mean:
#   Only common valid pixels of NUE and Harvested Area
#   Area > 0
#   Within the same country
#
# Features:
#   - Interactive JupyterLab controls
#   - Parallel years (joblib loky)
#   - NaN / NoData / Inf handling
#   - No interpolation or resampling
#   - Same country boundary and pixel selection
#   - Excel comparison output
# ============================================================

import os
import glob
import re
import time
import traceback

from pathlib import Path
from itertools import repeat

import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
import ipywidgets as widgets

from rasterstats import gen_zonal_stats
from joblib import Parallel, delayed
from IPython.display import display, clear_output


# ============================================================
# 1. USER CONFIGURATION
# ============================================================

SHP_PATH = (
    r"Z:/staff/Qiang_S/09_Datasets/"
    r"09OtherDataUsed/GDAM_GlobalCountryLevel_Shp/"
    r"GlobalCountryLevel_Shp.shp"
)

# ------------------------------------------------------------
# NUE GeoTIFF folder
# ------------------------------------------------------------

TIF_FOLDER = (
    #r"Z:\staff\Qiang_S\09_Datasets\03GlobalNUE_Results\03Global_NUE_Rice\03RiceNUE_M2_20261026V1_M6_150R"
    r"Z:\staff\Qiang_S\09_Datasets\03GlobalNUE_Results\01Global_NUE_Wheat\02WheatNUE_M2_20261026V1_M6_150R"
)

# ------------------------------------------------------------
# Harvested Area GeoTIFF folder
# CHANGE THIS PATH TO YOUR ACTUAL AREA DATA LOCATION
# ------------------------------------------------------------

AREA_FOLDER = (
    #r"Z:\staff\Qiang_S\09_Datasets\09OtherDataUsed\GCSN_HarvestedArea_1995_2020\02StandardizedDatasets\02MaizeHarvestedArea"
    r"Z:\staff\Qiang_S\09_Datasets\09OtherDataUsed\GCSN_HarvestedArea_1995_2020\02StandardizedDatasets\01WheatHarvestedArea"
    #r"Z:\staff\Qiang_S\09_Datasets\09OtherDataUsed\GCSN_HarvestedArea_1995_2020\02StandardizedDatasets\03RiceHarvestedArea"
)

# Crop name used in Harvested Area filenames
CROP = "Wheat"

# Example:
# Maize_HarArea_1995.tif
# Maize_HarArea_1996.tif
# Maize_HarArea_2020.tif

AREA_FILE_PREFIX = f"{CROP}_HarArea_"

# ------------------------------------------------------------
# Output Excel
# ------------------------------------------------------------

OUT_EXCEL = os.path.join(
    TIF_FOLDER,
    f"{CROP}_NUE_Country_Mean_Comparison.xlsx"
)

# ------------------------------------------------------------
# NoData settings
# ------------------------------------------------------------

NODATA = -9999.0

ADDITIONAL_NODATA = []

# ------------------------------------------------------------
# Parallel years
# ------------------------------------------------------------

N_WORKERS = 10

# None = all available years
# Example: [1995, 1996, 2020]
YEARS_TO_RUN = None

# ------------------------------------------------------------
# Country statistics settings
# ------------------------------------------------------------

ALL_TOUCHED = False

# Apply percentile / STD filtering only if N > 10
MIN_PIXELS_FOR_FILTER = 10

# ------------------------------------------------------------
# Harvested Area settings
# ------------------------------------------------------------

# Only positive harvested area pixels are used
AREA_THRESHOLD = 0.0

# True: Require an Area file for every processed NUE year
REQUIRE_AREA_MATCH = True

# ============================================================
# 2. READ AND CLEAN RASTER
# ============================================================

def read_clean_raster(src):

    """
    Recognize:
      - GeoTIFF metadata NoData
      - NaN
      - Inf / -Inf
      - -9999
      - Additional NoData

    Invalid pixels -> NODATA
    """

    masked = src.read(
        1,
        masked=True
    )

    data = np.array(
        masked.astype(np.float32).filled(np.nan),
        dtype=np.float32,
        copy=True
    )

    invalid = (
        ~np.isfinite(data)
        | (data == NODATA)
    )

    for nd in ADDITIONAL_NODATA:
        invalid |= (data == nd)

    data[invalid] = NODATA

    valid_count = int(
        np.count_nonzero(~invalid)
    )

    return data, valid_count

# ============================================================
# 3. CHECK GRID ALIGNMENT
# ============================================================

def check_grid_alignment(src_nue, src_area):

    """
    Ensure both rasters have exactly the same pixel grid.
    No implicit reprojection or resampling.
    """

    if src_nue.crs is None or src_area.crs is None:
        raise ValueError("Missing raster CRS")

    if src_nue.crs != src_area.crs:
        raise ValueError(
            "NUE and Harvested Area CRS mismatch"
        )

    if (
        src_nue.width != src_area.width
        or src_nue.height != src_area.height
    ):
        raise ValueError(
            "NUE and Harvested Area dimensions mismatch"
        )

    if not src_nue.transform.almost_equals(
        src_area.transform,
        precision=1e-10
    ):
        raise ValueError(
            "NUE and Harvested Area raster grids "
            "are not aligned"
        )


# ============================================================
# 4. COMPUTE ORIGINAL COUNTRY METRICS
# ============================================================

def compute_country_metrics(masked_array, config):

    values = np.ma.asarray(
        masked_array
    ).compressed()

    values = np.asarray(
        values,
        dtype=np.float64
    )

    valid = (
        np.isfinite(values)
        & (values != NODATA)
    )

    for nd in ADDITIONAL_NODATA:
        valid &= (values != nd)

    values = values[valid]

    n = values.size

    trim_tag = config["trim_tag"]

    result = {
        "Valid_Pixels": int(n),
        "value": None,
        f"Mean_Trimmed_{trim_tag}": None,
        f"N_Trimmed_{trim_tag}": None,
        "Mean_STD_2": None,
        "N_STD_2": None,
        "Mean_STD_3": None,
        "N_STD_3": None
    }

    if n == 0:
        return result

    # --------------------------------------------------------
    # Option 3: Arithmetic mean
    # --------------------------------------------------------

    original_mean = float(
        np.mean(
            values,
            dtype=np.float64
        )
    )

    if config["enable_all"]:
        result["value"] = original_mean

    # --------------------------------------------------------
    # Option 1: Percentile trimmed mean
    # --------------------------------------------------------

    if config["enable_trimmed"]:

        mean_col = f"Mean_Trimmed_{trim_tag}"
        count_col = f"N_Trimmed_{trim_tag}"

        if n <= MIN_PIXELS_FOR_FILTER:

            result[mean_col] = original_mean
            result[count_col] = int(n)

        else:

            tail = (
                100.0 - config["keep_percent"]
            ) / 2.0

            p_low, p_high = np.percentile(
                values,
                [tail, 100.0 - tail]
            )

            retained = values[
                (values >= p_low)
                & (values <= p_high)
            ]

            result[count_col] = int(
                retained.size
            )

            if retained.size > 0:

                result[mean_col] = float(
                    np.mean(
                        retained,
                        dtype=np.float64
                    )
                )

    # --------------------------------------------------------
    # Option 2: Standard deviation mean
    # --------------------------------------------------------

    if config["enable_std"]:

        if n <= MIN_PIXELS_FOR_FILTER:

            for k in config["std_multipliers"]:

                result[f"Mean_STD_{k}"] = original_mean
                result[f"N_STD_{k}"] = int(n)

        else:

            std_val = float(
                np.std(
                    values,
                    dtype=np.float64,
                    ddof=0
                )
            )

            for k in config["std_multipliers"]:

                lower = original_mean - k * std_val
                upper = original_mean + k * std_val

                retained = values[
                    (values >= lower)
                    & (values <= upper)
                ]

                result[f"N_STD_{k}"] = int(
                    retained.size
                )

                if retained.size > 0:

                    result[f"Mean_STD_{k}"] = float(
                        np.mean(
                            retained,
                            dtype=np.float64
                        )
                    )

    return result


# ============================================================
# 5. NEW: HARVESTED-AREA-WEIGHTED MEAN
# ============================================================

def compute_area_weighted_mean(
    nue_masked,
    area_masked
):

    """
    Calculate harvested-area-weighted mean
    using common valid pixels only.

    Conditions:
      - Both pixels are inside the same country
      - NUE is valid
      - Harvested Area is valid
      - Harvested Area > AREA_THRESHOLD

    No interpolation / filling.

    Returns:
      Mean_Area_Weighted
      N_Area_Weighted
      Harvested_Area_Used
    """

    output = {
        "Mean_Area_Weighted": None,
        "N_Area_Weighted": 0,
        "Harvested_Area_Used": 0.0
    }

    if nue_masked is None or area_masked is None:
        return output

    nue_ma = np.ma.asarray(nue_masked)
    area_ma = np.ma.asarray(area_masked)

    if nue_ma.shape != area_ma.shape:
        raise ValueError(
            "Country NUE and Area mini rasters "
            "have different shapes"
        )

    nue_values = np.asarray(
        np.ma.getdata(nue_ma),
        dtype=np.float64
    )

    area_values = np.asarray(
        np.ma.getdata(area_ma),
        dtype=np.float64
    )

    # --------------------------------------------------------
    # Valid NUE pixels
    # --------------------------------------------------------

    nue_valid = (
        ~np.ma.getmaskarray(nue_ma)
        & np.isfinite(nue_values)
        & (nue_values != NODATA)
    )

    # --------------------------------------------------------
    # Valid Harvested Area pixels
    # --------------------------------------------------------

    area_valid = (
        ~np.ma.getmaskarray(area_ma)
        & np.isfinite(area_values)
        & (area_values != NODATA)
        & (area_values > AREA_THRESHOLD)
    )

    for nd in ADDITIONAL_NODATA:

        nue_valid &= (nue_values != nd)
        area_valid &= (area_values != nd)

    # --------------------------------------------------------
    # Common valid pixels
    # --------------------------------------------------------

    common_mask = nue_valid & area_valid

    n_common = int(
        np.count_nonzero(common_mask)
    )

    output["N_Area_Weighted"] = n_common

    if n_common == 0:
        return output

    # Extract corresponding values
    nue_common = nue_values[common_mask]
    area_common = area_values[common_mask]

    # --------------------------------------------------------
    # Calculate weighted mean
    # --------------------------------------------------------

    total_area = float(
        np.sum(
            area_common,
            dtype=np.float64
        )
    )

    output["Harvested_Area_Used"] = total_area

    if total_area <= 0:
        return output

    weighted_sum = float(
        np.sum(
            nue_common * area_common,
            dtype=np.float64
        )
    )

    output["Mean_Area_Weighted"] = (
        weighted_sum / total_area
    )

    return output


# ============================================================
# 6. LOAD COUNTRY SHAPEFILE
# ============================================================

def load_countries():

    if not os.path.isfile(SHP_PATH):
        raise FileNotFoundError(SHP_PATH)

    gdf = gpd.read_file(SHP_PATH)

    if gdf.crs is None:
        raise ValueError("Shapefile CRS missing")

    gdf = gdf.to_crs("EPSG:4326")

    gdf = gdf.reset_index(drop=True)

    for field in ["M49_Code", "COUNTRY"]:

        if field not in gdf.columns:
            raise ValueError(
                f"Missing field: {field}"
            )

    return gdf


# ============================================================
# 7. FIND NUE AND HARVESTED AREA FILES
# ============================================================

def extract_year(filename):

    match = re.search(
        r"_(19\d{2}|20\d{2})\.(?:tif|tiff)$",
        Path(filename).name,
        re.IGNORECASE
    )

    if match:
        return int(match.group(1))

    return None


def find_year_files(config):

    if not os.path.isdir(TIF_FOLDER):
        raise FileNotFoundError(TIF_FOLDER)

    # --------------------------------------------------------
    # NUE files
    # --------------------------------------------------------

    nue_files = {}

    for path in sorted(Path(TIF_FOLDER).iterdir()):

        if not path.is_file():
            continue

        if path.suffix.lower() not in (".tif", ".tiff"):
            continue

        year = extract_year(path.name)

        if year is None:
            continue

        if (
            YEARS_TO_RUN is not None
            and year not in YEARS_TO_RUN
        ):
            continue

        if year in nue_files:
            raise ValueError(
                f"Duplicate NUE year: {year}"
            )

        nue_files[year] = str(path)

    if not nue_files:
        raise RuntimeError(
            "No matching NUE GeoTIFF files found"
        )

    # --------------------------------------------------------
    # Harvested Area files
    # --------------------------------------------------------

    area_files = {}

    if config["enable_weighted"]:

        if not os.path.isdir(AREA_FOLDER):
            raise FileNotFoundError(
                f"Area folder not found: {AREA_FOLDER}"
            )

        # Recursive search supports crop subfolders
        for path in sorted(
            Path(AREA_FOLDER).rglob("*")
        ):

            if not path.is_file():
                continue

            if path.suffix.lower() not in (".tif", ".tiff"):
                continue

            if not path.name.lower().startswith(
                AREA_FILE_PREFIX.lower()
            ):
                continue

            year = extract_year(path.name)

            if year is None:
                continue

            if year in area_files:
                raise ValueError(
                    f"Duplicate Area year {year}: "
                    f"{area_files[year]} and {path}"
                )

            area_files[year] = str(path)

    # --------------------------------------------------------
    # Match files by year
    # --------------------------------------------------------

    year_files = []

    for year in sorted(nue_files):

        area_path = area_files.get(year)

        if (
            config["enable_weighted"]
            and REQUIRE_AREA_MATCH
            and area_path is None
        ):
            raise FileNotFoundError(
                f"Missing Harvested Area for {year}"
            )

        year_files.append(
            (
                year,
                nue_files[year],
                area_path
            )
        )

    return year_files


# ============================================================
# 8. PROCESS ONE YEAR
# ============================================================

def process_one_year(
    year,
    nue_tif,
    area_tif,
    countries,
    config
):

    t0 = time.perf_counter()

    # --------------------------------------------------------
    # Read NUE
    # --------------------------------------------------------

    with rasterio.open(nue_tif) as src_nue:

        if src_nue.count != 1:
            raise ValueError(
                "NUE GeoTIFF must be single-band"
            )

        if src_nue.crs is None:
            raise ValueError("NUE raster CRS missing")

        raster_crs = src_nue.crs
        affine = src_nue.transform

        nue_data, nue_valid_count = read_clean_raster(
            src_nue
        )

        # ----------------------------------------------------
        # Read Harvested Area if enabled
        # ----------------------------------------------------

        area_data = None

        if (
            config["enable_weighted"]
            and area_tif is not None
        ):

            with rasterio.open(area_tif) as src_area:

                if src_area.count != 1:
                    raise ValueError(
                        "Area GeoTIFF must be single-band"
                    )

                check_grid_alignment(
                    src_nue,
                    src_area
                )

                area_data, _ = read_clean_raster(
                    src_area
                )

    # --------------------------------------------------------
    # Country CRS
    # --------------------------------------------------------

    if countries.crs != raster_crs:
        gdf_year = countries.to_crs(raster_crs)
    else:
        gdf_year = countries

    # --------------------------------------------------------
    # Initialize all country records
    # --------------------------------------------------------

    records = []

    for _, row in gdf_year.iterrows():

        records.append({
            "M49_Code": row["M49_Code"],
            "COUNTRY": row["COUNTRY"],
            "year": year,
            "Valid_Pixels": 0,
            "Mean_Area_Weighted": None,
            "N_Area_Weighted": 0,
            "Harvested_Area_Used": 0.0
        })

    # --------------------------------------------------------
    # Identify valid geometries
    # --------------------------------------------------------

    valid_positions = []

    for i, geom in enumerate(gdf_year.geometry):

        if geom is not None and not geom.is_empty:
            valid_positions.append(i)

    geometries = (
        gdf_year.geometry
        .iloc[valid_positions]
        .tolist()
    )

    # --------------------------------------------------------
    # Zonal statistics: same country boundaries
    # --------------------------------------------------------

    if geometries:

        # NUE zonal statistics
        nue_iterator = gen_zonal_stats(
            geometries,
            nue_data,
            affine=affine,
            stats=["mean"],
            nodata=NODATA,
            all_touched=ALL_TOUCHED,
            geojson_out=False,
            raster_out=True
        )

        # Area zonal statistics:
        # Use identical geometries and affine transform

        if area_data is not None:

            area_iterator = gen_zonal_stats(
                geometries,
                area_data,
                affine=affine,
                stats=["count"],
                nodata=NODATA,
                all_touched=ALL_TOUCHED,
                geojson_out=False,
                raster_out=True
            )

        else:

            area_iterator = repeat(None)

        # ----------------------------------------------------
        # Process each country
        # ----------------------------------------------------

        for pos, stat_nue, stat_area in zip(
            valid_positions,
            nue_iterator,
            area_iterator
        ):

            nue_mini = stat_nue.get(
                "mini_raster_array"
            )

            # -----------------------------------------------
            # Original statistical methods
            # -----------------------------------------------

            if nue_mini is not None:

                metrics = compute_country_metrics(
                    nue_mini,
                    config
                )

                # Preserve original rasterstats mean
                if (
                    config["enable_all"]
                    and stat_nue.get("mean") is not None
                ):

                    original_mean = float(
                        stat_nue["mean"]
                    )

                    if np.isfinite(original_mean):
                        metrics["value"] = original_mean

                records[pos].update(metrics)

            # -----------------------------------------------
            # NEW: Harvested-Area-Weighted Mean
            # -----------------------------------------------

            if stat_area is not None:

                area_mini = stat_area.get(
                    "mini_raster_array"
                )

                # Check mini-raster spatial alignment
                nue_aff = stat_nue.get(
                    "mini_raster_affine"
                )

                area_aff = stat_area.get(
                    "mini_raster_affine"
                )

                if (
                    nue_aff is not None
                    and area_aff is not None
                    and not nue_aff.almost_equals(
                        area_aff,
                        precision=1e-10
                    )
                ):
                    raise ValueError(
                        f"{year}: Country mini-raster "
                        "alignment mismatch"
                    )

                weighted_metrics = (
                    compute_area_weighted_mean(
                        nue_mini,
                        area_mini
                    )
                )

                records[pos].update(
                    weighted_metrics
                )

    elapsed = time.perf_counter() - t0

    print(
        f"[{year}] Completed | "
        f"Countries={len(records)} | "
        f"NUE valid pixels={nue_valid_count:,} | "
        f"Time={elapsed:.1f}s",
        flush=True
    )

    return year, records, nue_valid_count


# ============================================================
# 9. MAIN PROCESSING
# ============================================================

def run_country_mean_comparison(config):

    start_time = time.perf_counter()

    print("=" * 70)
    print("GLOBAL NUE - COUNTRY MEAN COMPARISON")
    print("=" * 70)

    print(f"Arithmetic Mean: {config['enable_all']}")
    print(f"Trimmed Mean: {config['enable_trimmed']}")
    print(f"STD Mean: {config['enable_std']}")
    print(f"Area Weighted Mean: {config['enable_weighted']}")
    print(f"Crop: {CROP}")

    countries = load_countries()

    year_files = find_year_files(config)

    n_jobs = min(
        N_WORKERS,
        len(year_files)
    )

    print(f"Countries: {len(countries)}")
    print(f"Years: {[x[0] for x in year_files]}")
    print(f"Workers: {n_jobs}")

    # --------------------------------------------------------
    # Parallel processing by year
    # --------------------------------------------------------

    if n_jobs == 1:

        parallel_results = [
            process_one_year(
                year,
                nue_tif,
                area_tif,
                countries,
                config
            )
            for year, nue_tif, area_tif in year_files
        ]

    else:

        parallel_results = Parallel(
            n_jobs=n_jobs,
            backend="loky",
            verbose=10,
            batch_size=1,
            pre_dispatch="n_jobs"
        )(
            delayed(process_one_year)(
                year,
                nue_tif,
                area_tif,
                countries,
                config
            )
            for year, nue_tif, area_tif in year_files
        )

    # --------------------------------------------------------
    # Combine years
    # --------------------------------------------------------

    parallel_results.sort(
        key=lambda x: x[0]
    )

    records = []

    for _, year_records, _ in parallel_results:
        records.extend(year_records)

    df = pd.DataFrame(records)

    # --------------------------------------------------------
    # Output columns
    # --------------------------------------------------------

    trim_tag = config["trim_tag"]

    output_columns = [
        "M49_Code",
        "COUNTRY",
        "year",
        "value",
        "Valid_Pixels",
        f"Mean_Trimmed_{trim_tag}",
        f"N_Trimmed_{trim_tag}",
        "Mean_STD_2",
        "N_STD_2",
        "Mean_STD_3",
        "N_STD_3",

        # NEW columns
        "Mean_Area_Weighted",
        "N_Area_Weighted",
        "Harvested_Area_Used"
    ]

    for col in output_columns:

        if col not in df.columns:
            df[col] = np.nan

    df = df[output_columns]

    # --------------------------------------------------------
    # Export Excel
    # --------------------------------------------------------

    os.makedirs(
        os.path.dirname(OUT_EXCEL),
        exist_ok=True
    )

    df.to_excel(
        OUT_EXCEL,
        index=False,
        engine="openpyxl",
        sheet_name="Country Mean Comparison"
    )

    # --------------------------------------------------------
    # Summary
    # --------------------------------------------------------

    elapsed = time.perf_counter() - start_time

    print("\n" + "=" * 70)
    print("COMPARISON COMPLETED")
    print("=" * 70)

    print(f"Total records: {len(df):,}")

    for col in [
        "value",
        f"Mean_Trimmed_{trim_tag}",
        "Mean_STD_2",
        "Mean_STD_3",
        "Mean_Area_Weighted"
    ]:

        print(
            f"{col}: "
            f"{df[col].notna().sum():,} valid records"
        )

    print(f"Total time: {elapsed:.2f}s")
    print(f"Output Excel: {OUT_EXCEL}")

    display(df.head(15))

    return df


# ============================================================
# 10. JUPYTERLAB INTERACTIVE CONTROLS
# ============================================================

# ------------------------------------------------------------
# Option 1: Percentile trimmed mean
# ------------------------------------------------------------

checkbox_trimmed = widgets.Checkbox(
    value=False,
    description="Option 1: Percentile Trimmed Mean",
    indent=False,
    layout=widgets.Layout(width="370px")
)

percent_widget = widgets.BoundedFloatText(
    value=95.0,
    min=1.0,
    max=100.0,
    step=1.0,
    description="Keep (%)",
    layout=widgets.Layout(width="230px")
)

# ------------------------------------------------------------
# Option 2: Standard deviation mean
# ------------------------------------------------------------

checkbox_std = widgets.Checkbox(
    value=False,
    description="Option 2: Standard Deviation Mean",
    indent=False,
    layout=widgets.Layout(width="370px")
)

checkbox_std2 = widgets.Checkbox(
    value=True,
    description="2 sigma",
    indent=False
)

checkbox_std3 = widgets.Checkbox(
    value=False,
    description="3 sigma",
    indent=False
)

# ------------------------------------------------------------
# Option 3: Arithmetic mean
# ------------------------------------------------------------

checkbox_all = widgets.Checkbox(
    value=True,
    description="Option 3: Arithmetic Mean",
    indent=False,
    layout=widgets.Layout(width="370px")
)

# ------------------------------------------------------------
# Option 4: Harvested-area-weighted mean
# ------------------------------------------------------------

checkbox_weighted = widgets.Checkbox(
    value=True,
    description="Option 4: Harvested-Area-Weighted Mean",
    indent=False,
    layout=widgets.Layout(width="430px")
)

# ------------------------------------------------------------
# Run button
# ------------------------------------------------------------

run_button = widgets.Button(
    description="Run All Selected Methods",
    button_style="success",
    icon="play",
    layout=widgets.Layout(width="270px")
)

output_widget = widgets.Output()


# ============================================================
# 11. BUTTON CALLBACK
# ============================================================

def on_run_clicked(button):

    enable_trimmed = checkbox_trimmed.value
    enable_std = checkbox_std.value
    enable_all = checkbox_all.value
    enable_weighted = checkbox_weighted.value

    std_multipliers = []

    if checkbox_std2.value:
        std_multipliers.append(2)

    if checkbox_std3.value:
        std_multipliers.append(3)

    if not any([
        enable_trimmed,
        enable_std,
        enable_all,
        enable_weighted
    ]):

        with output_widget:
            clear_output(wait=True)
            print("Select at least one method.")
        return

    if enable_std and not std_multipliers:

        with output_widget:
            clear_output(wait=True)
            print("Select 2 sigma and/or 3 sigma.")
        return

    keep_percent = float(percent_widget.value)

    trim_tag = (
        f"{keep_percent:g}"
        .replace(".", "p")
    )

    config = {
        "enable_trimmed": enable_trimmed,
        "keep_percent": keep_percent,
        "trim_tag": trim_tag,
        "enable_std": enable_std,
        "std_multipliers": std_multipliers,
        "enable_all": enable_all,
        "enable_weighted": enable_weighted
    }

    run_button.disabled = True

    with output_widget:

        clear_output(wait=True)

        try:

            global country_mean_results

            country_mean_results = (
                run_country_mean_comparison(
                    config
                )
            )

        except Exception:

            traceback.print_exc()

        finally:

            run_button.disabled = False


run_button.on_click(on_run_clicked)


# ============================================================
# 12. DISPLAY CONTROL PANEL
# ============================================================

control_panel = widgets.VBox([

    widgets.HTML(
        "<h3>Global NUE - Country Mean Comparison</h3>"
    ),

    checkbox_trimmed,
    percent_widget,

    widgets.HTML("<hr>"),

    checkbox_std,

    widgets.HBox([
        checkbox_std2,
        checkbox_std3
    ]),

    widgets.HTML("<hr>"),

    checkbox_all,

    widgets.HTML("<hr>"),

    checkbox_weighted,

    widgets.HTML("<hr>"),

    run_button
])

display(control_panel)
display(output_widget)

print(
    "Select one or more methods, "
    "then click Run All Selected Methods."
)

In [ ]:
# ============================================================
# Global NUE - Parallel Country Mean Comparison
# Mean three selectable methods:
#
# Option 1: Percentile Trimmed Mean (default 95%)
# Option 2: Standard Deviation Mean (2 sigma / 3 sigma)
# Option 3: Arithmetic Mean (default enabled)
#
# Multiple methods can run simultaneously.
#
# Features:
#   - Interactive JupyterLab controls
#   - Parallel processing across years
#   - One GeoTIFF read per year
#   - NaN / Inf / NoData / -9999 handling
#   - Country-based zonal statistics
#   - Minimum valid pixel condition (>10)
#   - Export comparison results into one Excel
#
# Requirements:
# numpy pandas geopandas rasterio rasterstats
# joblib openpyxl ipywidgets
# ============================================================

import os
import glob
import re
import time
import traceback

import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
import ipywidgets as widgets

from rasterstats import gen_zonal_stats
from joblib import Parallel, delayed
from IPython.display import display, clear_output


# ============================================================
# 1. USER CONFIGURATION
# ============================================================

SHP_PATH = (
    r"Z:/staff/Qiang_S/09_Datasets/"
    r"09OtherDataUsed/GDAM_GlobalCountryLevel_Shp/"
    r"GlobalCountryLevel_Shp.shp"
)

TIF_FOLDER = (
    r"Z:\staff\Qiang_S\09_Datasets"
    r"\03GlobalNUE_Results\02Global_NUE_Maize"
    r"\02MaizeNUE_M2_20261022V1_M6_160C"
)

OUT_EXCEL = os.path.join(
    TIF_FOLDER,
    "Maize_NUE_Country_Mean_Comparison.xlsx"
)

NODATA = -9999.0
ADDITIONAL_NODATA = []

# Parallel years
N_WORKERS = 10

# None = all years
# Example: [1995, 1996, 2020]
YEARS_TO_RUN = None

# Country pixel inclusion rule
ALL_TOUCHED = False

# STD / Percentile filtering only when N > 10
MIN_PIXELS_FOR_FILTER = 10

# ============================================================
# 2. STATISTICAL FUNCTIONS
# ============================================================

def compute_country_metrics(
    masked_array,
    config
):
    """
    Compute all selected methods from the SAME pixels.

    No raster modification.
    No iterative sigma clipping.

    Returns a dictionary of mean values and
    retained pixel counts.
    """

    values = np.ma.asarray(
        masked_array
    ).compressed()

    values = np.asarray(
        values,
        dtype=np.float64
    )

    # Extra protection
    valid = (
        np.isfinite(values)
        & (values != NODATA)
    )

    for nd in ADDITIONAL_NODATA:
        valid &= (values != nd)

    values = values[valid]

    n = values.size

    result = {
        "Valid_Pixels": int(n),
        "value": None
    }

    trim_tag = config["trim_tag"]

    trim_mean_col = f"Mean_Trimmed_{trim_tag}"
    trim_n_col = f"N_Trimmed_{trim_tag}"

    result[trim_mean_col] = None
    result[trim_n_col] = None

    for k in [2, 3]:
        result[f"Mean_STD_{k}"] = None
        result[f"N_STD_{k}"] = None

    if n == 0:
        return result

    # --------------------------------------------------------
    # Initial arithmetic mean
    # --------------------------------------------------------

    original_mean = float(
        np.mean(values, dtype=np.float64)
    )

    if config["enable_all"]:
        result["value"] = original_mean

    # --------------------------------------------------------
    # Option 1: Percentile Trimmed Mean
    # --------------------------------------------------------

    if config["enable_trimmed"]:

        if n <= MIN_PIXELS_FOR_FILTER:

            # Use original mean for small samples
            result[trim_mean_col] = original_mean
            result[trim_n_col] = int(n)

        else:

            keep_percent = config["keep_percent"]

            tail_percent = (
                100.0 - keep_percent
            ) / 2.0

            lower_p, upper_p = np.percentile(
                values,
                [
                    tail_percent,
                    100.0 - tail_percent
                ]
            )

            trimmed_values = values[
                (values >= lower_p)
                & (values <= upper_p)
            ]

            if trimmed_values.size > 0:

                result[trim_mean_col] = float(
                    np.mean(
                        trimmed_values,
                        dtype=np.float64
                    )
                )

                result[trim_n_col] = int(
                    trimmed_values.size
                )

            else:

                result[trim_n_col] = 0

    # --------------------------------------------------------
    # Option 2: Standard Deviation Mean
    # --------------------------------------------------------

    if config["enable_std"]:

        selected_multipliers = (
            config["std_multipliers"]
        )

        if n <= MIN_PIXELS_FOR_FILTER:

            for k in selected_multipliers:

                result[f"Mean_STD_{k}"] = original_mean
                result[f"N_STD_{k}"] = int(n)

        else:

            # Calculate initial STD only once
            std_val = float(
                np.std(
                    values,
                    dtype=np.float64,
                    ddof=0
                )
            )

            for k in selected_multipliers:

                lower_limit = (
                    original_mean - k * std_val
                )

                upper_limit = (
                    original_mean + k * std_val
                )

                sigma_values = values[
                    (values >= lower_limit)
                    & (values <= upper_limit)
                ]

                if sigma_values.size > 0:

                    result[f"Mean_STD_{k}"] = float(
                        np.mean(
                            sigma_values,
                            dtype=np.float64
                        )
                    )

                    result[f"N_STD_{k}"] = int(
                        sigma_values.size
                    )

                else:

                    result[f"N_STD_{k}"] = 0

    return result


# ============================================================
# 3. LOAD SHAPEFILE
# ============================================================

def load_countries():

    if not os.path.isfile(SHP_PATH):
        raise FileNotFoundError(SHP_PATH)

    gdf = gpd.read_file(SHP_PATH)

    if gdf.crs is None:
        raise ValueError("Shapefile CRS missing")

    gdf = gdf.to_crs("EPSG:4326")

    gdf = gdf.reset_index(drop=True)

    for field in ["M49_Code", "COUNTRY"]:

        if field not in gdf.columns:
            raise ValueError(
                f"Missing shapefile field: {field}"
            )

    return gdf


# ============================================================
# 4. FIND TIFF FILES
# ============================================================

def find_tif_files():

    tif_files = sorted(
        p for p in glob.glob(
            os.path.join(TIF_FOLDER, "*")
        )
        if p.lower().endswith((".tif", ".tiff"))
    )

    year_files = []

    for tif in tif_files:

        fname = os.path.basename(tif)

        match = re.search(
            r"(\d{4})\.(?:tif|tiff)$",
            fname,
            re.IGNORECASE
        )

        if not match:
            continue

        year = int(match.group(1))

        if (
            YEARS_TO_RUN is not None
            and year not in YEARS_TO_RUN
        ):
            continue

        year_files.append(
            (year, tif)
        )

    year_files.sort(
        key=lambda x: (x[0], x[1])
    )

    if not year_files:
        raise RuntimeError(
            "No matching GeoTIFF files found."
        )

    return year_files


# ============================================================
# 5. PROCESS ONE YEAR
# ============================================================

def process_one_year(
    year,
    tif,
    countries,
    config
):

    t0 = time.perf_counter()

    # --------------------------------------------------------
    # 5.1 Read and clean raster
    # --------------------------------------------------------

    with rasterio.open(tif) as src:

        if src.count != 1:
            raise ValueError(
                f"{tif}: Expected single-band raster"
            )

        if src.crs is None:
            raise ValueError(
                f"{tif}: Raster CRS missing"
            )

        raster_crs = src.crs
        affine = src.transform

        data = (
            src.read(1, masked=True)
            .astype(np.float32)
            .filled(np.nan)
        )

        data = np.asarray(
            data,
            dtype=np.float32
        )

    invalid = (
        ~np.isfinite(data)
        | (data == NODATA)
    )

    for nd in ADDITIONAL_NODATA:
        invalid |= (data == nd)

    data[invalid] = NODATA

    global_valid = int(
        np.count_nonzero(~invalid)
    )

    # --------------------------------------------------------
    # 5.2 Align country CRS
    # --------------------------------------------------------

    if countries.crs != raster_crs:
        gdf_year = countries.to_crs(
            raster_crs
        )
    else:
        gdf_year = countries

    # --------------------------------------------------------
    # 5.3 Initialize country records
    # --------------------------------------------------------

    records = []

    for _, row in gdf_year.iterrows():

        records.append({
            "M49_Code": row["M49_Code"],
            "COUNTRY": row["COUNTRY"],
            "year": year
        })

    # --------------------------------------------------------
    # 5.4 Identify valid geometries
    # --------------------------------------------------------

    valid_positions = []

    for i, geom in enumerate(
        gdf_year.geometry
    ):

        if geom is not None and not geom.is_empty:
            valid_positions.append(i)

    geometries = (
        gdf_year.geometry
        .iloc[valid_positions]
        .tolist()
    )

    # --------------------------------------------------------
    # 5.5 Country zonal statistics
    # --------------------------------------------------------

    # gen_zonal_stats processes one polygon at a time.
    #
    # raster_out=True provides the original raster values
    # belonging to each polygon.
    #
    # All selected statistical methods then use the same
    # valid pixels, avoiding repeated raster reads.

    if geometries:

        stats_iterator = gen_zonal_stats(
            geometries,
            data,
            affine=affine,
            stats=["mean"],
            nodata=NODATA,
            all_touched=ALL_TOUCHED,
            geojson_out=False,
            raster_out=True
        )

        for pos, stat in zip(
            valid_positions,
            stats_iterator
        ):

            country_data = stat.get(
                "mini_raster_array"
            )

            if country_data is None:
                continue

            metrics = compute_country_metrics(
                country_data,
                config
            )

            # Preserve the original rasterstats
            # arithmetic mean where requested.
            if (
                config["enable_all"]
                and stat.get("mean") is not None
            ):
                original_zonal_mean = float(
                    stat["mean"]
                )

                if np.isfinite(original_zonal_mean):
                    metrics["value"] = original_zonal_mean

            records[pos].update(metrics)

    # --------------------------------------------------------
    # 5.6 Finish
    # --------------------------------------------------------

    elapsed = time.perf_counter() - t0

    print(
        f"[{year}] Completed | "
        f"Countries={len(records)} | "
        f"Global valid pixels={global_valid:,} | "
        f"Time={elapsed:.1f}s",
        flush=True
    )

    return year, records, global_valid


# ============================================================
# 6. MAIN PROCESSING FUNCTION
# ============================================================

def run_country_mean_comparison(config):

    start_time = time.perf_counter()

    print("=" * 70)
    print("GLOBAL NUE - COUNTRY MEAN COMPARISON")
    print("=" * 70)

    print("Selected methods:")

    print(
        f"  Arithmetic mean: "
        f"{config['enable_all']}"
    )

    print(
        f"  Percentile trimmed mean: "
        f"{config['enable_trimmed']}"
    )

    if config["enable_trimmed"]:
        print(
            f"  Keep percentile: "
            f"{config['keep_percent']}%"
        )

    print(
        f"  STD mean: "
        f"{config['enable_std']}"
    )

    if config["enable_std"]:
        print(
            f"  STD multipliers: "
            f"{config['std_multipliers']}"
        )

    print(
        f"  Min pixels for filtering: "
        f"> {MIN_PIXELS_FOR_FILTER}"
    )

    print("=" * 70)

    # --------------------------------------------------------
    # Load inputs
    # --------------------------------------------------------

    countries = load_countries()

    year_files = find_tif_files()

    n_jobs = min(
        N_WORKERS,
        len(year_files)
    )

    print(f"Countries: {len(countries)}")
    print(f"GeoTIFF files: {len(year_files)}")
    print(f"Parallel workers: {n_jobs}")

    # --------------------------------------------------------
    # Parallel processing by year
    # --------------------------------------------------------

    if n_jobs == 1:

        parallel_results = [
            process_one_year(
                year,
                tif,
                countries,
                config
            )
            for year, tif in year_files
        ]

    else:

        parallel_results = Parallel(
            n_jobs=n_jobs,
            backend="loky",
            verbose=10,
            batch_size=1,
            pre_dispatch="n_jobs"
        )(
            delayed(process_one_year)(
                year,
                tif,
                countries,
                config
            )
            for year, tif in year_files
        )

    # --------------------------------------------------------
    # Merge results
    # --------------------------------------------------------

    parallel_results.sort(
        key=lambda x: x[0]
    )

    records = []

    for _, year_records, _ in parallel_results:
        records.extend(year_records)

    df = pd.DataFrame(records)

    # --------------------------------------------------------
    # Define output columns
    # --------------------------------------------------------

    trim_tag = config["trim_tag"]

    output_columns = [
        "M49_Code",
        "COUNTRY",
        "year",
        "value",
        "Valid_Pixels",
        f"Mean_Trimmed_{trim_tag}",
        f"N_Trimmed_{trim_tag}",
        "Mean_STD_2",
        "N_STD_2",
        "Mean_STD_3",
        "N_STD_3"
    ]

    # Add missing columns as empty values
    for col in output_columns:

        if col not in df.columns:
            df[col] = np.nan

    df = df[output_columns]

    # --------------------------------------------------------
    # Save Excel
    # --------------------------------------------------------

    os.makedirs(
        os.path.dirname(OUT_EXCEL),
        exist_ok=True
    )

    df.to_excel(
        OUT_EXCEL,
        index=False,
        engine="openpyxl",
        sheet_name="Country Mean Comparison"
    )

    # --------------------------------------------------------
    # Final report
    # --------------------------------------------------------

    elapsed = (
        time.perf_counter()
        - start_time
    )

    print("\n" + "=" * 70)
    print("COMPARISON COMPLETED")
    print("=" * 70)

    print(f"Total records: {len(df):,}")

    for col in [
        "value",
        f"Mean_Trimmed_{trim_tag}",
        "Mean_STD_2",
        "Mean_STD_3"
    ]:

        print(
            f"{col}: "
            f"{df[col].notna().sum():,} valid records"
        )

    print(f"\nTotal time: {elapsed:.2f}s")
    print(f"Output Excel: {OUT_EXCEL}")

    display(df.head(15))

    return df


# ============================================================
# 7. JUPYTERLAB INTERACTIVE CONTROLS
# ============================================================

# ------------------------------------------------------------
# Option 1: Percentile trimmed mean
# ------------------------------------------------------------

checkbox_trimmed = widgets.Checkbox(
    value=False,
    description="Option 1: Percentile Trimmed Mean",
    indent=False,
    layout=widgets.Layout(width="360px")
)

percent_widget = widgets.BoundedFloatText(
    value=95.0,
    min=1.0,
    max=100.0,
    step=1.0,
    description="Keep (%)",
    layout=widgets.Layout(width="220px")
)

# ------------------------------------------------------------
# Option 2: Standard deviation mean
# ------------------------------------------------------------

checkbox_std = widgets.Checkbox(
    value=False,
    description="Option 2: Standard Deviation Mean",
    indent=False,
    layout=widgets.Layout(width="360px")
)

checkbox_std2 = widgets.Checkbox(
    value=True,
    description="2 sigma",
    indent=False,
    layout=widgets.Layout(width="130px")
)

checkbox_std3 = widgets.Checkbox(
    value=False,
    description="3 sigma",
    indent=False,
    layout=widgets.Layout(width="130px")
)

# ------------------------------------------------------------
# Option 3: Arithmetic mean (default True)
# ------------------------------------------------------------

checkbox_all = widgets.Checkbox(
    value=True,
    description="Option 3: Arithmetic Mean (All Valid Pixels)",
    indent=False,
    layout=widgets.Layout(width="420px")
)

# ------------------------------------------------------------
# Run button
# ------------------------------------------------------------

run_button = widgets.Button(
    description="Run All Selected Methods",
    button_style="success",
    icon="play",
    layout=widgets.Layout(width="260px")
)

output_widget = widgets.Output()


# ============================================================
# 8. BUTTON CALLBACK
# ============================================================

def on_run_clicked(button):

    # Read selected modes
    enable_trimmed = checkbox_trimmed.value
    enable_std = checkbox_std.value
    enable_all = checkbox_all.value

    std_multipliers = []

    if checkbox_std2.value:
        std_multipliers.append(2)

    if checkbox_std3.value:
        std_multipliers.append(3)

    if not (
        enable_trimmed
        or enable_std
        or enable_all
    ):
        with output_widget:
            clear_output(wait=True)
            print("ERROR: Select at least one method.")
        return

    if enable_std and not std_multipliers:
        with output_widget:
            clear_output(wait=True)
            print(
                "ERROR: Select 2 sigma and/or 3 sigma."
            )
        return

    keep_percent = float(
        percent_widget.value
    )

    trim_tag = (
        f"{keep_percent:g}"
        .replace(".", "p")
    )

    # Configuration passed to all parallel workers
    config = {
        "enable_trimmed": enable_trimmed,
        "keep_percent": keep_percent,
        "trim_tag": trim_tag,
        "enable_std": enable_std,
        "std_multipliers": std_multipliers,
        "enable_all": enable_all
    }

    run_button.disabled = True

    with output_widget:

        clear_output(wait=True)

        try:

            # Results also available as:
            # country_mean_results
            global country_mean_results

            country_mean_results = (
                run_country_mean_comparison(
                    config
                )
            )

        except Exception:

            traceback.print_exc()

        finally:

            run_button.disabled = False


run_button.on_click(on_run_clicked)


# ============================================================
# 9. DISPLAY CONTROL PANEL
# ============================================================

control_panel = widgets.VBox([
    widgets.HTML(
        "<h3>Global NUE - Country Mean Comparison</h3>"
    ),

    checkbox_trimmed,
    percent_widget,

    widgets.HTML("<hr>"),

    checkbox_std,

    widgets.HBox([
        checkbox_std2,
        checkbox_std3
    ]),

    widgets.HTML("<hr>"),

    checkbox_all,

    widgets.HTML("<hr>"),

    run_button
])

display(control_panel)
display(output_widget)

print(
    "Select one or more methods, "
    "then click 'Run All Selected Methods'."
)


In [ ]:

# ============================================================
# Tool: Global NUE Annual Mean Comparison, mean4
#
# Method 1: Arithmetic Mean
# Method 2: 95% Percentile Trimmed Mean
# Method 3: Mean within +/- 2 Standard Deviations
# Method 4: Harvested-Area-Weighted NUE (NEW)
#
# Features:
# 1. Match NUE and harvested area by year
# 2. Verify identical spatial grids
# 3. Exclude NaN / Inf / NoData / -9999
# 4. Preserve original 3 mean calculations
# 5. Calculate harvested-area-weighted NUE
# 6. Export results to mean.txt
#
# JupyterLab - Single Cell
# ============================================================

import os
import re
import glob

import numpy as np
import rasterio

from pathlib import Path


# ============================================================
# 1. USER CONFIGURATION
# ============================================================

# ------------------------------------------------------------
# NUE GeoTIFF folder
# ------------------------------------------------------------

INPUT_FOLDER = (
    r"Z:\staff\Qiang_S\09_Datasets"
    r"\03GlobalNUE_Results\02Global_NUE_Maize"
    r"\02MaizeNUE_M2_20261022V1_M6_150R"
)

# ------------------------------------------------------------
# Harvested Area GeoTIFF folder
# IMPORTANT: Change to your actual folder
# ------------------------------------------------------------

AREA_FOLDER = (
    r"Z:\staff\Qiang_S\09_Datasets\09OtherDataUsed\GCSN_HarvestedArea_1995_2020\02StandardizedDatasets\02MaizeHarvestedArea"
    
)

# Crop selection
# "Wheat", "Maize", "Rice"
CROP = "Maize"

# Harvested Area filename examples:
# Maize_HarArea_1995.tif
# Maize_HarArea_1996.tif
# Maize_HarArea_2020.tif

AREA_FILENAME_PATTERN = (
    f"{CROP}_HarArea_*.tif"
)

# NUE file pattern
NUE_FILENAME_PATTERN = "*.tif"

# Output file
OUTPUT_TXT = os.path.join(
    INPUT_FOLDER,
    "mean3.txt"
)

# ------------------------------------------------------------
# NoData
# ------------------------------------------------------------

ADDITIONAL_NODATA = [-9999.0]

# ------------------------------------------------------------
# Percentile Trimmed Mean
# ------------------------------------------------------------

TRIM_PERCENT = 95.0

# ------------------------------------------------------------
# Standard deviation multiplier
# ------------------------------------------------------------

STD_MULTIPLIER = 2.0

# ------------------------------------------------------------
# Area-weighted NUE settings
# ------------------------------------------------------------

# Only harvested area > threshold is valid
AREA_THRESHOLD = 0.0

# True: Stop if a matching harvested area file is missing
REQUIRE_AREA_MATCH = True


# ============================================================
# 2. INPUT VALIDATION
# ============================================================

if not os.path.isdir(INPUT_FOLDER):
    raise FileNotFoundError(
        f"NUE folder not found:\n{INPUT_FOLDER}"
    )

if not os.path.isdir(AREA_FOLDER):
    raise FileNotFoundError(
        f"Harvested area folder not found:\n{AREA_FOLDER}"
    )

if not (0 < TRIM_PERCENT <= 100):
    raise ValueError(
        "TRIM_PERCENT must be within (0, 100]"
    )

if STD_MULTIPLIER <= 0:
    raise ValueError(
        "STD_MULTIPLIER must be positive"
    )


# ============================================================
# 3. UTILITY FUNCTIONS
# ============================================================

def extract_year(filepath):
    """
    Extract year from filename ending in _YYYY.tif.

    Examples:
      NUE_median_Opt_1995.tif -> 1995
      Maize_HarArea_1995.tif -> 1995
    """

    stem = Path(filepath).stem

    match = re.search(
        r"_(19\d{2}|20\d{2})$",
        stem
    )

    if match:
        return int(match.group(1))

    return None


def read_valid_raster(src):
    """
    Read raster and create valid pixel mask.

    Invalid values:
      - GeoTIFF metadata NoData
      - NaN
      - Inf / -Inf
      - Additional NoData values

    Return:
      values, valid_mask
    """

    data = src.read(
        1,
        masked=True
    )

    values = data.data

    valid = (
        ~np.ma.getmaskarray(data)
    ).copy()

    valid &= np.isfinite(values)

    for nd in ADDITIONAL_NODATA:
        valid &= (values != nd)

    return values, valid


def check_grid_alignment(src_nue, src_area):
    """
    Ensure NUE and harvested area rasters
    have the same spatial grid.

    No automatic resampling is performed.
    """

    if src_nue.crs is None or src_area.crs is None:
        raise ValueError(
            "Raster CRS is missing"
        )

    if src_nue.crs != src_area.crs:
        raise ValueError(
            "NUE and Area CRS mismatch"
        )

    if (
        src_nue.width != src_area.width
        or src_nue.height != src_area.height
    ):
        raise ValueError(
            "NUE and Area raster dimensions mismatch"
        )

    if not src_nue.transform.almost_equals(
        src_area.transform,
        precision=1e-10
    ):
        raise ValueError(
            "NUE and Area raster transforms mismatch"
        )


def format_value(value):
    """
    Format numerical results for TXT output.
    """

    if not np.isfinite(value):
        return "NaN"

    return f"{value:.6f}"


# ============================================================
# 4. COLLECT NUE FILES
# ============================================================

nue_files = sorted(
    glob.glob(
        os.path.join(
            INPUT_FOLDER,
            NUE_FILENAME_PATTERN
        )
    )
)

if not nue_files:
    raise RuntimeError(
        "No NUE GeoTIFF files found!"
    )

nue_year_files = {}

for tif in nue_files:

    year = extract_year(tif)

    if year is None:
        continue

    if year in nue_year_files:
        raise ValueError(
            f"Multiple NUE files for year {year}:\n"
            f"{nue_year_files[year]}\n{tif}"
        )

    nue_year_files[year] = tif


# ============================================================
# 5. COLLECT HARVESTED AREA FILES
# ============================================================

# Recursively search for harvested area GeoTIFF files

area_files = sorted(
    Path(AREA_FOLDER).rglob(
        AREA_FILENAME_PATTERN
    )
)

area_year_files = {}

for path in area_files:

    year = extract_year(path)

    if year is None:
        continue

    if year in area_year_files:
        raise ValueError(
            f"Duplicate harvested area for {year}:\n"
            f"{area_year_files[year]}\n{path}"
        )

    area_year_files[year] = str(path)


# ============================================================
# 6. INITIALIZATION
# ============================================================

years = sorted(
    nue_year_files.keys()
)

if not years:
    raise RuntimeError(
        "No valid NUE years found!"
    )

print("=" * 75)
print("GLOBAL NUE - ANNUAL MEAN COMPARISON")
print("=" * 75)

print(f"Crop: {CROP}")
print(f"NUE files: {len(nue_year_files)}")
print(f"Harvested area files: {len(area_year_files)}")

print(f"Years: {years}")

print(f"Trimmed Mean: {TRIM_PERCENT}%")
print(f"STD Multiplier: {STD_MULTIPLIER}")

print("Area Weighted Mean: Enabled")

print("=" * 75)

results = []


# ============================================================
# 7. LOOP THROUGH YEARS
# ============================================================

for year in years:

    nue_tif = nue_year_files[year]

    area_tif = area_year_files.get(year)

    print("\n" + "-" * 75)
    print(f"Processing year: {year}")

    # --------------------------------------------------------
    # 7.1 Check matching harvested area
    # --------------------------------------------------------

    if area_tif is None:

        if REQUIRE_AREA_MATCH:
            raise FileNotFoundError(
                f"No harvested area file for year {year}"
            )

        else:
            print(
                f"WARNING: Area file missing for {year}"
            )

    # --------------------------------------------------------
    # 7.2 Read NUE raster
    # --------------------------------------------------------

    with rasterio.open(nue_tif) as src_nue:

        nue_data, nue_valid = read_valid_raster(
            src_nue
        )

        total_pixels = nue_data.size

        valid_values = nue_data[
            nue_valid
        ].astype(
            np.float64,
            copy=False
        )

        valid_count = valid_values.size

        # ====================================================
        # Method 1: Arithmetic Mean
        # ====================================================

        if valid_count > 0:

            mean_val = float(
                np.mean(
                    valid_values,
                    dtype=np.float64
                )
            )

        else:

            mean_val = np.nan

        # ====================================================
        # Method 2: Percentile Trimmed Mean
        # ====================================================

        trimmed_mean = np.nan
        trimmed_count = 0

        if valid_count > 0:

            tail_percent = (
                100.0 - TRIM_PERCENT
            ) / 2.0

            p_low, p_high = np.percentile(
                valid_values,
                [
                    tail_percent,
                    100.0 - tail_percent
                ]
            )

            trimmed_mask = (
                (valid_values >= p_low)
                & (valid_values <= p_high)
            )

            trimmed_values = valid_values[
                trimmed_mask
            ]

            trimmed_count = trimmed_values.size

            if trimmed_count > 0:

                trimmed_mean = float(
                    np.mean(
                        trimmed_values,
                        dtype=np.float64
                    )
                )

        # ====================================================
        # Method 3: Mean within +/- k STD
        # ====================================================

        std_mean = np.nan
        std_count = 0

        if valid_count > 0:

            std_val = float(
                np.std(
                    valid_values,
                    dtype=np.float64,
                    ddof=0
                )
            )

            std_lower = (
                mean_val
                - STD_MULTIPLIER * std_val
            )

            std_upper = (
                mean_val
                + STD_MULTIPLIER * std_val
            )

            std_mask = (
                (valid_values >= std_lower)
                & (valid_values <= std_upper)
            )

            std_values = valid_values[
                std_mask
            ]

            std_count = std_values.size

            if std_count > 0:

                std_mean = float(
                    np.mean(
                        std_values,
                        dtype=np.float64
                    )
                )

        # ====================================================
        # Method 4: Harvested-Area-Weighted NUE
        # ====================================================

        area_weighted_mean = np.nan

        weighted_pixel_count = 0
        total_harvested_area = 0.0

        if area_tif is not None:

            with rasterio.open(
                area_tif
            ) as src_area:

                # Check spatial grid before calculation
                check_grid_alignment(
                    src_nue,
                    src_area
                )

                area_data, area_valid = read_valid_raster(
                    src_area
                )

                # --------------------------------------------
                # Common valid mask
                # --------------------------------------------

                common_mask = (
                    nue_valid
                    & area_valid
                    & (area_data > AREA_THRESHOLD)
                )

                weighted_pixel_count = int(
                    np.count_nonzero(common_mask)
                )

                # --------------------------------------------
                # Calculate weighted NUE
                # --------------------------------------------

                if weighted_pixel_count > 0:

                    nue_common = nue_data[
                        common_mask
                    ].astype(
                        np.float64,
                        copy=False
                    )

                    area_common = area_data[
                        common_mask
                    ].astype(
                        np.float64,
                        copy=False
                    )

                    # Sum harvested area
                    total_harvested_area = float(
                        np.sum(
                            area_common,
                            dtype=np.float64
                        )
                    )

                    if total_harvested_area > 0:

                        # Sum(NUE * Harvested Area)
                        weighted_sum = float(
                            np.sum(
                                nue_common * area_common,
                                dtype=np.float64
                            )
                        )

                        area_weighted_mean = (
                            weighted_sum
                            / total_harvested_area
                        )

    # ========================================================
    # 8. PRINT RESULTS
    # ========================================================

    print(
        f"{year} | "
        f"Arithmetic={mean_val:.6f} | "
        f"Trimmed95={trimmed_mean:.6f} | "
        f"STD{STD_MULTIPLIER:g}={std_mean:.6f} | "
        f"AreaWeighted={area_weighted_mean:.6f}"
    )

    print(
        f"       Valid NUE pixels: {valid_count:,} | "
        f"Weighted pixels: {weighted_pixel_count:,}"
    )

    print(
        f"       Total harvested area used: "
        f"{total_harvested_area:,.2f}"
    )

    # ========================================================
    # 9. STORE RESULTS
    # ========================================================

    results.append({
        "year": year,
        "mean": mean_val,
        "trimmed_mean_95": trimmed_mean,
        "std2_mean": std_mean,
        "valid_pixels": valid_count,
        "trimmed_pixels": trimmed_count,
        "std2_pixels": std_count,
        "area_weighted_mean": area_weighted_mean
    })


# ============================================================
# 10. SORT BY YEAR
# ============================================================

results = sorted(
    results,
    key=lambda x: x["year"]
)


# ============================================================
# 11. EXPORT TXT
# ============================================================

with open(
    OUTPUT_TXT,
    "w",
    encoding="utf-8"
) as f:

    f.write(
        "year,mean,trimmed_mean_95,std2_mean,"
        "valid_pixels,trimmed_pixels,std2_pixels,"
        "area_weighted_mean\n"
    )

    for result in results:

        f.write(
            f"{result['year']},"
            f"{format_value(result['mean'])},"
            f"{format_value(result['trimmed_mean_95'])},"
            f"{format_value(result['std2_mean'])},"
            f"{result['valid_pixels']},"
            f"{result['trimmed_pixels']},"
            f"{result['std2_pixels']},"
            f"{format_value(result['area_weighted_mean'])}\n"
        )


# ============================================================
# 12. FINAL SUMMARY
# ============================================================

print("\n" + "=" * 75)
print("GLOBAL NUE MEAN COMPARISON COMPLETED")
print("=" * 75)

print(f"Processed years: {len(results)}")

print("Methods:")
print("  1. Arithmetic Mean")
print(f"  2. {TRIM_PERCENT}% Percentile Trimmed Mean")
print(f"  3. Mean +/- {STD_MULTIPLIER:g} STD")
print("  4. Harvested-Area-Weighted Mean")

print(f"\nResults saved to:\n{OUTPUT_TXT}")

print("=" * 75)

In [ ]:
# ===================2026.10.26=========================================
# Tool: Regional NUE Zonal Mean + Harvested-Area-Weighted Mean 4US HAWM
#
# For China / USA counties or arbitrary SHP polygons
#
# Original:
#   - Preserve ALL shapefile attributes
#   - rasterstats country/region zonal mean
#   - NaN / Inf / NoData handling
#   - Excel output
#
# NEW:
#   - Match NUE and harvested area by year
#   - Calculate area-weighted NUE for each polygon
#   - Use ONLY common valid pixels
#   - No filling, no interpolation, no resampling
#   - Record common pixel count and area used
#
# Directly executable in ONE JupyterLab Cell
# ============================================================

import os
import re
import glob
import time

from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio

from rasterstats import gen_zonal_stats
from IPython.display import display


# ============================================================
# 1. USER CONFIGURATION
# ============================================================

# ------------------------------------------------------------
# Shapefile
# ------------------------------------------------------------

SHP_FOLDER = (
    # r"Z:\staff\Qiang_S\09_Datasets\09OtherDataUsed"
    # r"\GDAM_CountyLevel\gadm41_CHN_shp"
    r"Z:\staff\Qiang_S\09_Datasets\09OtherDataUsed"
    r"\GDAM_CountyLevel\gadm41_USA_shp"
)

# SHAPE_NAME = "gadm41_CHN_0.shp"
# SHAPE_NAME = "gadm41_CHN_3.shp"
SHAPE_NAME = "gadm41_USA_2.shp"

SHP_PATH = os.path.join(
    SHP_FOLDER,
    SHAPE_NAME
)

# ------------------------------------------------------------
# NUE GeoTIFF folder
# ------------------------------------------------------------

TIF_FOLDER = (
    r"Z:\staff\Qiang_S\09_Datasets"
    r"\03GlobalNUE_Results\02Global_NUE_Maize"
    r"\02MaizeNUE_M2_20261022V1_M6_150N"
)

# ------------------------------------------------------------
# Harvested Area GeoTIFF folder
#
# IMPORTANT:
# Change this to the actual location of your
# harvested-area GeoTIFF files.
#
# The code also searches its subfolders.
# ------------------------------------------------------------

AREA_FOLDER = (
    r"Z:\staff\Qiang_S\09_Datasets"
    r"\09OtherDataUsed"
    r"\GCSN_HarvestedArea_1995_2020"
    r"\02StandardizedDatasets"
)

# ------------------------------------------------------------
# Harvested Area filename matching
#
# Default examples:
#   Maize_HarArea_1995.tif
#   Maize_HarArea_1996.tif
#   Maize_HarArea_2020.tif
#
# Change the regex if your actual filenames differ.
# ------------------------------------------------------------

CROP = "Maize"

AREA_FILENAME_REGEX = (
    rf"^{re.escape(CROP)}_HarArea_"
    rf"(19\d{{2}}|20\d{{2}})\.(?:tif|tiff)$"
)

# ------------------------------------------------------------
# Excel output
# ------------------------------------------------------------

OUT_EXCEL = os.path.join(
    TIF_FOLDER,
    f"{Path(SHAPE_NAME).stem}_ZonalMean_AreaWeighted.xlsx"
)

# ------------------------------------------------------------
# NoData handling
# ------------------------------------------------------------

NODATA = -9999.0

ADDITIONAL_NODATA = []

# ------------------------------------------------------------
# Zonal statistics
# ------------------------------------------------------------

# False: use original pixel inclusion rule
# True : include every pixel touched by polygon

ALL_TOUCHED = False

RASTER_BAND = 1

# ------------------------------------------------------------
# Year range
# ------------------------------------------------------------

START_YEAR = 1995
END_YEAR = 2019

# ------------------------------------------------------------
# Harvested-area settings
# ------------------------------------------------------------

# Only positive harvested area is eligible
AREA_THRESHOLD = 0.0

# True: stop if a matching area file is missing
# False: leave area-weighted result empty for that year

REQUIRE_AREA_MATCH = True


# ============================================================
# 2. INITIAL VALIDATION
# ============================================================

start_time = time.perf_counter()

if not os.path.isfile(SHP_PATH):
    raise FileNotFoundError(
        f"Shapefile not found:\n{SHP_PATH}"
    )

if not os.path.isdir(TIF_FOLDER):
    raise FileNotFoundError(
        f"NUE folder not found:\n{TIF_FOLDER}"
    )

if not os.path.isdir(AREA_FOLDER):
    raise FileNotFoundError(
        "Harvested Area folder not found. "
        "Please update AREA_FOLDER:\n"
        f"{AREA_FOLDER}"
    )

if START_YEAR > END_YEAR:
    raise ValueError(
        "START_YEAR must be <= END_YEAR"
    )

if not np.isfinite(AREA_THRESHOLD):
    raise ValueError(
        "AREA_THRESHOLD must be finite"
    )


# ============================================================
# 3. LOAD SHAPEFILE
# ============================================================

print("=" * 75)
print("REGIONAL NUE MEAN + HARVESTED AREA WEIGHTED MEAN")
print("=" * 75)

print(f"Shapefile: {SHAPE_NAME}")

gdf = gpd.read_file(SHP_PATH)

if gdf.empty:
    raise ValueError(
        "Shapefile contains no features"
    )

if gdf.crs is None:
    raise ValueError(
        "Shapefile CRS is missing"
    )

geometry_col = gdf.geometry.name

# Preserve ALL original attribute fields
attribute_columns = [
    col for col in gdf.columns
    if col != geometry_col
]

# New output fields
NEW_COLUMNS = [
    "mean",
    "year",
    "area_weighted_mean",
    "common_pixels",
    "harvested_area_used"
]

# Prevent overwriting existing SHP fields
conflicting_fields = [
    col for col in attribute_columns
    if col.lower() in {
        x.lower() for x in NEW_COLUMNS
    }
]

if conflicting_fields:
    raise ValueError(
        "Shapefile contains conflicting fields: "
        f"{conflicting_fields}. "
        "Rename them before processing."
    )

gdf = gdf.reset_index(drop=True)

attributes_df = (
    gdf[attribute_columns]
    .copy()
    .reset_index(drop=True)
)

print(f"Total polygons : {len(gdf):,}")
print(f"Original CRS   : {gdf.crs}")
print(f"Attribute cols : {len(attribute_columns)}")


# ============================================================
# 4. FIND NUE FILES
# ============================================================

def extract_nue_year(filename):

    stem = Path(filename).stem

    match = re.search(
        r"(?:^|_)(19\d{2}|20\d{2})$",
        stem
    )

    if match:
        return int(match.group(1))

    return None


nue_year_files = {}

all_nue_tifs = sorted(
    glob.glob(
        os.path.join(TIF_FOLDER, "*.tif")
    )
    +
    glob.glob(
        os.path.join(TIF_FOLDER, "*.tiff")
    )
)

for tif in all_nue_tifs:

    year = extract_nue_year(tif)

    if year is None:
        continue

    if not (START_YEAR <= year <= END_YEAR):
        continue

    if year in nue_year_files:
        raise ValueError(
            f"Duplicate NUE files for {year}:\n"
            f"{nue_year_files[year]}\n"
            f"{tif}"
        )

    nue_year_files[year] = tif

if not nue_year_files:
    raise RuntimeError(
        "No NUE GeoTIFF files match the year range"
    )


# ============================================================
# 5. FIND HARVESTED AREA FILES
# ============================================================

area_regex = re.compile(
    AREA_FILENAME_REGEX,
    flags=re.IGNORECASE
)

area_year_files = {}

# Search recursively to support crop subfolders
for root, _, filenames in os.walk(AREA_FOLDER):

    for filename in filenames:

        match = area_regex.fullmatch(filename)

        if match is None:
            continue

        year = int(match.group(1))

        if not (START_YEAR <= year <= END_YEAR):
            continue

        area_path = os.path.join(
            root,
            filename
        )

        if year in area_year_files:
            raise ValueError(
                f"Duplicate Harvested Area files "
                f"for year {year}:\n"
                f"{area_year_files[year]}\n"
                f"{area_path}"
            )

        area_year_files[year] = area_path


# ============================================================
# 6. MATCH NUE AND AREA BY YEAR
# ============================================================

year_files = []

missing_area_years = []

for year in sorted(nue_year_files):

    nue_tif = nue_year_files[year]

    area_tif = area_year_files.get(year)

    if area_tif is None:
        missing_area_years.append(year)

    year_files.append(
        (year, nue_tif, area_tif)
    )

print(f"\nMatched NUE years : {len(year_files)}")
print(f"Area files found  : {len(area_year_files)}")

if missing_area_years:

    print(
        "Missing Harvested Area years:",
        missing_area_years
    )

    if REQUIRE_AREA_MATCH:
        raise FileNotFoundError(
            "Missing Harvested Area GeoTIFF "
            "for one or more NUE years. "
            "Check AREA_FOLDER and "
            "AREA_FILENAME_REGEX."
        )

print("\nYears to process:")
print([x[0] for x in year_files])


# ============================================================
# 7. READ AND CLEAN RASTER
# ============================================================

def read_clean_raster(src):

    """
    Standardize invalid pixels to -9999.

    Recognizes:
      GeoTIFF metadata NoData
      NaN / Inf
      Explicit -9999
      ADDITIONAL_NODATA

    Returns:
      data, valid_count, invalid_count
    """

    masked_data = src.read(
        RASTER_BAND,
        masked=True
    )

    data = np.array(
        masked_data.data,
        dtype=np.float32,
        copy=True
    )

    invalid_mask = (
        np.ma.getmaskarray(masked_data)
        .copy()
    )

    invalid_mask |= ~np.isfinite(data)

    invalid_mask |= (data == NODATA)

    for nd in ADDITIONAL_NODATA:
        invalid_mask |= (data == nd)

    data[invalid_mask] = NODATA

    valid_count = int(
        np.count_nonzero(~invalid_mask)
    )

    invalid_count = int(
        np.count_nonzero(invalid_mask)
    )

    return (
        data,
        valid_count,
        invalid_count
    )


# ============================================================
# 8. VERIFY NUE / HARVESTED AREA GRID ALIGNMENT
# ============================================================

def check_grid_alignment(src_nue, src_area):

    """
    No resampling is performed.

    Both rasters must have identical:
      - CRS
      - width / height
      - affine transform
    """

    if (
        src_nue.crs is None
        or src_area.crs is None
    ):
        raise ValueError(
            "NUE or Harvested Area CRS missing"
        )

    if src_nue.crs != src_area.crs:
        raise ValueError(
            "NUE / Harvested Area CRS mismatch"
        )

    if (
        src_nue.width != src_area.width
        or src_nue.height != src_area.height
    ):
        raise ValueError(
            "NUE / Harvested Area raster "
            "dimensions mismatch"
        )

    if not src_nue.transform.almost_equals(
        src_area.transform,
        precision=1e-10
    ):
        raise ValueError(
            "NUE / Harvested Area pixel grids "
            "are not aligned"
        )


# ============================================================
# 9. AREA-WEIGHTED MEAN FUNCTION
# ============================================================

def calculate_area_weighted_mean(
    nue_mini,
    area_mini
):

    """
    Calculate weighted mean using ONLY
    spatially corresponding valid pixels.

    NUE:
        Must be finite and not NoData

    Harvested Area:
        Must be finite, not NoData, and > 0

    Both masked arrays must represent the SAME
    polygon and the SAME pixel window.

    Returns:
        weighted_mean
        common_pixel_count
        harvested_area_sum
    """

    if nue_mini is None or area_mini is None:
        return None, 0, 0.0

    nue_ma = np.ma.asarray(nue_mini)
    area_ma = np.ma.asarray(area_mini)

    if nue_ma.shape != area_ma.shape:
        raise ValueError(
            "NUE / Area polygon mini-raster "
            "shape mismatch"
        )

    nue_values = np.asarray(
        np.ma.getdata(nue_ma),
        dtype=np.float64
    )

    area_values = np.asarray(
        np.ma.getdata(area_ma),
        dtype=np.float64
    )

    # --------------------------------------------------------
    # 9.1 Valid NUE pixels
    # --------------------------------------------------------

    valid_nue = (
        ~np.ma.getmaskarray(nue_ma)
        & np.isfinite(nue_values)
        & (nue_values != NODATA)
    )

    # --------------------------------------------------------
    # 9.2 Valid Harvested Area pixels
    # --------------------------------------------------------

    valid_area = (
        ~np.ma.getmaskarray(area_ma)
        & np.isfinite(area_values)
        & (area_values != NODATA)
        & (area_values > AREA_THRESHOLD)
    )

    for nd in ADDITIONAL_NODATA:

        valid_nue &= (nue_values != nd)

        valid_area &= (area_values != nd)

    # --------------------------------------------------------
    # 9.3 Common valid pixels
    # --------------------------------------------------------

    common_mask = (
        valid_nue & valid_area
    )

    common_count = int(
        np.count_nonzero(common_mask)
    )

    if common_count == 0:
        return None, 0, 0.0

    nue_common = nue_values[common_mask]

    area_common = area_values[common_mask]

    # --------------------------------------------------------
    # 9.4 Weighted mean
    # --------------------------------------------------------

    area_sum = float(
        np.sum(
            area_common,
            dtype=np.float64
        )
    )

    if not np.isfinite(area_sum) or area_sum <= 0:
        return None, common_count, area_sum

    weighted_sum = float(
        np.sum(
            nue_common * area_common,
            dtype=np.float64
        )
    )

    weighted_mean = (
        weighted_sum / area_sum
    )

    if not np.isfinite(weighted_mean):
        weighted_mean = None

    return (
        weighted_mean,
        common_count,
        area_sum
    )


# ============================================================
# 10. PREPARE PROCESSING
# ============================================================

polygon_cache = {}

result_frames = []

print("\n" + "=" * 75)
print("START REGIONAL ZONAL STATISTICS")
print("=" * 75)


# ============================================================
# 11. PROCESS EACH YEAR
# ============================================================

for index, (year, nue_tif, area_tif) in enumerate(
    year_files,
    start=1
):

    year_start = time.perf_counter()

    print("\n" + "-" * 75)

    print(
        f"[{index}/{len(year_files)}] "
        f"Processing year {year}"
    )

    print(
        f"NUE: {os.path.basename(nue_tif)}"
    )

    if area_tif is not None:
        print(
            f"Area: {os.path.basename(area_tif)}"
        )

    # --------------------------------------------------------
    # 11.1 Read NUE
    # --------------------------------------------------------

    with rasterio.open(nue_tif) as src_nue:

        if src_nue.count <= RASTER_BAND - 1:
            raise ValueError(
                f"Invalid NUE raster band: {nue_tif}"
            )

        raster_crs = src_nue.crs

        if raster_crs is None:
            raise ValueError(
                f"NUE CRS missing: {nue_tif}"
            )

        affine = src_nue.transform

        (
            nue_data,
            nue_valid_count,
            nue_invalid_count
        ) = read_clean_raster(src_nue)

        # ----------------------------------------------------
        # 11.2 Read Harvested Area
        # ----------------------------------------------------

        area_data = None

        if area_tif is not None:

            with rasterio.open(area_tif) as src_area:

                if src_area.count <= RASTER_BAND - 1:
                    raise ValueError(
                        f"Invalid Area raster band: {area_tif}"
                    )

                # Stop if grids do not align
                check_grid_alignment(
                    src_nue,
                    src_area
                )

                (
                    area_data,
                    area_valid_count,
                    area_invalid_count
                ) = read_clean_raster(src_area)

    print(
        f"Valid NUE pixels: "
        f"{nue_valid_count:,}"
    )

    # --------------------------------------------------------
    # 11.3 Reproject polygons if needed
    # --------------------------------------------------------

    crs_key = raster_crs.to_wkt()

    if crs_key not in polygon_cache:

        if gdf.crs == raster_crs:

            gdf_projected = gdf

        else:

            gdf_projected = gdf.to_crs(
                raster_crs
            )

        polygon_cache[crs_key] = (
            gdf_projected
        )

    gdf_projected = polygon_cache[crs_key]

    # --------------------------------------------------------
    # 11.4 Identify valid polygon geometries
    # --------------------------------------------------------

    geometry_valid = (
        gdf_projected.geometry.notna()
        & ~gdf_projected.geometry.is_empty
    )

    valid_indices = np.flatnonzero(
        geometry_valid.to_numpy()
    )

    valid_geometries = (
        gdf_projected.geometry
        .iloc[valid_indices]
        .tolist()
    )

    # --------------------------------------------------------
    # 11.5 Initialize result arrays
    # --------------------------------------------------------

    mean_values = [None] * len(gdf)

    weighted_values = [None] * len(gdf)

    common_counts = [0] * len(gdf)

    area_sums = [0.0] * len(gdf)

    # --------------------------------------------------------
    # 11.6 Country/region zonal statistics
    # --------------------------------------------------------

    if (
        nue_valid_count > 0
        and len(valid_geometries) > 0
    ):

        # Original NUE zonal mean method,
        # with raster_out=True to obtain the
        # exact pixels inside each polygon.

        nue_iterator = gen_zonal_stats(
            valid_geometries,
            nue_data,
            affine=affine,
            stats=["mean"],
            nodata=NODATA,
            all_touched=ALL_TOUCHED,
            geojson_out=False,
            raster_out=True
        )

        # Harvested Area uses identical geometries,
        # affine transform and pixel inclusion rule.

        if area_data is not None:

            area_iterator = gen_zonal_stats(
                valid_geometries,
                area_data,
                affine=affine,
                stats=["count"],
                nodata=NODATA,
                all_touched=ALL_TOUCHED,
                geojson_out=False,
                raster_out=True
            )

        else:

            area_iterator = (
                None for _ in valid_indices
            )

        # ----------------------------------------------------
        # 11.7 Process each polygon
        # ----------------------------------------------------

        for original_idx, nue_stat, area_stat in zip(
            valid_indices,
            nue_iterator,
            area_iterator
        ):

            # -----------------------------------------------
            # Original arithmetic zonal mean
            # -----------------------------------------------

            mean_val = nue_stat.get("mean")

            if (
                mean_val is not None
                and np.isfinite(mean_val)
            ):

                mean_values[original_idx] = float(
                    mean_val
                )

            # -----------------------------------------------
            # New area-weighted zonal mean
            # -----------------------------------------------

            if area_stat is None:
                continue

            nue_mini = nue_stat.get(
                "mini_raster_array"
            )

            area_mini = area_stat.get(
                "mini_raster_array"
            )

            # Verify mini-raster spatial alignment
            nue_mini_affine = nue_stat.get(
                "mini_raster_affine"
            )

            area_mini_affine = area_stat.get(
                "mini_raster_affine"
            )

            if (
                nue_mini_affine is None
                or area_mini_affine is None
            ):
                raise ValueError(
                    f"{year}: Missing mini-raster affine "
                    f"for polygon {original_idx}"
                )

            if not nue_mini_affine.almost_equals(
                area_mini_affine,
                precision=1e-10
            ):
                raise ValueError(
                    f"{year}: NUE and Area mini-rasters "
                    f"are not aligned for polygon "
                    f"{original_idx}"
                )

            (
                weighted_mean,
                common_count,
                area_sum
            ) = calculate_area_weighted_mean(
                nue_mini,
                area_mini
            )

            weighted_values[original_idx] = (
                weighted_mean
            )

            common_counts[original_idx] = (
                common_count
            )

            area_sums[original_idx] = (
                area_sum
            )

    # --------------------------------------------------------
    # 11.8 Create annual DataFrame
    # --------------------------------------------------------

    year_df = attributes_df.copy()

    # Original fields
    year_df["mean"] = mean_values

    year_df["year"] = year

    # NEW fields
    year_df["area_weighted_mean"] = (
        weighted_values
    )

    year_df["common_pixels"] = (
        common_counts
    )

    year_df["harvested_area_used"] = (
        area_sums
    )

    result_frames.append(year_df)

    # --------------------------------------------------------
    # 11.9 Annual summary
    # --------------------------------------------------------

    valid_original = sum(
        v is not None
        for v in mean_values
    )

    valid_weighted = sum(
        v is not None
        for v in weighted_values
    )

    total_common = sum(common_counts)

    elapsed = time.perf_counter() - year_start

    print(
        f"Original mean regions : "
        f"{valid_original:,}"
    )

    print(
        f"Weighted mean regions : "
        f"{valid_weighted:,}"
    )

    print(
        f"Common pixel counts   : "
        f"{total_common:,}"
    )

    print(
        f"Year {year} completed "
        f"in {elapsed:.1f}s"
    )


# ============================================================
# 12. MERGE RESULTS
# ============================================================

df_result = pd.concat(
    result_frames,
    ignore_index=True
)

# Original columns + weighted comparison fields
output_columns = (
    attribute_columns
    + [
        "mean",
        "year",
        "area_weighted_mean",
        "common_pixels",
        "harvested_area_used"
    ]
)

df_result = df_result[output_columns]

# ------------------------------------------------------------
# Ensure no invalid numerical results
# ------------------------------------------------------------

for col in [
    "mean",
    "area_weighted_mean",
    "harvested_area_used"
]:

    df_result[col] = pd.to_numeric(
        df_result[col],
        errors="coerce"
    )

    df_result[col] = (
        df_result[col]
        .replace(
            [np.inf, -np.inf],
            np.nan
        )
    )

# ------------------------------------------------------------
# Convert complex attribute values for Excel
# ------------------------------------------------------------

for col in attribute_columns:

    df_result[col] = df_result[col].map(
        lambda v: (
            str(v)
            if isinstance(
                v,
                (list, tuple, dict, np.ndarray)
            )
            else v
        )
    )


# ============================================================
# 13. EXPORT EXCEL
# ============================================================

# Excel maximum data rows + header
if len(df_result) > 1_048_575:
    raise ValueError(
        "Excel row limit exceeded. "
        "Split the output by year."
    )

df_result.to_excel(
    OUT_EXCEL,
    index=False,
    sheet_name="Zonal Mean",
    engine="openpyxl",
    na_rep=""
)


# ============================================================
# 14. FINAL SUMMARY
# ============================================================

elapsed_total = (
    time.perf_counter() - start_time
)

print("\n" + "=" * 75)
print("REGIONAL NUE STATISTICS COMPLETED")
print("=" * 75)

print(f"Shapefile        : {SHAPE_NAME}")
print(f"Polygons         : {len(gdf):,}")
print(f"Years            : {len(year_files)}")
print(f"Total records    : {len(df_result):,}")

print(
    "Valid means      : "
    f"{df_result['mean'].notna().sum():,}"
)

print(
    "Weighted means   : "
    f"{df_result['area_weighted_mean'].notna().sum():,}"
)

print(
    "No weighted mean : "
    f"{df_result['area_weighted_mean'].isna().sum():,}"
)

print(
    f"Processing time  : "
    f"{elapsed_total:.2f}s"
)

print("\nOutput columns:")
print(df_result.columns.tolist())

print(f"\nExcel saved to:\n{OUT_EXCEL}")

print("=" * 75)

display(df_result.head(10))

In [ ]:
# ============================2026.10.23================================
# Tool GeoTIFF Zonal Mean Extraction by Shapefile Polygons, for 4China and 4US
#
# Purpose:
#   1. Read user-specified Shapefile
#   2. Preserve ALL original attribute fields
#   3. Calculate mean for EACH polygon and EACH year
#   4. Automatically identify NaN / NoData / Inf
#   5. Export one Excel file
#
# Output:
#   All original SHP attributes + mean + year
#
# Method:
#   rasterstats.zonal_stats(stats=["mean"])
#
# No interpolation / filling / raster resampling
#
# Directly executable in JupyterLab
# ============================================================

import os
import glob
import re
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio

from rasterstats import zonal_stats
from pathlib import Path


# ============================================================
# 1. USER CONFIGURATION
# ============================================================

# ------------------------------------------------------------
# Shapefile folder
# ------------------------------------------------------------

SHP_FOLDER = (
    #r"Z:\staff\Qiang_S\09_Datasets\09OtherDataUsed\GDAM_CountyLevel\gadm41_CHN_shp"
    r"Z:\staff\Qiang_S\09_Datasets\09OtherDataUsed\GDAM_CountyLevel\gadm41_USA_shp"
)

# Shapefile filename - USER CONFIGURABLE
#SHAPE_NAME = "gadm41_CHN_0.shp"
#SHAPE_NAME = "gadm41_CHN_3.shp"
SHAPE_NAME = "gadm41_USA_2.shp"

# Complete shapefile path
SHP_PATH = os.path.join(
    SHP_FOLDER,
    SHAPE_NAME
)

# ------------------------------------------------------------
# GeoTIFF folder
# ------------------------------------------------------------

TIF_FOLDER = (
    r"Z:\staff\Qiang_S\09_Datasets\03GlobalNUE_Results\02Global_NUE_Maize\02MaizeNUE_M2_20261022V1_M6_180C" 
)

# ------------------------------------------------------------
# Output Excel
# ------------------------------------------------------------

OUT_EXCEL = os.path.join(
    TIF_FOLDER,
    f"{Path(SHAPE_NAME).stem}_ZonalMean.xlsx"
)

# ------------------------------------------------------------
# NoData settings
# ------------------------------------------------------------

# Standardized NoData for internal calculations
NODATA = -9999.0

# Optional additional invalid values
ADDITIONAL_NODATA = []

# ------------------------------------------------------------
# Zonal statistics settings
# ------------------------------------------------------------

# False = pixel center within polygon
# True = all pixels touched by polygon
ALL_TOUCHED = False

# Raster band
RASTER_BAND = 1

# Optional year range
START_YEAR = 1995
END_YEAR = 2019


# ============================================================
# 2. VALIDATE INPUTS
# ============================================================

if not os.path.isfile(SHP_PATH):
    raise FileNotFoundError(
        f"Shapefile not found:\n{SHP_PATH}"
    )

if not os.path.isdir(TIF_FOLDER):
    raise FileNotFoundError(
        f"GeoTIFF folder not found:\n{TIF_FOLDER}"
    )

os.makedirs(
    os.path.dirname(OUT_EXCEL),
    exist_ok=True
)


# ============================================================
# 3. LOAD SHAPEFILE
# ============================================================

print("=" * 70)
print("GEOTIFF ZONAL MEAN EXTRACTION")
print("=" * 70)

print(f"Shapefile: {SHAPE_NAME}")

gdf = gpd.read_file(SHP_PATH)

if gdf.empty:
    raise ValueError(
        "Shapefile contains no features."
    )

if gdf.crs is None:
    raise ValueError(
        "Shapefile CRS is missing."
    )

# Keep original attributes
geometry_col = gdf.geometry.name

attribute_columns = [
    col for col in gdf.columns
    if col != geometry_col
]

# Avoid overwriting existing attribute fields
reserved_columns = {"mean", "year"}

if any(
    col.lower() in reserved_columns
    for col in attribute_columns
):
    raise ValueError(
        "Shapefile already contains a 'mean' or 'year' "
        "attribute. Rename the original field first "
        "to avoid overwriting its values."
    )

# Preserve original polygon order
gdf = gdf.reset_index(drop=True)

print(f"Total polygons: {len(gdf):,}")
print(f"Original CRS: {gdf.crs}")
print(f"Attribute fields: {len(attribute_columns)}")

print("\nOriginal attribute fields:")
for col in attribute_columns:
    print(f"  {col}")


# ============================================================
# 4. IDENTIFY GEOTIFF FILES
# ============================================================

tif_files = sorted(
    glob.glob(
        os.path.join(TIF_FOLDER, "*.tif")
    )
    +
    glob.glob(
        os.path.join(TIF_FOLDER, "*.tiff")
    )
)

# Extract matching years
year_files = []

for tif in tif_files:

    filename = os.path.basename(tif)
    stem = Path(filename).stem

    # Example:
    # NUE_1998.tif
    # NUE_median_Opt_1998.tif

    match = re.search(
        r"(?:^|_)(19\d{2}|20\d{2})$",
        stem
    )

    if not match:
        print(f"Skipping (no year): {filename}")
        continue

    year = int(match.group(1))

    if not (START_YEAR <= year <= END_YEAR):
        continue

    year_files.append((year, tif))

year_files.sort(key=lambda x: (x[0], x[1]))

if not year_files:
    raise RuntimeError(
        "No GeoTIFF files matched the requested year range."
    )

print(f"\nMatched GeoTIFF files: {len(year_files)}")


# ============================================================
# 5. FUNCTION - CLEAN RASTER NODATA
# ============================================================

def read_clean_raster(src):
    """
    Read raster and identify invalid pixels.

    Invalid pixels:
        - GeoTIFF metadata NoData
        - NaN
        - +Inf / -Inf
        - -9999
        - Additional user-defined NoData

    All invalid pixels are converted to -9999.

    Original valid values remain unchanged,
    apart from float32 conversion when needed.
    """

    # Read raster with its defined mask
    masked_data = src.read(
        RASTER_BAND,
        masked=True
    )

    data = np.array(
        masked_data.data,
        dtype=np.float32,
        copy=True
    )

    # Metadata-defined invalid pixels
    invalid_mask = np.ma.getmaskarray(
        masked_data
    ).copy()

    # NaN / Inf
    invalid_mask |= ~np.isfinite(data)

    # Explicit NoData
    invalid_mask |= (data == NODATA)

    # Additional invalid values
    for value in ADDITIONAL_NODATA:
        invalid_mask |= (data == value)

    # Standardize invalid pixels
    data[invalid_mask] = NODATA

    valid_count = int(
        np.count_nonzero(~invalid_mask)
    )

    invalid_count = int(
        np.count_nonzero(invalid_mask)
    )

    return data, valid_count, invalid_count


# ============================================================
# 6. PREPARE POLYGON ATTRIBUTES
# ============================================================

# Export all attributes except geometry itself
attributes_df = (
    gdf[attribute_columns]
    .copy()
    .reset_index(drop=True)
)

# Cache CRS-transformed polygons
polygon_cache = {}

# Results
result_frames = []


# ============================================================
# 7. LOOP THROUGH ALL GEOTIFF FILES
# ============================================================

for index, (year, tif) in enumerate(
    year_files,
    start=1
):

    print("\n" + "-" * 70)

    print(
        f"[{index}/{len(year_files)}] "
        f"Processing year {year}"
    )

    print(f"Raster: {os.path.basename(tif)}")

    # --------------------------------------------------------
    # 7.1 Read raster
    # --------------------------------------------------------

    with rasterio.open(tif) as src:

        raster_crs = src.crs

        if raster_crs is None:
            raise ValueError(
                f"Missing raster CRS: {tif}"
            )

        affine = src.transform

        (
            raster_data,
            valid_count,
            invalid_count
        ) = read_clean_raster(src)

    print(f"Raster shape: {raster_data.shape}")

    print(
        f"Valid pixels: {valid_count:,}"
    )

    print(
        f"Invalid pixels: {invalid_count:,}"
    )

    # --------------------------------------------------------
    # 7.2 Reproject polygons if required
    # --------------------------------------------------------

    crs_key = raster_crs.to_wkt()

    if crs_key not in polygon_cache:

        if gdf.crs == raster_crs:
            gdf_projected = gdf
        else:
            gdf_projected = gdf.to_crs(
                raster_crs
            )

        polygon_cache[crs_key] = (
            gdf_projected
        )

    gdf_projected = polygon_cache[crs_key]

    # --------------------------------------------------------
    # 7.3 Identify valid geometries
    # --------------------------------------------------------

    geometry_valid = (
        gdf_projected.geometry.notna()
        & ~gdf_projected.geometry.is_empty
    )

    valid_indices = np.flatnonzero(
        geometry_valid.to_numpy()
    )

    valid_geometries = (
        gdf_projected.geometry
        .iloc[valid_indices]
        .tolist()
    )

    # --------------------------------------------------------
    # 7.4 Zonal mean statistics
    # --------------------------------------------------------

    # Initialize all polygon means as empty
    mean_values = [None] * len(gdf)

    if valid_count > 0 and len(valid_geometries) > 0:

        stats = zonal_stats(
            valid_geometries,
            raster_data,
            affine=affine,
            stats=["mean"],
            nodata=NODATA,
            all_touched=ALL_TOUCHED,
            geojson_out=False
        )

        # Match results to original polygon indices
        for original_idx, stat in zip(
            valid_indices,
            stats
        ):

            mean_val = stat.get("mean")

            if (
                mean_val is not None
                and np.isfinite(mean_val)
            ):

                mean_values[original_idx] = float(
                    mean_val
                )

    # --------------------------------------------------------
    # 7.5 Create annual DataFrame
    # --------------------------------------------------------

    # Keep all original attribute fields
    year_df = attributes_df.copy()

    # Add mean
    year_df["mean"] = mean_values

    # Add year
    year_df["year"] = year

    # Append annual results
    result_frames.append(year_df)

    # --------------------------------------------------------
    # 7.6 Year summary
    # --------------------------------------------------------

    valid_polygons = sum(
        val is not None
        for val in mean_values
    )

    missing_polygons = (
        len(gdf) - valid_polygons
    )

    print(
        f"Polygons with valid mean: "
        f"{valid_polygons:,}"
    )

    print(
        f"Polygons without valid mean: "
        f"{missing_polygons:,}"
    )

    print(f"Year {year} completed.")


# ============================================================
# 8. MERGE RESULTS
# ============================================================

df_result = pd.concat(
    result_frames,
    ignore_index=True
)

# Preserve original attribute field order
# and append exactly two additional fields
output_columns = (
    attribute_columns
    + ["mean", "year"]
)

df_result = df_result[output_columns]

# Ensure mean contains no Inf
df_result["mean"] = pd.to_numeric(
    df_result["mean"],
    errors="coerce"
)

df_result["mean"] = df_result["mean"].replace(
    [np.inf, -np.inf],
    np.nan
)

# Convert non-scalar attribute types if present
# so they can be written safely to Excel
for col in attribute_columns:

    df_result[col] = df_result[col].map(
        lambda v: (
            str(v)
            if isinstance(
                v,
                (list, tuple, dict, np.ndarray)
            )
            else v
        )
    )


# ============================================================
# 9. EXPORT EXCEL
# ============================================================

# Excel row limit: 1,048,576 including header
if len(df_result) > 1_048_575:
    raise ValueError(
        "Result exceeds the Excel worksheet row limit. "
        "Consider splitting the output by year."
    )

df_result.to_excel(
    OUT_EXCEL,
    index=False,
    sheet_name="Zonal Mean",
    na_rep=""
)


# ============================================================
# 10. FINAL VALIDATION
# ============================================================

print("\n" + "=" * 70)
print("ZONAL MEAN EXTRACTION COMPLETED")
print("=" * 70)

print(f"Shapefile: {SHAPE_NAME}")

print(f"Polygons: {len(gdf):,}")

print(f"GeoTIFF files: {len(year_files)}")

print(f"Total records: {len(df_result):,}")

print(
    "Valid mean values: "
    f"{df_result['mean'].notna().sum():,}"
)

print(
    "Missing mean values: "
    f"{df_result['mean'].isna().sum():,}"
)

print("\nOutput columns:")
print(df_result.columns.tolist())

print(f"\nExcel saved to:\n{OUT_EXCEL}")

print("=" * 70)

# Display first rows in JupyterLab
display(df_result.head(10))


In [ ]:
import rasterio
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import os
from datetime import datetime

def grid_to_grid_analysis(tif_a_path, tif_b_path, output_dir, val_min=None, val_max=None):
    """
    针对 4320x2160 分辨率优化的栅格散点分析
    """
    # 1. 创建输出路径
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
    
    start_time = datetime.now()
    print(f"[{start_time.strftime('%H:%M:%S')}] 正在加载栅格数据...")

    # 2. 读取数据
    with rasterio.open(tif_a_path) as src_a, rasterio.open(tif_b_path) as src_b:
        data_a = src_a.read(1).flatten()
        data_b = src_b.read(1).flatten()
        nodata_a = src_a.nodata
        nodata_b = src_b.nodata

    # 3. 掩码过滤 (有效值 + 指定范围)
    mask = (data_a != nodata_a) & (data_b != nodata_b) & (~np.isnan(data_a)) & (~np.isnan(data_b))
    
    if val_min is not None:
        mask &= (data_a >= val_min) & (data_b >= val_min)
    if val_max is not None:
        mask &= (data_a <= val_max) & (data_b <= val_max)

    x = data_a[mask]
    y = data_b[mask]
    
    # 释放大内存
    del data_a, data_b
    print(f"有效样本量: {len(x):,}")

    # 4. 统计计算
    print("正在计算回归指标...")
    slope, intercept, r_value, p_value, std_err = stats.linregress(x, y)
    pearson_corr, _ = stats.pearsonr(x, y)
    r_squared = r_value**2

    # 5. 导出数据 (CSV)
    csv_name = f"scatter_data_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"
    csv_path = os.path.join(output_dir, csv_name)
    pd.DataFrame({'Value_A': x, 'Value_B': y}).to_csv(csv_path, index=False)

    # 6. 绘图 (Hexbin 密度散点图)
    print("正在生成可视化图表...")
    plt.figure(figsize=(10, 8), dpi=100)
    
    # 使用 hexbin 显示密度，cmap 可选 'viridis', 'YlGnBu', 'magma'
    hb = plt.hexbin(x, y, gridsize=100, cmap='viridis', bins='log', mincnt=1)
    cb = plt.colorbar(hb, label='Log10(Count)')
    
    # 1:1 线
    axis_max = max(x.max(), y.max())
    axis_min = min(x.min(), y.min())
    plt.plot([axis_min, axis_max], [axis_min, axis_max], 'r--', lw=2, label='1:1 Line')
    
    # 回归线
    plt.plot(x, slope * x + intercept, 'k-', lw=1, label='Regression Line')

    # 统计信息标注 (LaTeX 格式)
    stats_text = (
        f"$R = {pearson_corr:.3f}$\n"
        f"$R^2 = {r_squared:.3f}$\n"
        f"$y = {slope:.2f}x + {intercept:.2f}$\n"
        f"$n = {len(x):,}$"
    )
    plt.gca().text(0.05, 0.95, stats_text, transform=plt.gca().transAxes,
                   fontsize=12, verticalalignment='top', bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

    plt.xlabel(f"Data A: {os.path.basename(tif_a_path)}")
    plt.ylabel(f"Data B: {os.path.basename(tif_b_path)}")
    plt.title("Grid-to-Grid Density Scatter Plot")
    plt.legend(loc='lower right')
    plt.grid(alpha=0.3)

    # 保存图片
    img_path = os.path.join(output_dir, "scatter_density_plot.png")
    plt.savefig(img_path, bbox_inches='tight')
    plt.show()

    end_time = datetime.now()
    print(f"[{end_time.strftime('%H:%M:%S')}] 处理完成！总耗时: {end_time - start_time}")

# ================= Jupyter 执行设置 =================
config = {
    #"tif_a_path": r"Z:/staff/Qiang_S/09_Datasets/09OtherDataUsed/GlobalNUE_4_Validation/GlobalNUEPaper1_datafromAuthorUnify/Wheat_NUE.tif",
    #"tif_b_path": r"Z:/staff/Qiang_S/09_Datasets/03GlobalNUE_Results/01Global_NUE_Wheat/01WheatNUE_V1_180_M5/NUE_median_Opt_2020.tif",
    
    "tif_a_path": r"Z:/staff/Qiang_S/09_Datasets/09OtherDataUsed/GlobalNUE_4_Validation/GlobalNUEPaper1_datafromAuthorUnify2/Zscore_Rice_NUE.tif",
    "tif_b_path": r"Z:/staff/Qiang_S/09_Datasets/09OtherDataUsed/GlobalNUE_4_Validation/GlobalNUEPaper1_datafromAuthorUnify2/Zscore_RiceNUE_median_Opt_2018.tif",
    "output_dir": r"Z:/staff/Qiang_S/09_Datasets/09OtherDataUsed/GlobalNUE_4_Validation/GlobalNUEPaper1_datafromAuthorUnify2",
    "val_min": 0,    # 仅统计正值区域
    "val_max": 4   # 设定上限过滤异常值
}

grid_to_grid_analysis(**config)

In [ ]:
import rasterio
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import pearsonr
import os

# =====================================
# 1. 参数设置（修改这里）
# =====================================
tif_A = r"Z:/staff/Qiang_S/09_Datasets/09OtherDataUsed/GlobalNUE_4_Validation/GlobalNUEPaper1_datafromAuthorUnify2/Zscore_Wheat_NUE.tif"
tif_B = r"Z:/staff/Qiang_S/09_Datasets/09OtherDataUsed/GlobalNUE_4_Validation/GlobalNUEPaper1_datafromAuthorUnify2/Zscore_NUE_median_Opt_2018.tif"

out_dir = r"Z:/staff/Qiang_S/09_Datasets/09OtherDataUsed/GlobalNUE_4_Validation/GlobalNUEPaper1_datafromAuthorUnify2/GPTResult"
os.makedirs(out_dir, exist_ok=True)

nodata = -9999

# 数值筛选范围（None 表示不限制）
value_min = None     # 如 0
value_max = 4     # 如 100

# 输出文件名
fig_name = "GridScatterWheat.png"
csv_name = "GridScatterWheat.csv"

# =====================================
# 2. 读取栅格
# =====================================
with rasterio.open(tif_A) as srcA:
    A = srcA.read(1).astype("float32")

with rasterio.open(tif_B) as srcB:
    B = srcB.read(1).astype("float32")

print("Raster shape:", A.shape)

# =====================================
# 3. NoData → NaN
# =====================================
A[A == nodata] = np.nan
B[B == nodata] = np.nan

# =====================================
# 4. 提取共同有效像元
# =====================================
mask = np.isfinite(A) & np.isfinite(B)

# 数值范围限制
if value_min is not None:
    mask &= (A >= value_min) & (B >= value_min)

if value_max is not None:
    mask &= (A <= value_max) & (B <= value_max)

A_valid = A[mask]
B_valid = B[mask]

print(f"Valid pixels: {len(A_valid):,}")

# =====================================
# 5. 统计计算
# =====================================
corr, _ = pearsonr(A_valid, B_valid)
r2 = corr ** 2

print(f"Correlation (R) = {corr:.4f}")
print(f"R² = {r2:.4f}")

# =====================================
# 6. 导出 CSV
# =====================================
df = pd.DataFrame({
    "A_value": A_valid,
    "B_value": B_valid
})

csv_path = os.path.join(out_dir, csv_name)
df.to_csv(csv_path, index=False)

print("CSV saved:", csv_path)

# =====================================
# 7. 绘制 Grid-to-Grid Scatter
# =====================================
plt.figure(figsize=(7,7))

plt.scatter(
    A_valid,
    B_valid,
    s=1,
    alpha=0.3
)

# ---- 1:1 line ----
vmin = np.nanmin([A_valid.min(), B_valid.min()])
vmax = np.nanmax([A_valid.max(), B_valid.max()])

plt.plot([vmin, vmax], [vmin, vmax], linestyle="--")

# ---- 标注统计量 ----
plt.text(
    0.05, 0.95,
    f"Corr = {corr:.3f}\nR² = {r2:.3f}\nN = {len(A_valid):,}",
    transform=plt.gca().transAxes,
    verticalalignment="top",
    bbox=dict(facecolor="white", alpha=0.8)
)

plt.xlabel("Dataset A")
plt.ylabel("Dataset B")
plt.title("Grid-to-Grid Scatter Plot")

plt.xlim(vmin, vmax)
plt.ylim(vmin, vmax)

plt.grid(True)

# =====================================
# 8. 保存图片
# =====================================
fig_path = os.path.join(out_dir, fig_name)

plt.savefig(fig_path, dpi=300, bbox_inches="tight")
plt.show()

print("Figure saved:", fig_path)

In [ ]:
# validation field scale
# ============================================================
# Extract GeoTIFF Pixel Values Using Excel Coordinates & Year
#
# Excel required columns:
#   Latitude
#   Longitude
#   Harvest year
#
# GeoTIFF filename examples:
#   NUE_1995.tif
#   NUE_1996.tif
#   NUE_2020.tif
#
# Output:
#   Original Excel columns +
#   Raster_Value
#   Raster_File
#   Extract_Status
#
# No interpolation or resampling.
# NaN / NoData handling included.
#
# Run directly in JupyterLab
# ============================================================

import os
import re
import numpy as np
import pandas as pd
import rasterio

from pathlib import Path
from rasterio.crs import CRS
from rasterio.warp import transform as crs_transform


# ============================================================
# 1. USER CONFIGURATION
# ============================================================

# Input Excel
INPUT_EXCEL = (
    r"Z:\staff\Qiang_S\09_Datasets\09OtherDataUsed\09_Meta_AnalysisFields"
    r"\DatabaseWheat_With_Local_Fertilization.xlsx"
)

# Excel sheet
# 0 = first sheet
SHEET_NAME = 0

# GeoTIFF folder
TIF_FOLDER = (
    r"Z:\staff\Qiang_S\09_Datasets\03GlobalNUE_Results\09CropResults4Validation1021V2_200_M6\wheat"
)

# Output Excel
OUTPUT_EXCEL = (
    r"Z:\staff\Qiang_S\09_Datasets\09OtherDataUsed\09_Meta_AnalysisFields"
    r"\01Extracted_GeoTIFF_ValuesPFPNv2.xlsx"
)

# Excel column names
LAT_COLUMN = "Latitude"
LON_COLUMN = "Longitude"
YEAR_COLUMN = "Harvest year"

# GeoTIFF filename prefix (optional)
# None = accept any xxx_YYYY.tif
# "NUE_" = only accept names starting with NUE_
TIF_PREFIX = None

# Raster band
RASTER_BAND = 1

# Additional NoData values
ADDITIONAL_NODATA = [-9999.0]

# Output column names
VALUE_COLUMN = "Raster_Value"
FILE_COLUMN = "Raster_File"
STATUS_COLUMN = "Extract_Status"


# ============================================================
# 2. VALIDATE PATHS
# ============================================================

if not os.path.isfile(INPUT_EXCEL):
    raise FileNotFoundError(
        f"Excel not found: {INPUT_EXCEL}"
    )

if not os.path.isdir(TIF_FOLDER):
    raise FileNotFoundError(
        f"GeoTIFF folder not found: {TIF_FOLDER}"
    )

if (
    os.path.abspath(INPUT_EXCEL).lower()
    == os.path.abspath(OUTPUT_EXCEL).lower()
):
    raise ValueError(
        "OUTPUT_EXCEL must differ from INPUT_EXCEL."
    )

os.makedirs(
    os.path.dirname(os.path.abspath(OUTPUT_EXCEL)),
    exist_ok=True
)


# ============================================================
# 3. READ EXCEL
# ============================================================

print("=" * 70)
print("EXTRACT GEOTIFF VALUES BY COORDINATES AND YEAR")
print("=" * 70)

df = pd.read_excel(
    INPUT_EXCEL,
    sheet_name=SHEET_NAME
)

# Remove leading/trailing spaces in column names
df.columns = [
    str(col).strip()
    for col in df.columns
]

required_columns = [
    LAT_COLUMN,
    LON_COLUMN,
    YEAR_COLUMN
]

for col in required_columns:

    if col not in df.columns:
        raise ValueError(
            f"Missing Excel column: {col}\n"
            f"Available columns: {df.columns.tolist()}"
        )

# Prevent overwriting existing columns
for col in [
    VALUE_COLUMN,
    FILE_COLUMN,
    STATUS_COLUMN
]:
    if col in df.columns:
        raise ValueError(
            f"Output column already exists: {col}"
        )

print(f"Excel rows: {len(df):,}")
print(f"Excel columns: {len(df.columns)}")


# ============================================================
# 4. PARSE HARVEST YEAR
# ============================================================

def parse_year(value):
    """
    Supported examples:
        1996
        1996.0
        "1996"
        datetime / Excel date
        "1996-07-15"

    Invalid or missing year -> NaN
    """

    if pd.isna(value):
        return np.nan

    if isinstance(value, pd.Timestamp):
        return value.year

    text = str(value).strip()

    # Numeric year
    match = re.fullmatch(
        r"(19\d{2}|20\d{2})(?:\.0+)?",
        text
    )

    if match:
        return int(match.group(1))

    # Date-like value
    try:

        date_value = pd.to_datetime(
            text,
            errors="coerce"
        )

        if pd.notna(date_value):
            return int(date_value.year)

    except Exception:
        pass

    return np.nan


df["_match_year"] = df[YEAR_COLUMN].apply(
    parse_year
)

# Convert coordinates
df["_lat_numeric"] = pd.to_numeric(
    df[LAT_COLUMN],
    errors="coerce"
)

df["_lon_numeric"] = pd.to_numeric(
    df[LON_COLUMN],
    errors="coerce"
)


# ============================================================
# 5. FIND GEOTIFF FILES AND MATCH YEARS
# ============================================================

tif_folder = Path(TIF_FOLDER)

# Recursively search all subfolders
tif_files = sorted(
    p for p in tif_folder.rglob("*")
    if p.is_file()
    and p.suffix.lower() in (".tif", ".tiff")
)

year_to_tif = {}

for tif in tif_files:

    filename = tif.name

    if TIF_PREFIX is not None:

        if not filename.lower().startswith(
            TIF_PREFIX.lower()
        ):
            continue

    # Last four digits are the year
    # Example: NUE_1996.tif -> 1996
    match = re.search(
        r"_(19\d{2}|20\d{2})\.tiff?$",
        filename,
        re.IGNORECASE
    )

    if match is None:
        continue

    year = int(match.group(1))

    # Prevent ambiguous matching
    if year in year_to_tif:

        raise ValueError(
            f"Multiple GeoTIFF files found for {year}:\n"
            f"  {year_to_tif[year]}\n"
            f"  {tif}\n"
            "Please set TIF_PREFIX or use a folder "
            "containing one raster per year."
        )

    year_to_tif[year] = tif


if not year_to_tif:
    raise RuntimeError(
        "No GeoTIFF files matching xxx_YYYY.tif found."
    )

print(f"\nMatched GeoTIFF files: {len(year_to_tif)}")
print(f"Available years: {sorted(year_to_tif)}")


# ============================================================
# 6. INITIALIZE OUTPUT COLUMNS
# ============================================================

df[VALUE_COLUMN] = np.nan
df[FILE_COLUMN] = ""
df[STATUS_COLUMN] = ""

# Validate years
valid_year = df["_match_year"].notna()

# Validate coordinate values and ranges
lat = df["_lat_numeric"]
lon = df["_lon_numeric"]

valid_coord = (
    np.isfinite(lat)
    & np.isfinite(lon)
    & lat.between(-90, 90)
    & lon.between(-180, 180)
)

# Initialize status
df.loc[
    ~valid_year,
    STATUS_COLUMN
] = "INVALID_YEAR"

df.loc[
    valid_year & ~valid_coord,
    STATUS_COLUMN
] = "INVALID_COORDINATES"

ready_mask = valid_year & valid_coord

df.loc[
    ready_mask,
    STATUS_COLUMN
] = "RASTER_YEAR_NOT_FOUND"


# ============================================================
# 7. PROCESS EACH YEAR
# ============================================================

required_years = sorted(
    int(y)
    for y in df.loc[
        ready_mask,
        "_match_year"
    ].unique()
)

for year in required_years:

    # Indices of Excel records for this year
    selected_indices = df.index[
        ready_mask
        & (df["_match_year"] == year)
    ].tolist()

    if year not in year_to_tif:

        print(
            f"[{year}] Raster not found | "
            f"Points: {len(selected_indices)}"
        )

        continue

    tif_path = year_to_tif[year]

    print("\n" + "-" * 70)
    print(f"Processing year: {year}")
    print(f"Raster: {tif_path.name}")
    print(f"Points: {len(selected_indices)}")

    # --------------------------------------------------------
    # 7.1 Open corresponding GeoTIFF
    # --------------------------------------------------------

    with rasterio.open(tif_path) as src:

        if src.crs is None:
            raise ValueError(
                f"GeoTIFF CRS missing: {tif_path}"
            )

        if RASTER_BAND > src.count:
            raise ValueError(
                f"Band {RASTER_BAND} not found in {tif_path.name}"
            )

        # Record filename for matched points
        df.loc[
            selected_indices,
            FILE_COLUMN
        ] = tif_path.name

        # ----------------------------------------------------
        # 7.2 Get coordinates from Excel
        # ----------------------------------------------------

        latitudes = df.loc[
            selected_indices,
            "_lat_numeric"
        ].to_numpy(dtype=float)

        longitudes = df.loc[
            selected_indices,
            "_lon_numeric"
        ].to_numpy(dtype=float)

        # ----------------------------------------------------
        # 7.3 Transform WGS84 coordinates to raster CRS
        # ----------------------------------------------------

        source_crs = CRS.from_epsg(4326)

        xs, ys = crs_transform(
            source_crs,
            src.crs,
            longitudes.tolist(),
            latitudes.tolist()
        )

        # ----------------------------------------------------
        # 7.4 Validate whether point is within raster
        # ----------------------------------------------------

        valid_points = []
        sample_coordinates = []

        for excel_index, x, y in zip(
            selected_indices,
            xs,
            ys
        ):

            if not (
                np.isfinite(x)
                and np.isfinite(y)
            ):

                df.at[
                    excel_index,
                    STATUS_COLUMN
                ] = "INVALID_TRANSFORM"

                continue

            # Get corresponding raster row and column
            row, col = src.index(x, y)

            if (
                row < 0
                or row >= src.height
                or col < 0
                or col >= src.width
            ):

                df.at[
                    excel_index,
                    STATUS_COLUMN
                ] = "OUTSIDE_RASTER"

                continue

            valid_points.append(excel_index)

            sample_coordinates.append(
                (x, y)
            )

        # ----------------------------------------------------
        # 7.5 Sample raster values
        # ----------------------------------------------------

        samples = src.sample(
            sample_coordinates,
            indexes=RASTER_BAND,
            masked=True
        )

        for excel_index, sample in zip(
            valid_points,
            samples
        ):

            # Check raster mask
            if np.any(np.ma.getmaskarray(sample)):

                df.at[
                    excel_index,
                    STATUS_COLUMN
                ] = "NODATA"

                continue

            value = float(sample[0])

            # NaN / Inf
            if not np.isfinite(value):

                df.at[
                    excel_index,
                    STATUS_COLUMN
                ] = "NODATA"

                continue

            # Additional NoData
            if any(
                value == nd
                for nd in ADDITIONAL_NODATA
            ):

                df.at[
                    excel_index,
                    STATUS_COLUMN
                ] = "NODATA"

                continue

            # -----------------------------------------------
            # Valid value: save original pixel value
            # -----------------------------------------------

            df.at[
                excel_index,
                VALUE_COLUMN
            ] = value

            df.at[
                excel_index,
                STATUS_COLUMN
            ] = "SUCCESS"

    success_count = int(
        (
            df.loc[
                selected_indices,
                STATUS_COLUMN
            ] == "SUCCESS"
        ).sum()
    )

    print(
        f"[{year}] Success: "
        f"{success_count}/{len(selected_indices)}"
    )


# ============================================================
# 8. REMOVE TEMPORARY COLUMNS
# ============================================================

df.drop(
    columns=[
        "_match_year",
        "_lat_numeric",
        "_lon_numeric"
    ],
    inplace=True
)


# ============================================================
# 9. EXPORT EXCEL
# ============================================================

df.to_excel(
    OUTPUT_EXCEL,
    index=False,
    sheet_name="Extracted_Values",
    na_rep=""
)


# ============================================================
# 10. FINAL SUMMARY
# ============================================================

print("\n" + "=" * 70)
print("GEOTIFF EXTRACTION COMPLETED")
print("=" * 70)

print(f"Total Excel rows: {len(df):,}")

print(
    f"Successfully extracted: "
    f"{(df[STATUS_COLUMN] == 'SUCCESS').sum():,}"
)

print("\nExtraction status summary:")

print(
    df[STATUS_COLUMN]
    .value_counts(dropna=False)
    .to_string()
)

print(f"\nOutput Excel:\n{OUTPUT_EXCEL}")

print("=" * 70)

display(df.head(10))


In [ ]:

# ============================================================
# Year-by-Year GeoTIFF Masking - Intersection Only used 2match
#
# Dataset A: Reference mask (e.g. Area_1995.tif)
# Dataset B: Target data    (e.g. NUE_1995.tif)
#
# Rules:
#   A valid, B valid   -> Keep original B value
#   A valid, B NoData  -> NoData
#   A invalid, B valid -> NoData
#   A invalid, B NoData -> NoData
#
# No interpolation, no filling, no resampling.
# ============================================================

from pathlib import Path
import re
import csv

import numpy as np
import rasterio


# ============================================================
# 1. USER CONFIGURATION
# ============================================================

# Dataset A: Reference mask
FOLDER_A = Path(
    #r"Z:\staff\Qiang_S\09_Datasets\09OtherDataUsed\GCSN_HarvestedArea_1995_2020\02StandardizedDatasets\01WheatHarvestedArea"
    r"Z:\staff\Qiang_S\09_Datasets\09OtherDataUsed\GCSN_HarvestedArea_1995_2020\02StandardizedDatasets\02MaizeHarvestedArea"
    #r"Z:\staff\Qiang_S\09_Datasets\09OtherDataUsed\GCSN_HarvestedArea_1995_2020\02StandardizedDatasets\03RiceHarvestedArea"
)

# Dataset B: Target data
FOLDER_B = Path(
    r"Z:\staff\Qiang_S\09_Datasets\03GlobalNUE_Results\02Global_NUE_Maize\02MaizeNUE_M2_20261027V2_Median"
    #r"Z:\staff\Qiang_S\09_Datasets\03GlobalNUE_Results\03Global_NUE_Rice\03RiceNUE_M2_20261026V1_Median"
)

# Output folder
OUTPUT_FOLDER = Path(
    r"Z:\staff\Qiang_S\09_Datasets\03GlobalNUE_Results\02Global_NUE_Maize\02MaizeNUE_M2_20261027V2_Median_Intersection"
)

# ------------------------------------------------------------
# Mask definition
# ------------------------------------------------------------

# "positive": A > 0 is valid (recommended for harvested area)
# "non_nodata": Any finite A value, including zero, is valid
MASK_MODE = "positive"

MASK_THRESHOLD = 0.0

# Optional additional NoData values
# GeoTIFF metadata NoData is automatically recognized
ADDITIONAL_NODATA_A = [-9999.0]
ADDITIONAL_NODATA_B = [-9999.0]

# Output settings
OUTPUT_NODATA = np.nan
OVERWRITE = False


# ============================================================
# 2. INITIALIZE
# ============================================================

FOLDER_A = FOLDER_A.resolve()
FOLDER_B = FOLDER_B.resolve()
OUTPUT_FOLDER = OUTPUT_FOLDER.resolve()

if not FOLDER_A.is_dir():
    raise FileNotFoundError(f"Folder A not found: {FOLDER_A}")

if not FOLDER_B.is_dir():
    raise FileNotFoundError(f"Folder B not found: {FOLDER_B}")

if MASK_MODE not in ("positive", "non_nodata"):
    raise ValueError("Invalid MASK_MODE")

if OUTPUT_FOLDER in (FOLDER_A, FOLDER_B):
    raise ValueError("Output folder must differ from input folders")

OUTPUT_FOLDER.mkdir(parents=True, exist_ok=True)


# ============================================================
# 3. COLLECT FILES BY YEAR
# ============================================================

def extract_year(filename):

    stem = Path(filename).stem

    match = re.search(
        r"_(19\d{2}|20\d{2})$",
        stem
    )

    return int(match.group(1)) if match else None


def collect_files(folder):

    files = {}

    for path in sorted(folder.iterdir()):

        if not path.is_file():
            continue

        if path.suffix.lower() not in (".tif", ".tiff"):
            continue

        year = extract_year(path.name)

        if year is None:
            continue

        if year in files:
            raise ValueError(
                f"Duplicate year {year}: "
                f"{files[year].name}, {path.name}"
            )

        files[year] = path

    return files


files_A = collect_files(FOLDER_A)
files_B = collect_files(FOLDER_B)

common_years = sorted(
    set(files_A) & set(files_B)
)

print("=" * 65)
print("GEOTIFF INTERSECTION MASKING")
print("=" * 65)

print(f"A files       : {len(files_A)}")
print(f"B files       : {len(files_B)}")
print(f"Matched years : {len(common_years)}")
print(f"Years         : {common_years}")

print("Missing A:", sorted(set(files_B) - set(files_A)))
print("Missing B:", sorted(set(files_A) - set(files_B)))

if not common_years:
    raise ValueError("No matched years found")


# ============================================================
# 4. CHECK GRID ALIGNMENT
# ============================================================

def check_grid(src_a, src_b):

    if src_a.crs is None or src_b.crs is None:
        raise ValueError("Missing raster CRS")

    if src_a.crs != src_b.crs:
        raise ValueError("CRS mismatch")

    if (
        src_a.width != src_b.width
        or src_a.height != src_b.height
    ):
        raise ValueError("Raster dimensions mismatch")

    if not src_a.transform.almost_equals(
        src_b.transform,
        precision=1e-10
    ):
        raise ValueError("Raster transform mismatch")

    if src_a.count != 1 or src_b.count != 1:
        raise ValueError("Only single-band GeoTIFF supported")


# ============================================================
# 5. READ DATA AND IDENTIFY VALID PIXELS
# ============================================================

def read_valid_data(src, additional_nodata):

    # Automatically mask metadata-defined NoData pixels
    data = src.read(1, masked=True)

    values = data.astype(np.float32).filled(np.nan)

    valid = np.isfinite(values)

    for nd in additional_nodata:
        valid &= (values != nd)

    return values, valid


# ============================================================
# 6. PROCESS ONE YEAR
# ============================================================

def process_year(year, path_a, path_b):

    output_path = OUTPUT_FOLDER / path_b.name

    if output_path.exists() and not OVERWRITE:

        print(f"{year}: SKIPPED - Already exists")

        return {
            "year": year,
            "status": "SKIPPED"
        }

    with rasterio.open(path_a) as src_a, \
         rasterio.open(path_b) as src_b:

        # Ensure identical grids
        check_grid(src_a, src_b)

        # ----------------------------------------------------
        # Read A
        # ----------------------------------------------------

        a_data, a_valid = read_valid_data(
            src_a,
            ADDITIONAL_NODATA_A
        )

        # Define mask
        if MASK_MODE == "positive":

            mask_a = (
                a_valid
                & (a_data > MASK_THRESHOLD)
            )

        else:
            mask_a = a_valid

        # ----------------------------------------------------
        # Read B
        # ----------------------------------------------------

        b_data, b_valid = read_valid_data(
            src_b,
            ADDITIONAL_NODATA_B
        )

        # ----------------------------------------------------
        # Intersection: A valid AND B valid
        # ----------------------------------------------------

        intersection_mask = mask_a & b_valid

        # Initialize all pixels as NoData
        output = np.full(
            b_data.shape,
            np.nan,
            dtype=np.float32
        )

        # Only copy valid B values inside A
        output[intersection_mask] = b_data[intersection_mask]

        # ----------------------------------------------------
        # Statistics
        # ----------------------------------------------------

        total_pixels = b_data.size

        a_pixels = int(np.count_nonzero(mask_a))

        b_pixels = int(np.count_nonzero(b_valid))

        intersection_pixels = int(
            np.count_nonzero(intersection_mask)
        )

        # A valid, B missing
        a_only = int(
            np.count_nonzero(mask_a & (~b_valid))
        )

        # B valid, outside A
        b_removed = int(
            np.count_nonzero((~mask_a) & b_valid)
        )

        # ----------------------------------------------------
        # Export GeoTIFF
        # ----------------------------------------------------

        profile = src_b.profile.copy()

        profile.update(
            driver="GTiff",
            dtype="float32",
            count=1,
            nodata=OUTPUT_NODATA,
            compress="deflate",
            predictor=3,
            tiled=True,
            blockxsize=256,
            blockysize=256,
            BIGTIFF="IF_SAFER"
        )

        with rasterio.open(
            output_path,
            "w",
            **profile
        ) as dst:

            dst.write(output, 1)

        print(
            f"{year} | "
            f"A={a_pixels:,} | "
            f"B={b_pixels:,} | "
            f"Intersection={intersection_pixels:,} | "
            f"Removed_B={b_removed:,} | "
            f"A_without_B={a_only:,}"
        )

        return {
            "year": year,
            "status": "SUCCESS",
            "total_pixels": total_pixels,
            "A_valid": a_pixels,
            "B_valid": b_pixels,
            "intersection": intersection_pixels,
            "B_removed": b_removed,
            "A_without_B": a_only,
            "output_file": str(output_path)
        }


# ============================================================
# 7. BATCH PROCESS
# ============================================================

results = []

for index, year in enumerate(common_years, start=1):

    print(f"\n[{index}/{len(common_years)}] Processing {year}")

    try:

        result = process_year(
            year,
            files_A[year],
            files_B[year]
        )

        results.append(result)

    except Exception as e:

        print(f"ERROR {year}: {e}")

        results.append({
            "year": year,
            "status": "FAILED",
            "error": str(e)
        })


# ============================================================
# 8. EXPORT PROCESSING LOG
# ============================================================

log_path = OUTPUT_FOLDER / "Intersection_Mask_Log.csv"

fieldnames = [
    "year",
    "status",
    "total_pixels",
    "A_valid",
    "B_valid",
    "intersection",
    "B_removed",
    "A_without_B",
    "output_file",
    "error"
]

with open(
    log_path,
    "w",
    newline="",
    encoding="utf-8-sig"
) as f:

    writer = csv.DictWriter(
        f,
        fieldnames=fieldnames
    )

    writer.writeheader()
    writer.writerows(results)


# ============================================================
# 9. FINAL SUMMARY
# ============================================================

print("\n" + "=" * 65)
print("INTERSECTION MASKING COMPLETED")
print("=" * 65)

for status in ("SUCCESS", "SKIPPED", "FAILED"):

    count = sum(
        r["status"] == status
        for r in results
    )

    print(f"{status}: {count}")

print(f"\nOutput folder: {OUTPUT_FOLDER}")
print(f"Log file: {log_path}")

print("=" * 65)


In [ ]:
# further refine, 
#现有代码可正常运行（异常值移除），优化要求：
#1. 是否执行标准差移除，增加一个true or false 的开关
#2. 超过数值上线识别，增加选择，选择移除，或选择将用最近邻值替代极大值。
#3. 低于数值下线识别，增加选择，选择移除，选择用最近邻替代。
#4. 导出文件保持不变，仍是识别到的栅格个数和比例。
# ============================================================
# 3remove parallel, NUE
# Global Wheat NUE: 3-step outlier removal, parallel by year
# Can be pasted into ONE JupyterLab cell (Windows + joblib/loky).
# Based on user's NaN-fixed code; statistical logic unchanged.
# ============================================================

import os
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from rasterio.features import geometry_mask
from joblib import Parallel, delayed
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

# ======================== CONFIG ============================

A_FOLDER = (
    # r"Z:\staff\Qiang_S\09_Datasets\03GlobalNUE_Results\01Global_NUE_Wheat\02WheatNUE_M2_20261021V2_Intersection"
    r"Z:\staff\Qiang_S\09_Datasets\03GlobalNUE_Results\02Global_NUE_Maize\02MaizeNUE_M2_20261022V1_Intersection"
)
MASK_FOLDER = (
     # r"Z:\staff\Qiang_S\09_Datasets\03GlobalNUE_Results\01Global_NUE_Wheat\02WheatNUE_M2_20261021V2"
    r"Z:\staff\Qiang_S\09_Datasets\03GlobalNUE_Results\02Global_NUE_Maize\02MaizeNUE_M2_20261022V1"
)
OUT_FOLDER = (
    r"Z:\staff\Qiang_S\09_Datasets\03GlobalNUE_Results\02Global_NUE_Maize\02MaizeNUE_M2_20261022V1_M6_180C"
)

SHP_PATH = (
    r"Z:\staff\Qiang_S\09_Datasets\09OtherDataUsed\GDAM_GlobalCountryLevel_Shp/GlobalCountryLevel_Shp.shp"
)

M49_FIELD = "M49_Code"
NAME_FIELD = "COUNTRY"
EXCEL_OUT = os.path.join(OUT_FOLDER, "outlier_removal_summary_V1_M6_180C.xlsx")

START_YEAR = 1995
END_YEAR = 2020
# None: all years in START_YEAR...END_YEAR; or e.g. [1995, 1997, 2005]
YEARS_TO_RUN = None
# Simultaneous years; start with 2 or 4 for files on mapped/network Z: drive
PARALLEL_YEARS = 4

NODATA = -9999
MIN_COUNT = 6
LOWER_HARD = 0.1
UPPER_HARD = 1.8
STD_FACTOR = 2.0
MAX_REMOVE_RATIO = 0.05

# ======================== HELPERS ===========================
def read_raster_fixed(src):
    data = src.read(1, masked=True).astype(np.float32).filled(np.nan)
    data = np.asarray(data, dtype=np.float32)
    invalid = (~np.isfinite(data)) | (data == NODATA)
    data[invalid] = NODATA
    return data


def get_valid_mask(data):
    return np.isfinite(data) & (data != NODATA)


def empty_country_record(year, m49, name):
    return {
        "Year": year, "M49": m49, "Country": name,
        "Valid_before_step3": 0, "Mean": np.nan, "Std": np.nan,
        "Upper_sigma": np.nan, "Candidate_removed": 0,
        "Max_allowed_remove": 0, "Step3_removed": 0,
        "Step3_removed_%": 0.0,
    }


def process_one_year(year):
    """Independent worker. Only returns small statistics, not raster arrays."""
    annual_record = None
    country_records = []
    A_tif = os.path.join(A_FOLDER, f"NUE_M2_median_{year}.tif")
    N_tif = os.path.join(MASK_FOLDER, f"NUE_M2_n_{year}.tif")
    OUT_tif = os.path.join(OUT_FOLDER, f"NUE_median_Opt_{year}.tif")

    try:
        if not os.path.exists(A_tif):
            return {"year": year, "status": "MISSING_A", "message": A_tif,
                    "annual": None, "countries": []}
        if not os.path.exists(N_tif):
            return {"year": year, "status": "MISSING_N", "message": N_tif,
                    "annual": None, "countries": []}

        # Load Shapefile independently inside each process (no GDAL objects shared)
        gdf = gpd.read_file(SHP_PATH)
        if gdf.crs is None:
            raise ValueError("Country shapefile has no CRS")
        if gdf.crs.to_epsg() != 4326:
            gdf = gdf.to_crs(epsg=4326)
        gdf = gdf.sort_values(by=M49_FIELD).reset_index(drop=True)

        with rasterio.open(A_tif) as src_a:
            A = read_raster_fixed(src_a)
            profile = src_a.profile.copy()
            transform = src_a.transform
            crs = src_a.crs
            height, width = A.shape

        with rasterio.open(N_tif) as src_n:
            N = read_raster_fixed(src_n)

        if A.shape != N.shape:
            raise ValueError(f"Shape mismatch: A={A.shape}, N={N.shape}")

        # ---------------- Step 1: N < 6 or N missing ----------------
        valid_mask = get_valid_mask(A)
        initial_count = int(np.count_nonzero(valid_mask))
        n_valid_mask = get_valid_mask(N)
        step1_mask = valid_mask & (~n_valid_mask | (N < MIN_COUNT))
        step1_removed = int(np.count_nonzero(step1_mask))
        A[step1_mask] = NODATA
        valid_mask = get_valid_mask(A)

        # ---------------- Step 2: hard thresholds -------------------
        step2_upper = (A > UPPER_HARD) & valid_mask
        step2_lower = (A < LOWER_HARD) & valid_mask
        step2_removed = int(np.count_nonzero(step2_upper) +
                            np.count_nonzero(step2_lower))
        A[step2_upper] = NODATA
        A[step2_lower] = NODATA
        valid_mask = get_valid_mask(A)

        # ---------------- Step 3: per-country 2 sigma ---------------
        step3_total_removed = 0
        if crs is None:
            raise ValueError("Raster CRS is missing")
        gdf_reproj = gdf.to_crs(crs) if gdf.crs != crs else gdf

        # Country order and sequential updates within one year are unchanged
        for _, country_row in gdf_reproj.iterrows():
            m49 = country_row[M49_FIELD]
            name = (country_row[NAME_FIELD] if NAME_FIELD and
                    NAME_FIELD in country_row else str(m49))
            geom = country_row.geometry
            if geom is None or geom.is_empty:
                country_records.append(empty_country_record(year, m49, name))
                continue
            try:
                country_mask_arr = ~geometry_mask(
                    [geom], transform=transform, invert=False,
                    out_shape=(height, width)
                )
            except Exception:
                country_records.append(empty_country_record(year, m49, name))
                continue

            in_country_valid = country_mask_arr & valid_mask
            pixel_indices = np.where(in_country_valid)
            n_valid = len(pixel_indices[0])
            if n_valid == 0:
                country_records.append(empty_country_record(year, m49, name))
                continue

            vals = A[pixel_indices]
            mean_val = float(np.mean(vals, dtype=np.float64))
            std_val = float(np.std(vals, dtype=np.float64))
            upper_sigma = mean_val + STD_FACTOR * std_val
            candidate_mask = in_country_valid & (A > upper_sigma)
            n_candidate = int(np.count_nonzero(candidate_mask))
            max_allowed = int(np.floor(n_valid * MAX_REMOVE_RATIO))
            if n_candidate <= max_allowed:
                actual_removed = n_candidate
                A[candidate_mask] = NODATA
            else:
                actual_removed = max_allowed
                if max_allowed > 0:
                    cand_rows, cand_cols = np.where(candidate_mask)
                    cand_vals = A[cand_rows, cand_cols]
                    sort_idx = np.argsort(-cand_vals)[:max_allowed]
                    A[cand_rows[sort_idx], cand_cols[sort_idx]] = NODATA

            valid_mask = get_valid_mask(A)
            step3_total_removed += actual_removed
            country_records.append({
                "Year": year, "M49": m49, "Country": name,
                "Valid_before_step3": n_valid,
                "Mean": round(mean_val, 6),
                "Std": round(std_val, 6),
                "Upper_sigma": round(upper_sigma, 6),
                "Candidate_removed": n_candidate,
                "Max_allowed_remove": max_allowed,
                "Step3_removed": actual_removed,
                "Step3_removed_%": (round(actual_removed / n_valid * 100, 4)
                                       if n_valid > 0 else 0.0),
            })

        # ---------------- Annual summary / save ---------------------
        final_count = int(np.count_nonzero(get_valid_mask(A)))
        total_removed = initial_count - final_count
        removed_ratio = (total_removed / initial_count * 100
                         if initial_count > 0 else 0)
        expected_removed = step1_removed + step2_removed + step3_total_removed
        count_check = total_removed - expected_removed
        annual_record = {
            "Year": year, "Initial_Valid": initial_count,
            "Step1_removed": step1_removed,
            "Step1_%": (round(step1_removed / initial_count * 100, 4)
                        if initial_count else 0),
            "Step2_removed": step2_removed,
            "Step2_%": (round(step2_removed / initial_count * 100, 4)
                        if initial_count else 0),
            "Step3_removed": step3_total_removed,
            "Step3_%": (round(step3_total_removed / initial_count * 100, 4)
                        if initial_count else 0),
            "Final_Valid": final_count,
            "Total_Removed": total_removed,
            "Removed_Ratio_%": round(removed_ratio, 4),
        }

        profile.update(dtype="float32", nodata=NODATA, compress="lzw")
        with rasterio.open(OUT_tif, "w", **profile) as dst:
            dst.write(A.astype(np.float32), 1)

        return {"year": year, "status": "SUCCESS", "message": "",
                "annual": annual_record, "countries": country_records,
                "check": count_check, "output": OUT_tif}

    except Exception as e:
        return {"year": year, "status": "FAILED", "message": repr(e),
                "annual": None, "countries": []}


# =================== PARALLEL PROCESSING ===================
os.makedirs(OUT_FOLDER, exist_ok=True)
if not os.path.isfile(SHP_PATH):
    raise FileNotFoundError(f"SHP file not found: {SHP_PATH}")
if not isinstance(PARALLEL_YEARS, int) or PARALLEL_YEARS < 1:
    raise ValueError("PARALLEL_YEARS must be an integer >= 1")

YEARS = (list(range(START_YEAR, END_YEAR + 1))
         if YEARS_TO_RUN is None else sorted(set(map(int, YEARS_TO_RUN))))
if not YEARS:
    raise ValueError("No years requested")

print("=" * 70)
print("Global Wheat NUE - Parallel 3-Step Outlier Removal")
print(f"Years: {YEARS}")
print(f"Concurrent years: {min(PARALLEL_YEARS, len(YEARS))}")
print("Processing model: joblib loky processes (Windows/Jupyter supported)")
print("=" * 70, flush=True)

# When only one year is requested, avoid worker startup overhead.
if PARALLEL_YEARS == 1 or len(YEARS) == 1:
    results = [process_one_year(y) for y in YEARS]
else:
    results = Parallel(
        n_jobs=min(PARALLEL_YEARS, len(YEARS)),
        backend="loky",
        batch_size=1,
        pre_dispatch="n_jobs",
        verbose=10,
    )(delayed(process_one_year)(y) for y in YEARS)

results.sort(key=lambda item: item["year"])
records_year = []
records_country = []
for result in results:
    year = result["year"]
    status = result["status"]
    if status == "SUCCESS":
        records_year.append(result["annual"])
        records_country.extend(result["countries"])
        a = result["annual"]
        print(f"[{year}] Initial={a['Initial_Valid']:,} | "
              f"Step1={a['Step1_removed']:,} | Step2={a['Step2_removed']:,} | "
              f"Step3={a['Step3_removed']:,} | Final={a['Final_Valid']:,} | "
              f"Check={result['check']}")
    else:
        print(f"[{year}] {status}: {result['message']}")

# ======================== EXCEL REPORT =====================
# Excel output is written by the MAIN process only.
if not records_year:
    print("No successful years: Excel not generated.")
else:
    df_year = pd.DataFrame(records_year).sort_values("Year").reset_index(drop=True)
    df_country = (pd.DataFrame(records_country)
                  .sort_values(["Year", "M49"])
                  .reset_index(drop=True))
    wb = Workbook()
    thin = Side(style="thin", color="AAAAAA")
    border = Border(left=thin, right=thin, top=thin, bottom=thin)

    def hdr_cell(ws, row, col, value, bg="2F75B6", fg="FFFFFF"):
        c = ws.cell(row=row, column=col, value=value)
        c.font = Font(bold=True, color=fg, name="Arial", size=10)
        c.fill = PatternFill("solid", start_color=bg)
        c.alignment = Alignment(horizontal="center", wrap_text=True)
        c.border = border
        return c

    def data_cell(ws, row, col, value, bg="FFFFFF", fmt=None):
        if isinstance(value, np.generic):
            value = value.item()
        c = ws.cell(row=row, column=col, value=value)
        c.font = Font(name="Arial", size=10)
        c.fill = PatternFill("solid", start_color=bg)
        c.alignment = Alignment(horizontal="center")
        c.border = border
        if fmt:
            c.number_format = fmt
        return c

    # Sheet 1 - Annual Summary (original fields, colors and total formulas)
    ws1 = wb.active
    ws1.title = "Annual Summary"
    col_colors_year = {
        "Year": "FFFFFF", "Initial_Valid": "FFFFFF",
        "Step1_removed": "D6E4F0", "Step1_%": "D6E4F0",
        "Step2_removed": "D5E8D4", "Step2_%": "D5E8D4",
        "Step3_removed": "FFF2CC", "Step3_%": "FFF2CC",
        "Final_Valid": "FFFFFF", "Total_Removed": "F4CCCC",
        "Removed_Ratio_%": "F4CCCC",
    }
    cols_y = list(df_year.columns)
    for ci, col in enumerate(cols_y, 1):
        hdr_cell(ws1, 1, ci, col)
    for ri, row in enumerate(df_year.itertuples(index=False), 2):
        for ci, (col, val) in enumerate(zip(cols_y, row), 1):
            fmt = "0.00" if "%" in col else ("#,##0" if col != "Year" else "0")
            data_cell(ws1, ri, ci, val, bg=col_colors_year.get(col, "FFFFFF"), fmt=fmt)

    sr = len(df_year) + 2
    data_cell(ws1, sr, 1, "TOTAL/AVG", bg="D9D9D9").font = Font(
        bold=True, name="Arial", size=10)
    for ci, col in enumerate(cols_y[1:], 2):
        cl = get_column_letter(ci)
        rng = f"{cl}2:{cl}{len(df_year) + 1}"
        formula = f"=AVERAGE({rng})" if "%" in col else f"=SUM({rng})"
        c = ws1.cell(row=sr, column=ci, value=formula)
        c.font = Font(bold=True, name="Arial", size=10)
        c.fill = PatternFill("solid", start_color="D9D9D9")
        c.alignment = Alignment(horizontal="center")
        c.border = border
        c.number_format = "0.00" if "%" in col else "#,##0"
    for ci, col in enumerate(cols_y, 1):
        ws1.column_dimensions[get_column_letter(ci)].width = max(len(col) + 4, 14)
    ws1.row_dimensions[1].height = 36
    ws1.freeze_panes = "B2"

    # Sheet 2 - Country Detail
    ws2 = wb.create_sheet("Country Detail")
    cols_c = list(df_country.columns)
    col_colors_cty = {
        "Year": "FFFFFF", "M49": "FFFFFF", "Country": "FFFFFF",
        "Valid_before_step3": "FFFFFF", "Mean": "EAD1DC",
        "Std": "EAD1DC", "Upper_sigma": "EAD1DC",
        "Candidate_removed": "FFF2CC", "Max_allowed_remove": "FFF2CC",
        "Step3_removed": "F4CCCC", "Step3_removed_%": "F4CCCC",
    }
    for ci, col in enumerate(cols_c, 1):
        hdr_cell(ws2, 1, ci, col)
    for ri, row in enumerate(df_country.itertuples(index=False), 2):
        for ci, (col, val) in enumerate(zip(cols_c, row), 1):
            if "%" in col:
                fmt = "0.00"
            elif col in ("Mean", "Std", "Upper_sigma"):
                fmt = "0.000000"
            elif col not in ("Year", "M49", "Country"):
                fmt = "#,##0"
            else:
                fmt = None
            display_val = (None if isinstance(val, (float, np.floating))
                           and np.isnan(val) else val)
            data_cell(ws2, ri, ci, display_val,
                      bg=col_colors_cty.get(col, "FFFFFF"), fmt=fmt)
    for ci, col in enumerate(cols_c, 1):
        ws2.column_dimensions[get_column_letter(ci)].width = max(len(col) + 3, 12)
    ws2.column_dimensions["C"].width = 22
    ws2.row_dimensions[1].height = 36
    ws2.freeze_panes = "D2"

    # Sheet 3 - Parameters (original parameters + parallel settings)
    ws3 = wb.create_sheet("Parameters")
    params = [
        ("Parameter", "Value"),
        ("MIN_COUNT (Step1)", MIN_COUNT),
        ("LOWER_HARD (Step2)", LOWER_HARD),
        ("UPPER_HARD (Step2)", UPPER_HARD),
        ("STD_FACTOR (Step3)", STD_FACTOR),
        ("MAX_REMOVE_RATIO (Step3)", MAX_REMOVE_RATIO),
        ("M49_FIELD", M49_FIELD),
        ("SHP_PATH", SHP_PATH),
        ("START_YEAR", START_YEAR),
        ("END_YEAR", END_YEAR),
        ("NODATA", NODATA),
        ("PARALLEL_YEARS", PARALLEL_YEARS),
        ("YEARS_TO_RUN", "All" if YEARS_TO_RUN is None else str(YEARS)),
    ]
    for ri, (k, v) in enumerate(params, 1):
        for ci, val in enumerate((k, v), 1):
            c = ws3.cell(row=ri, column=ci, value=val)
            c.font = Font(bold=(ri == 1), name="Arial", size=10,
                          color="FFFFFF" if ri == 1 else "000000")
            c.fill = PatternFill("solid", start_color="2F75B6" if ri == 1 else "F5F5F5")
            c.border = border
            c.alignment = Alignment(horizontal="left")
    ws3.column_dimensions["A"].width = 28
    ws3.column_dimensions["B"].width = 50
    wb.save(EXCEL_OUT)
    print(f"Excel saved: {EXCEL_OUT}")

    # Original count reconciliation, now performed after merging years
    df_check = df_year.copy()
    df_check["Check"] = (df_check["Initial_Valid"] - df_check["Step1_removed"]
                         - df_check["Step2_removed"] - df_check["Step3_removed"]
                         - df_check["Final_Valid"])
    print("\nFINAL VALIDATION SUMMARY")
    print(df_check[["Year", "Initial_Valid", "Step1_removed", "Step2_removed",
                    "Step3_removed", "Final_Valid", "Check"]].to_string(index=False))
    print("PASS: All annual pixel counts are consistent."
          if (df_check["Check"] == 0).all()
          else "WARNING: Some annual pixel counts are inconsistent.")

print("\nAll requested years finished.")

In [ ]:
# ============================================================
# 3remove parallel, PFPN
# Global Wheat NUE: 3-step outlier removal, parallel by year
# Can be pasted into ONE JupyterLab cell (Windows + joblib/loky).
# Based on user's NaN-fixed code; statistical logic unchanged.
# ============================================================

import os
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from rasterio.features import geometry_mask
from joblib import Parallel, delayed
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

# ======================== CONFIG ============================

A_FOLDER = (
    r"Z:\staff\Qiang_S\09_Datasets\03GlobalNUE_Results\01Global_NUE_Wheat\02WheatNUE_M2_20261021V2_PFPN_Intersection"
)
MASK_FOLDER = (
    r"Z:\staff\Qiang_S\09_Datasets\03GlobalNUE_Results\01Global_NUE_Wheat\02WheatNUE_M2_20261021V2"
)
OUT_FOLDER = (
    r"Z:\staff\Qiang_S\09_Datasets\03GlobalNUE_Results\01Global_NUE_Wheat\02WheatNUE_M2_20261021V2PFPN_M6_500C"
)

SHP_PATH = (
    r"Z:\staff\Qiang_S\09_Datasets\09OtherDataUsed\GDAM_GlobalCountryLevel_Shp/GlobalCountryLevel_Shp.shp"
)

M49_FIELD = "M49_Code"
NAME_FIELD = "COUNTRY"
EXCEL_OUT = os.path.join(OUT_FOLDER, "outlier_removal_summary_V1_M6_500C.xlsx")

START_YEAR = 1995
END_YEAR = 2020
# None: all years in START_YEAR...END_YEAR; or e.g. [1995, 1997, 2005]
YEARS_TO_RUN = None
# Simultaneous years; start with 2 or 4 for files on mapped/network Z: drive
PARALLEL_YEARS = 4

NODATA = -9999
MIN_COUNT = 6
LOWER_HARD = 1
UPPER_HARD = 200
STD_FACTOR = 2.0
MAX_REMOVE_RATIO = 0.05

# ======================== HELPERS ===========================
def read_raster_fixed(src):
    data = src.read(1, masked=True).astype(np.float32).filled(np.nan)
    data = np.asarray(data, dtype=np.float32)
    invalid = (~np.isfinite(data)) | (data == NODATA)
    data[invalid] = NODATA
    return data


def get_valid_mask(data):
    return np.isfinite(data) & (data != NODATA)


def empty_country_record(year, m49, name):
    return {
        "Year": year, "M49": m49, "Country": name,
        "Valid_before_step3": 0, "Mean": np.nan, "Std": np.nan,
        "Upper_sigma": np.nan, "Candidate_removed": 0,
        "Max_allowed_remove": 0, "Step3_removed": 0,
        "Step3_removed_%": 0.0,
    }


def process_one_year(year):
    """Independent worker. Only returns small statistics, not raster arrays."""
    annual_record = None
    country_records = []
    A_tif = os.path.join(A_FOLDER, f"PFPN_M2_median_{year}.tif")
    N_tif = os.path.join(MASK_FOLDER, f"NUE_M2_n_{year}.tif")
    OUT_tif = os.path.join(OUT_FOLDER, f"NUE_median_Opt_{year}.tif")

    try:
        if not os.path.exists(A_tif):
            return {"year": year, "status": "MISSING_A", "message": A_tif,
                    "annual": None, "countries": []}
        if not os.path.exists(N_tif):
            return {"year": year, "status": "MISSING_N", "message": N_tif,
                    "annual": None, "countries": []}

        # Load Shapefile independently inside each process (no GDAL objects shared)
        gdf = gpd.read_file(SHP_PATH)
        if gdf.crs is None:
            raise ValueError("Country shapefile has no CRS")
        if gdf.crs.to_epsg() != 4326:
            gdf = gdf.to_crs(epsg=4326)
        gdf = gdf.sort_values(by=M49_FIELD).reset_index(drop=True)

        with rasterio.open(A_tif) as src_a:
            A = read_raster_fixed(src_a)
            profile = src_a.profile.copy()
            transform = src_a.transform
            crs = src_a.crs
            height, width = A.shape

        with rasterio.open(N_tif) as src_n:
            N = read_raster_fixed(src_n)

        if A.shape != N.shape:
            raise ValueError(f"Shape mismatch: A={A.shape}, N={N.shape}")

        # ---------------- Step 1: N < 6 or N missing ----------------
        valid_mask = get_valid_mask(A)
        initial_count = int(np.count_nonzero(valid_mask))
        n_valid_mask = get_valid_mask(N)
        step1_mask = valid_mask & (~n_valid_mask | (N < MIN_COUNT))
        step1_removed = int(np.count_nonzero(step1_mask))
        A[step1_mask] = NODATA
        valid_mask = get_valid_mask(A)

        # ---------------- Step 2: hard thresholds -------------------
        step2_upper = (A > UPPER_HARD) & valid_mask
        step2_lower = (A < LOWER_HARD) & valid_mask
        step2_removed = int(np.count_nonzero(step2_upper) +
                            np.count_nonzero(step2_lower))
        A[step2_upper] = NODATA
        A[step2_lower] = NODATA
        valid_mask = get_valid_mask(A)

        # ---------------- Step 3: per-country 2 sigma ---------------
        step3_total_removed = 0
        if crs is None:
            raise ValueError("Raster CRS is missing")
        gdf_reproj = gdf.to_crs(crs) if gdf.crs != crs else gdf

        # Country order and sequential updates within one year are unchanged
        for _, country_row in gdf_reproj.iterrows():
            m49 = country_row[M49_FIELD]
            name = (country_row[NAME_FIELD] if NAME_FIELD and
                    NAME_FIELD in country_row else str(m49))
            geom = country_row.geometry
            if geom is None or geom.is_empty:
                country_records.append(empty_country_record(year, m49, name))
                continue
            try:
                country_mask_arr = ~geometry_mask(
                    [geom], transform=transform, invert=False,
                    out_shape=(height, width)
                )
            except Exception:
                country_records.append(empty_country_record(year, m49, name))
                continue

            in_country_valid = country_mask_arr & valid_mask
            pixel_indices = np.where(in_country_valid)
            n_valid = len(pixel_indices[0])
            if n_valid == 0:
                country_records.append(empty_country_record(year, m49, name))
                continue

            vals = A[pixel_indices]
            mean_val = float(np.mean(vals, dtype=np.float64))
            std_val = float(np.std(vals, dtype=np.float64))
            upper_sigma = mean_val + STD_FACTOR * std_val
            candidate_mask = in_country_valid & (A > upper_sigma)
            n_candidate = int(np.count_nonzero(candidate_mask))
            max_allowed = int(np.floor(n_valid * MAX_REMOVE_RATIO))
            if n_candidate <= max_allowed:
                actual_removed = n_candidate
                A[candidate_mask] = NODATA
            else:
                actual_removed = max_allowed
                if max_allowed > 0:
                    cand_rows, cand_cols = np.where(candidate_mask)
                    cand_vals = A[cand_rows, cand_cols]
                    sort_idx = np.argsort(-cand_vals)[:max_allowed]
                    A[cand_rows[sort_idx], cand_cols[sort_idx]] = NODATA

            valid_mask = get_valid_mask(A)
            step3_total_removed += actual_removed
            country_records.append({
                "Year": year, "M49": m49, "Country": name,
                "Valid_before_step3": n_valid,
                "Mean": round(mean_val, 6),
                "Std": round(std_val, 6),
                "Upper_sigma": round(upper_sigma, 6),
                "Candidate_removed": n_candidate,
                "Max_allowed_remove": max_allowed,
                "Step3_removed": actual_removed,
                "Step3_removed_%": (round(actual_removed / n_valid * 100, 4)
                                       if n_valid > 0 else 0.0),
            })

        # ---------------- Annual summary / save ---------------------
        final_count = int(np.count_nonzero(get_valid_mask(A)))
        total_removed = initial_count - final_count
        removed_ratio = (total_removed / initial_count * 100
                         if initial_count > 0 else 0)
        expected_removed = step1_removed + step2_removed + step3_total_removed
        count_check = total_removed - expected_removed
        annual_record = {
            "Year": year, "Initial_Valid": initial_count,
            "Step1_removed": step1_removed,
            "Step1_%": (round(step1_removed / initial_count * 100, 4)
                        if initial_count else 0),
            "Step2_removed": step2_removed,
            "Step2_%": (round(step2_removed / initial_count * 100, 4)
                        if initial_count else 0),
            "Step3_removed": step3_total_removed,
            "Step3_%": (round(step3_total_removed / initial_count * 100, 4)
                        if initial_count else 0),
            "Final_Valid": final_count,
            "Total_Removed": total_removed,
            "Removed_Ratio_%": round(removed_ratio, 4),
        }

        profile.update(dtype="float32", nodata=NODATA, compress="lzw")
        with rasterio.open(OUT_tif, "w", **profile) as dst:
            dst.write(A.astype(np.float32), 1)

        return {"year": year, "status": "SUCCESS", "message": "",
                "annual": annual_record, "countries": country_records,
                "check": count_check, "output": OUT_tif}

    except Exception as e:
        return {"year": year, "status": "FAILED", "message": repr(e),
                "annual": None, "countries": []}


# =================== PARALLEL PROCESSING ===================
os.makedirs(OUT_FOLDER, exist_ok=True)
if not os.path.isfile(SHP_PATH):
    raise FileNotFoundError(f"SHP file not found: {SHP_PATH}")
if not isinstance(PARALLEL_YEARS, int) or PARALLEL_YEARS < 1:
    raise ValueError("PARALLEL_YEARS must be an integer >= 1")

YEARS = (list(range(START_YEAR, END_YEAR + 1))
         if YEARS_TO_RUN is None else sorted(set(map(int, YEARS_TO_RUN))))
if not YEARS:
    raise ValueError("No years requested")

print("=" * 70)
print("Global Wheat NUE - Parallel 3-Step Outlier Removal")
print(f"Years: {YEARS}")
print(f"Concurrent years: {min(PARALLEL_YEARS, len(YEARS))}")
print("Processing model: joblib loky processes (Windows/Jupyter supported)")
print("=" * 70, flush=True)

# When only one year is requested, avoid worker startup overhead.
if PARALLEL_YEARS == 1 or len(YEARS) == 1:
    results = [process_one_year(y) for y in YEARS]
else:
    results = Parallel(
        n_jobs=min(PARALLEL_YEARS, len(YEARS)),
        backend="loky",
        batch_size=1,
        pre_dispatch="n_jobs",
        verbose=10,
    )(delayed(process_one_year)(y) for y in YEARS)

results.sort(key=lambda item: item["year"])
records_year = []
records_country = []
for result in results:
    year = result["year"]
    status = result["status"]
    if status == "SUCCESS":
        records_year.append(result["annual"])
        records_country.extend(result["countries"])
        a = result["annual"]
        print(f"[{year}] Initial={a['Initial_Valid']:,} | "
              f"Step1={a['Step1_removed']:,} | Step2={a['Step2_removed']:,} | "
              f"Step3={a['Step3_removed']:,} | Final={a['Final_Valid']:,} | "
              f"Check={result['check']}")
    else:
        print(f"[{year}] {status}: {result['message']}")

# ======================== EXCEL REPORT =====================
# Excel output is written by the MAIN process only.
if not records_year:
    print("No successful years: Excel not generated.")
else:
    df_year = pd.DataFrame(records_year).sort_values("Year").reset_index(drop=True)
    df_country = (pd.DataFrame(records_country)
                  .sort_values(["Year", "M49"])
                  .reset_index(drop=True))
    wb = Workbook()
    thin = Side(style="thin", color="AAAAAA")
    border = Border(left=thin, right=thin, top=thin, bottom=thin)

    def hdr_cell(ws, row, col, value, bg="2F75B6", fg="FFFFFF"):
        c = ws.cell(row=row, column=col, value=value)
        c.font = Font(bold=True, color=fg, name="Arial", size=10)
        c.fill = PatternFill("solid", start_color=bg)
        c.alignment = Alignment(horizontal="center", wrap_text=True)
        c.border = border
        return c

    def data_cell(ws, row, col, value, bg="FFFFFF", fmt=None):
        if isinstance(value, np.generic):
            value = value.item()
        c = ws.cell(row=row, column=col, value=value)
        c.font = Font(name="Arial", size=10)
        c.fill = PatternFill("solid", start_color=bg)
        c.alignment = Alignment(horizontal="center")
        c.border = border
        if fmt:
            c.number_format = fmt
        return c

    # Sheet 1 - Annual Summary (original fields, colors and total formulas)
    ws1 = wb.active
    ws1.title = "Annual Summary"
    col_colors_year = {
        "Year": "FFFFFF", "Initial_Valid": "FFFFFF",
        "Step1_removed": "D6E4F0", "Step1_%": "D6E4F0",
        "Step2_removed": "D5E8D4", "Step2_%": "D5E8D4",
        "Step3_removed": "FFF2CC", "Step3_%": "FFF2CC",
        "Final_Valid": "FFFFFF", "Total_Removed": "F4CCCC",
        "Removed_Ratio_%": "F4CCCC",
    }
    cols_y = list(df_year.columns)
    for ci, col in enumerate(cols_y, 1):
        hdr_cell(ws1, 1, ci, col)
    for ri, row in enumerate(df_year.itertuples(index=False), 2):
        for ci, (col, val) in enumerate(zip(cols_y, row), 1):
            fmt = "0.00" if "%" in col else ("#,##0" if col != "Year" else "0")
            data_cell(ws1, ri, ci, val, bg=col_colors_year.get(col, "FFFFFF"), fmt=fmt)

    sr = len(df_year) + 2
    data_cell(ws1, sr, 1, "TOTAL/AVG", bg="D9D9D9").font = Font(
        bold=True, name="Arial", size=10)
    for ci, col in enumerate(cols_y[1:], 2):
        cl = get_column_letter(ci)
        rng = f"{cl}2:{cl}{len(df_year) + 1}"
        formula = f"=AVERAGE({rng})" if "%" in col else f"=SUM({rng})"
        c = ws1.cell(row=sr, column=ci, value=formula)
        c.font = Font(bold=True, name="Arial", size=10)
        c.fill = PatternFill("solid", start_color="D9D9D9")
        c.alignment = Alignment(horizontal="center")
        c.border = border
        c.number_format = "0.00" if "%" in col else "#,##0"
    for ci, col in enumerate(cols_y, 1):
        ws1.column_dimensions[get_column_letter(ci)].width = max(len(col) + 4, 14)
    ws1.row_dimensions[1].height = 36
    ws1.freeze_panes = "B2"

    # Sheet 2 - Country Detail
    ws2 = wb.create_sheet("Country Detail")
    cols_c = list(df_country.columns)
    col_colors_cty = {
        "Year": "FFFFFF", "M49": "FFFFFF", "Country": "FFFFFF",
        "Valid_before_step3": "FFFFFF", "Mean": "EAD1DC",
        "Std": "EAD1DC", "Upper_sigma": "EAD1DC",
        "Candidate_removed": "FFF2CC", "Max_allowed_remove": "FFF2CC",
        "Step3_removed": "F4CCCC", "Step3_removed_%": "F4CCCC",
    }
    for ci, col in enumerate(cols_c, 1):
        hdr_cell(ws2, 1, ci, col)
    for ri, row in enumerate(df_country.itertuples(index=False), 2):
        for ci, (col, val) in enumerate(zip(cols_c, row), 1):
            if "%" in col:
                fmt = "0.00"
            elif col in ("Mean", "Std", "Upper_sigma"):
                fmt = "0.000000"
            elif col not in ("Year", "M49", "Country"):
                fmt = "#,##0"
            else:
                fmt = None
            display_val = (None if isinstance(val, (float, np.floating))
                           and np.isnan(val) else val)
            data_cell(ws2, ri, ci, display_val,
                      bg=col_colors_cty.get(col, "FFFFFF"), fmt=fmt)
    for ci, col in enumerate(cols_c, 1):
        ws2.column_dimensions[get_column_letter(ci)].width = max(len(col) + 3, 12)
    ws2.column_dimensions["C"].width = 22
    ws2.row_dimensions[1].height = 36
    ws2.freeze_panes = "D2"

    # Sheet 3 - Parameters (original parameters + parallel settings)
    ws3 = wb.create_sheet("Parameters")
    params = [
        ("Parameter", "Value"),
        ("MIN_COUNT (Step1)", MIN_COUNT),
        ("LOWER_HARD (Step2)", LOWER_HARD),
        ("UPPER_HARD (Step2)", UPPER_HARD),
        ("STD_FACTOR (Step3)", STD_FACTOR),
        ("MAX_REMOVE_RATIO (Step3)", MAX_REMOVE_RATIO),
        ("M49_FIELD", M49_FIELD),
        ("SHP_PATH", SHP_PATH),
        ("START_YEAR", START_YEAR),
        ("END_YEAR", END_YEAR),
        ("NODATA", NODATA),
        ("PARALLEL_YEARS", PARALLEL_YEARS),
        ("YEARS_TO_RUN", "All" if YEARS_TO_RUN is None else str(YEARS)),
    ]
    for ri, (k, v) in enumerate(params, 1):
        for ci, val in enumerate((k, v), 1):
            c = ws3.cell(row=ri, column=ci, value=val)
            c.font = Font(bold=(ri == 1), name="Arial", size=10,
                          color="FFFFFF" if ri == 1 else "000000")
            c.fill = PatternFill("solid", start_color="2F75B6" if ri == 1 else "F5F5F5")
            c.border = border
            c.alignment = Alignment(horizontal="left")
    ws3.column_dimensions["A"].width = 28
    ws3.column_dimensions["B"].width = 50
    wb.save(EXCEL_OUT)
    print(f"Excel saved: {EXCEL_OUT}")

    # Original count reconciliation, now performed after merging years
    df_check = df_year.copy()
    df_check["Check"] = (df_check["Initial_Valid"] - df_check["Step1_removed"]
                         - df_check["Step2_removed"] - df_check["Step3_removed"]
                         - df_check["Final_Valid"])
    print("\nFINAL VALIDATION SUMMARY")
    print(df_check[["Year", "Initial_Valid", "Step1_removed", "Step2_removed",
                    "Step3_removed", "Final_Valid", "Check"]].to_string(index=False))
    print("PASS: All annual pixel counts are consistent."
          if (df_check["Check"] == 0).all()
          else "WARNING: Some annual pixel counts are inconsistent.")

print("\nAll requested years finished.")

In [ ]:
# =======2026.10.22=====================================================
# Global NUE - Parallel Outlier Processing Options 4PFPN

# Step 1: N < MIN_COUNT or N missing -> REMOVE

# Step 2:
#   Upper threshold -> REMOVE / NEAREST
#   Lower threshold -> REMOVE / NEAREST
#
# Step 3:
#   Optional country-level mean + STD_FACTOR * STD
#   -> REMOVE / NEAREST
#   Maximum selected proportion per country: 5%
#
# NaN / NoData safe
# Parallel processing by year
# Output: GeoTIFF + Excel (3 original sheets)
#
# JupyterLab single-cell executable
# ============================================================

import os
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio

from rasterio.features import geometry_mask
from scipy.ndimage import distance_transform_edt
from joblib import Parallel, delayed

from openpyxl import Workbook
from openpyxl.styles import (
    Font, PatternFill, Alignment, Border, Side
)
from openpyxl.utils import get_column_letter

# ============================================================
# 1. USER CONFIGURATION
# ===========================================================
# ------------------------------------------------------------
# Input / output
# ------------------------------------------------------------

A_FOLDER = (
    r"Z:\staff\Qiang_S\09_Datasets\03GlobalNUE_Results\01Global_NUE_Wheat\02WheatNUE_M2_20261021V2_PFPN_Intersection"
)

MASK_FOLDER = (
   r"Z:\staff\Qiang_S\09_Datasets\03GlobalNUE_Results\01Global_NUE_Wheat\02WheatNUE_M2_20261021V2"
)

OUT_FOLDER = (
    r"Z:\staff\Qiang_S\09_Datasets\03GlobalNUE_Results\01Global_NUE_Wheat\02MaizeNUE_M2_20261021V2PFPN_M6_200C"
)

SHP_PATH = (
    r"Z:\staff\Qiang_S\09_Datasets"
    r"\09OtherDataUsed\GDAM_GlobalCountryLevel_Shp"
    r"\GlobalCountryLevel_Shp.shp"
)



M49_FIELD = "M49_Code"
NAME_FIELD = "COUNTRY"

EXCEL_OUT = os.path.join(
    OUT_FOLDER,
    "outlier_removal_summary_V1_M6_160C.xlsx"
)

# ------------------------------------------------------------
# Years and parallel processing
# ------------------------------------------------------------

START_YEAR = 1995
END_YEAR = 2020

# None = all years
# Example: [1995, 1996, 2000, 2020]
YEARS_TO_RUN = None

PARALLEL_YEARS = 5

# ------------------------------------------------------------
# Common settings
# ------------------------------------------------------------

NODATA = -9999.0

# ------------------------------------------------------------
# Step 1: Minimum valid sample count
# ------------------------------------------------------------
MIN_COUNT = 6

# ------------------------------------------------------------
# Step 2: Hard thresholds
# ------------------------------------------------------------
LOWER_HARD = 1
UPPER_HARD = 200

# Options: "REMOVE" or "NEAREST"
UPPER_OUTLIER_ACTION = "NEAREST"
LOWER_OUTLIER_ACTION = "NEAREST"

# ------------------------------------------------------------
# Step 3: Country standard deviation
# ------------------------------------------------------------

# True = enable country standard deviation processing
# False = skip Step 3

APPLY_STD_REMOVAL = False #True

# Mean + STD_FACTOR * standard deviation
STD_FACTOR = 2.0
# Maximum selected outlier ratio per country
MAX_REMOVE_RATIO = 0.05
# NEW:
# "REMOVE"  = Set selected Step 3 outliers to NoData
# "NEAREST" = Replace with nearest qualified value inside the SAME country

STD_OUTLIER_ACTION = "REMOVE"

# ------------------------------------------------------------
# Nearest-neighbor settings
# ------------------------------------------------------------

# None = No maximum distance
# Example: 12 = maximum 12 raster pixels
MAX_NEAREST_DISTANCE_PIXELS = None

# If no qualified nearest value exists,
# the target becomes NoData.


# ============================================================
# 2. PARAMETER VALIDATION
# ============================================================

VALID_ACTIONS = {"REMOVE", "NEAREST"}

UPPER_OUTLIER_ACTION = UPPER_OUTLIER_ACTION.upper()
LOWER_OUTLIER_ACTION = LOWER_OUTLIER_ACTION.upper()
STD_OUTLIER_ACTION = STD_OUTLIER_ACTION.upper()

for name, action in [
    ("UPPER_OUTLIER_ACTION", UPPER_OUTLIER_ACTION),
    ("LOWER_OUTLIER_ACTION", LOWER_OUTLIER_ACTION),
    ("STD_OUTLIER_ACTION", STD_OUTLIER_ACTION)
]:
    if action not in VALID_ACTIONS:
        raise ValueError(
            f"{name} must be REMOVE or NEAREST"
        )

if LOWER_HARD >= UPPER_HARD:
    raise ValueError(
        "LOWER_HARD must be smaller than UPPER_HARD"
    )

if PARALLEL_YEARS < 1:
    raise ValueError("PARALLEL_YEARS must be >= 1")

if not (0 <= MAX_REMOVE_RATIO <= 1):
    raise ValueError(
        "MAX_REMOVE_RATIO must be between 0 and 1"
    )

if (
    MAX_NEAREST_DISTANCE_PIXELS is not None
    and MAX_NEAREST_DISTANCE_PIXELS < 0
):
    raise ValueError(
        "MAX_NEAREST_DISTANCE_PIXELS must be >= 0"
    )

if not os.path.isfile(SHP_PATH):
    raise FileNotFoundError(SHP_PATH)

os.makedirs(OUT_FOLDER, exist_ok=True)


# ============================================================
# 3. RASTER UTILITY FUNCTIONS
# ============================================================

def read_raster_fixed(src):
    """
    Read raster and standardize:
      Metadata NoData
      NaN / Inf
      -9999

    Valid values remain unchanged.
    """

    data = (
        src.read(1, masked=True)
        .astype(np.float32)
        .filled(np.nan)
    )

    data = np.asarray(
        data,
        dtype=np.float32
    )

    invalid = (
        ~np.isfinite(data)
        | (data == NODATA)
    )

    data[invalid] = NODATA

    return data


def get_valid_mask(data):

    return (
        np.isfinite(data)
        & (data != NODATA)
    )


def empty_country_record(year, m49, name):

    return {
        "Year": year,
        "M49": m49,
        "Country": name,
        "Valid_before_step3": 0,
        "Mean": np.nan,
        "Std": np.nan,
        "Upper_sigma": np.nan,
        "Candidate_removed": 0,
        "Max_allowed_remove": 0,
        "Step3_removed": 0,
        "Step3_removed_%": 0.0
    }


# ============================================================
# 4. NEAREST-NEIGHBOR REPLACEMENT
# ============================================================

def replace_by_nearest(
    raster,
    target_mask,
    donor_mask,
    max_distance=None
):
    """
    Replace selected target pixels with nearest donor values.

    Parameters
    ----------
    raster:
        2D NumPy array, modified in place.

    target_mask:
        True for pixels selected for replacement.

    donor_mask:
        True for qualified source pixels.

    max_distance:
        Maximum distance in pixel units.
        None = unlimited.

    Returns
    -------
    replaced_count:
        Number successfully replaced.

    failed_count:
        Number without an eligible donor.
        These pixels become NoData.

    Notes
    -----
    - Original donor values are copied exactly.
    - No averaging or interpolation.
    - Calculations use row/column Euclidean distance.
    - Processing uses a local bounding window.
    """

    target_rows, target_cols = np.where(target_mask)

    target_count = len(target_rows)

    if target_count == 0:
        return 0, 0

    donor_mask = (
        donor_mask
        & get_valid_mask(raster)
        & ~target_mask
    )

    donor_rows, donor_cols = np.where(donor_mask)

    # No qualified donors
    if len(donor_rows) == 0:

        raster[target_rows, target_cols] = NODATA

        return 0, target_count

    # --------------------------------------------------------
    # Bounding window containing targets and donors
    # --------------------------------------------------------

    row_min = int(min(
        target_rows.min(),
        donor_rows.min()
    ))

    row_max = int(max(
        target_rows.max(),
        donor_rows.max()
    )) + 1

    col_min = int(min(
        target_cols.min(),
        donor_cols.min()
    ))

    col_max = int(max(
        target_cols.max(),
        donor_cols.max()
    )) + 1

    local_donors = donor_mask[
        row_min:row_max,
        col_min:col_max
    ]

    local_rows = target_rows - row_min
    local_cols = target_cols - col_min

    # --------------------------------------------------------
    # Nearest eligible donor
    # --------------------------------------------------------

    if max_distance is None:

        nearest_indices = distance_transform_edt(
            ~local_donors,
            return_distances=False,
            return_indices=True
        )

        nearest_rows = (
            nearest_indices[
                0,
                local_rows,
                local_cols
            ] + row_min
        )

        nearest_cols = (
            nearest_indices[
                1,
                local_rows,
                local_cols
            ] + col_min
        )

        replacement_values = raster[
            nearest_rows,
            nearest_cols
        ].copy()

        raster[
            target_rows,
            target_cols
        ] = replacement_values

        return target_count, 0

    else:

        distances, nearest_indices = distance_transform_edt(
            ~local_donors,
            return_distances=True,
            return_indices=True
        )

        target_distances = distances[
            local_rows,
            local_cols
        ]

        within_distance = (
            target_distances <= max_distance
        )

        # Qualified targets
        valid_rows = target_rows[within_distance]
        valid_cols = target_cols[within_distance]

        nearest_rows = (
            nearest_indices[
                0,
                local_rows[within_distance],
                local_cols[within_distance]
            ] + row_min
        )

        nearest_cols = (
            nearest_indices[
                1,
                local_rows[within_distance],
                local_cols[within_distance]
            ] + col_min
        )

        replacement_values = raster[
            nearest_rows,
            nearest_cols
        ].copy()

        raster[
            valid_rows,
            valid_cols
        ] = replacement_values

        # Targets exceeding maximum distance
        failed_rows = target_rows[~within_distance]
        failed_cols = target_cols[~within_distance]

        raster[
            failed_rows,
            failed_cols
        ] = NODATA

        replaced_count = int(
            np.count_nonzero(within_distance)
        )

        failed_count = target_count - replaced_count

        return replaced_count, failed_count


# ============================================================
# 5. PROCESS ONE YEAR
# ============================================================

def process_one_year(year):

    country_records = []

    A_tif = os.path.join(
        A_FOLDER,
        f"NUE_M2_median_{year}.tif"
    )

    N_tif = os.path.join(
        MASK_FOLDER,
        f"NUE_M2_n_{year}.tif"
    )

    OUT_tif = os.path.join(
        OUT_FOLDER,
        f"NUE_median_Opt_{year}.tif"
    )

    try:

        if not os.path.exists(A_tif):
            return {
                "year": year,
                "status": "MISSING_A",
                "message": A_tif,
                "annual": None,
                "countries": []
            }

        if not os.path.exists(N_tif):
            return {
                "year": year,
                "status": "MISSING_N",
                "message": N_tif,
                "annual": None,
                "countries": []
            }

        # ----------------------------------------------------
        # Read country polygons within each process
        # ----------------------------------------------------

        gdf = gpd.read_file(SHP_PATH)

        if gdf.crs is None:
            raise ValueError("Shapefile CRS missing")

        if gdf.crs.to_epsg() != 4326:
            gdf = gdf.to_crs(epsg=4326)

        gdf = gdf.sort_values(
            by=M49_FIELD
        ).reset_index(drop=True)

        # ----------------------------------------------------
        # Read raster A
        # ----------------------------------------------------

        with rasterio.open(A_tif) as src_a:

            A = read_raster_fixed(src_a)

            profile = src_a.profile.copy()
            transform = src_a.transform
            crs = src_a.crs

            height, width = A.shape

        # ----------------------------------------------------
        # Read raster N
        # ----------------------------------------------------

        with rasterio.open(N_tif) as src_n:

            N = read_raster_fixed(src_n)

            if (
                src_n.crs != crs
                or src_n.transform != transform
            ):
                raise ValueError(
                    "A and N raster grids are not aligned"
                )

        if A.shape != N.shape:
            raise ValueError(
                f"Shape mismatch: A={A.shape}, N={N.shape}"
            )

        if crs is None:
            raise ValueError("Raster CRS missing")

        # ====================================================
        # STEP 1 - N < MIN_COUNT / Missing N
        # ====================================================

        valid_mask = get_valid_mask(A)

        initial_count = int(
            np.count_nonzero(valid_mask)
        )

        n_valid_mask = get_valid_mask(N)

        step1_mask = (
            valid_mask
            & (
                ~n_valid_mask
                | (N < MIN_COUNT)
            )
        )

        step1_removed = int(
            np.count_nonzero(step1_mask)
        )

        A[step1_mask] = NODATA

        valid_mask = get_valid_mask(A)

        # ====================================================
        # STEP 2 - HARD THRESHOLDS
        # ====================================================

        step2_upper = (
            (A > UPPER_HARD)
            & valid_mask
        )

        step2_lower = (
            (A < LOWER_HARD)
            & valid_mask
        )

        upper_count = int(
            np.count_nonzero(step2_upper)
        )

        lower_count = int(
            np.count_nonzero(step2_lower)
        )

        # Same reported field as original script:
        # Number of pixels identified outside hard thresholds
        step2_removed = upper_count + lower_count

        # ----------------------------------------------------
        # Determine remove / nearest targets
        # ----------------------------------------------------

        step2_remove_mask = np.zeros_like(
            valid_mask,
            dtype=bool
        )

        step2_nearest_mask = np.zeros_like(
            valid_mask,
            dtype=bool
        )

        if UPPER_OUTLIER_ACTION == "REMOVE":
            step2_remove_mask |= step2_upper
        else:
            step2_nearest_mask |= step2_upper

        if LOWER_OUTLIER_ACTION == "REMOVE":
            step2_remove_mask |= step2_lower
        else:
            step2_nearest_mask |= step2_lower

        # Qualified donor pixels are those inside
        # the original hard thresholds, after Step 1.

        step2_donors = (
            valid_mask
            & (A >= LOWER_HARD)
            & (A <= UPPER_HARD)
        )

        # ----------------------------------------------------
        # REMOVE
        # ----------------------------------------------------

        step2_direct_removed = int(
            np.count_nonzero(step2_remove_mask)
        )

        A[step2_remove_mask] = NODATA

        # ----------------------------------------------------
        # NEAREST
        # ----------------------------------------------------

        step2_replaced, step2_failed = replace_by_nearest(
            A,
            step2_nearest_mask,
            step2_donors,
            MAX_NEAREST_DISTANCE_PIXELS
        )

        step2_actual_removed = (
            step2_direct_removed
            + step2_failed
        )

        valid_mask = get_valid_mask(A)

        # ====================================================
        # STEP 3 - COUNTRY STANDARD DEVIATION
        # ====================================================

        # Reported identified/selected count
        step3_total_removed = 0

        # Actual deletion count
        step3_actual_removed = 0

        # Successful replacement count
        step3_total_replaced = 0

        if APPLY_STD_REMOVAL:

            if gdf.crs != crs:
                gdf_reproj = gdf.to_crs(crs)
            else:
                gdf_reproj = gdf

            # Preserve M49 country processing order
            for _, country_row in gdf_reproj.iterrows():

                m49 = country_row[M49_FIELD]

                name = (
                    country_row[NAME_FIELD]
                    if NAME_FIELD in country_row
                    else str(m49)
                )

                geom = country_row.geometry

                if geom is None or geom.is_empty:

                    country_records.append(
                        empty_country_record(
                            year, m49, name
                        )
                    )

                    continue

                # --------------------------------------------
                # Country raster mask
                # --------------------------------------------

                try:

                    country_mask_arr = ~geometry_mask(
                        [geom],
                        transform=transform,
                        invert=False,
                        out_shape=(height, width)
                    )

                except Exception:

                    country_records.append(
                        empty_country_record(
                            year, m49, name
                        )
                    )

                    continue

                # --------------------------------------------
                # Country valid pixels
                # --------------------------------------------

                in_country_valid = (
                    country_mask_arr
                    & valid_mask
                )

                pixel_indices = np.where(
                    in_country_valid
                )

                n_valid = len(pixel_indices[0])

                if n_valid == 0:

                    country_records.append(
                        empty_country_record(
                            year, m49, name
                        )
                    )

                    continue

                # --------------------------------------------
                # Mean and standard deviation
                # --------------------------------------------

                vals = A[pixel_indices]

                mean_val = float(
                    np.mean(
                        vals,
                        dtype=np.float64
                    )
                )

                std_val = float(
                    np.std(
                        vals,
                        dtype=np.float64
                    )
                )

                upper_sigma = (
                    mean_val
                    + STD_FACTOR * std_val
                )

                # --------------------------------------------
                # Identify Step 3 candidates
                # --------------------------------------------

                candidate_mask = (
                    in_country_valid
                    & (A > upper_sigma)
                )

                n_candidate = int(
                    np.count_nonzero(candidate_mask)
                )

                max_allowed = int(
                    np.floor(
                        n_valid * MAX_REMOVE_RATIO
                    )
                )

                # --------------------------------------------
                # Select outliers
                #
                # Same 5% cap and descending-value rule
                # as the original script
                # --------------------------------------------

                selected_mask = np.zeros_like(
                    valid_mask,
                    dtype=bool
                )

                if n_candidate <= max_allowed:

                    selected_mask = candidate_mask

                    actual_selected = n_candidate

                else:

                    actual_selected = max_allowed

                    if max_allowed > 0:

                        cand_rows, cand_cols = np.where(
                            candidate_mask
                        )

                        cand_vals = A[
                            cand_rows,
                            cand_cols
                        ]

                        sort_idx = np.argsort(
                            -cand_vals
                        )[:max_allowed]

                        selected_mask[
                            cand_rows[sort_idx],
                            cand_cols[sort_idx]
                        ] = True

                # --------------------------------------------
                # NEW: Step 3 action
                # --------------------------------------------

                if STD_OUTLIER_ACTION == "REMOVE":

                    A[selected_mask] = NODATA

                    country_replaced = 0
                    country_actual_removed = actual_selected

                else:

                    # Nearest-neighbor donors:
                    # 1. Inside the same country
                    # 2. Valid after Step 2
                    # 3. Not above the country's sigma limit
                    #
                    # Exclude ALL values above sigma as donors,
                    # not just selected outliers.

                    country_donors = (
                        in_country_valid
                        & (A <= upper_sigma)
                    )

                    (
                        country_replaced,
                        country_actual_removed
                    ) = replace_by_nearest(
                        A,
                        selected_mask,
                        country_donors,
                        MAX_NEAREST_DISTANCE_PIXELS
                    )

                # --------------------------------------------
                # Update global raster valid mask
                # --------------------------------------------

                valid_mask = get_valid_mask(A)

                # Preserve existing reported meaning:
                # Step3_removed = selected outlier count
                # even when action is NEAREST.

                step3_total_removed += actual_selected

                step3_actual_removed += (
                    country_actual_removed
                )

                step3_total_replaced += (
                    country_replaced
                )

                # --------------------------------------------
                # Original country Excel fields
                # --------------------------------------------

                country_records.append({
                    "Year": year,
                    "M49": m49,
                    "Country": name,
                    "Valid_before_step3": n_valid,
                    "Mean": round(mean_val, 6),
                    "Std": round(std_val, 6),
                    "Upper_sigma": round(
                        upper_sigma, 6
                    ),
                    "Candidate_removed": n_candidate,
                    "Max_allowed_remove": max_allowed,
                    "Step3_removed": actual_selected,
                    "Step3_removed_%": (
                        round(
                            actual_selected / n_valid * 100,
                            4
                        )
                        if n_valid > 0 else 0.0
                    )
                })

        # ====================================================
        # 6. ANNUAL SUMMARY
        # ====================================================

        final_count = int(
            np.count_nonzero(
                get_valid_mask(A)
            )
        )

        total_removed = (
            initial_count - final_count
        )

        removed_ratio = (
            total_removed / initial_count * 100
            if initial_count > 0 else 0
        )

        # True number of deleted pixels:
        # replacement does NOT count as actual deletion.

        expected_actual_removed = (
            step1_removed
            + step2_actual_removed
            + step3_actual_removed
        )

        count_check = (
            total_removed
            - expected_actual_removed
        )

        # Original Annual Summary column structure
        annual_record = {
            "Year": year,
            "Initial_Valid": initial_count,
            "Step1_removed": step1_removed,
            "Step1_%": (
                round(
                    step1_removed / initial_count * 100,
                    4
                )
                if initial_count else 0
            ),
            "Step2_removed": step2_removed,
            "Step2_%": (
                round(
                    step2_removed / initial_count * 100,
                    4
                )
                if initial_count else 0
            ),
            "Step3_removed": step3_total_removed,
            "Step3_%": (
                round(
                    step3_total_removed / initial_count * 100,
                    4
                )
                if initial_count else 0
            ),
            "Final_Valid": final_count,
            "Total_Removed": total_removed,
            "Removed_Ratio_%": round(
                removed_ratio, 4
            )
        }

        # ====================================================
        # 7. EXPORT GEOTIFF
        # ====================================================

        profile.update(
            dtype="float32",
            nodata=NODATA,
            compress="lzw"
        )

        with rasterio.open(
            OUT_tif,
            "w",
            **profile
        ) as dst:

            dst.write(
                A.astype(np.float32),
                1
            )

        # ----------------------------------------------------
        # Return results to main process
        # ----------------------------------------------------

        return {
            "year": year,
            "status": "SUCCESS",
            "message": "",
            "annual": annual_record,
            "countries": country_records,
            "check": count_check,
            "step2_replaced": step2_replaced,
            "step3_replaced": step3_total_replaced,
            "actual_removed": total_removed,
            "output": OUT_tif
        }

    except Exception as e:

        return {
            "year": year,
            "status": "FAILED",
            "message": repr(e),
            "annual": None,
            "countries": []
        }


# ============================================================
# 8. PARALLEL PROCESSING BY YEAR
# ============================================================

YEARS = (
    list(range(START_YEAR, END_YEAR + 1))
    if YEARS_TO_RUN is None
    else sorted(set(map(int, YEARS_TO_RUN)))
)

if not YEARS:
    raise ValueError("No years selected")

print("=" * 70)
print("GLOBAL NUE OUTLIER PROCESSING")
print("=" * 70)

print(f"Years: {YEARS}")
print(f"Parallel workers: {PARALLEL_YEARS}")

print(f"Step 1 MIN_COUNT: {MIN_COUNT}")

print(
    f"Step 2 Upper: {UPPER_HARD} "
    f"({UPPER_OUTLIER_ACTION})"
)

print(
    f"Step 2 Lower: {LOWER_HARD} "
    f"({LOWER_OUTLIER_ACTION})"
)

print(
    f"Step 3 Enabled: {APPLY_STD_REMOVAL}"
)

print(
    f"Step 3 Action: {STD_OUTLIER_ACTION}"
)

print(
    f"Step 3 STD_FACTOR: {STD_FACTOR}"
)

print("=" * 70, flush=True)


if PARALLEL_YEARS == 1 or len(YEARS) == 1:

    results = [
        process_one_year(year)
        for year in YEARS
    ]

else:

    results = Parallel(
        n_jobs=min(
            PARALLEL_YEARS,
            len(YEARS)
        ),
        backend="loky",
        batch_size=1,
        pre_dispatch="n_jobs",
        verbose=10
    )(
        delayed(process_one_year)(year)
        for year in YEARS
    )


# ============================================================
# 9. COLLECT PARALLEL RESULTS
# ============================================================

results.sort(
    key=lambda item: item["year"]
)

records_year = []
records_country = []

for result in results:

    year = result["year"]
    status = result["status"]

    if status == "SUCCESS":

        records_year.append(
            result["annual"]
        )

        records_country.extend(
            result["countries"]
        )

        a = result["annual"]

        print(
            f"[{year}] "
            f"Initial={a['Initial_Valid']:,} | "
            f"Step1={a['Step1_removed']:,} | "
            f"Step2={a['Step2_removed']:,} | "
            f"Step3={a['Step3_removed']:,} | "
            f"Step2_Replaced={result['step2_replaced']:,} | "
            f"Step3_Replaced={result['step3_replaced']:,} | "
            f"Final={a['Final_Valid']:,} | "
            f"Check={result['check']}"
        )

    else:

        print(
            f"[{year}] {status}: "
            f"{result['message']}"
        )


# ============================================================
# 10. EXCEL REPORT
# ============================================================

COUNTRY_COLS = [
    "Year",
    "M49",
    "Country",
    "Valid_before_step3",
    "Mean",
    "Std",
    "Upper_sigma",
    "Candidate_removed",
    "Max_allowed_remove",
    "Step3_removed",
    "Step3_removed_%"
]

if not records_year:

    print(
        "\nNo successful years. Excel not generated."
    )

else:

    df_year = (
        pd.DataFrame(records_year)
        .sort_values("Year")
        .reset_index(drop=True)
    )

    df_country = pd.DataFrame(
        records_country,
        columns=COUNTRY_COLS
    )

    if not df_country.empty:

        df_country = (
            df_country
            .sort_values(["Year", "M49"])
            .reset_index(drop=True)
        )

    wb = Workbook()

    thin = Side(
        style="thin",
        color="AAAAAA"
    )

    border = Border(
        left=thin,
        right=thin,
        top=thin,
        bottom=thin
    )

    # --------------------------------------------------------
    # Excel formatting helpers
    # --------------------------------------------------------

    def hdr_cell(
        ws, row, col, value,
        bg="2F75B6",
        fg="FFFFFF"
    ):

        c = ws.cell(
            row=row,
            column=col,
            value=value
        )

        c.font = Font(
            bold=True,
            color=fg,
            name="Arial",
            size=10
        )

        c.fill = PatternFill(
            "solid",
            start_color=bg
        )

        c.alignment = Alignment(
            horizontal="center",
            wrap_text=True
        )

        c.border = border

        return c


    def data_cell(
        ws, row, col, value,
        bg="FFFFFF",
        fmt=None
    ):

        if isinstance(value, np.generic):
            value = value.item()

        if (
            isinstance(value, float)
            and not np.isfinite(value)
        ):
            value = None

        c = ws.cell(
            row=row,
            column=col,
            value=value
        )

        c.font = Font(
            name="Arial",
            size=10
        )

        c.fill = PatternFill(
            "solid",
            start_color=bg
        )

        c.alignment = Alignment(
            horizontal="center"
        )

        c.border = border

        if fmt:
            c.number_format = fmt

        return c


    # ========================================================
    # SHEET 1: ANNUAL SUMMARY
    # ========================================================

    ws1 = wb.active

    ws1.title = "Annual Summary"

    col_colors_year = {
        "Year": "FFFFFF",
        "Initial_Valid": "FFFFFF",
        "Step1_removed": "D6E4F0",
        "Step1_%": "D6E4F0",
        "Step2_removed": "D5E8D4",
        "Step2_%": "D5E8D4",
        "Step3_removed": "FFF2CC",
        "Step3_%": "FFF2CC",
        "Final_Valid": "FFFFFF",
        "Total_Removed": "F4CCCC",
        "Removed_Ratio_%": "F4CCCC"
    }

    cols_y = list(df_year.columns)

    for ci, col in enumerate(cols_y, 1):
        hdr_cell(ws1, 1, ci, col)

    for ri, row in enumerate(
        df_year.itertuples(index=False),
        2
    ):

        for ci, (col, val) in enumerate(
            zip(cols_y, row),
            1
        ):

            fmt = (
                "0.00"
                if "%" in col
                else (
                    "#,##0"
                    if col != "Year"
                    else "0"
                )
            )

            data_cell(
                ws1,
                ri,
                ci,
                val,
                bg=col_colors_year.get(
                    col,
                    "FFFFFF"
                ),
                fmt=fmt
            )

    # Original TOTAL/AVG row
    sr = len(df_year) + 2

    data_cell(
        ws1,
        sr,
        1,
        "TOTAL/AVG",
        bg="D9D9D9"
    ).font = Font(
        bold=True,
        name="Arial",
        size=10
    )

    for ci, col in enumerate(
        cols_y[1:],
        2
    ):

        cl = get_column_letter(ci)

        rng = (
            f"{cl}2:"
            f"{cl}{len(df_year) + 1}"
        )

        formula = (
            f"=AVERAGE({rng})"
            if "%" in col
            else f"=SUM({rng})"
        )

        c = ws1.cell(
            row=sr,
            column=ci,
            value=formula
        )

        c.font = Font(
            bold=True,
            name="Arial",
            size=10
        )

        c.fill = PatternFill(
            "solid",
            start_color="D9D9D9"
        )

        c.alignment = Alignment(
            horizontal="center"
        )

        c.border = border

        c.number_format = (
            "0.00"
            if "%" in col
            else "#,##0"
        )

    for ci, col in enumerate(cols_y, 1):

        ws1.column_dimensions[
            get_column_letter(ci)
        ].width = max(
            len(col) + 4,
            14
        )

    ws1.row_dimensions[1].height = 36
    ws1.freeze_panes = "B2"

    # ========================================================
    # SHEET 2: COUNTRY DETAIL
    # ========================================================

    ws2 = wb.create_sheet(
        "Country Detail"
    )

    cols_c = COUNTRY_COLS

    col_colors_cty = {
        "Year": "FFFFFF",
        "M49": "FFFFFF",
        "Country": "FFFFFF",
        "Valid_before_step3": "FFFFFF",
        "Mean": "EAD1DC",
        "Std": "EAD1DC",
        "Upper_sigma": "EAD1DC",
        "Candidate_removed": "FFF2CC",
        "Max_allowed_remove": "FFF2CC",
        "Step3_removed": "F4CCCC",
        "Step3_removed_%": "F4CCCC"
    }

    for ci, col in enumerate(cols_c, 1):
        hdr_cell(ws2, 1, ci, col)

    for ri, row in enumerate(
        df_country.itertuples(index=False),
        2
    ):

        for ci, (col, val) in enumerate(
            zip(cols_c, row),
            1
        ):

            if "%" in col:
                fmt = "0.00"

            elif col in (
                "Mean",
                "Std",
                "Upper_sigma"
            ):
                fmt = "0.000000"

            elif col not in (
                "Year",
                "M49",
                "Country"
            ):
                fmt = "#,##0"

            else:
                fmt = None

            data_cell(
                ws2,
                ri,
                ci,
                val,
                bg=col_colors_cty.get(
                    col,
                    "FFFFFF"
                ),
                fmt=fmt
            )

    for ci, col in enumerate(cols_c, 1):

        ws2.column_dimensions[
            get_column_letter(ci)
        ].width = max(
            len(col) + 3,
            12
        )

    ws2.column_dimensions["C"].width = 22
    ws2.row_dimensions[1].height = 36
    ws2.freeze_panes = "D2"

    # ========================================================
    # SHEET 3: PARAMETERS
    # ========================================================

    ws3 = wb.create_sheet(
        "Parameters"
    )

    params = [
        ("Parameter", "Value"),
        ("MIN_COUNT (Step1)", MIN_COUNT),
        ("LOWER_HARD (Step2)", LOWER_HARD),
        ("UPPER_HARD (Step2)", UPPER_HARD),
        ("LOWER_OUTLIER_ACTION", LOWER_OUTLIER_ACTION),
        ("UPPER_OUTLIER_ACTION", UPPER_OUTLIER_ACTION),
        ("APPLY_STD_REMOVAL", APPLY_STD_REMOVAL),
        ("STD_FACTOR (Step3)", STD_FACTOR),
        ("STD_OUTLIER_ACTION", STD_OUTLIER_ACTION),
        ("MAX_REMOVE_RATIO (Step3)", MAX_REMOVE_RATIO),
        (
            "MAX_NEAREST_DISTANCE_PIXELS",
            "None" if MAX_NEAREST_DISTANCE_PIXELS is None
            else MAX_NEAREST_DISTANCE_PIXELS
        ),
        ("M49_FIELD", M49_FIELD),
        ("SHP_PATH", SHP_PATH),
        ("START_YEAR", START_YEAR),
        ("END_YEAR", END_YEAR),
        ("NODATA", NODATA),
        ("PARALLEL_YEARS", PARALLEL_YEARS),
        (
            "YEARS_TO_RUN",
            "All" if YEARS_TO_RUN is None
            else str(YEARS)
        )
    ]

    for ri, (k, v) in enumerate(
        params,
        1
    ):

        for ci, val in enumerate(
            (k, v),
            1
        ):

            c = ws3.cell(
                row=ri,
                column=ci,
                value=val
            )

            c.font = Font(
                bold=(ri == 1),
                name="Arial",
                size=10,
                color=(
                    "FFFFFF"
                    if ri == 1
                    else "000000"
                )
            )

            c.fill = PatternFill(
                "solid",
                start_color=(
                    "2F75B6"
                    if ri == 1
                    else "F5F5F5"
                )
            )

            c.border = border

            c.alignment = Alignment(
                horizontal="left"
            )

    ws3.column_dimensions["A"].width = 38
    ws3.column_dimensions["B"].width = 60

    # --------------------------------------------------------
    # Save Excel
    # --------------------------------------------------------

    wb.save(EXCEL_OUT)

    print(
        f"\nExcel saved: {EXCEL_OUT}"
    )

    # ========================================================
    # 11. FINAL VALIDATION
    # ========================================================

    check_rows = []

    for result in results:

        if result["status"] == "SUCCESS":

            check_rows.append({
                "Year": result["year"],
                "Step2_detected": (
                    result["annual"]["Step2_removed"]
                ),
                "Step3_detected": (
                    result["annual"]["Step3_removed"]
                ),
                "Step2_replaced": (
                    result["step2_replaced"]
                ),
                "Step3_replaced": (
                    result["step3_replaced"]
                ),
                "Actual_removed": (
                    result["actual_removed"]
                ),
                "Check": result["check"]
            })

    df_check = pd.DataFrame(
        check_rows
    )

    print("\n" + "=" * 70)
    print("FINAL VALIDATION SUMMARY")
    print("=" * 70)

    print(
        df_check.to_string(
            index=False
        )
    )

    if (df_check["Check"] == 0).all():

        print(
            "\nPASS: All annual pixel counts are consistent."
        )

    else:

        print(
            "\nWARNING: Pixel count inconsistency detected."
        )

print("\nAll requested years finished.")

In [ ]:
# ============================================================
# Batch GeoTIFF Processing + OPTIONAL Annual NUE Mask
#
# 1. Remove original NoData and specific invalid values
# 2. Multiply valid values by ratio
# 3. Remove values outside [range_min, range_max]
# 4. OPTIONAL: Apply annual NUE spatial mask
# 5. Preserve original GeoTIFF spatial properties
#
# JupyterLab - Single Cell
# ============================================================

from osgeo import gdal
import numpy as np
import os
import glob
import re

gdal.UseExceptions()


# ============================================================
# 1. USER PARAMETERS
# ============================================================

input_folder = (
    r"Z:\staff\Qiang_S\09_Datasets\03GlobalNUE_Results\NUE_Upload_Zenodo_V2.0\01WheatNUE_M2_20261030V1_M6_150R"
)

output_folder = (
    r"Z:\staff\Qiang_S\09_Datasets\03GlobalNUE_Results\NUE_Upload_Zenodo_V2.0\01_NUE_Wheat_V2.0\NUE_Wheat_Median_1995_2020"
)

# ------------------------------------------------------------
# NEW: ENABLE / DISABLE NUE MASK
#
# True  = Apply annual NUE mask
# False = Do not use NUE mask
# ------------------------------------------------------------

USE_NUE_MASK = True

# ------------------------------------------------------------
# Annual NUE mask settings
# Only used when USE_NUE_MASK = True
# ------------------------------------------------------------

nue_mask_folder = (
    r"Z:\staff\Qiang_S\09_Datasets\03GlobalNUE_Results\NUE_Upload_Zenodo_V2.0\01WheatNUE_M2_20261030V1_M6_150R"
)

NUE_MASK_PREFIX = "NUE_median_Opt_"
# NUE_Wheat_Median_
# False: Valid NUE = finite and not NoData
# True : Valid NUE = finite, not NoData, and > 0
MASK_REQUIRE_POSITIVE = False

# ------------------------------------------------------------
# Original numerical processing settings
# ------------------------------------------------------------

# Multiply original values by ratio
ratio = 100.0

# Remove specified values BEFORE multiplication
replace_values = [-1]

# Keep output values in this range (inclusive)
range_min = 0.0
range_max = 150.0

# Unified NoData
nodata_value = -9999.0

# Additional original invalid values
additional_nodata = [-9999.0]

# Block processing
BLOCK_SIZE = 512


# ============================================================
# 2. VALIDATE CONFIGURATION
# ============================================================

if not os.path.isdir(input_folder):
    raise FileNotFoundError(
        f"Input folder not found:\n{input_folder}"
    )

# NEW: Only check NUE mask folder when enabled
if USE_NUE_MASK:
    if not os.path.isdir(nue_mask_folder):
        raise FileNotFoundError(
            f"NUE mask folder not found:\n{nue_mask_folder}"
        )

if os.path.abspath(input_folder) == os.path.abspath(output_folder):
    raise ValueError(
        "Input and output folders must be different."
    )

if not np.isfinite(ratio):
    raise ValueError("ratio must be finite")

if not (
    np.isfinite(range_min)
    and np.isfinite(range_max)
    and range_min <= range_max
):
    raise ValueError("Invalid output value range")

if range_min <= nodata_value <= range_max:
    raise ValueError(
        "Output NoData must be outside the valid range"
    )

os.makedirs(output_folder, exist_ok=True)


# ============================================================
# 3. COLLECT INPUT FILES
# ============================================================

tif_files = sorted(
    glob.glob(os.path.join(input_folder, "*.tif"))
    +
    glob.glob(os.path.join(input_folder, "*.tiff"))
)

def extract_year(filepath):

    match = re.search(
        r"_(19\d{2}|20\d{2})\.(?:tif|tiff)$",
        os.path.basename(filepath),
        flags=re.IGNORECASE
    )

    return int(match.group(1)) if match else None


matched_files = []
skipped_files = []

for in_file in tif_files:

    year = extract_year(in_file)

    # --------------------------------------------------------
    # MASK ENABLED
    # Match corresponding annual NUE mask
    # --------------------------------------------------------

    if USE_NUE_MASK:

        if year is None:
            skipped_files.append(
                (os.path.basename(in_file), "Year not found")
            )
            continue

        mask_file = os.path.join(
            nue_mask_folder,
            f"{NUE_MASK_PREFIX}{year}.tif"
        )

        if not os.path.isfile(mask_file):
            skipped_files.append(
                (
                    os.path.basename(in_file),
                    f"NUE mask missing for {year}"
                )
            )
            continue

    # --------------------------------------------------------
    # MASK DISABLED
    # No mask file required
    # --------------------------------------------------------

    else:
        mask_file = None

    matched_files.append(
        (year, in_file, mask_file)
    )


print("=" * 75)
print("BATCH GEOTIFF PROCESSING")
print("=" * 75)

print(f"Input files      : {len(tif_files)}")
print(f"Files to process : {len(matched_files)}")
print(f"Skipped files    : {len(skipped_files)}")

print(
    f"NUE mask         : "
    f"{'ENABLED' if USE_NUE_MASK else 'DISABLED'}"
)

if USE_NUE_MASK:
    print(f"Mask > 0 required: {MASK_REQUIRE_POSITIVE}")

print(f"Multiply ratio   : {ratio}")
print(f"Valid range      : [{range_min}, {range_max}]")
print(f"Output NoData    : {nodata_value}")

print("=" * 75)

if not matched_files:
    raise RuntimeError(
        "No GeoTIFF files available for processing"
    )


# ============================================================
# 4. PROCESS FILES
# ============================================================

for i, (year, in_file, mask_file) in enumerate(
    matched_files,
    start=1
):

    fname = os.path.basename(in_file)
    out_file = os.path.join(output_folder, fname)

    print("\n" + "-" * 75)
    print(f"[{i}/{len(matched_files)}] {fname}")

    src = gdal.Open(in_file, gdal.GA_ReadOnly)

    if src is None:
        raise RuntimeError(f"Cannot open: {in_file}")

    if src.RasterCount != 1:
        raise ValueError(
            f"Expected single-band raster: {fname}"
        )

    src_band = src.GetRasterBand(1)
    src_nodata = src_band.GetNoDataValue()

    width = src.RasterXSize
    height = src.RasterYSize

    # --------------------------------------------------------
    # NEW: Open annual NUE mask only when enabled
    # --------------------------------------------------------

    mask_ds = None
    mask_band = None
    mask_nodata = None
    nue_has_mask = False

    if USE_NUE_MASK:

        mask_ds = gdal.Open(
            mask_file,
            gdal.GA_ReadOnly
        )

        if mask_ds is None:
            raise RuntimeError(
                f"Cannot open NUE mask: {mask_file}"
            )

        if (
            width != mask_ds.RasterXSize
            or height != mask_ds.RasterYSize
        ):
            raise ValueError(
                f"Raster dimensions differ: {fname}"
            )

        mask_band = mask_ds.GetRasterBand(1)
        mask_nodata = mask_band.GetNoDataValue()

        nue_has_mask = not (
            mask_band.GetMaskFlags()
            & gdal.GMF_ALL_VALID
        )

        print(
            f"  Annual NUE mask: "
            f"{os.path.basename(mask_file)}"
        )

    else:
        print("  NUE mask: DISABLED")

    # --------------------------------------------------------
    # Create output GeoTIFF
    # --------------------------------------------------------

    driver = gdal.GetDriverByName("GTiff")

    out_ds = driver.Create(
        out_file,
        width,
        height,
        1,
        gdal.GDT_Float32,
        options=[
            "COMPRESS=DEFLATE",
            "PREDICTOR=3",
            "TILED=YES",
            "BIGTIFF=IF_SAFER"
        ]
    )

    if out_ds is None:
        raise RuntimeError(
            f"Cannot create output: {out_file}"
        )

    # Preserve original spatial properties
    out_ds.SetGeoTransform(
        src.GetGeoTransform()
    )

    out_ds.SetProjection(
        src.GetProjection()
    )

    out_ds.SetMetadata(
        src.GetMetadata()
    )

    out_band = out_ds.GetRasterBand(1)

    out_band.SetNoDataValue(
        nodata_value
    )

    out_band.SetDescription(
        src_band.GetDescription()
    )

    out_band.SetUnitType(
        src_band.GetUnitType()
    )

    # Preserve band metadata but remove old statistics
    metadata = src_band.GetMetadata() or {}

    metadata = {
        k: v for k, v in metadata.items()
        if not k.upper().startswith("STATISTICS_")
    }

    out_band.SetMetadata(metadata)

    # --------------------------------------------------------
    # Counts
    # --------------------------------------------------------

    total_pixels = 0
    valid_pixels = 0

    removed_by_mask = 0
    removed_by_range = 0

    src_has_mask = not (
        src_band.GetMaskFlags()
        & gdal.GMF_ALL_VALID
    )

    # --------------------------------------------------------
    # Process raster by blocks
    # --------------------------------------------------------

    for yoff in range(0, height, BLOCK_SIZE):

        ysize = min(
            BLOCK_SIZE,
            height - yoff
        )

        for xoff in range(0, width, BLOCK_SIZE):

            xsize = min(
                BLOCK_SIZE,
                width - xoff
            )

            # ================================================
            # STEP 1: Read original values
            # ================================================

            arr = src_band.ReadAsArray(
                xoff,
                yoff,
                xsize,
                ysize
            ).astype(np.float64)

            total_pixels += arr.size

            # ================================================
            # STEP 2: Original valid pixel mask
            # ================================================

            input_valid = np.isfinite(arr)

            if src_nodata is not None:
                if np.isfinite(src_nodata):
                    input_valid &= (arr != src_nodata)

            for nd in additional_nodata:
                input_valid &= (arr != nd)

            # Remove specific values BEFORE multiplication
            for v in replace_values:
                input_valid &= (arr != v)

            # Respect source GDAL mask
            if src_has_mask:

                src_mask_arr = (
                    src_band.GetMaskBand().ReadAsArray(
                        xoff, yoff, xsize, ysize
                    )
                )

                input_valid &= (src_mask_arr != 0)

            # ================================================
            # STEP 3: Optional annual NUE mask
            # ================================================

            if USE_NUE_MASK:

                nue_arr = mask_band.ReadAsArray(
                    xoff,
                    yoff,
                    xsize,
                    ysize
                )

                mask_valid = np.isfinite(nue_arr)

                # Exclude metadata-defined NoData
                if mask_nodata is not None:
                    if np.isfinite(mask_nodata):
                        mask_valid &= (
                            nue_arr != mask_nodata
                        )

                # Exclude explicit -9999
                mask_valid &= (
                    nue_arr != nodata_value
                )

                # Respect NUE GDAL mask
                if nue_has_mask:

                    nue_mask_arr = (
                        mask_band.GetMaskBand().ReadAsArray(
                            xoff, yoff, xsize, ysize
                        )
                    )

                    mask_valid &= (
                        nue_mask_arr != 0
                    )

                # Optional additional condition
                if MASK_REQUIRE_POSITIVE:
                    mask_valid &= (nue_arr > 0)

            else:

                # Mask disabled:
                # Every spatial position is permitted.
                mask_valid = np.ones(
                    arr.shape,
                    dtype=bool
                )

            removed_by_mask += int(
                np.count_nonzero(
                    input_valid & ~mask_valid
                )
            )

            # ================================================
            # STEP 4: Combine valid masks
            # ================================================

            common_valid = (
                input_valid & mask_valid
            )

            # ================================================
            # STEP 5: Multiply ratio
            # ================================================

            scaled = np.full(
                arr.shape,
                np.nan,
                dtype=np.float64
            )

            scaled[common_valid] = (
                arr[common_valid] * ratio
            )

            # ================================================
            # STEP 6: Valid numerical range
            #
            # Keep:
            #   range_min <= value <= range_max
            # ================================================

            range_valid = (
                np.isfinite(scaled)
                & (scaled >= range_min)
                & (scaled <= range_max)
                & (
                    np.abs(scaled)
                    <= np.finfo(np.float32).max
                )
            )

            removed_by_range += int(
                np.count_nonzero(
                    common_valid & ~range_valid
                )
            )

            final_valid = (
                common_valid & range_valid
            )

            # ================================================
            # STEP 7: Output pixel values
            # ================================================

            output_arr = np.full(
                arr.shape,
                nodata_value,
                dtype=np.float32
            )

            output_arr[final_valid] = (
                scaled[final_valid].astype(np.float32)
            )

            valid_pixels += int(
                np.count_nonzero(final_valid)
            )

            # ================================================
            # STEP 8: Write GeoTIFF block
            # ================================================

            out_band.WriteArray(
                output_arr,
                xoff,
                yoff
            )

    # --------------------------------------------------------
    # Finalize current GeoTIFF
    # --------------------------------------------------------

    out_band.FlushCache()
    out_ds.FlushCache()

    out_band = None
    out_ds = None

    src_band = None
    mask_band = None

    src = None
    mask_ds = None

    print(f"  Total pixels       : {total_pixels:,}")
    print(f"  Removed by NUE mask: {removed_by_mask:,}")
    print(f"  Removed by range   : {removed_by_range:,}")
    print(f"  Final valid pixels : {valid_pixels:,}")

    if valid_pixels == 0:
        print(
            "  WARNING: No valid pixels remain!"
        )


# ============================================================
# 5. FINAL SUMMARY
# ============================================================

print("\n" + "=" * 75)
print("ALL GEOTIFF FILES FINISHED")
print("=" * 75)

print(f"Input files    : {len(tif_files)}")
print(f"Processed      : {len(matched_files)}")
print(f"Skipped        : {len(skipped_files)}")

print(
    "NUE mask       : "
    + ("ENABLED" if USE_NUE_MASK else "DISABLED")
)

if skipped_files:

    print("\nSkipped files:")

    for fname, reason in skipped_files:
        print(f"  {fname}: {reason}")

print(f"\nOutput directory:\n{output_folder}")
print("=" * 75)

In [ ]:
# ============================================================
# Batch GeoTIFF Processing + OPTIONAL Annual NUE Mask
#
# 1. Remove original NoData and specific invalid values
# 2. Multiply valid values by ratio
# 3. Remove values outside [range_min, range_max]
# 4. OPTIONAL: Apply annual NUE spatial mask
# 5. Preserve original GeoTIFF spatial properties
# 6. NEW: Configurable output GeoTIFF filenames
#
# JupyterLab - Single Cell
# ============================================================

from osgeo import gdal
import numpy as np
import os
import glob
import re

gdal.UseExceptions()


# ============================================================
# 1. USER PARAMETERS
# ============================================================

# ------------------------------------------------------------
# Input folder
# ------------------------------------------------------------

input_folder = (
    r"Z:\staff\Qiang_S\09_Datasets\03GlobalNUE_Results\NUE_Upload_Zenodo_V2.0\01WheatNUE_M2_20261030V1_Nup"
)

# ------------------------------------------------------------
# Output folder
# ------------------------------------------------------------

output_folder = (
    r"Z:\staff\Qiang_S\09_Datasets\03GlobalNUE_Results\NUE_Upload_Zenodo_V2.0\01_NUE_Wheat_V2.0\Noutput_Wheat_Median_1995_2020"
)

# ------------------------------------------------------------
# NEW: OUTPUT GEOTIFF FILENAME
#
# Examples:
#
# "New_NUE_{year}.tif"
#     -> New_NUE_1995.tif
#     -> New_NUE_2000.tif
#
# "Wheat_NUE_{year}.tif"
#     -> Wheat_NUE_2000.tif
#
# "NUE_median_Opt_{year}.tif"
#     -> NUE_median_Opt_2000.tif
#
# "{original_name}"
#     -> Keep original input filename
#
# IMPORTANT:
# When using {year}, each input file must
# have a filename ending in _YYYY.tif/.tiff.
# ------------------------------------------------------------

OUTPUT_FILENAME_TEMPLATE = "Noutput_Wheat_Median_{year}.tif"

# ============================================================
# 2. OPTIONAL NUE MASK SETTINGS
# ============================================================

# ------------------------------------------------------------
# ENABLE / DISABLE NUE MASK
#
# True  = Apply annual NUE mask
# False = Do not use NUE mask
# ------------------------------------------------------------

USE_NUE_MASK = True

# ------------------------------------------------------------
# Annual NUE mask folder
#
# Only used when USE_NUE_MASK = True
# ------------------------------------------------------------

nue_mask_folder = (
    r"Z:\staff\Qiang_S\09_Datasets\03GlobalNUE_Results\NUE_Upload_Zenodo_V2.0\01WheatNUE_M2_20261030V1_M6_150R"
)

# Annual mask filename:
# NUE_median_Opt_1995.tif

NUE_MASK_PREFIX = "NUE_median_Opt_"

# ------------------------------------------------------------
# Mask validity condition
#
# False:
#   NUE mask pixel must be finite and not NoData
#
# True:
#   NUE mask pixel must also be > 0
#
# Only effective when USE_NUE_MASK = True.
# ------------------------------------------------------------

MASK_REQUIRE_POSITIVE = False


# ============================================================
# 3. NUMERICAL PROCESSING SETTINGS
# ============================================================

# ------------------------------------------------------------
# Multiply original values by this ratio
# ------------------------------------------------------------

#ratio = 100.0 #for NUE
ratio = 1.0  # Noutput


# ------------------------------------------------------------
# Remove specific ORIGINAL values
#
# Checked BEFORE multiplication.
# ------------------------------------------------------------

replace_values = [-1]

# ------------------------------------------------------------
# Valid output range
#
# Keep:
#     range_min <= value <= range_max
#
# Remove:
#     value < range_min OR value > range_max
#
# Applied AFTER multiplication.
# ------------------------------------------------------------

range_min = 0.0
#range_max = 150.0 #for NUE
range_max = 10000.0 #for Noutput

# ------------------------------------------------------------
# Output NoData value
# ------------------------------------------------------------

nodata_value = -9999.0

# ------------------------------------------------------------
# Additional input NoData values
# ------------------------------------------------------------

additional_nodata = [-9999.0]

# ------------------------------------------------------------
# Block processing
# ------------------------------------------------------------

BLOCK_SIZE = 512


# ============================================================
# 4. VALIDATE CONFIGURATION
# ============================================================

if not os.path.isdir(input_folder):
    raise FileNotFoundError(
        f"Input folder not found:\n{input_folder}"
    )

if USE_NUE_MASK:
    if not os.path.isdir(nue_mask_folder):
        raise FileNotFoundError(
            f"NUE mask folder not found:\n{nue_mask_folder}"
        )

if os.path.abspath(input_folder) == os.path.abspath(output_folder):
    raise ValueError(
        "Input and output folders must be different."
    )

if not np.isfinite(ratio):
    raise ValueError(
        "ratio must be finite"
    )

if not (
    np.isfinite(range_min)
    and np.isfinite(range_max)
    and range_min <= range_max
):
    raise ValueError(
        "Invalid output value range"
    )

if range_min <= nodata_value <= range_max:
    raise ValueError(
        "Output NoData must be outside the valid range"
    )

if BLOCK_SIZE <= 0:
    raise ValueError(
        "BLOCK_SIZE must be positive"
    )

if not OUTPUT_FILENAME_TEMPLATE:
    raise ValueError(
        "OUTPUT_FILENAME_TEMPLATE cannot be empty"
    )

os.makedirs(
    output_folder,
    exist_ok=True
)


# ============================================================
# 5. COLLECT INPUT GEOTIFF FILES
# ============================================================

tif_files = sorted(
    glob.glob(
        os.path.join(input_folder, "*.tif")
    )
    +
    glob.glob(
        os.path.join(input_folder, "*.tiff")
    )
)


def extract_year(filepath):
    """
    Extract year from filename ending in:

        xxx_1995.tif
        xxx_2000.tif
        xxx_2020.tiff
    """

    match = re.search(
        r"_(19\d{2}|20\d{2})\.(?:tif|tiff)$",
        os.path.basename(filepath),
        flags=re.IGNORECASE
    )

    if match:
        return int(match.group(1))

    return None


# ============================================================
# 6. MATCH ANNUAL FILES AND PREPARE OUTPUT NAMES
# ============================================================

matched_files = []
skipped_files = []

# Prevent different input files from overwriting
# the same output filename.
planned_output_names = set()

for in_file in tif_files:

    fname = os.path.basename(in_file)

    year = extract_year(in_file)

    # --------------------------------------------------------
    # Match annual NUE mask only when enabled
    # --------------------------------------------------------

    if USE_NUE_MASK:

        if year is None:

            skipped_files.append(
                (
                    fname,
                    "Year not found in filename"
                )
            )

            continue

        mask_file = os.path.join(
            nue_mask_folder,
            f"{NUE_MASK_PREFIX}{year}.tif"
        )

        if not os.path.isfile(mask_file):

            skipped_files.append(
                (
                    fname,
                    f"NUE mask missing for {year}"
                )
            )

            continue

    else:

        mask_file = None

    # --------------------------------------------------------
    # NEW: Generate output filename
    # --------------------------------------------------------

    if (
        year is None
        and "{year}" in OUTPUT_FILENAME_TEMPLATE
    ):
        raise ValueError(
            f"Cannot extract year from {fname}. "
            "The output filename template requires {year}."
        )

    try:

        output_name = OUTPUT_FILENAME_TEMPLATE.format(
            year=year,
            original_name=fname
        )

    except (KeyError, ValueError) as e:

        raise ValueError(
            "Invalid OUTPUT_FILENAME_TEMPLATE. "
            "Use {year} and/or {original_name}."
        ) from e

    # Check filename and extension
    if (
        not output_name
        or os.path.basename(output_name) != output_name
        or "/" in output_name
        or "\\" in output_name
        or not output_name.lower().endswith(
            (".tif", ".tiff")
        )
    ):
        raise ValueError(
            f"Invalid output filename: {output_name}"
        )

    # Avoid output name collisions
    output_key = output_name.lower()

    if output_key in planned_output_names:

        raise ValueError(
            f"Duplicate output filename: {output_name}\n"
            "Different input files would overwrite each other."
        )

    planned_output_names.add(output_key)

    out_file = os.path.join(
        output_folder,
        output_name
    )

    matched_files.append(
        (
            year,
            in_file,
            mask_file,
            out_file
        )
    )


# ============================================================
# 7. INITIAL SUMMARY
# ============================================================

print("=" * 75)
print("BATCH GEOTIFF PROCESSING")
print("=" * 75)

print(f"Input files      : {len(tif_files)}")
print(f"Files to process : {len(matched_files)}")
print(f"Skipped files    : {len(skipped_files)}")

print(
    f"NUE mask         : "
    f"{'ENABLED' if USE_NUE_MASK else 'DISABLED'}"
)

if USE_NUE_MASK:

    print(
        f"Mask > 0 required: {MASK_REQUIRE_POSITIVE}"
    )

print(f"Multiply ratio   : {ratio}")

print(
    f"Valid range      : "
    f"[{range_min}, {range_max}]"
)

print(f"Output NoData    : {nodata_value}")

print(
    f"Output filename  : "
    f"{OUTPUT_FILENAME_TEMPLATE}"
)

print("=" * 75)

if not matched_files:

    raise RuntimeError(
        "No GeoTIFF files available for processing"
    )


# ============================================================
# 8. PROCESS FILES
# ============================================================

for i, (
    year,
    in_file,
    mask_file,
    out_file
) in enumerate(
    matched_files,
    start=1
):

    fname = os.path.basename(in_file)

    output_name = os.path.basename(out_file)

    print("\n" + "-" * 75)

    print(
        f"[{i}/{len(matched_files)}] "
        f"Processing year: {year}"
    )

    print(f"  Input : {fname}")

    print(f"  Output: {output_name}")

    # --------------------------------------------------------
    # Open input raster
    # --------------------------------------------------------

    src = gdal.Open(
        in_file,
        gdal.GA_ReadOnly
    )

    if src is None:

        raise RuntimeError(
            f"Cannot open: {in_file}"
        )

    if src.RasterCount != 1:

        raise ValueError(
            f"Expected single-band raster: {fname}"
        )

    src_band = src.GetRasterBand(1)

    src_nodata = src_band.GetNoDataValue()

    width = src.RasterXSize
    height = src.RasterYSize

    # --------------------------------------------------------
    # Open annual NUE mask if enabled
    # --------------------------------------------------------

    mask_ds = None
    mask_band = None
    mask_nodata = None
    nue_has_mask = False

    if USE_NUE_MASK:

        mask_ds = gdal.Open(
            mask_file,
            gdal.GA_ReadOnly
        )

        if mask_ds is None:

            raise RuntimeError(
                f"Cannot open NUE mask: {mask_file}"
            )

        if (
            width != mask_ds.RasterXSize
            or height != mask_ds.RasterYSize
        ):

            raise ValueError(
                f"Raster dimensions differ: {fname}"
            )

        mask_band = mask_ds.GetRasterBand(1)

        mask_nodata = mask_band.GetNoDataValue()

        nue_has_mask = not (
            mask_band.GetMaskFlags()
            & gdal.GMF_ALL_VALID
        )

        print(
            f"  Annual NUE mask: "
            f"{os.path.basename(mask_file)}"
        )

    else:

        print(
            "  NUE mask: DISABLED"
        )

    # --------------------------------------------------------
    # Create output GeoTIFF
    # --------------------------------------------------------

    driver = gdal.GetDriverByName("GTiff")

    out_ds = driver.Create(
        out_file,
        width,
        height,
        1,
        gdal.GDT_Float32,
        options=[
            "COMPRESS=DEFLATE",
            "PREDICTOR=3",
            "TILED=YES",
            "BIGTIFF=IF_SAFER"
        ]
    )

    if out_ds is None:

        raise RuntimeError(
            f"Cannot create output: {out_file}"
        )

    # --------------------------------------------------------
    # Preserve original GeoTIFF spatial properties
    # --------------------------------------------------------

    out_ds.SetGeoTransform(
        src.GetGeoTransform()
    )

    out_ds.SetProjection(
        src.GetProjection()
    )

    out_ds.SetMetadata(
        src.GetMetadata()
    )

    out_band = out_ds.GetRasterBand(1)

    out_band.SetNoDataValue(
        nodata_value
    )

    out_band.SetDescription(
        src_band.GetDescription()
    )

    out_band.SetUnitType(
        src_band.GetUnitType()
    )

    # Preserve band metadata, excluding obsolete stats
    metadata = src_band.GetMetadata() or {}

    metadata = {
        k: v
        for k, v in metadata.items()
        if not k.upper().startswith("STATISTICS_")
    }

    out_band.SetMetadata(
        metadata
    )

    # --------------------------------------------------------
    # Statistics
    # --------------------------------------------------------

    total_pixels = 0

    valid_pixels = 0

    removed_by_mask = 0

    removed_by_range = 0

    # Respect original GDAL mask
    src_has_mask = not (
        src_band.GetMaskFlags()
        & gdal.GMF_ALL_VALID
    )

    # --------------------------------------------------------
    # Process blocks
    # --------------------------------------------------------

    for yoff in range(
        0,
        height,
        BLOCK_SIZE
    ):

        ysize = min(
            BLOCK_SIZE,
            height - yoff
        )

        for xoff in range(
            0,
            width,
            BLOCK_SIZE
        ):

            xsize = min(
                BLOCK_SIZE,
                width - xoff
            )

            # ================================================
            # STEP 1: Read original values
            # ================================================

            arr = src_band.ReadAsArray(
                xoff,
                yoff,
                xsize,
                ysize
            ).astype(
                np.float64
            )

            total_pixels += arr.size

            # ================================================
            # STEP 2: Original valid pixel mask
            # ================================================

            input_valid = np.isfinite(
                arr
            )

            if src_nodata is not None:

                if np.isfinite(src_nodata):

                    input_valid &= (
                        arr != src_nodata
                    )

            for nd in additional_nodata:

                input_valid &= (
                    arr != nd
                )

            # Remove specific values before multiplication
            for v in replace_values:

                input_valid &= (
                    arr != v
                )

            # Respect source GDAL mask
            if src_has_mask:

                src_mask_arr = (
                    src_band.GetMaskBand().ReadAsArray(
                        xoff,
                        yoff,
                        xsize,
                        ysize
                    )
                )

                input_valid &= (
                    src_mask_arr != 0
                )

            # ================================================
            # STEP 3: Optional annual NUE mask
            # ================================================

            if USE_NUE_MASK:

                nue_arr = mask_band.ReadAsArray(
                    xoff,
                    yoff,
                    xsize,
                    ysize
                )

                mask_valid = np.isfinite(
                    nue_arr
                )

                # Exclude mask metadata NoData
                if mask_nodata is not None:

                    if np.isfinite(mask_nodata):

                        mask_valid &= (
                            nue_arr != mask_nodata
                        )

                # Exclude explicit -9999
                mask_valid &= (
                    nue_arr != nodata_value
                )

                # Respect mask raster GDAL validity mask
                if nue_has_mask:

                    nue_mask_arr = (
                        mask_band.GetMaskBand().ReadAsArray(
                            xoff,
                            yoff,
                            xsize,
                            ysize
                        )
                    )

                    mask_valid &= (
                        nue_mask_arr != 0
                    )

                # Optional positive NUE condition
                if MASK_REQUIRE_POSITIVE:

                    mask_valid &= (
                        nue_arr > 0
                    )

            else:

                # No spatial restriction when disabled
                mask_valid = np.ones(
                    arr.shape,
                    dtype=bool
                )

            removed_by_mask += int(
                np.count_nonzero(
                    input_valid & ~mask_valid
                )
            )

            # ================================================
            # STEP 4: Combine valid masks
            # ================================================

            common_valid = (
                input_valid
                & mask_valid
            )

            # ================================================
            # STEP 5: Multiply by ratio
            # ================================================

            scaled = np.full(
                arr.shape,
                np.nan,
                dtype=np.float64
            )

            scaled[common_valid] = (
                arr[common_valid] * ratio
            )

            # ================================================
            # STEP 6: Keep valid numerical range
            #
            # range_min <= scaled <= range_max
            # ================================================

            range_valid = (
                np.isfinite(scaled)
                & (scaled >= range_min)
                & (scaled <= range_max)
                & (
                    np.abs(scaled)
                    <= np.finfo(np.float32).max
                )
            )

            removed_by_range += int(
                np.count_nonzero(
                    common_valid
                    & ~range_valid
                )
            )

            final_valid = (
                common_valid
                & range_valid
            )

            # ================================================
            # STEP 7: Prepare output pixel values
            # ================================================

            output_arr = np.full(
                arr.shape,
                nodata_value,
                dtype=np.float32
            )

            output_arr[final_valid] = (
                scaled[final_valid].astype(
                    np.float32
                )
            )

            valid_pixels += int(
                np.count_nonzero(
                    final_valid
                )
            )

            # ================================================
            # STEP 8: Write GeoTIFF block
            # ================================================

            out_band.WriteArray(
                output_arr,
                xoff,
                yoff
            )

    # --------------------------------------------------------
    # Finalize current GeoTIFF
    # --------------------------------------------------------

    out_band.FlushCache()
    out_ds.FlushCache()

    out_band = None
    out_ds = None

    src_band = None
    mask_band = None

    src = None
    mask_ds = None

    # --------------------------------------------------------
    # Print annual statistics
    # --------------------------------------------------------

    print(
        f"  Total pixels       : "
        f"{total_pixels:,}"
    )

    print(
        f"  Removed by NUE mask: "
        f"{removed_by_mask:,}"
    )

    print(
        f"  Removed by range   : "
        f"{removed_by_range:,}"
    )

    print(
        f"  Final valid pixels : "
        f"{valid_pixels:,}"
    )

    print(
        f"  Saved as           : "
        f"{output_name}"
    )

    if valid_pixels == 0:

        print(
            "  WARNING: No valid pixels remain!"
        )


# ============================================================
# 9. FINAL SUMMARY
# ============================================================

print("\n" + "=" * 75)

print("ALL GEOTIFF FILES FINISHED")

print("=" * 75)

print(f"Input files : {len(tif_files)}")

print(f"Processed   : {len(matched_files)}")

print(f"Skipped     : {len(skipped_files)}")

print(
    f"NUE mask    : "
    f"{'ENABLED' if USE_NUE_MASK else 'DISABLED'}"
)

print(
    f"Output name : "
    f"{OUTPUT_FILENAME_TEMPLATE}"
)

if skipped_files:

    print("\nSkipped files:")

    for fname, reason in skipped_files:

        print(
            f"  {fname}: {reason}"
        )

print(
    f"\nOutput directory:\n"
    f"{output_folder}"
)

print("=" * 75)